# Notebook 07 - Complete 6-DOF UUV Model

## Purpose

Notebook 07 introduces the first integrated **six-degree-of-freedom underwater vehicle dynamics model** in HydroLab-3D.

Notebooks 00–06 deliberately developed the physical foundations independently:

- translational kinematics;
- roll, pitch, and yaw;
- body/world reference-frame transformations;
- force-driven translational dynamics;
- quadratic hydrodynamic drag;
- gravity and buoyancy;
- command-driven thruster force;
- ocean-current-relative motion.

This notebook combines those validated foundations into a single marine rigid-body model containing both **translation and rotation**.

The vehicle state will be represented by pose:

$$
\eta
=
\begin{bmatrix}
x \\
y \\
z \\
\phi \\
\theta \\
\psi
\end{bmatrix}
$$

and body-frame velocity:

$$
\nu
=
\begin{bmatrix}
u \\
v \\
w \\
p \\
q \\
r
\end{bmatrix}
$$

where:

- $x,y,z$ are world-frame position coordinates;
- $\phi,\theta,\psi$ are roll, pitch, and yaw;
- $u,v,w$ are body-frame translational velocities;
- $p,q,r$ are body-frame angular velocities.

The objective is not to reproduce every term available in a high-fidelity naval-architecture model.

The objective is to construct the **smallest technically serious 6-DOF UUV model that preserves the validated HydroLab-3D physical lineage** and is suitable as the ground-truth vehicle dynamics layer for later sensing, control, simulator-kernel, autonomy, and learning experiments.

---

## Why This Matters

Until this notebook, HydroLab-3D has intentionally treated important physical effects separately.

That decomposition was useful for validation, but a real underwater vehicle does not experience those effects independently.

A force applied away from the center of gravity can create a moment.

Angular velocity changes vehicle orientation.

Vehicle orientation changes how body-frame velocity appears in the world frame.

Gravity and buoyancy can generate both forces and moments.

Hydrodynamic effects depend on motion relative to the surrounding water.

Ocean current is defined by the environment in the world frame, while the marine vehicle dynamics are most naturally expressed in the vehicle body frame.

Notebook 07 therefore establishes the first complete causal chain:

```text
world-frame environment
        +
body-frame vehicle state
        ↓
current transformation
        ↓
water-relative body velocity
        ↓
hydrodynamic forces and moments
        +
rigid-body / added-mass dynamics
        +
gravity / buoyancy restoring effects
        +
actuator wrench
        ↓
body-frame linear and angular acceleration
        ↓
body-frame velocity update
        ↓
6-DOF kinematic transformation
        ↓
world-frame position and orientation update
```

This is the point where HydroLab-3D transitions from a collection of validated physical ingredients into a coherent underwater vehicle model.

---

## What We Will Implement

Notebook 07 will progressively develop and validate:

1. a six-component pose representation;

2. a six-component body-velocity representation;

3. the complete body-velocity to pose-rate transformation:

$$
\dot{\eta}
=
J(\eta)\nu
$$

4. rigid-body mass and rotational inertia;

5. a lightweight added-mass representation;

6. rigid-body Coriolis and centripetal effects;

7. added-mass Coriolis effects at the selected fidelity;

8. six-axis linear and quadratic hydrodynamic damping;

9. world-current to body-current transformation;

10. water-relative body velocity;

11. gravity and buoyancy restoring forces;

12. restoring moments caused by center-of-gravity and center-of-buoyancy offsets where modeled;

13. generalized body-frame actuator forces and moments;

14. deterministic explicit-Euler propagation of the complete state;

15. canonical translational and rotational validation cases;

16. reduction tests proving correspondence with the validated models from Notebooks 00–06;

17. production-code promotion only after the notebook-local formulation has passed;

18. focused automated regression and complete repository regression;

19. compact persistent scientific artifacts for the validated 6-DOF model.

Major terms will be introduced and tested separately before being assembled into the complete state derivative.

Notebook 07 will **not** begin by hiding all dynamics inside one monolithic function.

---

## What We Inherit From Notebooks 00–06

Notebook 07 treats the previous notebooks as engineering dependencies rather than material to be rediscovered.

### Notebook 00 - Basic 3D Kinematics

Validated deterministic world-frame position propagation:

$$
\mathbf{p}_{k+1}
=
\mathbf{p}_k
+
\mathbf{v}_k\Delta t
$$

Production capability:

`integrate_position()`

Notebook 07 preserves the physical meaning of world-frame position propagation while generalizing the state to include orientation.

---

### Notebook 01 - Orientation and Reference Frames

Validated HydroLab-3D world convention:

```text
+x_W = horizontal x
+y_W = horizontal y
+z_W = upward
```

and body-to-world rotation:

$$
R_B^W
=
R_z(\psi)
R_y(\theta)
R_x(\phi)
$$

Production capabilities include:

- `rotation_x()`
- `rotation_y()`
- `rotation_z()`
- `rotation_body_to_world()`

Notebook 07 will use this exact roll-pitch-yaw sequence.

No alternate Euler convention will be introduced.

---

### Notebook 02 - Basic Dynamics

Validated Newtonian translational dynamics:

$$
\mathbf{a}
=
\frac{\mathbf{F}}{m}
$$

and explicit-Euler velocity integration:

$$
\mathbf{v}_{k+1}
=
\mathbf{v}_k
+
\mathbf{a}_k\Delta t
$$

Production capabilities:

- `acceleration_from_force()`
- `integrate_velocity()`

The established numerical philosophy is retained when the dynamics are generalized to six axes.

---

### Notebook 03 - Hydrodynamic Drag

Validated reduced-order quadratic drag:

$$
\mathbf{F}_D
=
-C_D
\|\mathbf{v}\|
\mathbf{v}
$$

Production capability:

`quadratic_drag_force()`

The existing model established the essential physical requirements that drag:

- vanishes at zero relative velocity;
- opposes motion;
- is dissipative;
- exhibits quadratic speed scaling.

Notebook 07 will generalize damping to six components while explicitly preserving this physical meaning and recovering the previously validated translational behavior in an appropriate reduced case.

---

### Notebook 04 - Gravity and Buoyancy

Validated world-frame gravity:

$$
\mathbf{F}_G
=
\begin{bmatrix}
0 \\
0 \\
-mg
\end{bmatrix}
$$

and buoyancy:

$$
\mathbf{F}_B
=
\begin{bmatrix}
0 \\
0 \\
\rho gV
\end{bmatrix}
$$

Production capabilities:

- `gravity_force()`
- `buoyancy_force()`

Notebook 07 extends these force-only models into a generalized hydrostatic wrench capable of representing both forces and moments.

The existing reduced-order functions remain valid APIs and will not be mutated merely to accommodate the new model.

---

### Notebook 05 - Thruster Model

Validated normalized command saturation:

$$
u_{sat}
=
\operatorname{clip}(u,-1,1)
$$

with thrust magnitude:

$$
F_T
=
u_{sat}F_{max}
$$

and body-frame thrust:

$$
\mathbf{F}_{T,B}
=
F_T\hat{\mathbf{d}}_B
$$

together with:

$$
\mathbf{F}_{T,W}
=
R_B^W
\mathbf{F}_{T,B}
$$

Production capabilities:

- `thruster_force_body()`
- `thruster_force_world()`

Notebook 07 retains this validated command-to-force behavior and extends actuator application to a generalized six-component body wrench.

---

### Notebook 06 - Ocean Currents

Validated world-frame ocean current:

$$
\mathbf{v}_{c,W}
=
\begin{bmatrix}
v_{c,x} \\
v_{c,y} \\
v_{c,z}
\end{bmatrix}
$$

and water-relative translational velocity when both quantities are represented in the world frame:

$$
\mathbf{v}_{rel,W}
=
\mathbf{v}_W
-
\mathbf{v}_{c,W}
$$

Production capabilities:

- `uniform_current_velocity()`
- `relative_water_velocity()`

Notebook 06 established the essential invariant:

> Hydrodynamic resistance depends on vehicle motion relative to the surrounding water.

Notebook 07 preserves this invariant while moving the hydrodynamic calculation into the body-frame representation required by the complete marine model.

The complete HydroLab-3D regression baseline entering this notebook is:

```text
95 passed
```

---

## 6-DOF State Definition

The complete pose is:

$$
\boxed{
\eta
=
\begin{bmatrix}
x &
y &
z &
\phi &
\theta &
\psi
\end{bmatrix}^{T}
}
$$

The generalized body velocity is:

$$
\boxed{
\nu
=
\begin{bmatrix}
u &
v &
w &
p &
q &
r
\end{bmatrix}^{T}
}
$$

It is useful to partition these quantities as:

$$
\eta
=
\begin{bmatrix}
\mathbf{p}_W \\
\boldsymbol{\Theta}
\end{bmatrix}
$$

where:

$$
\mathbf{p}_W
=
\begin{bmatrix}
x \\
y \\
z
\end{bmatrix}
$$

and:

$$
\boldsymbol{\Theta}
=
\begin{bmatrix}
\phi \\
\theta \\
\psi
\end{bmatrix}
$$

while:

$$
\nu
=
\begin{bmatrix}
\mathbf{v}_B \\
\boldsymbol{\omega}_B
\end{bmatrix}
$$

with:

$$
\mathbf{v}_B
=
\begin{bmatrix}
u \\
v \\
w
\end{bmatrix}
$$

and:

$$
\boldsymbol{\omega}_B
=
\begin{bmatrix}
p \\
q \\
r
\end{bmatrix}
$$

The distinction is fundamental:

- $\eta$ describes **where the vehicle is and how it is oriented** relative to the world;
- $\nu$ describes **how the vehicle is translating and rotating** in its body frame.

These quantities must not be mixed merely because both contain six components.

---

## Reference-Frame Convention

HydroLab-3D retains the previously validated right-handed world convention:

```text
+x_W = horizontal x
+y_W = horizontal y
+z_W = upward
```

The body-to-world rotation remains:

$$
\boxed{
R_B^W
=
R_z(\psi)
R_y(\theta)
R_x(\phi)
}
$$

Therefore a body-frame translational vector transforms into the world frame according to:

$$
\mathbf{v}_W
=
R_B^W
\mathbf{v}_B
$$

Because the rotation matrix is orthogonal:

$$
(R_B^W)^{-1}
=
(R_B^W)^T
$$

so a world-frame vector is expressed in the body frame using:

$$
\mathbf{v}_B
=
(R_B^W)^T
\mathbf{v}_W
$$

This inverse transformation becomes especially important for ocean-current coupling.

No force, velocity, moment, or environmental quantity will be combined with another quantity until both are represented in the same reference frame.

---

## Kinematic Model

The six-degree-of-freedom pose evolves according to:

$$
\boxed{
\dot{\eta}
=
J(\eta)\nu
}
$$

For the HydroLab-3D Euler-angle convention:

$$
J(\eta)
=
\begin{bmatrix}
R_B^W & 0_{3\times3} \\
0_{3\times3} & T(\phi,\theta)
\end{bmatrix}
$$

The translational block is therefore exactly the already validated body-to-world rotation:

$$
\dot{\mathbf{p}}_W
=
R_B^W
\mathbf{v}_B
$$

The angular block maps body angular velocity:

$$
\boldsymbol{\omega}_B
=
\begin{bmatrix}
p \\
q \\
r
\end{bmatrix}
$$

into Euler-angle rates:

$$
\dot{\boldsymbol{\Theta}}
=
\begin{bmatrix}
\dot{\phi} \\
\dot{\theta} \\
\dot{\psi}
\end{bmatrix}
$$

using the transformation consistent with:

$$
R_B^W
=
R_z(\psi)R_y(\theta)R_x(\phi)
$$

For this convention:

$$
T(\phi,\theta)
=
\begin{bmatrix}
1 &
\sin\phi\tan\theta &
\cos\phi\tan\theta
\\
0 &
\cos\phi &
-\sin\phi
\\
0 &
\dfrac{\sin\phi}{\cos\theta} &
\dfrac{\cos\phi}{\cos\theta}
\end{bmatrix}
$$

Thus:

$$
\dot{\boldsymbol{\Theta}}
=
T(\phi,\theta)
\boldsymbol{\omega}_B
$$

This Euler-angle representation has the expected kinematic singularity as:

$$
\cos\theta
\rightarrow
0
$$

or equivalently near:

$$
\theta
=
\pm\frac{\pi}{2}
$$

Notebook 07 will explicitly validate and bound this representation rather than silently pretending Euler angles are globally nonsingular.

Quaternions may become appropriate in a future extension, but introducing a second orientation representation in this notebook would unnecessarily break the validated HydroLab-3D roll-pitch-yaw lineage.

---

## Dynamic Model

The target marine rigid-body structure is:

$$
\boxed{
M\dot{\nu}
+
C_{RB}(\nu)\nu
+
C_A(\nu_r)\nu_r
+
D(\nu_r)\nu_r
+
g(\eta)
=
\tau
+
\tau_{env}
}
$$

where:

- $M$ is the total generalized inertia matrix;
- $C_{RB}$ represents rigid-body Coriolis and centripetal effects;
- $C_A$ represents added-mass Coriolis and centripetal effects;
- $\nu_r$ is water-relative generalized velocity;
- $D(\nu_r)\nu_r$ represents hydrodynamic damping;
- $g(\eta)$ represents hydrostatic restoring forces and moments;
- $\tau$ is the actuator-generated body wrench;
- $\tau_{env}$ is reserved for additional explicitly modeled environmental wrenches not already represented through current-relative hydrodynamics.

The exact implemented formulation will be derived and validated term by term.

Ocean current will **not** be inserted simultaneously as both a relative-velocity effect and an arbitrary external force, because that would double-count the same physical mechanism.

The acceleration will eventually be obtained from:

$$
\dot{\nu}
=
M^{-1}
\left[
\tau
+
\tau_{env}
-
C_{RB}(\nu)\nu
-
C_A(\nu_r)\nu_r
-
D(\nu_r)\nu_r
-
g(\eta)
\right]
$$

provided each term has independently passed its required structural and physical checks.

---

## Ocean-Current / Relative-Velocity Convention

Notebook 06 expresses uniform ocean current in the world frame:

$$
\mathbf{v}_{c,W}
$$

while Notebook 07 expresses vehicle translational velocity inside:

$$
\nu
$$

in the body frame.

The current must therefore be transformed before subtraction.

Using the validated rotation:

$$
R_B^W
$$

the body-frame current velocity is:

$$
\boxed{
\mathbf{v}_{c,B}
=
(R_B^W)^T
\mathbf{v}_{c,W}
}
$$

For the initial uniform, non-rotating environmental current model, define the generalized body-frame current velocity as:

$$
\nu_c
=
\begin{bmatrix}
\mathbf{v}_{c,B} \\
0 \\
0 \\
0
\end{bmatrix}
$$

and therefore:

$$
\boxed{
\nu_r
=
\nu
-
\nu_c
}
$$

or explicitly:

$$
\nu_r
=
\begin{bmatrix}
u-u_c \\
v-v_c \\
w-w_c \\
p \\
q \\
r
\end{bmatrix}
$$

for this initial current representation.

A world-frame current that is constant in magnitude and direction generally becomes time-varying when expressed in a rotating vehicle body frame.

Notebook 07 will account for this through repeated evaluation of:

$$
\mathbf{v}_{c,B}
=
(R_B^W)^T
\mathbf{v}_{c,W}
$$

from the current vehicle orientation.

The central physical invariant inherited from Notebook 06 remains:

$$
\boxed{
\text{hydrodynamic damping depends on motion relative to the water}
}
$$

A vehicle whose translational body velocity exactly matches the transformed local water velocity should therefore have zero translational water-relative velocity and, in the matched reduced case, zero translational hydrodynamic drag.

---

## Mass and Added-Mass Concept

The total generalized inertia will be represented as:

$$
\boxed{
M
=
M_{RB}
+
M_A
}
$$

The rigid-body component represents physical vehicle mass and rotational inertia.

For the initial canonical model, a body-fixed reference located at the center of gravity permits a transparent rigid-body matrix of the form:

$$
M_{RB}
=
\begin{bmatrix}
mI_3 & 0 \\
0 & I_B
\end{bmatrix}
$$

with:

$$
I_B
=
\begin{bmatrix}
I_x & 0 & 0 \\
0 & I_y & 0 \\
0 & 0 & I_z
\end{bmatrix}
$$

for a first diagonal-inertia implementation.

Added mass represents the additional apparent inertia produced because acceleration of a submerged body also accelerates surrounding fluid.

HydroLab-3D does not require a dense experimentally identified added-mass matrix at this stage.

A diagonal representation is therefore an appropriate initial fidelity target:

$$
M_A
=
\operatorname{diag}
\left(
m_{A,u},
m_{A,v},
m_{A,w},
m_{A,p},
m_{A,q},
m_{A,r}
\right)
$$

using a sign convention that will be documented explicitly before implementation.

The total matrix must satisfy the numerical and physical properties required by the chosen formulation, including:

- shape $(6,6)$;
- finite entries;
- symmetry where required;
- invertibility;
- positive effective inertia for the canonical vehicle.

Cross-coupled inertia terms will not be introduced merely for mathematical ornamentation.

---

## Coriolis / Centripetal Concept

Six-degree-of-freedom motion introduces velocity-dependent coupling absent from the earlier reduced translational models.

Notebook 07 therefore distinguishes:

$$
C_{RB}(\nu)
$$

for rigid-body Coriolis and centripetal effects and:

$$
C_A(\nu_r)
$$

for added-mass Coriolis and centripetal effects.

These matrices will be derived from the actual selected mass representation rather than copied mechanically from an unrelated convention.

Where appropriate for the final definitions, the implementation will verify that the Coriolis contribution does not behave like artificial damping.

A structural energy check may take the form:

$$
\nu^T
C_{RB}(\nu)
\nu
\approx
0
$$

and an analogous relation for the selected added-mass formulation.

The exact property will only be enforced where it follows mathematically from the implemented matrix definition.

At zero generalized velocity, the velocity-dependent Coriolis contribution must vanish.

---

## Hydrodynamic Damping Concept

Notebook 03 validated the reduced translational law:

$$
\mathbf{F}_D
=
-C_D
\|\mathbf{v}\|
\mathbf{v}
$$

Notebook 07 requires resistance in all six generalized velocity components.

The initial model will therefore use a lightweight body-axis damping representation combining linear and quadratic terms.

Conceptually, the hydrodynamic damping wrench may be written as:

$$
\boxed{
\tau_D
=
-D_L\nu_r
-
D_Q
\left(
|\nu_r|
\odot
\nu_r
\right)
}
$$

where:

- $D_L$ contains linear damping coefficients;
- $D_Q$ contains quadratic damping coefficients;
- $\odot$ denotes element-wise multiplication.

For the first implementation these matrices may be diagonal so deliberately decoupled canonical cases remain interpretable.

The damping model must satisfy:

$$
\nu_r
=
0
\quad\Rightarrow\quad
\tau_D
=
0
$$

and, for physically dissipative coefficients:

$$
\nu_r^T\tau_D
\le
0
$$

The new representation must also possess a parameter configuration capable of recovering the validated Notebook 03 one-axis quadratic-drag behavior.

This reduction test is essential.

The old drag law is not discarded merely because the state vector is now six-dimensional.

---

## Gravity / Buoyancy Restoring Concept

Gravity and buoyancy are defined most naturally from known world-vertical forces.

With HydroLab-3D's upward-positive world $z$ convention:

$$
\mathbf{F}_{G,W}
=
\begin{bmatrix}
0 \\
0 \\
-mg
\end{bmatrix}
$$

and:

$$
\mathbf{F}_{B,W}
=
\begin{bmatrix}
0 \\
0 \\
\rho gV
\end{bmatrix}
$$

For body-frame dynamics these forces must be transformed consistently:

$$
\mathbf{F}_{G,B}
=
(R_B^W)^T
\mathbf{F}_{G,W}
$$

$$
\mathbf{F}_{B,B}
=
(R_B^W)^T
\mathbf{F}_{B,W}
$$

If:

$$
\mathbf{r}_G
$$

and:

$$
\mathbf{r}_B
$$

are the center-of-gravity and center-of-buoyancy positions relative to the body reference point, the corresponding moments are conceptually:

$$
\mathbf{M}_G
=
\mathbf{r}_G
\times
\mathbf{F}_{G,B}
$$

and:

$$
\mathbf{M}_B
=
\mathbf{r}_B
\times
\mathbf{F}_{B,B}
$$

The combined hydrostatic force and moment will form the generalized restoring term.

For exact neutral buoyancy:

$$
mg
=
\rho gV
$$

and coincident centers:

$$
\mathbf{r}_G
=
\mathbf{r}_B
$$

the appropriate canonical configuration must permit:

$$
g(\eta)
=
0
$$

within numerical tolerance.

A small CG/CB offset may then be used to verify passive hydrostatic restoring moments without turning Notebook 07 into a detailed hydrostatics study.

---

## Generalized Actuator Wrench

The complete body-frame actuator input is:

$$
\boxed{
\tau
=
\begin{bmatrix}
X \\
Y \\
Z \\
K \\
M \\
N
\end{bmatrix}
}
$$

where:

- $X,Y,Z$ are body-frame actuator forces;
- $K,M,N$ are body-frame actuator moments.

Notebook 05 already provides validated command-to-force generation.

For a body-frame thruster force:

$$
\mathbf{F}_{T,B}
$$

applied at body-fixed offset:

$$
\mathbf{r}_T
$$

relative to the chosen reference point, the generated moment is:

$$
\boxed{
\mathbf{M}_T
=
\mathbf{r}_T
\times
\mathbf{F}_{T,B}
}
$$

and the resulting generalized actuator wrench is:

$$
\tau_T
=
\begin{bmatrix}
\mathbf{F}_{T,B} \\
\mathbf{M}_T
\end{bmatrix}
$$

A thruster whose line of action passes through the reference point has:

$$
\mathbf{M}_T
=
0
$$

while an offset thruster can produce rotational acceleration.

Notebook 07 does not require a complete multi-thruster allocation optimizer.

Direct generalized wrench inputs may also be used for isolated six-axis validation where they make the physics clearer.

---

## Numerical Integration Convention

Notebook 07 retains HydroLab-3D's established explicit-Euler philosophy.

At timestep $k$, the intended deterministic sequence is conceptually:

1. evaluate the pose-dependent kinematic transformation from $\eta_k$;

2. evaluate world-to-body current transformation;

3. construct $\nu_{r,k}$;

4. evaluate inertia, Coriolis, damping, hydrostatic, actuator, and environmental terms from the beginning-of-step state;

5. compute:

$$
\dot{\nu}_k
$$

6. compute:

$$
\dot{\eta}_k
=
J(\eta_k)\nu_k
$$

7. propagate pose using beginning-of-step velocity information:

$$
\eta_{k+1}
=
\eta_k
+
\dot{\eta}_k\Delta t
$$

8. propagate body velocity:

$$
\nu_{k+1}
=
\nu_k
+
\dot{\nu}_k\Delta t
$$

This preserves the numerical meaning established in the earlier notebooks:

$$
\text{evaluate from state }k
\rightarrow
\text{propagate to state }k+1
$$

No adaptive external ODE solver will be introduced merely because the state dimension increased.

If the canonical 6-DOF experiments reveal a genuine stability limitation, that limitation will be characterized explicitly rather than silently changing numerical methods.

---

## Expected Result

A successful Notebook 07 will leave HydroLab-3D with a deterministic and transparent 6-DOF UUV model in which:

- body-frame linear motion changes world-frame position correctly;
- body angular velocity changes roll, pitch, and yaw correctly;
- rigid-body and added-mass inertia resist acceleration;
- Coriolis and centripetal terms introduce physically meaningful velocity coupling without artificial energy dissipation;
- hydrodynamic damping opposes motion relative to the surrounding water;
- gravity and buoyancy produce correct forces;
- CG/CB geometry can produce restoring moments;
- actuator forces create translation;
- actuator moments create rotation;
- ocean current is transformed consistently between frames;
- a vehicle moving with the local water experiences the correct reduction in hydrodynamic resistance;
- previously validated reduced translational behaviors remain recoverable as special cases.

The result should be a model rich enough for control and sensor development while remaining inexpensive enough for rapid headless experimentation.

---

## Validation Criteria

Notebook 07 is considered successful only if the implemented model demonstrates all essential structural and physical properties.

### State and Kinematics

- $\eta$ has exactly six components;
- $\nu$ has exactly six components;
- $J(\eta)$ has shape $(6,6)$;
- all kinematic quantities remain finite inside the supported Euler-angle domain;
- identity orientation maps body translation directly into the corresponding world axes;
- known roll, pitch, and yaw cases agree with the Notebook 01 convention;
- zero body angular velocity gives zero Euler-angle rate;
- analytically simple $p,q,r$ cases produce the expected Euler-angle rates.

### Mass and Added Mass

- all mass matrices have the intended dimensions;
- all parameters are finite;
- invalid mass and inertia inputs are rejected;
- symmetry is preserved where required;
- the canonical total inertia matrix is invertible;
- effective canonical inertia is physically valid.

### Coriolis and Centripetal Effects

- zero velocity gives zero Coriolis contribution;
- the implemented matrices satisfy their intended structural properties;
- the selected Coriolis model does not act as artificial damping;
- deliberately decoupled cases do not create unexplained cross-axis behavior.

### Hydrodynamic Damping

- zero water-relative velocity gives zero damping;
- damping opposes water-relative motion;
- damping is dissipative;
- diagonal coefficients do not create unintended cross-axis loads;
- matched translational parameters recover the validated Notebook 03 behavior.

### Gravity and Buoyancy

- identity-orientation force balance reproduces the Notebook 04 gravity/buoyancy result;
- exact neutral buoyancy can produce zero hydrostatic force;
- coincident CG and CB produce no hydrostatic moment;
- an intentionally introduced CG/CB offset produces a restoring moment in the physically expected direction where that feature is included.

### Ocean Current

- world-frame current transforms correctly into the body frame;
- current transformation changes consistently with vehicle orientation;
- water-relative velocity uses quantities expressed in a common frame;
- a matched vehicle/current translational velocity produces zero corresponding relative motion;
- the Notebook 06 current-aware reduced case remains recoverable.

### Actuation

- pure body force produces the expected translation;
- pure body moment produces the expected angular acceleration;
- force applied through the reference point produces zero application-point moment;
- offset force satisfies:

$$
\mathbf{M}
=
\mathbf{r}
\times
\mathbf{F}
$$

- Notebook 05 command saturation and force-direction semantics remain intact.

### Numerical Behavior

- all states remain finite;
- all matrices remain finite;
- all generalized forces and moments remain finite;
- all state derivatives remain finite;
- deterministic reruns reproduce the same numerical history;
- no unexplained motion appears from a zero-input equilibrium case.

### Engineering Validation

- the validated implementation is promoted only after notebook-local testing;
- production imports resolve to the intended files;
- canonical validation is rerun through production code;
- a fresh-process import check passes where appropriate;
- persistent artifacts are generated and checked;
- focused Notebook 07 regression passes;
- the complete HydroLab-3D regression suite passes.

The final test count will be taken from actual `pytest` output rather than predicted in advance.

---

## Reduction / Backward-Compatibility Criteria

Notebook 07 must demonstrate that the new model contains the previously validated HydroLab-3D physics as sensible limiting cases.

### Reduction to Notebook 00

With fixed orientation and constant body velocity:

$$
R_B^W
=
I
$$

the translational kinematics must reduce to:

$$
\mathbf{p}_{k+1}
=
\mathbf{p}_k
+
\mathbf{v}_k\Delta t
$$

---

### Reduction to Notebook 01

The translational block of:

$$
J(\eta)
$$

must reproduce:

$$
\mathbf{v}_W
=
R_B^W
\mathbf{v}_B
$$

using:

$$
R_B^W
=
R_z(\psi)
R_y(\theta)
R_x(\phi)
$$

---

### Reduction to Notebook 02

With:

- zero added mass;
- zero damping;
- neutral hydrostatics;
- zero current;
- no rotational coupling;
- pure axis-aligned translation;

the six-axis dynamics must recover:

$$
\mathbf{a}
=
\frac{\mathbf{F}}{m}
$$

---

### Reduction to Notebook 03

For one-axis translation with appropriately matched quadratic damping coefficients, the generalized damping model must recover:

$$
F_D
=
-C_D|v|v
$$

or the equivalent one-axis form of the validated isotropic quadratic model.

---

### Reduction to Notebook 04

At identity orientation, the hydrostatic model must reproduce:

$$
\mathbf{F}_G
=
\begin{bmatrix}
0 \\
0 \\
-mg
\end{bmatrix}
$$

and:

$$
\mathbf{F}_B
=
\begin{bmatrix}
0 \\
0 \\
\rho gV
\end{bmatrix}
$$

For exact neutral buoyancy with coincident CG and CB, the complete hydrostatic wrench must vanish.

---

### Reduction to Notebook 05

A force produced by the validated thruster model must enter the generalized body wrench without changing its validated command-to-thrust behavior.

If the force acts through the selected reference point:

$$
\mathbf{M}_T
=
0
$$

---

### Reduction to Notebook 06

For fixed identity orientation, translation-only motion, and uniform current:

$$
\mathbf{v}_{c,B}
=
\mathbf{v}_{c,W}
$$

and therefore the 6-DOF relative-velocity construction must reduce to:

$$
\mathbf{v}_{rel}
=
\mathbf{v}
-
\mathbf{v}_c
$$

in the common frame.

The established Notebook 06 invariant must remain true:

$$
\boxed{
\text{current shifts world-relative motion while hydrodynamics depend on water-relative motion}
}
$$

These reduction tests are not optional bookkeeping.

They establish that Notebook 07 is a generalization of the validated HydroLab-3D stack rather than an unrelated replacement model.

---

## Relationship to HydroLab-3D

HydroLab-3D follows an **algorithm-first selective-fidelity** philosophy.

The simulator is intended to spend computational effort on physical effects that materially influence robotics algorithms rather than on hydrodynamic detail that does not improve the research question.

Notebook 07 therefore occupies the following position:

```text
Notebooks 00–06
validated physical primitives
        ↓
Notebook 07
integrated 6-DOF vehicle dynamics
        ↓
Notebook 08
imperfect sensor observations
        ↓
Notebook 09
closed-loop control
        ↓
Notebook 10
simulator kernel
        ↓
autonomy / multi-UUV / learning / ROS 2
```

The 6-DOF model becomes the source of **ground-truth vehicle state** for subsequent simulator layers.

Its purpose is not merely to move a virtual submarine convincingly.

Its purpose is to provide a numerically trustworthy plant whose behavior can be observed, controlled, disturbed, learned from, and eventually connected to broader marine-autonomy experiments.

The governing engineering objective remains:

$$
\boxed{
\text{maximum useful marine-dynamics fidelity per unit of computational complexity}
}
$$

---

## Scope Boundary

Notebook 07 may include:

- six-component pose;
- six-component body velocity;
- body/world 6-DOF kinematics;
- rigid-body inertia;
- diagonal or otherwise justified added mass;
- rigid-body Coriolis and centripetal effects;
- added-mass Coriolis effects at justified fidelity;
- linear and quadratic six-axis damping;
- current-relative body velocity;
- gravity and buoyancy restoring forces;
- restoring moments;
- generalized actuator forces and moments;
- force-at-offset moment generation;
- deterministic explicit-Euler propagation;
- canonical translation and rotation experiments;
- reduction tests against Notebooks 00–06;
- production promotion;
- automated regression tests;
- compact scientific CSV and plot artifacts.

Notebook 07 will **not** become:

- CFD;
- Navier-Stokes simulation;
- turbulence simulation;
- wave simulation;
- vortex-shedding simulation;
- flexible-body dynamics;
- detailed propeller hydrodynamics;
- a full thruster-allocation optimizer;
- terrain collision;
- sensor simulation;
- sonar simulation;
- DVL simulation;
- IMU simulation;
- communication modeling;
- multi-UUV simulation;
- waypoint navigation;
- controller design;
- ROS 2 integration;
- reinforcement learning;
- a complete simulator kernel;
- a hardware-specific digital twin.

Those capabilities belong to later HydroLab-3D development stages.

---

## Expected Outputs

Notebook 07 will generate only artifacts that provide useful engineering evidence.

The final artifact set is expected to remain compact and may include:

```text
results/notebooks/07_six_dof_uuv/
├── data/
│   └── six_dof_validation.csv
└── plots/
    ├── six_dof_state_history.png
    ├── trajectory_orientation_validation.png
    ├── force_moment_validation.png
    └── reduced_model_regression.png
```

The exact filenames and number of plots will be finalized only after the canonical experiments have demonstrated which artifacts materially strengthen validation.

The principal CSV should preserve sufficient information to reconstruct the canonical experiment, potentially including:

- time;
- pose;
- body velocity;
- body acceleration;
- Euler-angle rates;
- world-frame current;
- body-frame current;
- water-relative generalized velocity;
- actuator wrench;
- damping wrench;
- hydrostatic wrench;
- Coriolis contributions;
- net generalized force and moment.

Decorative plots will not be generated.

---

## Output Directory

Notebook 07 owns:

```text
results/notebooks/07_six_dof_uuv/
```

All persistent numerical and visual artifacts produced by this notebook will remain inside that directory.

---

## Engineering Handoff Target

When Notebook 07 is complete, later HydroLab-3D development should be able to assume that the following foundations have already been validated:

- 3D translational kinematics;
- roll/pitch/yaw convention;
- body-to-world rotation;
- generalized six-component pose representation;
- generalized six-component body-velocity representation;
- body-velocity to pose-rate mapping;
- rigid-body mass and inertia;
- added mass at the selected HydroLab-3D fidelity;
- Coriolis and centripetal terms;
- six-axis hydrodynamic damping;
- gravity and buoyancy restoring forces;
- restoring moments where modeled;
- command-to-thrust conversion;
- thrust saturation;
- body-frame thruster direction;
- generalized actuator force and moment application;
- world-frame current representation;
- world-to-body current transformation;
- water-relative generalized velocity;
- current-aware hydrodynamic damping;
- deterministic six-degree-of-freedom state propagation;
- reduction to the previously validated translational special cases.

Notebook 08 may then treat the 6-DOF vehicle state as simulator ground truth and begin constructing imperfect observations without reopening foundational vehicle dynamics.

The next development stage is therefore:

```text
08_sensor_models.ipynb
```

where HydroLab-3D can begin introducing:

- IMU measurements;
- depth sensing;
- DVL measurements;
- an initial sonar abstraction;
- sensor noise;
- bias;
- drift;
- latency;
- dropout where appropriate.

Notebook 07 therefore serves as the engineering boundary between:

```text
physical vehicle truth
        ↓
imperfect robotic observation
```

Once this notebook is complete, HydroLab-3D should possess a validated lightweight marine plant capable of supporting the sensing and autonomy layers that follow.

---

## Step 1 - Repository Audit and 6-DOF Dependency Freeze

Before deriving or implementing any 6-DOF mathematics, we first verify the **actual production state inherited from Notebooks 00–06**.

This step is intentionally read-only.

Notebook 07 depends on several previously validated modules, and the 6-DOF formulation must extend those modules without silently changing their conventions or duplicating their functionality.

The dependency chain entering Notebook 07 is:

```text
kinematics.py
    integrate_position()
          │
          ├──────────────┐
          │              │
transforms.py        rigid_body.py
rotation_*()         acceleration_from_force()
rotation_body_to_world()
          │              │
          └──────┬───────┘
                 │
              drag.py
        quadratic_drag_force()
                 │
          buoyancy.py
 gravity_force() / buoyancy_force()
                 │
           thrusters.py
 thruster_force_body() / thruster_force_world()
                 │
           currents.py
 uniform_current_velocity()
 relative_water_velocity()
                 │
                 ▼
        Notebook 07 - 6 DOF
```

We also inspect:

- `src/hydrolab3d/dynamics/__init__.py`;
- `src/hydrolab3d/world/__init__.py`;
- the current `tests/` directory;
- any already-existing candidate files such as:
  - `dynamics/hydrodynamics.py`;
  - `vehicles/uuv.py`;
  - `core/state.py`.

The purpose is to determine what already exists before Notebook 07 creates or modifies anything.

### Conventions Frozen by Previous Validation

The following conventions are treated as inherited engineering constraints unless the repository inspection contradicts them.

#### World Frame

HydroLab-3D uses:

```text
+x_W = horizontal x
+y_W = horizontal y
+z_W = upward
```

Therefore gravity acts in the negative world-$z$ direction.

---

#### Orientation

The validated body-to-world rotation is:

$$
R_B^W
=
R_z(\psi)
R_y(\theta)
R_x(\phi)
$$

Notebook 07 must use this same roll-pitch-yaw sequence.

---

#### Existing Translational Integration

The previously validated explicit-Euler ordering is:

$$
\mathbf{p}_{k+1}
=
\mathbf{p}_k
+
\mathbf{v}_k\Delta t
$$

followed by:

$$
\mathbf{v}_{k+1}
=
\mathbf{v}_k
+
\mathbf{a}_k\Delta t
$$

The future 6-DOF integrator should preserve the same beginning-of-step evaluation philosophy unless numerical validation demonstrates a reason not to.

---

#### Existing Drag Meaning

Notebook 03 established that hydrodynamic drag opposes the supplied velocity:

$$
\mathbf{F}_D
=
-C_D
\|\mathbf{v}\|
\mathbf{v}
$$

Notebook 06 then established that the physically relevant velocity in moving water is the **water-relative velocity**.

Therefore Notebook 07 must preserve:

$$
\boxed{
\text{hydrodynamic resistance depends on motion relative to the water}
}
$$

rather than directly on world-relative vehicle velocity.

---

#### Current Convention

Ocean current is currently defined in the world frame.

For Notebook 07, a world-frame translational current:

$$
\mathbf{v}_{c,W}
$$

will eventually need to be represented in the body frame using:

$$
\mathbf{v}_{c,B}
=
(R_B^W)^T
\mathbf{v}_{c,W}
$$

before constructing body-frame relative velocity.

No world-frame current vector will be directly subtracted from a body-frame vehicle velocity.

---

### Why We Inspect Candidate Production Files Now

The README describes several future architectural locations, including:

```text
dynamics/hydrodynamics.py
vehicles/uuv.py
core/state.py
```

These are architectural suggestions, not permission to overwrite them.

Notebook 07 will therefore first determine whether each candidate:

- does not yet exist;
- exists but is empty or skeletal;
- already contains implementation;
- already participates in package exports or tests.

Only after notebook-local mathematics has been validated will we decide what functionality belongs in production code.

---

### Expected Result

This step should report:

1. the resolved HydroLab-3D repository root;
2. all inherited production modules required by Notebook 07;
3. the exact source locations of the inherited public functions;
4. the current dynamics and world package exports;
5. the current automated-test files;
6. the existence and contents of tentative Notebook 07 production targets;
7. confirmation that the repository is ready for notebook-local 6-DOF prototyping.

Nothing should be created or modified.

---

### Estimated Notebook 07 Development Size

Based on the existing HydroLab-3D architecture and the minimum-sufficient-decomposition rule, Notebook 07 is expected to require approximately:

$$
\boxed{
14\text{–}17
}
$$

Markdown + code implementation/validation combinations.

A likely progression is:

```text
1.  Repository audit and dependency freeze
2.  6-DOF state + kinematic transformation
3.  Kinematic analytical validation
4.  Rigid-body + added-mass matrices
5.  Coriolis / centripetal terms
6.  Six-axis damping
7.  Gravity / buoyancy restoring wrench
8.  Current transformation + relative velocity
9.  Generalized actuator wrench
10. Complete 6-DOF derivative
11. Deterministic state integration
12. Canonical force / moment experiments
13. Reduced-model regressions against 00–06
14. Production-target inspection and promotion
15. Production re-import + fresh-process validation
16. Persistent scientific artifacts
17. Focused + complete repository regression
```

Several adjacent items may be safely combined if their validation remains clear, so this is an engineering estimate rather than a fixed target.

The notebook will not be expanded merely to reach this count.

In [1]:
from pathlib import Path
import importlib
import inspect
import sys


# ---------------------------------------------------------------------
# Resolve HydroLab-3D repository root
# ---------------------------------------------------------------------

start = Path.cwd().resolve()
repo_root = None

for candidate in [start, *start.parents]:
    if (
        (candidate / "README.md").exists()
        and (candidate / "src").exists()
        and (candidate / "notebooks").exists()
    ):
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate the HydroLab-3D repository root from "
        f"{start}"
    )

src_path = repo_root / "src"

src_path_str = str(src_path)
if src_path_str not in sys.path:
    sys.path.insert(0, src_path_str)

importlib.invalidate_caches()


# ---------------------------------------------------------------------
# Required inherited production files
# ---------------------------------------------------------------------

required_files = {
    "kinematics": (
        src_path
        / "hydrolab3d"
        / "dynamics"
        / "kinematics.py"
    ),
    "transforms": (
        src_path
        / "hydrolab3d"
        / "utils"
        / "transforms.py"
    ),
    "rigid_body": (
        src_path
        / "hydrolab3d"
        / "dynamics"
        / "rigid_body.py"
    ),
    "drag": (
        src_path
        / "hydrolab3d"
        / "dynamics"
        / "drag.py"
    ),
    "buoyancy": (
        src_path
        / "hydrolab3d"
        / "dynamics"
        / "buoyancy.py"
    ),
    "thrusters": (
        src_path
        / "hydrolab3d"
        / "dynamics"
        / "thrusters.py"
    ),
    "currents": (
        src_path
        / "hydrolab3d"
        / "world"
        / "currents.py"
    ),
    "dynamics_init": (
        src_path
        / "hydrolab3d"
        / "dynamics"
        / "__init__.py"
    ),
    "world_init": (
        src_path
        / "hydrolab3d"
        / "world"
        / "__init__.py"
    ),
}

for name, path in required_files.items():
    if not path.exists():
        raise FileNotFoundError(
            f"Required inherited production file is missing "
            f"({name}): {path}"
        )

    if not path.is_file():
        raise RuntimeError(
            f"Expected a file for {name}, found another path type: "
            f"{path}"
        )


# ---------------------------------------------------------------------
# Import inherited public functionality
# ---------------------------------------------------------------------

from hydrolab3d.dynamics.kinematics import integrate_position
from hydrolab3d.utils.transforms import (
    rotation_x,
    rotation_y,
    rotation_z,
    rotation_body_to_world,
)
from hydrolab3d.dynamics.rigid_body import (
    acceleration_from_force,
    integrate_velocity,
)
from hydrolab3d.dynamics.drag import quadratic_drag_force
from hydrolab3d.dynamics.buoyancy import (
    gravity_force,
    buoyancy_force,
)
from hydrolab3d.dynamics.thrusters import (
    thruster_force_body,
    thruster_force_world,
)
from hydrolab3d.world.currents import (
    uniform_current_velocity,
    relative_water_velocity,
)


inherited_functions = {
    "integrate_position": integrate_position,
    "rotation_x": rotation_x,
    "rotation_y": rotation_y,
    "rotation_z": rotation_z,
    "rotation_body_to_world": rotation_body_to_world,
    "acceleration_from_force": acceleration_from_force,
    "integrate_velocity": integrate_velocity,
    "quadratic_drag_force": quadratic_drag_force,
    "gravity_force": gravity_force,
    "buoyancy_force": buoyancy_force,
    "thruster_force_body": thruster_force_body,
    "thruster_force_world": thruster_force_world,
    "uniform_current_velocity": uniform_current_velocity,
    "relative_water_velocity": relative_water_velocity,
}


# ---------------------------------------------------------------------
# Tentative Notebook 07 production targets
#
# These are inspected only. Nothing is created or modified.
# ---------------------------------------------------------------------

candidate_targets = {
    "hydrodynamics": (
        src_path
        / "hydrolab3d"
        / "dynamics"
        / "hydrodynamics.py"
    ),
    "uuv": (
        src_path
        / "hydrolab3d"
        / "vehicles"
        / "uuv.py"
    ),
    "state": (
        src_path
        / "hydrolab3d"
        / "core"
        / "state.py"
    ),
}


# ---------------------------------------------------------------------
# Test inventory
# ---------------------------------------------------------------------

tests_dir = repo_root / "tests"

if not tests_dir.exists():
    raise FileNotFoundError(
        f"Tests directory not found: {tests_dir}"
    )

test_files = sorted(tests_dir.glob("test_*.py"))


# ---------------------------------------------------------------------
# Report repository and inherited implementation
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 07 — STEP 1 REPOSITORY AUDIT")
print("=" * 80)

print(f"\nRepository root:\n  {repo_root}")

print("\nInherited production files:")
for name, path in required_files.items():
    print(
        f"  {name:<16} -> "
        f"{path.relative_to(repo_root)}"
    )


print("\nInherited function source locations:")

for name, function in inherited_functions.items():
    source_file = Path(
        inspect.getsourcefile(function)
    ).resolve()

    try:
        display_path = source_file.relative_to(repo_root)
    except ValueError:
        display_path = source_file

    print(
        f"  {name:<30} -> {display_path}"
    )


# ---------------------------------------------------------------------
# Package export inspection
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("CURRENT DYNAMICS PACKAGE EXPORTS")
print("=" * 80)

dynamics_init_source = required_files[
    "dynamics_init"
].read_text(encoding="utf-8")

print(dynamics_init_source)


print("\n" + "=" * 80)
print("CURRENT WORLD PACKAGE EXPORTS")
print("=" * 80)

world_init_source = required_files[
    "world_init"
].read_text(encoding="utf-8")

print(world_init_source)


# ---------------------------------------------------------------------
# Existing test files
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("CURRENT TEST INVENTORY")
print("=" * 80)

if not test_files:
    print("  No test_*.py files found.")
else:
    for path in test_files:
        print(f"  {path.relative_to(repo_root)}")


# ---------------------------------------------------------------------
# Candidate Notebook 07 production targets
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("TENTATIVE NOTEBOOK 07 PRODUCTION TARGETS")
print("=" * 80)

for name, path in candidate_targets.items():

    relative = path.relative_to(repo_root)

    if not path.exists():
        print(
            f"\n{name}: {relative}\n"
            "  STATUS: does not currently exist"
        )
        continue

    if not path.is_file():
        raise RuntimeError(
            f"Candidate target exists but is not a file: {path}"
        )

    source = path.read_text(encoding="utf-8")

    print(
        f"\n{name}: {relative}\n"
        f"  STATUS: existing file\n"
        f"  SIZE: {len(source)} characters"
    )

    print("-" * 80)

    if source.strip():
        print(source)
    else:
        print("<empty file>")

    print("-" * 80)


# ---------------------------------------------------------------------
# Final read-only audit assertions
# ---------------------------------------------------------------------

for function_name, function in inherited_functions.items():
    assert callable(function), (
        f"Inherited dependency is not callable: "
        f"{function_name}"
    )

assert len(test_files) > 0

print("\n" + "=" * 80)
print("STEP 1 AUDIT RESULT")
print("=" * 80)

print(
    "PASS: inherited Notebook 00–06 production dependencies "
    "were found and imported successfully."
)

print(
    "PASS: package exports and existing tests were inspected."
)

print(
    "PASS: tentative Notebook 07 production targets were "
    "inspected without modification."
)

print(
    "\nRepository is ready for notebook-local 6-DOF "
    "kinematic development."
)

HYDROLAB-3D NOTEBOOK 07 — STEP 1 REPOSITORY AUDIT

Repository root:
  /home/agniacolyte/HydroLab-3D

Inherited production files:
  kinematics       -> src/hydrolab3d/dynamics/kinematics.py
  transforms       -> src/hydrolab3d/utils/transforms.py
  rigid_body       -> src/hydrolab3d/dynamics/rigid_body.py
  drag             -> src/hydrolab3d/dynamics/drag.py
  buoyancy         -> src/hydrolab3d/dynamics/buoyancy.py
  thrusters        -> src/hydrolab3d/dynamics/thrusters.py
  currents         -> src/hydrolab3d/world/currents.py
  dynamics_init    -> src/hydrolab3d/dynamics/__init__.py
  world_init       -> src/hydrolab3d/world/__init__.py

Inherited function source locations:
  integrate_position             -> src/hydrolab3d/dynamics/kinematics.py
  rotation_x                     -> src/hydrolab3d/utils/transforms.py
  rotation_y                     -> src/hydrolab3d/utils/transforms.py
  rotation_z                     -> src/hydrolab3d/utils/transforms.py
  rotation_body_to_world      

## Step 2 - 6-DOF State Representation and Kinematic Transformation

The first mathematical component of the complete UUV model is the mapping between:

- body-frame translational and angular velocity;
- world-frame position rate;
- Euler-angle rate.

No dynamics are introduced yet.

This step answers only:

> Given the current pose and body-frame velocity of the vehicle, what is the instantaneous rate of change of its six-component pose?

---

### 6-DOF Pose

The vehicle pose is:

$$
\eta
=
\begin{bmatrix}
x \\
y \\
z \\
\phi \\
\theta \\
\psi
\end{bmatrix}
$$

where:

$$
\mathbf{p}_W
=
\begin{bmatrix}
x \\
y \\
z
\end{bmatrix}
$$

is the world-frame position and:

$$
\boldsymbol{\Theta}
=
\begin{bmatrix}
\phi \\
\theta \\
\psi
\end{bmatrix}
$$

contains roll, pitch, and yaw.

---

### Body-Frame Generalized Velocity

The generalized body velocity is:

$$
\nu
=
\begin{bmatrix}
u \\
v \\
w \\
p \\
q \\
r
\end{bmatrix}
$$

where:

$$
\mathbf{v}_B
=
\begin{bmatrix}
u \\
v \\
w
\end{bmatrix}
$$

is body-frame translational velocity and:

$$
\boldsymbol{\omega}_B
=
\begin{bmatrix}
p \\
q \\
r
\end{bmatrix}
$$

is body-frame angular velocity.

These are not pose derivatives directly.

In particular:

$$
\begin{bmatrix}
p \\
q \\
r
\end{bmatrix}
\neq
\begin{bmatrix}
\dot{\phi} \\
\dot{\theta} \\
\dot{\psi}
\end{bmatrix}
$$

except in special orientations.

---

### Translational Kinematics

Notebook 01 established the HydroLab-3D body-to-world rotation:

$$
R_B^W
=
R_z(\psi)
R_y(\theta)
R_x(\phi)
$$

Therefore body-frame translational velocity maps into world-frame position rate as:

$$
\boxed{
\dot{\mathbf{p}}_W
=
R_B^W
\mathbf{v}_B
}
$$

This is the direct 6-DOF continuation of the previously validated body/world transformation.

---

### Angular Kinematics

For the same Z-Y-X Euler convention:

$$
R_B^W
=
R_z(\psi)
R_y(\theta)
R_x(\phi)
$$

the body angular velocity and Euler-angle rates satisfy:

$$
\boldsymbol{\omega}_B
=
E(\phi,\theta)
\dot{\boldsymbol{\Theta}}
$$

with:

$$
E(\phi,\theta)
=
\begin{bmatrix}
1 & 0 & -\sin\theta \\
0 & \cos\phi & \sin\phi\cos\theta \\
0 & -\sin\phi & \cos\phi\cos\theta
\end{bmatrix}
$$

Therefore:

$$
\dot{\boldsymbol{\Theta}}
=
E^{-1}(\phi,\theta)
\boldsymbol{\omega}_B
$$

and:

$$
\boxed{
\dot{\boldsymbol{\Theta}}
=
T(\phi,\theta)
\boldsymbol{\omega}_B
}
$$

where:

$$
T(\phi,\theta)
=
\begin{bmatrix}
1 &
\sin\phi\tan\theta &
\cos\phi\tan\theta
\\
0 &
\cos\phi &
-\sin\phi
\\
0 &
\dfrac{\sin\phi}{\cos\theta} &
\dfrac{\cos\phi}{\cos\theta}
\end{bmatrix}
$$

---

### Complete 6-DOF Kinematic Transformation

Combining the translational and rotational mappings gives:

$$
\boxed{
\dot{\eta}
=
J(\eta)\nu
}
$$

with:

$$
J(\eta)
=
\begin{bmatrix}
R_B^W & 0_{3\times3}
\\
0_{3\times3} & T(\phi,\theta)
\end{bmatrix}
$$

Therefore:

$$
\dot{\eta}
=
\begin{bmatrix}
R_B^W\mathbf{v}_B
\\
T(\phi,\theta)\boldsymbol{\omega}_B
\end{bmatrix}
$$

This transformation is purely kinematic.

It contains no:

- mass;
- force;
- moment;
- drag;
- added mass;
- buoyancy;
- thruster dynamics;
- ocean-current dynamics.

Those enter later.

---

### Euler-Angle Singularity

The matrix:

$$
T(\phi,\theta)
$$

contains terms involving:

$$
\tan\theta
$$

and:

$$
\frac{1}{\cos\theta}
$$

so the Z-Y-X Euler representation becomes singular when:

$$
\cos\theta = 0
$$

or:

$$
\theta
=
\pm\frac{\pi}{2}
$$

This is the familiar pitch singularity of Euler-angle representations.

For HydroLab-3D's initial lightweight UUV model, Euler angles remain appropriate because:

- they are already the validated orientation representation;
- they are transparent for control and debugging;
- ordinary UUV operation can be kept away from the singular configuration;
- changing to quaternions now would introduce unnecessary architectural duplication.

The implementation will therefore reject poses sufficiently close to the singularity rather than allow numerically unbounded rates.

---

### Prototype Functions

This step introduces only notebook-local prototypes:

```text
euler_rate_matrix()
kinematic_matrix_6dof()
pose_rate_6dof()
```

They will not yet be promoted into `src/`.

The existing production function:

```text
rotation_body_to_world()
```

will be reused directly for the translational block.

This prevents duplication of the already validated Notebook 01 rotation convention.

---

### Validation Logic

This first implementation step checks structural properties only.

We expect:

1. the Euler-rate matrix to have shape:

$$
(3,3)
$$

2. the complete kinematic matrix to have shape:

$$
(6,6)
$$

3. both matrices to contain finite values for ordinary orientations;

4. the upper-left block of:

$$
J(\eta)
$$

to exactly equal the validated:

$$
R_B^W
$$

5. the off-diagonal blocks to be zero;

6. zero generalized velocity to produce:

$$
\dot{\eta}
=
0
$$

7. the identity orientation:

$$
\phi=\theta=\psi=0
$$

to produce:

$$
R_B^W=I_3
$$

and:

$$
T(0,0)=I_3
$$

so that:

$$
J(0)=I_6
$$

This gives an important sanity condition:

$$
\boxed{
\eta=0
\quad\Rightarrow\quad
\dot{\eta}=\nu
}
$$

for the identity orientation.

Analytical nontrivial angle/rate cases will be tested separately in the next step rather than mixed into this implementation step.

---

### Expected Result

The prototype should report:

```text
Euler-rate matrix shape : (3, 3)
6-DOF J matrix shape     : (6, 6)

Identity T:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

Identity J:
[[1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 1.]]

Zero-velocity pose rate:
[0. 0. 0. 0. 0. 0.]
```

together with successful assertions confirming the structural properties.

The next step will subject these equations to analytical orientation and angular-rate cases before any dynamic model is introduced.

In [2]:
import numpy as np

from hydrolab3d.utils.transforms import rotation_body_to_world


def euler_rate_matrix(roll, pitch, singularity_tolerance=1e-9):
    """
    Map body angular velocity [p, q, r] to Z-Y-X Euler-angle rates
    [roll_dot, pitch_dot, yaw_dot].

    Parameters
    ----------
    roll : float
        Roll angle phi [rad].
    pitch : float
        Pitch angle theta [rad].
    singularity_tolerance : float
        Minimum permitted absolute value of cos(theta).

    Returns
    -------
    numpy.ndarray
        3x3 Euler-rate transformation matrix T(phi, theta).
    """
    roll = float(roll)
    pitch = float(pitch)

    if not np.isfinite(roll) or not np.isfinite(pitch):
        raise ValueError("roll and pitch must be finite")

    singularity_tolerance = float(singularity_tolerance)

    if (
        not np.isfinite(singularity_tolerance)
        or singularity_tolerance <= 0.0
    ):
        raise ValueError(
            "singularity_tolerance must be finite and positive"
        )

    c_phi = np.cos(roll)
    s_phi = np.sin(roll)
    c_theta = np.cos(pitch)

    if abs(c_theta) <= singularity_tolerance:
        raise ValueError(
            "Euler-rate transformation is singular or too close "
            "to singular at this pitch angle"
        )

    t_theta = np.tan(pitch)

    return np.array(
        [
            [
                1.0,
                s_phi * t_theta,
                c_phi * t_theta,
            ],
            [
                0.0,
                c_phi,
                -s_phi,
            ],
            [
                0.0,
                s_phi / c_theta,
                c_phi / c_theta,
            ],
        ],
        dtype=float,
    )


def kinematic_matrix_6dof(pose):
    """
    Construct the HydroLab-3D 6-DOF kinematic transformation J(eta).

    pose = [x, y, z, roll, pitch, yaw]

    Returns
    -------
    numpy.ndarray
        6x6 matrix satisfying eta_dot = J(eta) @ nu.
    """
    pose = np.asarray(pose, dtype=float)

    if pose.shape != (6,):
        raise ValueError("pose must have shape (6,)")

    if not np.all(np.isfinite(pose)):
        raise ValueError("pose must contain only finite values")

    roll, pitch, yaw = pose[3:]

    R_bw = rotation_body_to_world(
        roll,
        pitch,
        yaw,
    )

    T_euler = euler_rate_matrix(
        roll,
        pitch,
    )

    J = np.zeros((6, 6), dtype=float)

    J[:3, :3] = R_bw
    J[3:, 3:] = T_euler

    return J


def pose_rate_6dof(pose, body_velocity):
    """
    Compute eta_dot from pose eta and body-frame generalized velocity nu.

    pose          = [x, y, z, roll, pitch, yaw]
    body_velocity = [u, v, w, p, q, r]
    """
    body_velocity = np.asarray(
        body_velocity,
        dtype=float,
    )

    if body_velocity.shape != (6,):
        raise ValueError(
            "body_velocity must have shape (6,)"
        )

    if not np.all(np.isfinite(body_velocity)):
        raise ValueError(
            "body_velocity must contain only finite values"
        )

    return (
        kinematic_matrix_6dof(pose)
        @ body_velocity
    )


# ---------------------------------------------------------------------
# Structural validation
# ---------------------------------------------------------------------

zero_pose = np.zeros(6)
zero_velocity = np.zeros(6)

T_identity = euler_rate_matrix(
    roll=0.0,
    pitch=0.0,
)

J_identity = kinematic_matrix_6dof(
    zero_pose
)

zero_pose_rate = pose_rate_6dof(
    zero_pose,
    zero_velocity,
)


assert T_identity.shape == (3, 3)
assert J_identity.shape == (6, 6)
assert zero_pose_rate.shape == (6,)

assert np.all(np.isfinite(T_identity))
assert np.all(np.isfinite(J_identity))
assert np.all(np.isfinite(zero_pose_rate))

np.testing.assert_allclose(
    T_identity,
    np.eye(3),
    atol=1e-15,
)

np.testing.assert_allclose(
    J_identity,
    np.eye(6),
    atol=1e-15,
)

np.testing.assert_allclose(
    zero_pose_rate,
    np.zeros(6),
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Check that the translational block exactly reuses Notebook 01
# ---------------------------------------------------------------------

test_pose = np.array(
    [
        1.0,
        -2.0,
        3.0,
        0.25,
        -0.20,
        0.60,
    ]
)

R_existing = rotation_body_to_world(
    test_pose[3],
    test_pose[4],
    test_pose[5],
)

J_test = kinematic_matrix_6dof(
    test_pose
)

np.testing.assert_allclose(
    J_test[:3, :3],
    R_existing,
    atol=1e-15,
)

np.testing.assert_allclose(
    J_test[:3, 3:],
    np.zeros((3, 3)),
    atol=1e-15,
)

np.testing.assert_allclose(
    J_test[3:, :3],
    np.zeros((3, 3)),
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Output
# ---------------------------------------------------------------------

print(
    "Euler-rate matrix shape :",
    T_identity.shape,
)

print(
    "6-DOF J matrix shape     :",
    J_identity.shape,
)

print("\nIdentity T:")
print(T_identity)

print("\nIdentity J:")
print(J_identity)

print("\nZero-velocity pose rate:")
print(zero_pose_rate)

print("\nNontrivial test pose:")
print(test_pose)

print("\nNontrivial J(eta):")
print(J_test)

print(
    "\nPASS: 6-DOF kinematic prototype has valid structure."
)

print(
    "PASS: identity orientation gives J = I6."
)

print(
    "PASS: translational block exactly matches "
    "the validated Notebook 01 body-to-world rotation."
)

print(
    "PASS: zero body velocity gives zero pose rate."
)

Euler-rate matrix shape : (3, 3)
6-DOF J matrix shape     : (6, 6)

Identity T:
[[ 1.  0.  0.]
 [ 0.  1. -0.]
 [ 0.  0.  1.]]

Identity J:
[[ 1.  0.  0.  0.  0.  0.]
 [ 0.  1.  0.  0.  0.  0.]
 [ 0.  0.  1.  0.  0.  0.]
 [ 0.  0.  0.  1.  0.  0.]
 [ 0.  0.  0.  0.  1. -0.]
 [ 0.  0.  0.  0.  0.  1.]]

Zero-velocity pose rate:
[0. 0. 0. 0. 0. 0.]

Nontrivial test pose:
[ 1.   -2.    3.    0.25 -0.2   0.6 ]

Nontrivial J(eta):
[[ 0.80888385 -0.58765565 -0.0191767   0.          0.          0.        ]
 [ 0.55338722  0.77192486 -0.31288113  0.          0.          0.        ]
 [ 0.19866933  0.24247235  0.94959868  0.          0.          0.        ]
 [ 0.          0.          0.          1.         -0.05015127 -0.19640827]
 [ 0.          0.          0.          0.          0.96891242 -0.24740396]
 [ 0.          0.          0.          0.          0.25243587  0.98861898]]

PASS: 6-DOF kinematic prototype has valid structure.
PASS: identity orientation gives J = I6.
PASS: translational block

## Step 3 - Analytical Validation of 6-DOF Kinematics

Step 2 established that the 6-DOF kinematic mapping has the correct structure.

This step validates the mathematics against cases where the expected result can be derived analytically.

The goal is to verify both parts of:

$$
\dot{\eta}
=
J(\eta)\nu
$$

namely:

$$
\dot{\mathbf{p}}_W
=
R_B^W\mathbf{v}_B
$$

and:

$$
\dot{\boldsymbol{\Theta}}
=
T(\phi,\theta)\boldsymbol{\omega}_B
$$

before the model is allowed to progress into mass, inertia, or force dynamics.

---

### Test 1 - Identity Orientation Translation

For:

$$
\phi=\theta=\psi=0
$$

the body and world frames coincide:

$$
R_B^W=I
$$

Therefore:

$$
\mathbf{v}_B
=
\begin{bmatrix}
u\\
v\\
w
\end{bmatrix}
$$

must produce:

$$
\dot{\mathbf{p}}_W
=
\begin{bmatrix}
u\\
v\\
w
\end{bmatrix}
$$

with no cross-axis leakage.

---

### Test 2 - Known 90 Degree Yaw

Set:

$$
\psi=\frac{\pi}{2}
$$

with:

$$
\phi=\theta=0
$$

For the validated HydroLab-3D convention:

$$
R_B^W
=
R_z\left(\frac{\pi}{2}\right)
$$

A body-forward velocity:

$$
\mathbf{v}_B
=
\begin{bmatrix}
1\\
0\\
0
\end{bmatrix}
$$

must map to:

$$
\dot{\mathbf{p}}_W
=
\begin{bmatrix}
0\\
1\\
0
\end{bmatrix}
$$

This provides direct continuity with Notebook 01.

---

### Test 3 - Pure Roll Rate at Zero Attitude

At:

$$
\phi=\theta=\psi=0
$$

the Euler-rate transformation becomes:

$$
T(0,0)=I
$$

Therefore for:

$$
\boldsymbol{\omega}_B
=
\begin{bmatrix}
p\\
0\\
0
\end{bmatrix}
$$

we expect:

$$
\begin{bmatrix}
\dot{\phi}\\
\dot{\theta}\\
\dot{\psi}
\end{bmatrix}
=
\begin{bmatrix}
p\\
0\\
0
\end{bmatrix}
$$

The same identity relationship applies independently to pure pitch and pure yaw body rates at zero attitude.

---

### Test 4 - Nonzero Roll with Pure Pitch Body Rate

Consider:

$$
\phi
=
30^\circ
$$

$$
\theta=0
$$

and:

$$
\boldsymbol{\omega}_B
=
\begin{bmatrix}
0\\
1\\
0
\end{bmatrix}
$$

Because:

$$
\tan(0)=0
$$

and:

$$
\cos(0)=1
$$

the transformation reduces to:

$$
T(\phi,0)
=
\begin{bmatrix}
1 & 0 & 0\\
0 & \cos\phi & -\sin\phi\\
0 & \sin\phi & \cos\phi
\end{bmatrix}
$$

Therefore:

$$
\dot{\phi}=0
$$

$$
\dot{\theta}
=
\cos(30^\circ)
=
\frac{\sqrt{3}}{2}
$$

and:

$$
\dot{\psi}
=
\sin(30^\circ)
=
\frac{1}{2}
$$

Thus the expected Euler-rate vector is:

$$
\boxed{
\dot{\boldsymbol{\Theta}}
=
\begin{bmatrix}
0\\
0.8660254\\
0.5
\end{bmatrix}
}
$$

This is an important demonstration that:

$$
q
\neq
\dot{\theta}
$$

for a generally oriented body.

---

### Test 5 - Nonzero Pitch with Pure Yaw Body Rate

Consider:

$$
\phi=0
$$

$$
\theta=30^\circ
$$

and:

$$
\boldsymbol{\omega}_B
=
\begin{bmatrix}
0\\
0\\
1
\end{bmatrix}
$$

For:

$$
\phi=0
$$

the Euler-rate matrix becomes:

$$
T(0,\theta)
=
\begin{bmatrix}
1 & 0 & \tan\theta\\
0 & 1 & 0\\
0 & 0 & \sec\theta
\end{bmatrix}
$$

Therefore:

$$
\dot{\phi}
=
\tan(30^\circ)
=
\frac{1}{\sqrt{3}}
$$

$$
\dot{\theta}=0
$$

$$
\dot{\psi}
=
\sec(30^\circ)
=
\frac{2}{\sqrt{3}}
$$

so the expected result is:

$$
\boxed{
\dot{\boldsymbol{\Theta}}
=
\begin{bmatrix}
0.57735027\\
0\\
1.15470054
\end{bmatrix}
}
$$

This verifies the expected coupling introduced by nonzero pitch.

---

### Test 6 - Euler Singularity Guard

The Z-Y-X Euler mapping becomes singular as:

$$
\theta
\rightarrow
\pm\frac{\pi}{2}
$$

The implementation must therefore reject:

$$
\theta=\frac{\pi}{2}
$$

rather than produce extremely large or undefined rates.

A successful exception is considered correct behavior.

---

### Validation Criteria

This step passes only if:

- identity translation is exact;
- the known 90-degree yaw transformation agrees with Notebook 01;
- pure roll, pitch, and yaw rates behave correctly at zero attitude;
- the analytically derived nonzero-roll case agrees numerically;
- the analytically derived nonzero-pitch case agrees numerically;
- the Euler singularity is rejected cleanly;
- all nonsingular computed pose rates remain finite.

If these tests pass, the complete HydroLab-3D 6-DOF kinematic transformation can be treated as notebook-validated and we can proceed to generalized mass and inertia.

In [3]:
import numpy as np


# ---------------------------------------------------------------------
# Test 1 - Identity orientation translation
# ---------------------------------------------------------------------

pose_identity = np.zeros(6)

nu_translation = np.array(
    [
        1.2,
        -0.7,
        0.35,
        0.0,
        0.0,
        0.0,
    ]
)

eta_dot_translation = pose_rate_6dof(
    pose_identity,
    nu_translation,
)

expected_translation = np.array(
    [
        1.2,
        -0.7,
        0.35,
        0.0,
        0.0,
        0.0,
    ]
)

np.testing.assert_allclose(
    eta_dot_translation,
    expected_translation,
    atol=1e-14,
)


# ---------------------------------------------------------------------
# Test 2 - 90 degree yaw
# ---------------------------------------------------------------------

pose_yaw_90 = np.array(
    [
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        np.pi / 2.0,
    ]
)

nu_forward = np.array(
    [
        1.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
    ]
)

eta_dot_yaw_90 = pose_rate_6dof(
    pose_yaw_90,
    nu_forward,
)

expected_yaw_90 = np.array(
    [
        0.0,
        1.0,
        0.0,
        0.0,
        0.0,
        0.0,
    ]
)

np.testing.assert_allclose(
    eta_dot_yaw_90,
    expected_yaw_90,
    atol=1e-14,
)


# ---------------------------------------------------------------------
# Test 3 - Pure angular rates at zero attitude
# ---------------------------------------------------------------------

angular_rate_cases = {
    "pure roll": np.array(
        [0.0, 0.0, 0.0, 0.4, 0.0, 0.0]
    ),
    "pure pitch": np.array(
        [0.0, 0.0, 0.0, 0.0, -0.6, 0.0]
    ),
    "pure yaw": np.array(
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.8]
    ),
}

angular_results = {}

for name, nu in angular_rate_cases.items():

    result = pose_rate_6dof(
        pose_identity,
        nu,
    )

    np.testing.assert_allclose(
        result,
        nu,
        atol=1e-14,
    )

    angular_results[name] = result


# ---------------------------------------------------------------------
# Test 4 - Nonzero roll, pure q body rate
# ---------------------------------------------------------------------

roll_30 = np.deg2rad(30.0)

pose_roll_30 = np.array(
    [
        0.0,
        0.0,
        0.0,
        roll_30,
        0.0,
        0.0,
    ]
)

nu_pure_q = np.array(
    [
        0.0,
        0.0,
        0.0,
        0.0,
        1.0,
        0.0,
    ]
)

eta_dot_roll_case = pose_rate_6dof(
    pose_roll_30,
    nu_pure_q,
)

expected_roll_case = np.array(
    [
        0.0,
        0.0,
        0.0,
        0.0,
        np.sqrt(3.0) / 2.0,
        0.5,
    ]
)

np.testing.assert_allclose(
    eta_dot_roll_case,
    expected_roll_case,
    atol=1e-14,
)


# ---------------------------------------------------------------------
# Test 5 - Nonzero pitch, pure r body rate
# ---------------------------------------------------------------------

pitch_30 = np.deg2rad(30.0)

pose_pitch_30 = np.array(
    [
        0.0,
        0.0,
        0.0,
        0.0,
        pitch_30,
        0.0,
    ]
)

nu_pure_r = np.array(
    [
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        1.0,
    ]
)

eta_dot_pitch_case = pose_rate_6dof(
    pose_pitch_30,
    nu_pure_r,
)

expected_pitch_case = np.array(
    [
        0.0,
        0.0,
        0.0,
        1.0 / np.sqrt(3.0),
        0.0,
        2.0 / np.sqrt(3.0),
    ]
)

np.testing.assert_allclose(
    eta_dot_pitch_case,
    expected_pitch_case,
    atol=1e-14,
)


# ---------------------------------------------------------------------
# Test 6 - Euler singularity guard
# ---------------------------------------------------------------------

singular_pose = np.array(
    [
        0.0,
        0.0,
        0.0,
        0.0,
        np.pi / 2.0,
        0.0,
    ]
)

singularity_rejected = False

try:
    kinematic_matrix_6dof(singular_pose)

except ValueError:
    singularity_rejected = True

assert singularity_rejected


# ---------------------------------------------------------------------
# Finite-value checks
# ---------------------------------------------------------------------

validated_results = [
    eta_dot_translation,
    eta_dot_yaw_90,
    eta_dot_roll_case,
    eta_dot_pitch_case,
    *angular_results.values(),
]

for result in validated_results:
    assert np.all(np.isfinite(result))


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("Identity translation:")
print(eta_dot_translation)

print("\n90-degree yaw, body-forward velocity:")
print(eta_dot_yaw_90)

print("\nPure angular-rate cases at zero attitude:")

for name, result in angular_results.items():
    print(f"  {name:<11}: {result}")

print("\n30-degree roll + pure q=1 rad/s:")
print("Numerical :", eta_dot_roll_case)
print("Expected  :", expected_roll_case)

print("\n30-degree pitch + pure r=1 rad/s:")
print("Numerical :", eta_dot_pitch_case)
print("Expected  :", expected_pitch_case)

print(
    "\nEuler singularity rejected:",
    singularity_rejected,
)

print(
    "\nPASS: translational kinematics match analytical cases."
)

print(
    "PASS: body angular velocity maps correctly to "
    "Euler-angle rates."
)

print(
    "PASS: nontrivial angular coupling matches "
    "analytical predictions."
)

print(
    "PASS: Euler singularity is rejected cleanly."
)

print(
    "\n6-DOF kinematic formulation is analytically validated."
)

Identity translation:
[ 1.2  -0.7   0.35  0.    0.    0.  ]

90-degree yaw, body-forward velocity:
[6.123234e-17 1.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00
 0.000000e+00]

Pure angular-rate cases at zero attitude:
  pure roll  : [0.  0.  0.  0.4 0.  0. ]
  pure pitch : [ 0.   0.   0.   0.  -0.6  0. ]
  pure yaw   : [0.  0.  0.  0.  0.  0.8]

30-degree roll + pure q=1 rad/s:
Numerical : [0.        0.        0.        0.        0.8660254 0.5      ]
Expected  : [0.        0.        0.        0.        0.8660254 0.5      ]

30-degree pitch + pure r=1 rad/s:
Numerical : [0.         0.         0.         0.57735027 0.         1.15470054]
Expected  : [0.         0.         0.         0.57735027 0.         1.15470054]

Euler singularity rejected: True

PASS: translational kinematics match analytical cases.
PASS: body angular velocity maps correctly to Euler-angle rates.
PASS: nontrivial angular coupling matches analytical predictions.
PASS: Euler singularity is rejected cleanly.

6-DO

## Step 4 - Rigid-Body and Added-Mass Matrices

With the complete 6-DOF kinematic transformation validated, we now introduce the generalized inertia that governs how forces and moments produce translational and angular acceleration.

This step does **not** yet compute acceleration.

It only constructs and validates the matrices that will later appear in:

$$
M\dot{\nu}
+
C(\cdot)\nu
+
D(\nu_r)\nu_r
+
g(\eta)
=
\tau
+
\tau_{env}
$$

The objective is to establish a physically interpretable and numerically valid mass representation before introducing Coriolis, damping, hydrostatics, or actuation.

---

### Generalized Inertia

For the Notebook 07 model, total generalized inertia is:

$$
\boxed{
M
=
M_{RB}
+
M_A
}
$$

where:

- $M_{RB}$ is rigid-body inertia;
- $M_A$ is added mass.

The first term represents the physical mass and rotational inertia of the vehicle itself.

The second represents the additional effective inertia caused by accelerating surrounding water.

---

### Body Reference Point

For the initial HydroLab-3D 6-DOF formulation, the body reference point is chosen at the vehicle center of gravity.

This deliberately simplifies the rigid-body inertia matrix.

For a more general body-fixed origin displaced from the center of gravity, translational and rotational inertia would contain coupling terms involving the skew-symmetric matrix of the CG offset.

That additional representation is unnecessary for the first integrated model because hydrostatic center-of-gravity and center-of-buoyancy geometry can still be represented separately when restoring moments are introduced.

Therefore, for this notebook:

$$
\boxed{
\mathbf{r}_G=\mathbf{0}
}
$$

with respect to the body-dynamics reference point.

---

### Rigid-Body Mass Matrix

Let:

$$
m>0
$$

be vehicle mass.

For the canonical Notebook 07 model we use principal body axes so that the inertia tensor is diagonal:

$$
I_B
=
\begin{bmatrix}
I_x & 0 & 0\\
0 & I_y & 0\\
0 & 0 & I_z
\end{bmatrix}
$$

with:

$$
I_x>0,
\qquad
I_y>0,
\qquad
I_z>0
$$

The rigid-body mass matrix becomes:

$$
\boxed{
M_{RB}
=
\begin{bmatrix}
mI_3 & 0_{3\times3}\\
0_{3\times3} & I_B
\end{bmatrix}
}
$$

or explicitly:

$$
M_{RB}
=
\begin{bmatrix}
m & 0 & 0 & 0 & 0 & 0\\
0 & m & 0 & 0 & 0 & 0\\
0 & 0 & m & 0 & 0 & 0\\
0 & 0 & 0 & I_x & 0 & 0\\
0 & 0 & 0 & 0 & I_y & 0\\
0 & 0 & 0 & 0 & 0 & I_z
\end{bmatrix}
$$

This representation provides transparent correspondence with Notebook 02.

If all other dynamic effects are removed and added mass is disabled, the translational equations reduce to:

$$
m\dot{u}=X
$$

$$
m\dot{v}=Y
$$

$$
m\dot{w}=Z
$$

and therefore:

$$
\dot{\mathbf{v}}
=
\frac{\mathbf{F}}{m}
$$

exactly matching the previously validated Newtonian model.

---

### Added Mass

A submerged vehicle accelerating through water must also accelerate some surrounding fluid.

This produces an effective inertia known as **added mass**.

A full hydrodynamic model may contain coupled added-mass coefficients, but HydroLab-3D does not yet have geometry-specific or experimentally identified hydrodynamic coefficients.

Notebook 07 therefore uses a diagonal added-mass model:

$$
M_A
=
\operatorname{diag}
\left(
m_{A,u},
m_{A,v},
m_{A,w},
m_{A,p},
m_{A,q},
m_{A,r}
\right)
$$

with nonnegative effective added-inertia entries.

This notebook uses the convention:

$$
\boxed{
M
=
M_{RB}
+
M_A
}
$$

where the supplied added-mass values themselves are positive effective inertias.

This is an important sign convention.

Some marine-dynamics literature instead defines hydrodynamic derivatives such as:

$$
X_{\dot{u}},
\quad
Y_{\dot{v}},
\quad
Z_{\dot{w}}
$$

as negative coefficients and then constructs the added-mass matrix using their negatives.

Notebook 07 will **not mix these conventions**.

The API developed here accepts direct positive added-inertia magnitudes.

For example:

$$
m_{A,u}=5\text{ kg}
$$

means that surge acceleration behaves as though an additional:

$$
5\text{ kg}
$$

of inertia is present.

---

### Canonical Validation Vehicle

For structural validation we choose a deliberately simple parameter set:

$$
m=20\text{ kg}
$$

and:

$$
I_B
=
\operatorname{diag}
\left(
1.2,\,
1.6,\,
2.0
\right)
\text{ kg m}^2
$$

The effective diagonal added mass is chosen as:

$$
M_A
=
\operatorname{diag}
\left(
5,\,
8,\,
10,\,
0.20,\,
0.30,\,
0.40
\right)
$$

with translational entries in kilograms and rotational entries in:

$$
\text{kg m}^2
$$

These parameters are not claimed to represent a certified real UUV.

They are transparent validation parameters chosen to produce a physically meaningful and numerically well-conditioned six-axis model.

The resulting total inertia should therefore be:

$$
M
=
\operatorname{diag}
\left(
25,\,
28,\,
30,\,
1.4,\,
1.9,\,
2.4
\right)
$$

---

### Required Structural Properties

The rigid-body matrix, added-mass matrix, and total matrix must each have shape:

$$
(6,6)
$$

For the selected diagonal representation they must also be symmetric:

$$
M^T=M
$$

The rigid-body and total matrices must be positive definite.

For any nonzero vector:

$$
\mathbf{x}\neq0
$$

positive definiteness requires:

$$
\mathbf{x}^TM\mathbf{x}>0
$$

Equivalently, all eigenvalues of the symmetric canonical mass matrix must be positive.

This ensures that the matrix can represent positive kinetic energy and can be safely inverted when acceleration is introduced.

---

### Generalized Kinetic Energy

For generalized body velocity:

$$
\nu
$$

the kinetic-energy expression associated with the total effective inertia is:

$$
\boxed{
E_K
=
\frac{1}{2}
\nu^TM\nu
}
$$

For every nonzero generalized velocity:

$$
E_K>0
$$

when:

$$
M
$$

is positive definite.

This provides a physically meaningful validation beyond merely checking matrix dimensions.

---

### Zero Added-Mass Reduction

Setting:

$$
M_A=0
$$

must produce:

$$
M=M_{RB}
$$

exactly.

This is an important backward-compatibility condition.

It allows Notebook 07 to recover the Notebook 02 translational inertia model whenever hydrodynamic added inertia is intentionally disabled.

---

### Invalid Parameters

The prototype will reject:

- zero or negative mass;
- non-finite mass;
- inertia vectors with the wrong shape;
- zero or negative rigid-body principal inertias;
- non-finite inertia values;
- added-mass vectors with the wrong shape;
- negative added-inertia magnitudes;
- non-finite added-mass values.

A zero added-mass coefficient is valid because an axis may intentionally disable that hydrodynamic effect.

---

### Prototype Functions

This step introduces notebook-local prototypes:

```text
rigid_body_mass_matrix()
added_mass_matrix()
total_mass_matrix()
```

These functions remain inside Notebook 07 for now.

Nothing is promoted into:

```text
src/hydrolab3d/
```

until the broader six-degree-of-freedom formulation has been validated.

---

### Validation Criteria

This step passes only if:

1. all generated matrices have shape $(6,6)$;

2. the canonical rigid-body matrix equals:

$$
\operatorname{diag}
\left(
20,\,
20,\,
20,\,
1.2,\,
1.6,\,
2.0
\right)
$$

3. the canonical added-mass matrix equals:

$$
\operatorname{diag}
\left(
5,\,
8,\,
10,\,
0.2,\,
0.3,\,
0.4
\right)
$$

4. total inertia equals:

$$
\operatorname{diag}
\left(
25,\,
28,\,
30,\,
1.4,\,
1.9,\,
2.4
\right)
$$

5. all matrices remain finite;

6. the required matrices are symmetric;

7. rigid-body and total inertia are positive definite;

8. the total matrix is invertible;

9. a nonzero generalized velocity produces positive kinetic energy;

10. zero added mass reproduces the rigid-body matrix exactly;

11. invalid parameters are rejected.

If these conditions pass, Notebook 07 has a validated generalized inertia representation suitable for deriving the 6-DOF Coriolis and acceleration terms.

In [4]:
import numpy as np


def rigid_body_mass_matrix(mass, inertia):
    """
    Construct the 6x6 rigid-body mass matrix for a body reference
    located at the center of gravity and aligned with principal axes.

    Parameters
    ----------
    mass : float
        Vehicle mass [kg].
    inertia : array_like, shape (3,)
        Principal moments of inertia [Ix, Iy, Iz] [kg m^2].

    Returns
    -------
    numpy.ndarray
        6x6 rigid-body mass matrix.
    """
    mass = float(mass)
    inertia = np.asarray(inertia, dtype=float)

    if not np.isfinite(mass) or mass <= 0.0:
        raise ValueError(
            "mass must be finite and strictly positive"
        )

    if inertia.shape != (3,):
        raise ValueError(
            "inertia must have shape (3,)"
        )

    if not np.all(np.isfinite(inertia)):
        raise ValueError(
            "inertia must contain only finite values"
        )

    if np.any(inertia <= 0.0):
        raise ValueError(
            "principal inertias must be strictly positive"
        )

    M_rb = np.zeros((6, 6), dtype=float)

    M_rb[:3, :3] = mass * np.eye(3)
    M_rb[3:, 3:] = np.diag(inertia)

    return M_rb


def added_mass_matrix(added_inertia):
    """
    Construct a diagonal effective added-mass matrix.

    Convention
    ----------
    added_inertia contains positive effective inertia magnitudes:

        [m_Au, m_Av, m_Aw, m_Ap, m_Aq, m_Ar]

    The total mass matrix is formed as:

        M = M_RB + M_A
    """
    added_inertia = np.asarray(
        added_inertia,
        dtype=float,
    )

    if added_inertia.shape != (6,):
        raise ValueError(
            "added_inertia must have shape (6,)"
        )

    if not np.all(np.isfinite(added_inertia)):
        raise ValueError(
            "added_inertia must contain only finite values"
        )

    if np.any(added_inertia < 0.0):
        raise ValueError(
            "added-inertia magnitudes must be nonnegative"
        )

    return np.diag(added_inertia)


def total_mass_matrix(
    mass,
    inertia,
    added_inertia,
):
    """
    Construct total effective generalized inertia:

        M = M_RB + M_A
    """
    M_rb = rigid_body_mass_matrix(
        mass,
        inertia,
    )

    M_a = added_mass_matrix(
        added_inertia,
    )

    return M_rb + M_a


# ---------------------------------------------------------------------
# Canonical validation parameters
# ---------------------------------------------------------------------

mass = 20.0

inertia = np.array(
    [
        1.2,
        1.6,
        2.0,
    ]
)

added_inertia = np.array(
    [
        5.0,
        8.0,
        10.0,
        0.20,
        0.30,
        0.40,
    ]
)


# ---------------------------------------------------------------------
# Construct matrices
# ---------------------------------------------------------------------

M_rb = rigid_body_mass_matrix(
    mass,
    inertia,
)

M_a = added_mass_matrix(
    added_inertia,
)

M = total_mass_matrix(
    mass,
    inertia,
    added_inertia,
)


expected_M_rb = np.diag(
    [
        20.0,
        20.0,
        20.0,
        1.2,
        1.6,
        2.0,
    ]
)

expected_M_a = np.diag(
    [
        5.0,
        8.0,
        10.0,
        0.20,
        0.30,
        0.40,
    ]
)

expected_M = np.diag(
    [
        25.0,
        28.0,
        30.0,
        1.4,
        1.9,
        2.4,
    ]
)


# ---------------------------------------------------------------------
# Structural checks
# ---------------------------------------------------------------------

assert M_rb.shape == (6, 6)
assert M_a.shape == (6, 6)
assert M.shape == (6, 6)

assert np.all(np.isfinite(M_rb))
assert np.all(np.isfinite(M_a))
assert np.all(np.isfinite(M))

np.testing.assert_allclose(
    M_rb,
    expected_M_rb,
    atol=1e-15,
)

np.testing.assert_allclose(
    M_a,
    expected_M_a,
    atol=1e-15,
)

np.testing.assert_allclose(
    M,
    expected_M,
    atol=1e-15,
)

np.testing.assert_allclose(
    M_rb,
    M_rb.T,
    atol=1e-15,
)

np.testing.assert_allclose(
    M_a,
    M_a.T,
    atol=1e-15,
)

np.testing.assert_allclose(
    M,
    M.T,
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Positive-definiteness and invertibility
# ---------------------------------------------------------------------

eig_M_rb = np.linalg.eigvalsh(M_rb)
eig_M = np.linalg.eigvalsh(M)

assert np.all(eig_M_rb > 0.0)
assert np.all(eig_M > 0.0)

det_M = np.linalg.det(M)

assert np.isfinite(det_M)
assert det_M > 0.0

M_inverse = np.linalg.inv(M)

np.testing.assert_allclose(
    M @ M_inverse,
    np.eye(6),
    atol=1e-12,
)


# ---------------------------------------------------------------------
# Kinetic-energy validation
# ---------------------------------------------------------------------

nu_test = np.array(
    [
        1.2,
        -0.4,
        0.3,
        0.2,
        -0.1,
        0.15,
    ]
)

kinetic_energy = 0.5 * nu_test @ M @ nu_test

assert np.isfinite(kinetic_energy)
assert kinetic_energy > 0.0


# ---------------------------------------------------------------------
# Zero-added-mass reduction
# ---------------------------------------------------------------------

M_zero_added = total_mass_matrix(
    mass,
    inertia,
    np.zeros(6),
)

np.testing.assert_allclose(
    M_zero_added,
    M_rb,
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Invalid-input validation
# ---------------------------------------------------------------------

invalid_cases_rejected = 0

invalid_calls = [
    lambda: rigid_body_mass_matrix(
        0.0,
        inertia,
    ),
    lambda: rigid_body_mass_matrix(
        -1.0,
        inertia,
    ),
    lambda: rigid_body_mass_matrix(
        np.nan,
        inertia,
    ),
    lambda: rigid_body_mass_matrix(
        mass,
        [1.0, 2.0],
    ),
    lambda: rigid_body_mass_matrix(
        mass,
        [1.0, 0.0, 2.0],
    ),
    lambda: rigid_body_mass_matrix(
        mass,
        [1.0, np.inf, 2.0],
    ),
    lambda: added_mass_matrix(
        [1.0, 2.0],
    ),
    lambda: added_mass_matrix(
        [1.0, 1.0, 1.0, 1.0, -0.1, 1.0],
    ),
    lambda: added_mass_matrix(
        [1.0, 1.0, np.nan, 1.0, 1.0, 1.0],
    ),
]

for call in invalid_calls:
    try:
        call()
    except ValueError:
        invalid_cases_rejected += 1

assert invalid_cases_rejected == len(invalid_calls)


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("Rigid-body mass matrix M_RB:")
print(M_rb)

print("\nAdded-mass matrix M_A:")
print(M_a)

print("\nTotal mass matrix M:")
print(M)

print("\nRigid-body eigenvalues:")
print(eig_M_rb)

print("\nTotal-mass eigenvalues:")
print(eig_M)

print(
    "\nDeterminant of total mass matrix:",
    det_M,
)

print(
    "Condition number of total mass matrix:",
    np.linalg.cond(M),
)

print("\nTest generalized velocity:")
print(nu_test)

print(
    "Generalized kinetic energy:",
    kinetic_energy,
)

print(
    "\nZero-added-mass reduction exact:",
    np.allclose(M_zero_added, M_rb),
)

print(
    "Invalid parameter cases rejected:",
    f"{invalid_cases_rejected}/{len(invalid_calls)}",
)

print(
    "\nPASS: rigid-body mass matrix is structurally valid."
)

print(
    "PASS: diagonal added-mass convention is structurally valid."
)

print(
    "PASS: total generalized inertia is symmetric, "
    "positive definite, and invertible."
)

print(
    "PASS: nonzero generalized velocity has positive "
    "kinetic energy."
)

print(
    "PASS: zero added mass reduces exactly to rigid-body inertia."
)

print(
    "PASS: invalid physical parameters are rejected cleanly."
)

Rigid-body mass matrix M_RB:
[[20.   0.   0.   0.   0.   0. ]
 [ 0.  20.   0.   0.   0.   0. ]
 [ 0.   0.  20.   0.   0.   0. ]
 [ 0.   0.   0.   1.2  0.   0. ]
 [ 0.   0.   0.   0.   1.6  0. ]
 [ 0.   0.   0.   0.   0.   2. ]]

Added-mass matrix M_A:
[[ 5.   0.   0.   0.   0.   0. ]
 [ 0.   8.   0.   0.   0.   0. ]
 [ 0.   0.  10.   0.   0.   0. ]
 [ 0.   0.   0.   0.2  0.   0. ]
 [ 0.   0.   0.   0.   0.3  0. ]
 [ 0.   0.   0.   0.   0.   0.4]]

Total mass matrix M:
[[25.   0.   0.   0.   0.   0. ]
 [ 0.  28.   0.   0.   0.   0. ]
 [ 0.   0.  30.   0.   0.   0. ]
 [ 0.   0.   0.   1.4  0.   0. ]
 [ 0.   0.   0.   0.   1.9  0. ]
 [ 0.   0.   0.   0.   0.   2.4]]

Rigid-body eigenvalues:
[ 1.2  1.6  2.  20.  20.  20. ]

Total-mass eigenvalues:
[ 1.4  1.9  2.4 25.  28.  30. ]

Determinant of total mass matrix: 134063.9999999999
Condition number of total mass matrix: 21.42857142857143

Test generalized velocity:
[ 1.2  -0.4   0.3   0.2  -0.1   0.15]
Generalized kinetic energy: 21.6545000

## Step 5 - Rigid-Body and Added-Mass Coriolis / Centripetal Terms

The generalized inertia matrices are now validated.

We next introduce the velocity-dependent terms that arise because the vehicle dynamics are expressed in a **rotating body-fixed reference frame**.

Even when no external force is applied, translation and rotation are not independent in body coordinates.

The 6-DOF dynamics therefore require Coriolis and centripetal terms.

For Notebook 07 we distinguish:

$$
C_{RB}(\nu)\nu
$$

for rigid-body effects and:

$$
C_A(\nu_r)\nu_r
$$

for added-mass effects.

These terms will be validated independently before they enter the complete vehicle derivative.

---

### Why Coriolis Terms Appear

The generalized body velocity is:

$$
\nu
=
\begin{bmatrix}
\mathbf{v}\\
\boldsymbol{\omega}
\end{bmatrix}
$$

where:

$$
\mathbf{v}
=
\begin{bmatrix}
u\\
v\\
w
\end{bmatrix}
$$

and:

$$
\boldsymbol{\omega}
=
\begin{bmatrix}
p\\
q\\
r
\end{bmatrix}
$$

Because these quantities are expressed in a rotating body frame, the derivative of momentum contains rotational transport terms.

For rigid-body translational momentum:

$$
\mathbf{p}_{RB}
=
m\mathbf{v}
$$

the body-frame transport contribution contains:

$$
\boldsymbol{\omega}
\times
\mathbf{p}_{RB}
$$

For rigid-body angular momentum:

$$
\mathbf{h}_{RB}
=
I_B\boldsymbol{\omega}
$$

the rotational contribution contains:

$$
\boldsymbol{\omega}
\times
\mathbf{h}_{RB}
$$

The same principle extends to added-mass momentum.

---

### Skew-Symmetric Cross-Product Matrix

For a vector:

$$
\mathbf{a}
=
\begin{bmatrix}
a_x\\
a_y\\
a_z
\end{bmatrix}
$$

define:

$$
S(\mathbf{a})
=
\begin{bmatrix}
0 & -a_z & a_y\\
a_z & 0 & -a_x\\
-a_y & a_x & 0
\end{bmatrix}
$$

such that:

$$
\boxed{
S(\mathbf{a})\mathbf{b}
=
\mathbf{a}\times\mathbf{b}
}
$$

The matrix satisfies:

$$
S(\mathbf{a})^T
=
-S(\mathbf{a})
$$

This primitive will be used repeatedly throughout the 6-DOF model.

---

### Momentum-Based Coriolis Construction

For a constant block-diagonal generalized inertia matrix:

$$
M
=
\begin{bmatrix}
M_{11} & 0\\
0 & M_{22}
\end{bmatrix}
$$

and generalized velocity:

$$
\nu
=
\begin{bmatrix}
\mathbf{v}\\
\boldsymbol{\omega}
\end{bmatrix}
$$

define generalized momentum:

$$
M\nu
=
\begin{bmatrix}
\mathbf{p}\\
\mathbf{h}
\end{bmatrix}
$$

where:

$$
\mathbf{p}
=
M_{11}\mathbf{v}
$$

and:

$$
\mathbf{h}
=
M_{22}\boldsymbol{\omega}
$$

For the diagonal, CG-centered formulation selected in Notebook 07, we use the skew-symmetric Coriolis representation:

$$
\boxed{
C(M,\nu)
=
\begin{bmatrix}
0_{3\times3}
&
-S(\mathbf{p})
\\
-S(\mathbf{p})
&
-S(\mathbf{h})
\end{bmatrix}
}
$$

This representation is useful because:

$$
C^T
=
-C
$$

and therefore:

$$
\boxed{
\nu^T C(M,\nu)\nu
=
0
}
$$

up to floating-point precision.

Thus the Coriolis term redistributes energy between degrees of freedom but does not behave like damping.

---

### Matrix Representation Is Not Unique

A Coriolis matrix is not uniquely determined by its appearance alone.

Different algebraically equivalent matrices may produce the same physical generalized force:

$$
C(\nu)\nu
$$

For example, the rigid-body translational contribution may be represented using either momentum-based cross-coupling blocks or expressions involving:

$$
mS(\boldsymbol{\omega})
$$

provided the resulting product is physically equivalent.

Notebook 07 therefore validates both:

1. structural properties of the selected matrix;
2. the actual generalized Coriolis wrench:

$$
C(\nu)\nu
$$

The second property is physically more important than matching a particular textbook matrix layout.

---

## Rigid-Body Coriolis Term

For the validated rigid-body matrix:

$$
M_{RB}
=
\begin{bmatrix}
mI_3 & 0\\
0 & I_B
\end{bmatrix}
$$

the momenta are:

$$
\mathbf{p}_{RB}
=
m\mathbf{v}
$$

and:

$$
\mathbf{h}_{RB}
=
I_B\boldsymbol{\omega}
$$

The resulting generalized Coriolis wrench should have translational component:

$$
\boxed{
\mathbf{f}_{C,RB}
=
\boldsymbol{\omega}
\times
\mathbf{p}_{RB}
}
$$

because:

$$
-S(\mathbf{p}_{RB})\boldsymbol{\omega}
=
\boldsymbol{\omega}
\times
\mathbf{p}_{RB}
$$

The rotational component is:

$$
-S(\mathbf{p}_{RB})\mathbf{v}
-
S(\mathbf{h}_{RB})\boldsymbol{\omega}
$$

Since:

$$
\mathbf{p}_{RB}
=
m\mathbf{v}
$$

the first part vanishes:

$$
-S(m\mathbf{v})\mathbf{v}
=
0
$$

and therefore:

$$
\boxed{
\boldsymbol{\tau}_{C,RB}
=
\boldsymbol{\omega}
\times
(I_B\boldsymbol{\omega})
}
$$

This is the familiar Euler rigid-body rotational coupling term.

---

## Added-Mass Coriolis Term

Notebook 07 uses positive effective added inertia:

$$
M_A
=
\operatorname{diag}
\left(
m_{A,u},
m_{A,v},
m_{A,w},
m_{A,p},
m_{A,q},
m_{A,r}
\right)
$$

For water-relative velocity:

$$
\nu_r
=
\begin{bmatrix}
\mathbf{v}_r\\
\boldsymbol{\omega}_r
\end{bmatrix}
$$

the added-mass generalized momentum is:

$$
M_A\nu_r
=
\begin{bmatrix}
\mathbf{p}_A\\
\mathbf{h}_A
\end{bmatrix}
$$

The selected added-mass Coriolis matrix is therefore:

$$
\boxed{
C_A(\nu_r)
=
\begin{bmatrix}
0
&
-S(\mathbf{p}_A)
\\
-S(\mathbf{p}_A)
&
-S(\mathbf{h}_A)
\end{bmatrix}
}
$$

and the generalized term becomes:

$$
C_A(\nu_r)\nu_r
$$

Its translational component is:

$$
\boxed{
\boldsymbol{\omega}_r
\times
\mathbf{p}_A
}
$$

while its rotational component is:

$$
\boxed{
\mathbf{v}_r
\times
\mathbf{p}_A
+
\boldsymbol{\omega}_r
\times
\mathbf{h}_A
}
$$

Unlike rigid-body translational momentum:

$$
\mathbf{p}_{RB}=m\mathbf{v}
$$

anisotropic added mass generally gives:

$$
\mathbf{p}_A
\not\parallel
\mathbf{v}_r
$$

so:

$$
\mathbf{v}_r
\times
\mathbf{p}_A
$$

does not necessarily vanish.

This is one way anisotropic marine inertia introduces genuine translational-rotational coupling.

---

## Current-Aware Interpretation

The rigid-body Coriolis term depends on actual vehicle body velocity:

$$
\boxed{
C_{RB}(\nu)\nu
}
$$

The added-mass hydrodynamic term will instead use water-relative generalized velocity:

$$
\boxed{
C_A(\nu_r)\nu_r
}
$$

This distinction is important.

The rigid vehicle's inertia does not disappear because surrounding water moves.

Hydrodynamic added-mass effects, however, belong to motion relative to the surrounding fluid in the current-aware formulation selected for HydroLab-3D.

The exact current transformation will be implemented in a later step.

For now, $\nu_r$ is supplied directly to isolate and validate the added-mass Coriolis mathematics.

---

## Zero-Velocity Requirement

At:

$$
\nu
=
0
$$

we must obtain:

$$
C_{RB}(0)
=
0
$$

and therefore:

$$
C_{RB}(0)0
=
0
$$

Likewise:

$$
\nu_r
=
0
$$

must give:

$$
C_A(0)=0
$$

and:

$$
C_A(0)0=0
$$

---

## Pure Translation Requirement

For rigid-body motion with:

$$
\boldsymbol{\omega}=0
$$

the rigid-body Coriolis wrench must vanish.

Because:

$$
\mathbf{p}_{RB}=m\mathbf{v}
$$

we have:

$$
\mathbf{v}\times\mathbf{p}_{RB}
=
\mathbf{v}\times m\mathbf{v}
=
0
$$

Therefore:

$$
\boxed{
C_{RB}(\nu)\nu=0
}
$$

for pure rigid-body translation in this CG-centered model.

For anisotropic added mass, however, pure relative translation may produce a rotational added-mass coupling because:

$$
\mathbf{v}_r
\times
M_{A,t}\mathbf{v}_r
$$

need not vanish.

That behavior is physical for the selected anisotropic effective-inertia representation and must not be mistaken for numerical cross-axis leakage.

---

## Energy Property

For the selected skew-symmetric construction:

$$
C^T=-C
$$

and therefore for any generalized velocity:

$$
\boxed{
\nu^TC\nu=0
}
$$

This means Coriolis and centripetal forces perform zero instantaneous generalized power.

They can redistribute motion between axes, but they cannot independently dissipate energy.

This sharply distinguishes them from the damping term that will be introduced next.

---

## Prototype Functions

This step introduces notebook-local prototypes:

```text
skew()
coriolis_from_block_diagonal_mass()
rigid_body_coriolis_matrix()
added_mass_coriolis_matrix()
```

Nothing is promoted into production code yet.

---

## Validation Criteria

This step passes only if:

1. `skew(a)` produces the same result as `np.cross(a, b)`;

2. the skew operator satisfies:

$$
S(\mathbf{a})^T=-S(\mathbf{a})
$$

3. rigid-body and added-mass Coriolis matrices have shape $(6,6)$;

4. all values remain finite;

5. both selected Coriolis matrices are skew-symmetric;

6. zero generalized velocity gives zero Coriolis matrix and zero Coriolis wrench;

7. pure rigid-body translation gives zero rigid-body Coriolis wrench;

8. a nontrivial rigid-body case matches the independently calculated:

$$
\boldsymbol{\omega}\times m\mathbf{v}
$$

and:

$$
\boldsymbol{\omega}\times I_B\boldsymbol{\omega}
$$

9. a nontrivial added-mass case matches the independently calculated:

$$
\boldsymbol{\omega}_r\times\mathbf{p}_A
$$

and:

$$
\mathbf{v}_r\times\mathbf{p}_A
+
\boldsymbol{\omega}_r\times\mathbf{h}_A
$$

10. the numerical generalized-power checks satisfy:

$$
\nu^TC_{RB}(\nu)\nu
\approx0
$$

and:

$$
\nu_r^TC_A(\nu_r)\nu_r
\approx0
$$

If these tests pass, the velocity-dependent inertial coupling required by the 6-DOF model can be treated as validated.

In [5]:
import numpy as np


def skew(vector):
    """
    Return the 3x3 skew-symmetric matrix S(a) satisfying:

        S(a) @ b = a x b
    """
    vector = np.asarray(vector, dtype=float)

    if vector.shape != (3,):
        raise ValueError(
            "vector must have shape (3,)"
        )

    if not np.all(np.isfinite(vector)):
        raise ValueError(
            "vector must contain only finite values"
        )

    x, y, z = vector

    return np.array(
        [
            [0.0, -z, y],
            [z, 0.0, -x],
            [-y, x, 0.0],
        ],
        dtype=float,
    )


def coriolis_from_block_diagonal_mass(
    mass_matrix,
    generalized_velocity,
):
    """
    Construct the skew-symmetric Coriolis matrix for the
    block-diagonal 6-DOF inertia representation used in Notebook 07.

    Parameters
    ----------
    mass_matrix : array_like, shape (6, 6)
        Constant block-diagonal generalized inertia matrix.

    generalized_velocity : array_like, shape (6,)
        [u, v, w, p, q, r].

    Returns
    -------
    numpy.ndarray
        6x6 Coriolis / centripetal matrix.
    """
    mass_matrix = np.asarray(
        mass_matrix,
        dtype=float,
    )

    generalized_velocity = np.asarray(
        generalized_velocity,
        dtype=float,
    )

    if mass_matrix.shape != (6, 6):
        raise ValueError(
            "mass_matrix must have shape (6, 6)"
        )

    if generalized_velocity.shape != (6,):
        raise ValueError(
            "generalized_velocity must have shape (6,)"
        )

    if not np.all(np.isfinite(mass_matrix)):
        raise ValueError(
            "mass_matrix must contain only finite values"
        )

    if not np.all(np.isfinite(generalized_velocity)):
        raise ValueError(
            "generalized_velocity must contain only finite values"
        )

    # This Notebook 07 formulation assumes a block-diagonal
    # translational / rotational inertia representation.
    np.testing.assert_allclose(
        mass_matrix[:3, 3:],
        np.zeros((3, 3)),
        atol=1e-15,
    )

    np.testing.assert_allclose(
        mass_matrix[3:, :3],
        np.zeros((3, 3)),
        atol=1e-15,
    )

    generalized_momentum = (
        mass_matrix @ generalized_velocity
    )

    linear_momentum = generalized_momentum[:3]
    angular_momentum = generalized_momentum[3:]

    C = np.zeros((6, 6), dtype=float)

    C[:3, 3:] = -skew(linear_momentum)
    C[3:, :3] = -skew(linear_momentum)
    C[3:, 3:] = -skew(angular_momentum)

    return C


def rigid_body_coriolis_matrix(
    mass,
    inertia,
    body_velocity,
):
    """
    Construct C_RB(nu) for the CG-centered rigid-body model.
    """
    M_rb_local = rigid_body_mass_matrix(
        mass,
        inertia,
    )

    return coriolis_from_block_diagonal_mass(
        M_rb_local,
        body_velocity,
    )


def added_mass_coriolis_matrix(
    added_inertia,
    relative_velocity,
):
    """
    Construct C_A(nu_r) for the diagonal positive-effective
    added-mass convention selected for Notebook 07.
    """
    M_a_local = added_mass_matrix(
        added_inertia,
    )

    return coriolis_from_block_diagonal_mass(
        M_a_local,
        relative_velocity,
    )


# ---------------------------------------------------------------------
# Validate skew operator
# ---------------------------------------------------------------------

a = np.array(
    [1.2, -0.7, 0.4]
)

b = np.array(
    [-0.3, 1.5, 0.8]
)

cross_matrix_result = skew(a) @ b
cross_numpy_result = np.cross(a, b)

np.testing.assert_allclose(
    cross_matrix_result,
    cross_numpy_result,
    atol=1e-15,
)

np.testing.assert_allclose(
    skew(a).T,
    -skew(a),
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Zero-velocity validation
# ---------------------------------------------------------------------

nu_zero = np.zeros(6)

C_rb_zero = rigid_body_coriolis_matrix(
    mass,
    inertia,
    nu_zero,
)

C_a_zero = added_mass_coriolis_matrix(
    added_inertia,
    nu_zero,
)

np.testing.assert_allclose(
    C_rb_zero,
    np.zeros((6, 6)),
    atol=1e-15,
)

np.testing.assert_allclose(
    C_a_zero,
    np.zeros((6, 6)),
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Pure rigid-body translation
# ---------------------------------------------------------------------

nu_pure_translation = np.array(
    [
        1.2,
        -0.4,
        0.3,
        0.0,
        0.0,
        0.0,
    ]
)

C_rb_translation = rigid_body_coriolis_matrix(
    mass,
    inertia,
    nu_pure_translation,
)

wrench_rb_translation = (
    C_rb_translation
    @ nu_pure_translation
)

np.testing.assert_allclose(
    wrench_rb_translation,
    np.zeros(6),
    atol=1e-14,
)


# ---------------------------------------------------------------------
# Nontrivial rigid-body validation
# ---------------------------------------------------------------------

nu_rb = np.array(
    [
        1.2,
        -0.4,
        0.3,
        0.2,
        -0.1,
        0.15,
    ]
)

C_rb = rigid_body_coriolis_matrix(
    mass,
    inertia,
    nu_rb,
)

wrench_rb = C_rb @ nu_rb

v_rb = nu_rb[:3]
omega_rb = nu_rb[3:]

linear_momentum_rb = mass * v_rb
angular_momentum_rb = inertia * omega_rb

expected_force_rb = np.cross(
    omega_rb,
    linear_momentum_rb,
)

expected_moment_rb = np.cross(
    omega_rb,
    angular_momentum_rb,
)

expected_wrench_rb = np.concatenate(
    [
        expected_force_rb,
        expected_moment_rb,
    ]
)

np.testing.assert_allclose(
    wrench_rb,
    expected_wrench_rb,
    atol=1e-14,
)


# ---------------------------------------------------------------------
# Nontrivial added-mass validation
# ---------------------------------------------------------------------

nu_relative = np.array(
    [
        0.8,
        -0.5,
        0.25,
        0.12,
        -0.08,
        0.18,
    ]
)

C_a = added_mass_coriolis_matrix(
    added_inertia,
    nu_relative,
)

wrench_a = C_a @ nu_relative

M_a_local = added_mass_matrix(
    added_inertia
)

momentum_a = M_a_local @ nu_relative

linear_momentum_a = momentum_a[:3]
angular_momentum_a = momentum_a[3:]

v_relative = nu_relative[:3]
omega_relative = nu_relative[3:]

expected_force_a = np.cross(
    omega_relative,
    linear_momentum_a,
)

expected_moment_a = (
    np.cross(
        v_relative,
        linear_momentum_a,
    )
    +
    np.cross(
        omega_relative,
        angular_momentum_a,
    )
)

expected_wrench_a = np.concatenate(
    [
        expected_force_a,
        expected_moment_a,
    ]
)

np.testing.assert_allclose(
    wrench_a,
    expected_wrench_a,
    atol=1e-14,
)


# ---------------------------------------------------------------------
# Structural validation
# ---------------------------------------------------------------------

assert C_rb.shape == (6, 6)
assert C_a.shape == (6, 6)

assert np.all(np.isfinite(C_rb))
assert np.all(np.isfinite(C_a))

np.testing.assert_allclose(
    C_rb.T,
    -C_rb,
    atol=1e-14,
)

np.testing.assert_allclose(
    C_a.T,
    -C_a,
    atol=1e-14,
)


# ---------------------------------------------------------------------
# Generalized-power validation
# ---------------------------------------------------------------------

power_rb = nu_rb @ C_rb @ nu_rb

power_a = (
    nu_relative
    @ C_a
    @ nu_relative
)

assert abs(power_rb) < 1e-12
assert abs(power_a) < 1e-12


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("Skew cross-product validation:")
print("Matrix result :", cross_matrix_result)
print("NumPy result  :", cross_numpy_result)

print("\nRigid-body Coriolis matrix C_RB:")
print(C_rb)

print("\nRigid-body Coriolis wrench:")
print(wrench_rb)

print("\nExpected rigid-body wrench:")
print(expected_wrench_rb)

print(
    "\nRigid-body generalized power:",
    power_rb,
)

print("\nAdded-mass Coriolis matrix C_A:")
print(C_a)

print("\nAdded-mass Coriolis wrench:")
print(wrench_a)

print("\nExpected added-mass wrench:")
print(expected_wrench_a)

print(
    "\nAdded-mass generalized power:",
    power_a,
)

print(
    "\nPure rigid-body translation wrench:"
)
print(wrench_rb_translation)

print(
    "\nPASS: skew operator reproduces the vector cross product."
)

print(
    "PASS: rigid-body Coriolis matrix is skew-symmetric."
)

print(
    "PASS: rigid-body Coriolis wrench matches "
    "independent Newton-Euler cross-product calculations."
)

print(
    "PASS: added-mass Coriolis matrix is skew-symmetric."
)

print(
    "PASS: added-mass Coriolis wrench matches "
    "independent momentum cross-product calculations."
)

print(
    "PASS: Coriolis generalized power is zero "
    "within numerical precision."
)

print(
    "PASS: pure rigid-body translation produces "
    "zero Coriolis wrench."
)

Skew cross-product validation:
Matrix result : [-1.16 -1.08  1.59]
NumPy result  : [-1.16 -1.08  1.59]

Rigid-body Coriolis matrix C_RB:
[[  0.     0.     0.    -0.     6.     8.  ]
 [  0.     0.     0.    -6.    -0.    24.  ]
 [  0.     0.     0.    -8.   -24.    -0.  ]
 [ -0.     6.     8.    -0.     0.3    0.16]
 [ -6.    -0.    24.    -0.3   -0.     0.24]
 [ -8.   -24.    -0.    -0.16  -0.24  -0.  ]]

Rigid-body Coriolis wrench:
[ 0.6    2.4    0.8   -0.006 -0.024 -0.008]

Expected rigid-body wrench:
[ 0.6    2.4    0.8   -0.006 -0.024 -0.008]

Rigid-body generalized power: -2.6107588313450947e-16

Added-mass Coriolis matrix C_A:
[[ 0.     0.     0.    -0.     2.5    4.   ]
 [ 0.     0.     0.    -2.5   -0.     4.   ]
 [ 0.     0.     0.    -4.    -4.    -0.   ]
 [-0.     2.5    4.    -0.     0.072  0.024]
 [-2.5   -0.     4.    -0.072 -0.     0.024]
 [-4.    -4.    -0.    -0.024 -0.024 -0.   ]]

Added-mass Coriolis wrench:
[ 0.52     0.42    -0.16    -0.25144 -1.00432 -1.20096]

E

## Step 6 - Six-Axis Linear and Quadratic Hydrodynamic Damping

With generalized inertia and Coriolis effects validated, we now introduce the first explicitly dissipative term in the 6-DOF marine model.

Notebook 03 validated a reduced translational quadratic drag law:

$$
\mathbf{F}_D
=
-C_D
\|\mathbf{v}\|
\mathbf{v}
$$

Notebook 06 then established that hydrodynamic resistance must depend on velocity **relative to the surrounding water**.

Notebook 07 now requires a generalized damping wrench acting on:

$$
\nu_r
=
\begin{bmatrix}
u_r\\
v_r\\
w_r\\
p_r\\
q_r\\
r_r
\end{bmatrix}
$$

The objective is to obtain a transparent six-axis model that:

- supports translational and rotational damping;
- remains computationally lightweight;
- is explicitly dissipative;
- introduces no unintended cross-axis forces when configured diagonally;
- preserves a demonstrable reduction to the already validated Notebook 03 drag model.

---

### Selected Damping Model

For Notebook 07 we use diagonal linear and quadratic damping coefficients.

Let:

$$
d_L
=
\begin{bmatrix}
d_{L,u}\\
d_{L,v}\\
d_{L,w}\\
d_{L,p}\\
d_{L,q}\\
d_{L,r}
\end{bmatrix}
$$

and:

$$
d_Q
=
\begin{bmatrix}
d_{Q,u}\\
d_{Q,v}\\
d_{Q,w}\\
d_{Q,p}\\
d_{Q,q}\\
d_{Q,r}
\end{bmatrix}
$$

The generalized hydrodynamic damping wrench is defined directly as:

$$
\boxed{
\tau_D(\nu_r)
=
-d_L\odot\nu_r
-
d_Q\odot|\nu_r|\odot\nu_r
}
$$

where:

$$
\odot
$$

denotes element-wise multiplication.

Equivalently, using diagonal matrices:

$$
D_L
=
\operatorname{diag}(d_L)
$$

and:

$$
D_Q(\nu_r)
=
\operatorname{diag}
\left(
d_Q\odot|\nu_r|
\right)
$$

we may write:

$$
\boxed{
\tau_D
=
-
\left[
D_L
+
D_Q(\nu_r)
\right]\nu_r
}
$$

The implementation will return the physical damping wrench itself.

This means the later dynamics equation can be assembled transparently using:

$$
+\tau_D
$$

on the applied-wrench side rather than constructing a positive resistance matrix and subtracting it somewhere else.

This explicit sign choice reduces ambiguity.

---

## Physical Interpretation

For any individual degree of freedom:

$$
\nu_{r,i}>0
$$

must generate:

$$
\tau_{D,i}<0
$$

and:

$$
\nu_{r,i}<0
$$

must generate:

$$
\tau_{D,i}>0
$$

provided at least one corresponding damping coefficient is positive.

Thus damping always opposes relative motion.

At:

$$
\nu_r=0
$$

we require:

$$
\boxed{
\tau_D=0
}
$$

exactly.

---

## Dissipativity

The instantaneous generalized power contributed by damping is:

$$
P_D
=
\nu_r^T\tau_D
$$

Substituting the selected law gives:

$$
P_D
=
-
\sum_i
d_{L,i}\nu_{r,i}^2
-
\sum_i
d_{Q,i}|\nu_{r,i}|^3
$$

For nonnegative coefficients:

$$
d_{L,i}\ge0
$$

and:

$$
d_{Q,i}\ge0
$$

therefore:

$$
\boxed{
P_D\le0
}
$$

Damping may remove kinetic energy or contribute zero power, but it cannot inject energy into the vehicle.

This property distinguishes hydrodynamic damping from the Coriolis terms validated in Step 5, for which:

$$
P_C\approx0
$$

---

## Axis Decoupling

Because Notebook 07 initially uses diagonal damping coefficients, motion in one generalized coordinate must not directly generate damping in another coordinate.

For example, pure surge:

$$
\nu_r
=
\begin{bmatrix}
u_r\\
0\\
0\\
0\\
0\\
0
\end{bmatrix}
$$

must produce:

$$
\tau_D
=
\begin{bmatrix}
X_D\\
0\\
0\\
0\\
0\\
0
\end{bmatrix}
$$

Similarly, pure yaw rate must produce only a yaw damping moment.

This deliberately transparent structure provides a clean baseline before any future experimentally identified cross-coupled damping is considered.

---

## Linear Versus Quadratic Damping

The linear contribution is:

$$
\tau_{D,L}
=
-d_L\odot\nu_r
$$

while the quadratic contribution is:

$$
\tau_{D,Q}
=
-d_Q\odot|\nu_r|\odot\nu_r
$$

Linear damping is useful for representing low-speed viscous effects in a lightweight model.

Quadratic damping dominates more strongly as speed increases.

HydroLab-3D does not claim that this diagonal combination fully identifies the hydrodynamics of a particular real UUV.

It is a selective-fidelity model intended to preserve the effects most relevant to control and autonomy experiments.

---

## Reduction to Notebook 03

Notebook 03 used the isotropic vector law:

$$
\mathbf{F}_D
=
-C_D
\|\mathbf{v}\|
\mathbf{v}
$$

For general simultaneous 3D motion, that norm-coupled law is not identical to independent axis-wise quadratic damping.

Notebook 07 therefore does **not** claim global algebraic equivalence between the two models.

However, for a one-axis translational case:

$$
\mathbf{v}
=
\begin{bmatrix}
u\\
0\\
0
\end{bmatrix}
$$

we have:

$$
\|\mathbf{v}\|
=
|u|
$$

and Notebook 03 becomes:

$$
F_{D,x}
=
-C_D|u|u
$$

The Notebook 07 surge damping law becomes:

$$
X_D
=
-d_{L,u}u
-
d_{Q,u}|u|u
$$

Setting:

$$
d_{L,u}=0
$$

and:

$$
d_{Q,u}=C_D
$$

gives:

$$
\boxed{
X_D
=
-C_D|u|u
}
$$

which exactly reproduces the Notebook 03 force in the one-axis reduction case.

This is the required backward-compatibility bridge.

The same correspondence applies independently to pure sway or pure heave when the corresponding quadratic coefficient is matched.

---

## Current-Aware Meaning

The damping input is:

$$
\nu_r
$$

rather than blindly using:

$$
\nu
$$

For the translational components, $\nu_r$ will later be constructed from vehicle body velocity minus transformed body-frame current velocity.

Thus:

$$
\mathbf{v}_{r,B}
=
\mathbf{v}_B
-
\mathbf{v}_{c,B}
$$

The rotational current components remain zero for the initial uniform non-rotating current model, so:

$$
\boldsymbol{\omega}_r
=
\boldsymbol{\omega}
$$

under that approximation.

This step does not yet perform the current transformation.

It validates the damping function independently for an already supplied relative velocity.

---

## Canonical Damping Coefficients

For numerical validation, use:

$$
d_L
=
\begin{bmatrix}
2.0\\
3.0\\
4.0\\
0.10\\
0.15\\
0.20
\end{bmatrix}
$$

and:

$$
d_Q
=
\begin{bmatrix}
5.0\\
6.0\\
7.0\\
0.30\\
0.40\\
0.50
\end{bmatrix}
$$

These values are validation parameters rather than identified coefficients from a specific physical vehicle.

They are chosen to make every axis numerically distinguishable.

---

## Prototype Functions

This step introduces:

```text
hydrodynamic_damping_wrench()
```

as a notebook-local prototype.

No production file is modified.

---

## Validation Criteria

This step passes only if:

1. the returned wrench has shape $(6,)$;

2. zero relative velocity gives exactly zero damping;

3. all output values remain finite;

4. positive relative velocity produces an opposing negative damping component;

5. negative relative velocity produces an opposing positive damping component;

6. pure-axis motion produces no cross-axis leakage;

7. the damping power satisfies:

$$
\nu_r^T\tau_D\le0
$$

8. velocity reversal produces the expected odd symmetry:

$$
\tau_D(-\nu_r)
=
-\tau_D(\nu_r)
$$

9. with linear damping disabled, doubling one-axis speed produces four times the quadratic damping magnitude;

10. the pure-surge reduction with:

$$
d_{L,u}=0
$$

and:

$$
d_{Q,u}=C_D
$$

exactly agrees with the production Notebook 03 function:

```text
quadratic_drag_force()
```

for matched one-axis motion;

11. negative or non-finite damping coefficients are rejected;

12. malformed relative-velocity inputs are rejected.

If these checks pass, the generalized damping law will preserve the validated physical meaning of the earlier drag model while extending HydroLab-3D to six dissipative degrees of freedom.

In [6]:
import numpy as np

from hydrolab3d.dynamics.drag import quadratic_drag_force


def hydrodynamic_damping_wrench(
    relative_velocity,
    linear_damping,
    quadratic_damping,
):
    """
    Compute the six-component hydrodynamic damping wrench.

    tau_D = -d_L * nu_r - d_Q * |nu_r| * nu_r

    Parameters
    ----------
    relative_velocity : array_like, shape (6,)
        Water-relative generalized body velocity
        [u_r, v_r, w_r, p, q, r].

    linear_damping : array_like, shape (6,)
        Nonnegative linear damping coefficients.

    quadratic_damping : array_like, shape (6,)
        Nonnegative quadratic damping coefficients.

    Returns
    -------
    numpy.ndarray
        Generalized damping wrench
        [X_D, Y_D, Z_D, K_D, M_D, N_D].
    """
    relative_velocity = np.asarray(
        relative_velocity,
        dtype=float,
    )

    linear_damping = np.asarray(
        linear_damping,
        dtype=float,
    )

    quadratic_damping = np.asarray(
        quadratic_damping,
        dtype=float,
    )

    if relative_velocity.shape != (6,):
        raise ValueError(
            "relative_velocity must have shape (6,)"
        )

    if linear_damping.shape != (6,):
        raise ValueError(
            "linear_damping must have shape (6,)"
        )

    if quadratic_damping.shape != (6,):
        raise ValueError(
            "quadratic_damping must have shape (6,)"
        )

    if not np.all(np.isfinite(relative_velocity)):
        raise ValueError(
            "relative_velocity must contain only finite values"
        )

    if not np.all(np.isfinite(linear_damping)):
        raise ValueError(
            "linear_damping must contain only finite values"
        )

    if not np.all(np.isfinite(quadratic_damping)):
        raise ValueError(
            "quadratic_damping must contain only finite values"
        )

    if np.any(linear_damping < 0.0):
        raise ValueError(
            "linear damping coefficients must be nonnegative"
        )

    if np.any(quadratic_damping < 0.0):
        raise ValueError(
            "quadratic damping coefficients must be nonnegative"
        )

    linear_term = (
        -linear_damping * relative_velocity
    )

    quadratic_term = (
        -quadratic_damping
        * np.abs(relative_velocity)
        * relative_velocity
    )

    return linear_term + quadratic_term


# ---------------------------------------------------------------------
# Canonical coefficients
# ---------------------------------------------------------------------

d_linear = np.array(
    [
        2.0,
        3.0,
        4.0,
        0.10,
        0.15,
        0.20,
    ]
)

d_quadratic = np.array(
    [
        5.0,
        6.0,
        7.0,
        0.30,
        0.40,
        0.50,
    ]
)


# ---------------------------------------------------------------------
# Zero relative velocity
# ---------------------------------------------------------------------

nu_r_zero = np.zeros(6)

tau_d_zero = hydrodynamic_damping_wrench(
    nu_r_zero,
    d_linear,
    d_quadratic,
)

np.testing.assert_array_equal(
    tau_d_zero,
    np.zeros(6),
)


# ---------------------------------------------------------------------
# General nontrivial relative velocity
# ---------------------------------------------------------------------

nu_r = np.array(
    [
        1.2,
        -0.5,
        0.30,
        0.20,
        -0.10,
        0.15,
    ]
)

tau_d = hydrodynamic_damping_wrench(
    nu_r,
    d_linear,
    d_quadratic,
)

expected_tau_d = (
    -d_linear * nu_r
    -d_quadratic * np.abs(nu_r) * nu_r
)

np.testing.assert_allclose(
    tau_d,
    expected_tau_d,
    atol=1e-15,
)

assert tau_d.shape == (6,)
assert np.all(np.isfinite(tau_d))


# ---------------------------------------------------------------------
# Damping must oppose every nonzero component
# ---------------------------------------------------------------------

nonzero = np.abs(nu_r) > 0.0

assert np.all(
    tau_d[nonzero] * nu_r[nonzero] < 0.0
)


# ---------------------------------------------------------------------
# Dissipativity
# ---------------------------------------------------------------------

damping_power = nu_r @ tau_d

assert damping_power < 0.0


# ---------------------------------------------------------------------
# Reversal symmetry
# ---------------------------------------------------------------------

tau_d_reversed = hydrodynamic_damping_wrench(
    -nu_r,
    d_linear,
    d_quadratic,
)

np.testing.assert_allclose(
    tau_d_reversed,
    -tau_d,
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Pure-axis leakage test
# ---------------------------------------------------------------------

nu_r_pure_surge = np.array(
    [
        1.5,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
    ]
)

tau_pure_surge = hydrodynamic_damping_wrench(
    nu_r_pure_surge,
    d_linear,
    d_quadratic,
)

assert tau_pure_surge[0] < 0.0

np.testing.assert_allclose(
    tau_pure_surge[1:],
    np.zeros(5),
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Pure quadratic scaling
# ---------------------------------------------------------------------

zero_linear = np.zeros(6)

quadratic_only = np.array(
    [
        5.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
    ]
)

nu_speed_1 = np.array(
    [
        1.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
    ]
)

nu_speed_2 = np.array(
    [
        2.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
    ]
)

tau_speed_1 = hydrodynamic_damping_wrench(
    nu_speed_1,
    zero_linear,
    quadratic_only,
)

tau_speed_2 = hydrodynamic_damping_wrench(
    nu_speed_2,
    zero_linear,
    quadratic_only,
)

quadratic_scaling_ratio = (
    abs(tau_speed_2[0])
    / abs(tau_speed_1[0])
)

np.testing.assert_allclose(
    quadratic_scaling_ratio,
    4.0,
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Reduction to Notebook 03 production drag
# ---------------------------------------------------------------------

C_D = 5.0

velocity_3d = np.array(
    [
        1.7,
        0.0,
        0.0,
    ]
)

notebook03_drag = quadratic_drag_force(
    velocity_3d,
    C_D,
)

nu_r_reduction = np.array(
    [
        1.7,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
    ]
)

linear_reduction = np.zeros(6)

quadratic_reduction = np.array(
    [
        C_D,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
    ]
)

notebook07_damping = hydrodynamic_damping_wrench(
    nu_r_reduction,
    linear_reduction,
    quadratic_reduction,
)

np.testing.assert_allclose(
    notebook07_damping[:3],
    notebook03_drag,
    atol=1e-15,
)

np.testing.assert_allclose(
    notebook07_damping[3:],
    np.zeros(3),
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Negative-speed reduction also checks direction symmetry
# ---------------------------------------------------------------------

velocity_3d_negative = np.array(
    [
        -1.7,
        0.0,
        0.0,
    ]
)

notebook03_drag_negative = quadratic_drag_force(
    velocity_3d_negative,
    C_D,
)

nu_r_reduction_negative = np.array(
    [
        -1.7,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
    ]
)

notebook07_damping_negative = (
    hydrodynamic_damping_wrench(
        nu_r_reduction_negative,
        linear_reduction,
        quadratic_reduction,
    )
)

np.testing.assert_allclose(
    notebook07_damping_negative[:3],
    notebook03_drag_negative,
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Invalid-input validation
# ---------------------------------------------------------------------

invalid_calls = [
    lambda: hydrodynamic_damping_wrench(
        [1.0, 2.0],
        d_linear,
        d_quadratic,
    ),
    lambda: hydrodynamic_damping_wrench(
        nu_r,
        [1.0, 2.0],
        d_quadratic,
    ),
    lambda: hydrodynamic_damping_wrench(
        nu_r,
        d_linear,
        [1.0, 2.0],
    ),
    lambda: hydrodynamic_damping_wrench(
        [1.0, 0.0, np.nan, 0.0, 0.0, 0.0],
        d_linear,
        d_quadratic,
    ),
    lambda: hydrodynamic_damping_wrench(
        nu_r,
        [-1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        d_quadratic,
    ),
    lambda: hydrodynamic_damping_wrench(
        nu_r,
        d_linear,
        [0.0, 0.0, 0.0, 0.0, np.inf, 0.0],
    ),
    lambda: hydrodynamic_damping_wrench(
        nu_r,
        d_linear,
        [0.0, -0.1, 0.0, 0.0, 0.0, 0.0],
    ),
]

invalid_rejected = 0

for call in invalid_calls:
    try:
        call()
    except ValueError:
        invalid_rejected += 1

assert invalid_rejected == len(invalid_calls)


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("General relative velocity nu_r:")
print(nu_r)

print("\nGeneral damping wrench tau_D:")
print(tau_d)

print(
    "\nGeneralized damping power:",
    damping_power,
)

print("\nReversed-velocity damping wrench:")
print(tau_d_reversed)

print("\nPure-surge damping wrench:")
print(tau_pure_surge)

print(
    "\nQuadratic speed-scaling ratio "
    "|tau(2 m/s)| / |tau(1 m/s)|:",
    quadratic_scaling_ratio,
)

print("\nNotebook 03 production drag:")
print(notebook03_drag)

print(
    "Notebook 07 reduced damping force:"
)
print(notebook07_damping[:3])

print("\nNegative-speed Notebook 03 drag:")
print(notebook03_drag_negative)

print(
    "Negative-speed Notebook 07 reduction:"
)
print(notebook07_damping_negative[:3])

print(
    "\nInvalid parameter/input cases rejected:",
    f"{invalid_rejected}/{len(invalid_calls)}",
)

print(
    "\nPASS: zero relative velocity gives zero damping."
)

print(
    "PASS: damping opposes every active relative-motion axis."
)

print(
    "PASS: damping is strictly dissipative for the "
    "nonzero canonical case."
)

print(
    "PASS: velocity reversal produces odd-symmetric damping."
)

print(
    "PASS: diagonal damping produces no cross-axis leakage."
)

print(
    "PASS: quadratic damping scales with speed squared."
)

print(
    "PASS: one-axis Notebook 07 quadratic damping "
    "exactly reproduces Notebook 03 production drag."
)

print(
    "PASS: invalid damping parameters and malformed "
    "inputs are rejected cleanly."
)

General relative velocity nu_r:
[ 1.2  -0.5   0.3   0.2  -0.1   0.15]

General damping wrench tau_D:
[-9.6      3.      -1.83    -0.032    0.019   -0.04125]

Generalized damping power: -13.583487499999997

Reversed-velocity damping wrench:
[ 9.6     -3.       1.83     0.032   -0.019    0.04125]

Pure-surge damping wrench:
[-14.25  -0.    -0.    -0.    -0.    -0.  ]

Quadratic speed-scaling ratio |tau(2 m/s)| / |tau(1 m/s)|: 4.0

Notebook 03 production drag:
[-14.45  -0.    -0.  ]
Notebook 07 reduced damping force:
[-14.45  -0.    -0.  ]

Negative-speed Notebook 03 drag:
[14.45 -0.   -0.  ]
Negative-speed Notebook 07 reduction:
[14.45 -0.   -0.  ]

Invalid parameter/input cases rejected: 7/7

PASS: zero relative velocity gives zero damping.
PASS: damping opposes every active relative-motion axis.
PASS: damping is strictly dissipative for the nonzero canonical case.
PASS: velocity reversal produces odd-symmetric damping.
PASS: diagonal damping produces no cross-axis leakage.
PASS: quadra

## Step 7 - Gravity, Buoyancy, and the 6-DOF Restoring Wrench

Notebook 04 validated gravity and buoyancy as world-frame translational forces.

Notebook 07 must now extend those same physical quantities into a complete body-frame **restoring wrench** containing both:

- force;
- moment.

The goal is to preserve the validated Notebook 04 force model while adding the minimum geometry required for hydrostatic rotational behavior.

No vehicle dynamics are integrated yet.

This step validates only the hydrostatic wrench:

$$
\tau_H(\eta)
=
\begin{bmatrix}
\mathbf{F}_{H,B}\\
\mathbf{M}_{H,B}
\end{bmatrix}
$$

generated by gravity and buoyancy.

---

### Inherited World-Frame Force Convention

HydroLab-3D uses:

```text
+x_W = horizontal x
+y_W = horizontal y
+z_W = upward
```

Notebook 04 therefore established gravity as:

$$
\boxed{
\mathbf{F}_{G,W}
=
\begin{bmatrix}
0\\
0\\
-mg
\end{bmatrix}
}
$$

and buoyancy as:

$$
\boxed{
\mathbf{F}_{B,W}
=
\begin{bmatrix}
0\\
0\\
\rho gV
\end{bmatrix}
}
$$

where:

- $m$ is vehicle mass;
- $\rho$ is fluid density;
- $V$ is displaced volume;
- $g$ is gravitational acceleration magnitude.

Notebook 07 does not alter these definitions.

---

## Transformation Into the Body Frame

The marine dynamics are expressed in body coordinates.

Therefore the world-frame forces must be transformed using the already validated body-to-world rotation:

$$
R_B^W
$$

Since:

$$
(R_B^W)^{-1}
=
(R_B^W)^T
$$

we obtain:

$$
\boxed{
\mathbf{F}_{G,B}
=
(R_B^W)^T
\mathbf{F}_{G,W}
}
$$

and:

$$
\boxed{
\mathbf{F}_{B,B}
=
(R_B^W)^T
\mathbf{F}_{B,W}
}
$$

The total body-frame hydrostatic force is:

$$
\boxed{
\mathbf{F}_{H,B}
=
\mathbf{F}_{G,B}
+
\mathbf{F}_{B,B}
}
$$

---

## Hydrostatic Moments

Let:

$$
\mathbf{r}_G
$$

be the body-frame position of the center of gravity relative to the chosen body-dynamics reference point.

Let:

$$
\mathbf{r}_B
$$

be the body-frame position of the center of buoyancy relative to the same reference point.

The corresponding moments are:

$$
\boxed{
\mathbf{M}_{G,B}
=
\mathbf{r}_G
\times
\mathbf{F}_{G,B}
}
$$

and:

$$
\boxed{
\mathbf{M}_{B,B}
=
\mathbf{r}_B
\times
\mathbf{F}_{B,B}
}
$$

Therefore the total hydrostatic moment is:

$$
\boxed{
\mathbf{M}_{H,B}
=
\mathbf{M}_{G,B}
+
\mathbf{M}_{B,B}
}
$$

and the complete restoring wrench is:

$$
\boxed{
\tau_H
=
\begin{bmatrix}
\mathbf{F}_{H,B}\\
\mathbf{M}_{H,B}
\end{bmatrix}
}
$$

---

## Sign Convention

This notebook-local function will return the **physical hydrostatic wrench acting on the vehicle**.

That means:

- gravity enters with its actual downward force;
- buoyancy enters with its actual upward force;
- the resulting moment uses the physical cross product.

Later, when the full dynamics are assembled, this wrench will be added directly to the applied generalized forces:

$$
M\dot{\nu}
=
\cdots
+
\tau_H
+
\cdots
$$

rather than defining a positive restoring vector and then subtracting it.

This convention keeps the individual physical terms directly inspectable.

---

## Neutral Buoyancy

Exact neutral buoyancy requires:

$$
mg
=
\rho gV
$$

which reduces to:

$$
\boxed{
m=\rho V
}
$$

For the canonical HydroLab-3D mass:

$$
m=20\text{ kg}
$$

and:

$$
\rho=1000\text{ kg/m}^3
$$

the neutral displaced volume is:

$$
V
=
\frac{20}{1000}
=
0.020\text{ m}^3
$$

This is exactly the neutral-buoyancy parameter combination used previously.

At any orientation, gravity and buoyancy have equal and opposite world-frame forces.

If they are also applied at the same body point:

$$
\mathbf{r}_G
=
\mathbf{r}_B
$$

then both force and moment cancel:

$$
\boxed{
\tau_H=0
}
$$

This must hold regardless of orientation.

---

## Coincident Centers

If:

$$
\mathbf{r}_G
=
\mathbf{r}_B
=
\mathbf{0}
$$

then both forces act through the body reference point.

The hydrostatic moment is therefore:

$$
\mathbf{M}_{H,B}=0
$$

even when the vehicle is not neutrally buoyant.

The remaining hydrostatic behavior is purely translational.

This provides the direct reduction to Notebook 04.

---

## Reduction to Notebook 04

At identity orientation:

$$
R_B^W=I
$$

so:

$$
\mathbf{F}_{G,B}
=
\mathbf{F}_{G,W}
$$

and:

$$
\mathbf{F}_{B,B}
=
\mathbf{F}_{B,W}
$$

Therefore, with both centers at the body origin, the translational part of the 6-DOF hydrostatic wrench must exactly equal:

$$
\mathbf{F}_G
+
\mathbf{F}_B
$$

from the existing production functions:

```text
gravity_force()
buoyancy_force()
```

This is the required backward-compatibility test.

---

## Passive Restoring Moment From CG / CB Separation

A common statically stable underwater configuration places the center of buoyancy above the center of gravity.

Under HydroLab-3D's body convention, consider:

$$
\mathbf{r}_G
=
\begin{bmatrix}
0\\
0\\
0
\end{bmatrix}
$$

and:

$$
\mathbf{r}_B
=
\begin{bmatrix}
0\\
0\\
d
\end{bmatrix}
$$

with:

$$
d>0
$$

For an upright vehicle:

$$
\phi=\theta=0
$$

the buoyancy force is parallel to the $z$ offset, so:

$$
\mathbf{r}_B
\times
\mathbf{F}_{B,B}
=
0
$$

and no hydrostatic moment exists.

Now apply a small positive roll:

$$
\phi>0
$$

while retaining exact neutral buoyancy.

The world-up buoyancy vector appears tilted in body coordinates.

For the selected convention, a positive roll should produce a hydrostatic roll moment opposing the displacement.

Thus for a small positive roll:

$$
\boxed{
K_H<0
}
$$

and for a corresponding negative roll:

$$
\boxed{
K_H>0
}
$$

This is the essential signature of passive roll stability.

The same principle applies to pitch for suitable CG/CB geometry.

---

## Canonical Restoring Configuration

For validation use:

$$
m=20\text{ kg}
$$

$$
\rho=1000\text{ kg/m}^3
$$

$$
V=0.020\text{ m}^3
$$

$$
g=9.81\text{ m/s}^2
$$

so:

$$
mg
=
\rho gV
=
196.2\text{ N}
$$

For the restoring-moment case use:

$$
\mathbf{r}_G
=
\begin{bmatrix}
0\\
0\\
0
\end{bmatrix}
$$

and:

$$
\mathbf{r}_B
=
\begin{bmatrix}
0\\
0\\
0.05
\end{bmatrix}
\text{ m}
$$

The maximum characteristic hydrostatic moment scale is therefore:

$$
\rho gV(0.05)
=
9.81\text{ N m}
$$

The actual moment depends on orientation.

---

## Prototype Function

This step introduces:

```text
hydrostatic_wrench()
```

as a notebook-local prototype.

It will reuse the existing production functions:

```text
gravity_force()
buoyancy_force()
rotation_body_to_world()
```

rather than rewriting their validated physical meaning.

---

## Validation Criteria

This step passes only if:

1. the returned hydrostatic wrench has shape $(6,)$;

2. all values remain finite;

3. identity orientation with coincident centers reproduces the production Notebook 04 force balance exactly;

4. positive buoyancy produces positive world/body $z$ force at identity orientation;

5. negative buoyancy produces negative world/body $z$ force at identity orientation;

6. exact neutral buoyancy with coincident CG and CB produces:

$$
\tau_H=0
$$

7. neutral buoyancy with coincident centers remains zero at a nontrivial orientation;

8. coincident centers produce zero hydrostatic moment even for a non-neutral vehicle;

9. a positive small roll with CB above CG produces an opposing roll moment;

10. the corresponding negative roll reverses the restoring-moment sign;

11. zero roll with vertical CB offset produces zero restoring moment;

12. gravity and buoyancy magnitudes continue to match the production Notebook 04 implementation;

13. malformed vectors and invalid physical parameters are rejected cleanly.

If these tests pass, Notebook 07 will have generalized the validated force-only buoyancy model into a physically meaningful 6-DOF hydrostatic restoring wrench.

In [7]:
import numpy as np

from hydrolab3d.dynamics.buoyancy import (
    gravity_force,
    buoyancy_force,
)
from hydrolab3d.utils.transforms import (
    rotation_body_to_world,
)


def hydrostatic_wrench(
    pose,
    mass,
    fluid_density,
    displaced_volume,
    center_of_gravity,
    center_of_buoyancy,
    gravity=9.81,
):
    """
    Compute the physical gravity + buoyancy wrench in the body frame.

    Parameters
    ----------
    pose : array_like, shape (6,)
        [x, y, z, roll, pitch, yaw].

    mass : float
        Vehicle mass [kg].

    fluid_density : float
        Fluid density [kg/m^3].

    displaced_volume : float
        Displaced volume [m^3].

    center_of_gravity : array_like, shape (3,)
        CG position relative to body reference point [m].

    center_of_buoyancy : array_like, shape (3,)
        CB position relative to body reference point [m].

    gravity : float
        Positive gravitational acceleration magnitude [m/s^2].

    Returns
    -------
    numpy.ndarray
        Body-frame generalized hydrostatic wrench:
        [X, Y, Z, K, M, N].
    """
    pose = np.asarray(
        pose,
        dtype=float,
    )

    center_of_gravity = np.asarray(
        center_of_gravity,
        dtype=float,
    )

    center_of_buoyancy = np.asarray(
        center_of_buoyancy,
        dtype=float,
    )

    if pose.shape != (6,):
        raise ValueError(
            "pose must have shape (6,)"
        )

    if center_of_gravity.shape != (3,):
        raise ValueError(
            "center_of_gravity must have shape (3,)"
        )

    if center_of_buoyancy.shape != (3,):
        raise ValueError(
            "center_of_buoyancy must have shape (3,)"
        )

    if not np.all(np.isfinite(pose)):
        raise ValueError(
            "pose must contain only finite values"
        )

    if not np.all(np.isfinite(center_of_gravity)):
        raise ValueError(
            "center_of_gravity must contain only finite values"
        )

    if not np.all(np.isfinite(center_of_buoyancy)):
        raise ValueError(
            "center_of_buoyancy must contain only finite values"
        )

    mass = float(mass)
    fluid_density = float(fluid_density)
    displaced_volume = float(displaced_volume)
    gravity = float(gravity)

    if not np.isfinite(mass) or mass <= 0.0:
        raise ValueError(
            "mass must be finite and strictly positive"
        )

    if (
        not np.isfinite(fluid_density)
        or fluid_density <= 0.0
    ):
        raise ValueError(
            "fluid_density must be finite and strictly positive"
        )

    if (
        not np.isfinite(displaced_volume)
        or displaced_volume <= 0.0
    ):
        raise ValueError(
            "displaced_volume must be finite and strictly positive"
        )

    if not np.isfinite(gravity) or gravity <= 0.0:
        raise ValueError(
            "gravity must be finite and strictly positive"
        )

    roll, pitch, yaw = pose[3:]

    R_bw = rotation_body_to_world(
        roll,
        pitch,
        yaw,
    )

    # Reuse Notebook 04 production force definitions.
    F_g_world = gravity_force(
        mass,
        gravity,
    )

    F_b_world = buoyancy_force(
        fluid_density,
        displaced_volume,
        gravity,
    )

    # World -> body.
    F_g_body = R_bw.T @ F_g_world
    F_b_body = R_bw.T @ F_b_world

    F_hydrostatic_body = (
        F_g_body + F_b_body
    )

    M_g_body = np.cross(
        center_of_gravity,
        F_g_body,
    )

    M_b_body = np.cross(
        center_of_buoyancy,
        F_b_body,
    )

    M_hydrostatic_body = (
        M_g_body + M_b_body
    )

    return np.concatenate(
        [
            F_hydrostatic_body,
            M_hydrostatic_body,
        ]
    )


# ---------------------------------------------------------------------
# Canonical parameters
# ---------------------------------------------------------------------

mass_h = 20.0
rho_h = 1000.0
gravity_h = 9.81

neutral_volume = (
    mass_h / rho_h
)

r_g_origin = np.zeros(3)
r_b_origin = np.zeros(3)

pose_identity_h = np.zeros(6)


# ---------------------------------------------------------------------
# Production Notebook 04 force magnitudes
# ---------------------------------------------------------------------

F_g_production = gravity_force(
    mass_h,
    gravity_h,
)

F_b_neutral_production = buoyancy_force(
    rho_h,
    neutral_volume,
    gravity_h,
)

np.testing.assert_allclose(
    F_g_production,
    np.array([0.0, 0.0, -196.2]),
    atol=1e-12,
)

np.testing.assert_allclose(
    F_b_neutral_production,
    np.array([0.0, 0.0, 196.2]),
    atol=1e-12,
)


# ---------------------------------------------------------------------
# Exact neutral buoyancy, coincident centers
# ---------------------------------------------------------------------

tau_neutral_identity = hydrostatic_wrench(
    pose_identity_h,
    mass_h,
    rho_h,
    neutral_volume,
    r_g_origin,
    r_b_origin,
    gravity_h,
)

np.testing.assert_allclose(
    tau_neutral_identity,
    np.zeros(6),
    atol=1e-12,
)


# ---------------------------------------------------------------------
# Notebook 04 reduction: positive buoyancy
# ---------------------------------------------------------------------

positive_volume = 0.022

tau_positive = hydrostatic_wrench(
    pose_identity_h,
    mass_h,
    rho_h,
    positive_volume,
    r_g_origin,
    r_b_origin,
    gravity_h,
)

expected_positive_force = (
    gravity_force(
        mass_h,
        gravity_h,
    )
    +
    buoyancy_force(
        rho_h,
        positive_volume,
        gravity_h,
    )
)

np.testing.assert_allclose(
    tau_positive[:3],
    expected_positive_force,
    atol=1e-12,
)

np.testing.assert_allclose(
    tau_positive[3:],
    np.zeros(3),
    atol=1e-12,
)

assert tau_positive[2] > 0.0


# ---------------------------------------------------------------------
# Notebook 04 reduction: negative buoyancy
# ---------------------------------------------------------------------

negative_volume = 0.018

tau_negative = hydrostatic_wrench(
    pose_identity_h,
    mass_h,
    rho_h,
    negative_volume,
    r_g_origin,
    r_b_origin,
    gravity_h,
)

expected_negative_force = (
    gravity_force(
        mass_h,
        gravity_h,
    )
    +
    buoyancy_force(
        rho_h,
        negative_volume,
        gravity_h,
    )
)

np.testing.assert_allclose(
    tau_negative[:3],
    expected_negative_force,
    atol=1e-12,
)

np.testing.assert_allclose(
    tau_negative[3:],
    np.zeros(3),
    atol=1e-12,
)

assert tau_negative[2] < 0.0


# ---------------------------------------------------------------------
# Neutral buoyancy remains force/moment-free at arbitrary orientation
# when CG and CB coincide
# ---------------------------------------------------------------------

pose_arbitrary = np.array(
    [
        1.0,
        -2.0,
        3.0,
        np.deg2rad(25.0),
        np.deg2rad(-15.0),
        np.deg2rad(40.0),
    ]
)

tau_neutral_arbitrary = hydrostatic_wrench(
    pose_arbitrary,
    mass_h,
    rho_h,
    neutral_volume,
    r_g_origin,
    r_b_origin,
    gravity_h,
)

np.testing.assert_allclose(
    tau_neutral_arbitrary,
    np.zeros(6),
    atol=1e-12,
)


# ---------------------------------------------------------------------
# Coincident centers -> no moment even when non-neutral
# ---------------------------------------------------------------------

tau_non_neutral_oriented = hydrostatic_wrench(
    pose_arbitrary,
    mass_h,
    rho_h,
    positive_volume,
    r_g_origin,
    r_b_origin,
    gravity_h,
)

np.testing.assert_allclose(
    tau_non_neutral_oriented[3:],
    np.zeros(3),
    atol=1e-12,
)


# ---------------------------------------------------------------------
# Restoring moment: center of buoyancy above CG
# ---------------------------------------------------------------------

r_b_stable = np.array(
    [
        0.0,
        0.0,
        0.05,
    ]
)

roll_angle = np.deg2rad(10.0)

pose_positive_roll = np.array(
    [
        0.0,
        0.0,
        0.0,
        roll_angle,
        0.0,
        0.0,
    ]
)

pose_negative_roll = np.array(
    [
        0.0,
        0.0,
        0.0,
        -roll_angle,
        0.0,
        0.0,
    ]
)

tau_positive_roll = hydrostatic_wrench(
    pose_positive_roll,
    mass_h,
    rho_h,
    neutral_volume,
    r_g_origin,
    r_b_stable,
    gravity_h,
)

tau_negative_roll = hydrostatic_wrench(
    pose_negative_roll,
    mass_h,
    rho_h,
    neutral_volume,
    r_g_origin,
    r_b_stable,
    gravity_h,
)

tau_zero_roll = hydrostatic_wrench(
    pose_identity_h,
    mass_h,
    rho_h,
    neutral_volume,
    r_g_origin,
    r_b_stable,
    gravity_h,
)


# Positive roll must produce negative restoring roll moment.
assert tau_positive_roll[3] < 0.0

# Negative roll must produce positive restoring roll moment.
assert tau_negative_roll[3] > 0.0

# Equal-and-opposite orientation should reverse the moment.
np.testing.assert_allclose(
    tau_negative_roll[3],
    -tau_positive_roll[3],
    atol=1e-12,
)

# Upright configuration has no hydrostatic moment.
np.testing.assert_allclose(
    tau_zero_roll[3:],
    np.zeros(3),
    atol=1e-12,
)


# ---------------------------------------------------------------------
# Analytical small/restoring roll moment check
#
# With neutral buoyancy, CG at origin, CB = [0,0,d],
# pure roll phi, pitch=yaw=0:
#
# K = -B * d * sin(phi)
# ---------------------------------------------------------------------

B = rho_h * gravity_h * neutral_volume
d = r_b_stable[2]

expected_roll_moment = (
    -B * d * np.sin(roll_angle)
)

np.testing.assert_allclose(
    tau_positive_roll[3],
    expected_roll_moment,
    atol=1e-12,
)


# ---------------------------------------------------------------------
# Finite-value validation
# ---------------------------------------------------------------------

validated_hydrostatic_wrenches = [
    tau_neutral_identity,
    tau_positive,
    tau_negative,
    tau_neutral_arbitrary,
    tau_non_neutral_oriented,
    tau_positive_roll,
    tau_negative_roll,
    tau_zero_roll,
]

for wrench in validated_hydrostatic_wrenches:
    assert wrench.shape == (6,)
    assert np.all(np.isfinite(wrench))


# ---------------------------------------------------------------------
# Invalid-input checks
# ---------------------------------------------------------------------

invalid_hydro_calls = [
    lambda: hydrostatic_wrench(
        [0.0, 0.0],
        mass_h,
        rho_h,
        neutral_volume,
        r_g_origin,
        r_b_origin,
        gravity_h,
    ),
    lambda: hydrostatic_wrench(
        pose_identity_h,
        0.0,
        rho_h,
        neutral_volume,
        r_g_origin,
        r_b_origin,
        gravity_h,
    ),
    lambda: hydrostatic_wrench(
        pose_identity_h,
        mass_h,
        -1000.0,
        neutral_volume,
        r_g_origin,
        r_b_origin,
        gravity_h,
    ),
    lambda: hydrostatic_wrench(
        pose_identity_h,
        mass_h,
        rho_h,
        0.0,
        r_g_origin,
        r_b_origin,
        gravity_h,
    ),
    lambda: hydrostatic_wrench(
        pose_identity_h,
        mass_h,
        rho_h,
        neutral_volume,
        [0.0, 0.0],
        r_b_origin,
        gravity_h,
    ),
    lambda: hydrostatic_wrench(
        pose_identity_h,
        mass_h,
        rho_h,
        neutral_volume,
        r_g_origin,
        [0.0, np.nan, 0.0],
        gravity_h,
    ),
    lambda: hydrostatic_wrench(
        pose_identity_h,
        mass_h,
        rho_h,
        neutral_volume,
        r_g_origin,
        r_b_origin,
        -9.81,
    ),
]

invalid_hydro_rejected = 0

for call in invalid_hydro_calls:
    try:
        call()
    except ValueError:
        invalid_hydro_rejected += 1

assert invalid_hydro_rejected == len(
    invalid_hydro_calls
)


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("Production gravity force:")
print(F_g_production)

print("\nProduction neutral buoyancy force:")
print(F_b_neutral_production)

print("\nNeutral identity hydrostatic wrench:")
print(tau_neutral_identity)

print("\nPositive-buoyancy hydrostatic wrench:")
print(tau_positive)

print("\nNegative-buoyancy hydrostatic wrench:")
print(tau_negative)

print(
    "\nNeutral arbitrary-orientation wrench "
    "(coincident CG/CB):"
)
print(tau_neutral_arbitrary)

print("\nPositive 10-degree roll restoring wrench:")
print(tau_positive_roll)

print("\nNegative 10-degree roll restoring wrench:")
print(tau_negative_roll)

print(
    "\nAnalytical positive-roll moment:",
    expected_roll_moment,
)

print(
    "Numerical positive-roll moment :",
    tau_positive_roll[3],
)

print(
    "\nInvalid physical/input cases rejected:",
    f"{invalid_hydro_rejected}/"
    f"{len(invalid_hydro_calls)}",
)

print(
    "\nPASS: Notebook 04 gravity and buoyancy "
    "forces are preserved exactly."
)

print(
    "PASS: neutral buoyancy with coincident CG/CB "
    "produces zero hydrostatic wrench."
)

print(
    "PASS: coincident centers produce no "
    "hydrostatic moment."
)

print(
    "PASS: positive and negative buoyancy retain "
    "the validated Notebook 04 force direction."
)

print(
    "PASS: CB-above-CG geometry produces an "
    "opposing passive roll-restoring moment."
)

print(
    "PASS: analytical and numerical restoring "
    "moments agree."
)

print(
    "PASS: invalid hydrostatic parameters and "
    "malformed inputs are rejected cleanly."
)

Production gravity force:
[   0.     0.  -196.2]

Production neutral buoyancy force:
[  0.    0.  196.2]

Neutral identity hydrostatic wrench:
[0. 0. 0. 0. 0. 0.]

Positive-buoyancy hydrostatic wrench:
[ 0.    0.   19.62  0.    0.    0.  ]

Negative-buoyancy hydrostatic wrench:
[  0.     0.   -19.62   0.     0.     0.  ]

Neutral arbitrary-orientation wrench (coincident CG/CB):
[0. 0. 0. 0. 0. 0.]

Positive 10-degree roll restoring wrench:
[ 0.          0.          0.         -1.70348862  0.          0.        ]

Negative 10-degree roll restoring wrench:
[0.         0.         0.         1.70348862 0.         0.        ]

Analytical positive-roll moment: -1.703488622912587
Numerical positive-roll moment : -1.7034886229125867

Invalid physical/input cases rejected: 7/7

PASS: Notebook 04 gravity and buoyancy forces are preserved exactly.
PASS: neutral buoyancy with coincident CG/CB produces zero hydrostatic wrench.
PASS: coincident centers produce no hydrostatic moment.
PASS: positive a

## Step 8 - Ocean-Current Transformation and Generalized Water-Relative Velocity

Notebook 06 established the central hydrodynamic invariant:

$$
\boxed{
\text{hydrodynamic effects depend on motion relative to the surrounding water}
}
$$

However, Notebook 06 expressed both vehicle velocity and ocean current in the world frame.

Notebook 07 expresses the marine dynamics using body-frame generalized velocity:

$$
\nu
=
\begin{bmatrix}
u\\
v\\
w\\
p\\
q\\
r
\end{bmatrix}
$$

while the existing ocean-current model still supplies:

$$
\mathbf{v}_{c,W}
$$

in the HydroLab-3D world frame.

These quantities cannot be subtracted directly.

This step therefore establishes the required frame-consistent bridge:

```text
world-frame current
        ↓
world-to-body rotation
        ↓
body-frame current
        ↓
generalized current velocity
        ↓
water-relative generalized velocity
```

No hydrodynamic damping or vehicle integration is performed yet.

The objective is to validate the current transformation itself before it enters any dynamic term.

---

## Inherited Notebook 06 Convention

Notebook 06 validated a world-frame uniform current:

$$
\mathbf{v}_{c,W}
=
\begin{bmatrix}
v_{c,x}\\
v_{c,y}\\
v_{c,z}
\end{bmatrix}
$$

and, when vehicle velocity was also represented in the world frame:

$$
\boxed{
\mathbf{v}_{rel,W}
=
\mathbf{v}_W
-
\mathbf{v}_{c,W}
}
$$

The production utilities are:

```text
uniform_current_velocity()
relative_water_velocity()
```

Notebook 07 preserves this physical meaning.

Only the frame in which the hydrodynamic calculation is performed changes.

---

## World-to-Body Current Transformation

Notebook 01 established:

$$
\mathbf{v}_W
=
R_B^W
\mathbf{v}_B
$$

with:

$$
R_B^W
=
R_z(\psi)
R_y(\theta)
R_x(\phi)
$$

Since the rotation matrix is orthogonal:

$$
(R_B^W)^{-1}
=
(R_B^W)^T
$$

a world-frame ocean-current vector is expressed in body coordinates using:

$$
\boxed{
\mathbf{v}_{c,B}
=
(R_B^W)^T
\mathbf{v}_{c,W}
}
$$

This transformation must be recomputed as vehicle orientation changes.

A current that is constant in the world frame is generally **not constant in body coordinates** for a rotating vehicle.

---

## Generalized Current Velocity

The current model validated in Notebook 06 represents translational water motion only.

It does not represent a rotating fluid parcel or ambient angular velocity.

Therefore the generalized current velocity used by the initial 6-DOF model is:

$$
\boxed{
\nu_c
=
\begin{bmatrix}
\mathbf{v}_{c,B}\\
0\\
0\\
0
\end{bmatrix}
}
$$

or explicitly:

$$
\nu_c
=
\begin{bmatrix}
u_c\\
v_c\\
w_c\\
0\\
0\\
0
\end{bmatrix}
$$

This is a deliberate selective-fidelity approximation.

The ocean current affects translational water-relative motion, while the vehicle's body angular rates remain relative to a non-rotating local water frame.

Thus:

$$
p_r=p
$$

$$
q_r=q
$$

$$
r_r=r
$$

for the current model used in Notebook 07.

---

## Generalized Water-Relative Velocity

The body-frame generalized water-relative velocity is:

$$
\boxed{
\nu_r
=
\nu-\nu_c
}
$$

Therefore:

$$
\nu_r
=
\begin{bmatrix}
u-u_c\\
v-v_c\\
w-w_c\\
p\\
q\\
r
\end{bmatrix}
$$

This quantity will later feed:

$$
C_A(\nu_r)\nu_r
$$

and:

$$
\tau_D(\nu_r)
$$

so the added-mass and damping terms operate on motion relative to the surrounding fluid.

The rigid-body Coriolis term remains based on:

$$
\nu
$$

rather than:

$$
\nu_r
$$

because rigid-body inertia belongs to the vehicle itself.

---

## Identity-Orientation Reduction

At:

$$
\phi=\theta=\psi=0
$$

we have:

$$
R_B^W=I
$$

and therefore:

$$
\mathbf{v}_{c,B}
=
\mathbf{v}_{c,W}
$$

For translational vehicle velocity:

$$
\mathbf{v}_B
=
\mathbf{v}_W
$$

the Notebook 07 construction becomes:

$$
\mathbf{v}_{r,B}
=
\mathbf{v}_B
-
\mathbf{v}_{c,B}
$$

which reduces exactly to the Notebook 06 relation:

$$
\boxed{
\mathbf{v}_{rel}
=
\mathbf{v}
-
\mathbf{v}_c
}
$$

This provides the direct backward-compatibility case.

---

## Known 90-Degree Yaw Case

Consider:

$$
\psi
=
90^\circ
$$

with:

$$
\phi=\theta=0
$$

Suppose the world current is:

$$
\mathbf{v}_{c,W}
=
\begin{bmatrix}
1\\
0\\
0
\end{bmatrix}
$$

The vehicle's body $x$ axis now points along world $+y$.

Therefore world $+x$ appears along body $-y$.

The expected transformed current is:

$$
\boxed{
\mathbf{v}_{c,B}
=
\begin{bmatrix}
0\\
-1\\
0
\end{bmatrix}
}
$$

This provides a simple analytical orientation test.

---

## Rotation Must Preserve Current Magnitude

A pure reference-frame rotation cannot alter vector magnitude.

Therefore:

$$
\boxed{
\|\mathbf{v}_{c,B}\|
=
\|\mathbf{v}_{c,W}\|
}
$$

for every valid orientation.

This is a useful structural check that the transformation is a rotation rather than an unintended scaling or projection.

---

## Vehicle Moving With the Water

Suppose:

$$
\mathbf{v}_B
=
\mathbf{v}_{c,B}
$$

Then:

$$
\mathbf{v}_{r,B}
=
0
$$

and therefore the translational components of:

$$
\nu_r
$$

must vanish exactly within numerical tolerance.

This is the body-frame form of Notebook 06's physical invariant.

If angular velocity is also zero:

$$
\boldsymbol{\omega}=0
$$

then:

$$
\boxed{
\nu_r=0
}
$$

and the hydrodynamic damping validated in Step 6 must subsequently produce:

$$
\tau_D=0
$$

---

## Stationary Vehicle in Moving Water

For a vehicle stationary relative to the body/world kinematic state:

$$
\mathbf{v}_B=0
$$

with nonzero current:

$$
\mathbf{v}_{c,B}\neq0
$$

we obtain:

$$
\mathbf{v}_{r,B}
=
-\mathbf{v}_{c,B}
$$

Thus the vehicle is not stationary relative to the fluid.

When this relative velocity is later passed to the damping law, the resulting hydrodynamic force will act in the direction of the current, reproducing the physical behavior validated in Notebook 06.

---

## Arbitrary Orientation Consistency

For a general orientation:

$$
(\phi,\theta,\psi)\neq0
$$

we compute:

$$
\mathbf{v}_{c,B}
=
(R_B^W)^T
\mathbf{v}_{c,W}
$$

A round-trip transformation should satisfy:

$$
\boxed{
R_B^W\mathbf{v}_{c,B}
=
\mathbf{v}_{c,W}
}
$$

within floating-point tolerance.

This confirms frame consistency independently of any particular axis-aligned case.

---

## Relationship to the Existing Notebook 06 Production API

We will continue to use:

```text
uniform_current_velocity()
```

to construct the world-frame current.

For the identity-orientation reduction case, we will compare the new translational body-relative velocity directly against:

```text
relative_water_velocity()
```

This establishes that Notebook 07 extends rather than replaces the validated Notebook 06 semantics.

The new notebook-local functionality is only responsible for:

1. transforming the existing world-frame current into the body frame;
2. embedding it into six dimensions;
3. subtracting it from generalized body velocity.

---

## Prototype Functions

This step introduces:

```text
current_velocity_body()
generalized_current_velocity()
relative_velocity_6dof()
```

as notebook-local prototypes.

They will reuse:

```text
rotation_body_to_world()
uniform_current_velocity()
relative_water_velocity()
```

where appropriate.

Nothing is promoted into production code yet.

---

## Validation Criteria

This step passes only if:

1. body-frame current has shape $(3,)$;

2. generalized current velocity has shape $(6,)$;

3. generalized relative velocity has shape $(6,)$;

4. all results remain finite;

5. identity orientation leaves the world-frame current unchanged;

6. the 90-degree yaw case transforms world $+x$ current into body $-y$ current;

7. current magnitude is invariant under frame rotation;

8. arbitrary-orientation round-trip transformation recovers the original world current;

9. the final three components of generalized current velocity are exactly zero;

10. a vehicle translating exactly with the body-frame water velocity has zero translational relative velocity;

11. a vehicle moving with the water and having zero angular velocity produces:

$$
\nu_r=0
$$

12. a stationary vehicle in moving water produces:

$$
\mathbf{v}_{r,B}
=
-\mathbf{v}_{c,B}
$$

13. the identity-orientation translation-only case exactly reproduces the Notebook 06 production:

$$
\mathbf{v}_{rel}
=
\mathbf{v}
-
\mathbf{v}_c
$$

14. malformed states, currents, and poses are rejected cleanly.

If these checks pass, HydroLab-3D will have a validated frame-consistent water-relative velocity suitable for both six-axis damping and added-mass Coriolis effects.

In [8]:
import numpy as np

from hydrolab3d.utils.transforms import (
    rotation_body_to_world,
)
from hydrolab3d.world.currents import (
    uniform_current_velocity,
    relative_water_velocity,
)


def current_velocity_body(
    pose,
    current_velocity_world,
):
    """
    Express a world-frame translational ocean current in the
    vehicle body frame.

    v_c,B = R_B^W.T @ v_c,W

    Parameters
    ----------
    pose : array_like, shape (6,)
        [x, y, z, roll, pitch, yaw].

    current_velocity_world : array_like, shape (3,)
        Ocean-current velocity expressed in the world frame.

    Returns
    -------
    numpy.ndarray
        Current velocity expressed in the body frame.
    """
    pose = np.asarray(
        pose,
        dtype=float,
    )

    current_velocity_world = np.asarray(
        current_velocity_world,
        dtype=float,
    )

    if pose.shape != (6,):
        raise ValueError(
            "pose must have shape (6,)"
        )

    if current_velocity_world.shape != (3,):
        raise ValueError(
            "current_velocity_world must have shape (3,)"
        )

    if not np.all(np.isfinite(pose)):
        raise ValueError(
            "pose must contain only finite values"
        )

    if not np.all(
        np.isfinite(current_velocity_world)
    ):
        raise ValueError(
            "current_velocity_world must contain only finite values"
        )

    roll, pitch, yaw = pose[3:]

    R_bw = rotation_body_to_world(
        roll,
        pitch,
        yaw,
    )

    return R_bw.T @ current_velocity_world


def generalized_current_velocity(
    pose,
    current_velocity_world,
):
    """
    Construct the six-component generalized current velocity.

    The current model is translational only:

        nu_c = [u_c, v_c, w_c, 0, 0, 0]
    """
    v_c_body = current_velocity_body(
        pose,
        current_velocity_world,
    )

    nu_c = np.zeros(6, dtype=float)
    nu_c[:3] = v_c_body

    return nu_c


def relative_velocity_6dof(
    pose,
    body_velocity,
    current_velocity_world,
):
    """
    Compute generalized body-frame water-relative velocity:

        nu_r = nu - nu_c
    """
    body_velocity = np.asarray(
        body_velocity,
        dtype=float,
    )

    if body_velocity.shape != (6,):
        raise ValueError(
            "body_velocity must have shape (6,)"
        )

    if not np.all(np.isfinite(body_velocity)):
        raise ValueError(
            "body_velocity must contain only finite values"
        )

    nu_c = generalized_current_velocity(
        pose,
        current_velocity_world,
    )

    return body_velocity - nu_c


# ---------------------------------------------------------------------
# Canonical world current from Notebook 06 production API
# ---------------------------------------------------------------------

current_world = uniform_current_velocity(
    np.array(
        [
            0.50,
            -0.25,
            0.10,
        ]
    )
)

assert current_world.shape == (3,)
assert np.all(np.isfinite(current_world))


# ---------------------------------------------------------------------
# Identity-orientation transformation
# ---------------------------------------------------------------------

pose_identity_current = np.zeros(6)

current_body_identity = current_velocity_body(
    pose_identity_current,
    current_world,
)

np.testing.assert_allclose(
    current_body_identity,
    current_world,
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Generalized current structure
# ---------------------------------------------------------------------

nu_c_identity = generalized_current_velocity(
    pose_identity_current,
    current_world,
)

assert nu_c_identity.shape == (6,)

np.testing.assert_allclose(
    nu_c_identity[:3],
    current_world,
    atol=1e-15,
)

np.testing.assert_array_equal(
    nu_c_identity[3:],
    np.zeros(3),
)


# ---------------------------------------------------------------------
# Known 90-degree yaw transformation
# ---------------------------------------------------------------------

pose_yaw_90_current = np.array(
    [
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        np.pi / 2.0,
    ]
)

world_x_current = np.array(
    [
        1.0,
        0.0,
        0.0,
    ]
)

body_current_yaw_90 = current_velocity_body(
    pose_yaw_90_current,
    world_x_current,
)

expected_body_current_yaw_90 = np.array(
    [
        0.0,
        -1.0,
        0.0,
    ]
)

np.testing.assert_allclose(
    body_current_yaw_90,
    expected_body_current_yaw_90,
    atol=1e-14,
)


# ---------------------------------------------------------------------
# Arbitrary orientation:
# magnitude preservation and round-trip transformation
# ---------------------------------------------------------------------

pose_arbitrary_current = np.array(
    [
        3.0,
        -1.0,
        2.0,
        np.deg2rad(25.0),
        np.deg2rad(-15.0),
        np.deg2rad(40.0),
    ]
)

current_body_arbitrary = current_velocity_body(
    pose_arbitrary_current,
    current_world,
)

roll_c, pitch_c, yaw_c = (
    pose_arbitrary_current[3:]
)

R_bw_current = rotation_body_to_world(
    roll_c,
    pitch_c,
    yaw_c,
)

current_world_round_trip = (
    R_bw_current @ current_body_arbitrary
)

np.testing.assert_allclose(
    np.linalg.norm(current_body_arbitrary),
    np.linalg.norm(current_world),
    atol=1e-14,
)

np.testing.assert_allclose(
    current_world_round_trip,
    current_world,
    atol=1e-14,
)


# ---------------------------------------------------------------------
# Vehicle moving exactly with the water
# ---------------------------------------------------------------------

nu_with_water = np.zeros(6)
nu_with_water[:3] = current_body_arbitrary

nu_r_with_water = relative_velocity_6dof(
    pose_arbitrary_current,
    nu_with_water,
    current_world,
)

np.testing.assert_allclose(
    nu_r_with_water,
    np.zeros(6),
    atol=1e-14,
)


# ---------------------------------------------------------------------
# Stationary vehicle in moving water
# ---------------------------------------------------------------------

nu_stationary = np.zeros(6)

nu_r_stationary = relative_velocity_6dof(
    pose_arbitrary_current,
    nu_stationary,
    current_world,
)

np.testing.assert_allclose(
    nu_r_stationary[:3],
    -current_body_arbitrary,
    atol=1e-14,
)

np.testing.assert_array_equal(
    nu_r_stationary[3:],
    np.zeros(3),
)


# ---------------------------------------------------------------------
# Angular velocity remains unchanged by translational current model
# ---------------------------------------------------------------------

nu_rotating = np.array(
    [
        current_body_arbitrary[0],
        current_body_arbitrary[1],
        current_body_arbitrary[2],
        0.20,
        -0.10,
        0.15,
    ]
)

nu_r_rotating = relative_velocity_6dof(
    pose_arbitrary_current,
    nu_rotating,
    current_world,
)

np.testing.assert_allclose(
    nu_r_rotating[:3],
    np.zeros(3),
    atol=1e-14,
)

np.testing.assert_allclose(
    nu_r_rotating[3:],
    nu_rotating[3:],
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Direct reduction to Notebook 06 at identity orientation
# ---------------------------------------------------------------------

vehicle_world_velocity = np.array(
    [
        1.2,
        -0.4,
        0.3,
    ]
)

relative_notebook06 = relative_water_velocity(
    vehicle_world_velocity,
    current_world,
)

nu_identity_vehicle = np.array(
    [
        vehicle_world_velocity[0],
        vehicle_world_velocity[1],
        vehicle_world_velocity[2],
        0.0,
        0.0,
        0.0,
    ]
)

relative_notebook07 = relative_velocity_6dof(
    pose_identity_current,
    nu_identity_vehicle,
    current_world,
)

np.testing.assert_allclose(
    relative_notebook07[:3],
    relative_notebook06,
    atol=1e-15,
)

np.testing.assert_allclose(
    relative_notebook07[3:],
    np.zeros(3),
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Connect directly to Step 6 damping invariant
# ---------------------------------------------------------------------

tau_d_with_water = hydrodynamic_damping_wrench(
    nu_r_with_water,
    d_linear,
    d_quadratic,
)

np.testing.assert_allclose(
    tau_d_with_water,
    np.zeros(6),
    atol=1e-14,
)


# ---------------------------------------------------------------------
# Finite-value checks
# ---------------------------------------------------------------------

current_validation_results = [
    current_body_identity,
    nu_c_identity,
    body_current_yaw_90,
    current_body_arbitrary,
    current_world_round_trip,
    nu_r_with_water,
    nu_r_stationary,
    nu_r_rotating,
    relative_notebook07,
    tau_d_with_water,
]

for result in current_validation_results:
    assert np.all(np.isfinite(result))


# ---------------------------------------------------------------------
# Invalid-input validation
# ---------------------------------------------------------------------

invalid_current_calls = [
    lambda: current_velocity_body(
        [0.0, 0.0],
        current_world,
    ),
    lambda: current_velocity_body(
        pose_identity_current,
        [1.0, 2.0],
    ),
    lambda: current_velocity_body(
        [
            0.0,
            0.0,
            0.0,
            np.nan,
            0.0,
            0.0,
        ],
        current_world,
    ),
    lambda: current_velocity_body(
        pose_identity_current,
        [1.0, np.inf, 0.0],
    ),
    lambda: relative_velocity_6dof(
        pose_identity_current,
        [1.0, 2.0],
        current_world,
    ),
    lambda: relative_velocity_6dof(
        pose_identity_current,
        [
            0.0,
            0.0,
            0.0,
            0.0,
            np.nan,
            0.0,
        ],
        current_world,
    ),
]

invalid_current_rejected = 0

for call in invalid_current_calls:
    try:
        call()
    except ValueError:
        invalid_current_rejected += 1

assert invalid_current_rejected == len(
    invalid_current_calls
)


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("World-frame canonical current:")
print(current_world)

print("\nIdentity-orientation body current:")
print(current_body_identity)

print("\n90-degree yaw, world +x current -> body current:")
print(body_current_yaw_90)

print("\nArbitrary-orientation body current:")
print(current_body_arbitrary)

print(
    "\nWorld current magnitude:",
    np.linalg.norm(current_world),
)

print(
    "Body current magnitude :",
    np.linalg.norm(current_body_arbitrary),
)

print("\nRound-trip world current:")
print(current_world_round_trip)

print(
    "\nRelative velocity when vehicle moves with water:"
)
print(nu_r_with_water)

print(
    "\nRelative velocity of stationary vehicle in moving water:"
)
print(nu_r_stationary)

print(
    "\nRelative velocity with matched translation "
    "and nonzero body rotation:"
)
print(nu_r_rotating)

print("\nNotebook 06 production relative velocity:")
print(relative_notebook06)

print(
    "Notebook 07 identity-orientation reduction:"
)
print(relative_notebook07[:3])

print(
    "\nDamping wrench when vehicle moves exactly "
    "with water and does not rotate:"
)
print(tau_d_with_water)

print(
    "\nInvalid state/current cases rejected:",
    f"{invalid_current_rejected}/"
    f"{len(invalid_current_calls)}",
)

print(
    "\nPASS: world-frame current transforms correctly "
    "into the vehicle body frame."
)

print(
    "PASS: rotation preserves ocean-current magnitude."
)

print(
    "PASS: arbitrary body/world current round trip "
    "is numerically consistent."
)

print(
    "PASS: generalized current contains no ambient "
    "angular velocity."
)

print(
    "PASS: vehicle translation matching the water "
    "gives zero translational relative velocity."
)

print(
    "PASS: stationary vehicle sees the opposite "
    "body-frame current as relative velocity."
)

print(
    "PASS: identity-orientation reduction exactly "
    "reproduces Notebook 06 production semantics."
)

print(
    "PASS: zero generalized relative velocity produces "
    "zero Step 6 hydrodynamic damping."
)

World-frame canonical current:
[ 0.5  -0.25  0.1 ]

Identity-orientation body current:
[ 0.5  -0.25  0.1 ]

90-degree yaw, world +x current -> body current:
[ 6.123234e-17 -1.000000e+00  0.000000e+00]

Arbitrary-orientation body current:
[ 0.24063167 -0.44834624  0.25215481]

World current magnitude: 0.5678908345800273
Body current magnitude : 0.5678908345800273

Round-trip world current:
[ 0.5  -0.25  0.1 ]

Relative velocity when vehicle moves with water:
[0. 0. 0. 0. 0. 0.]

Relative velocity of stationary vehicle in moving water:
[-0.24063167  0.44834624 -0.25215481  0.          0.          0.        ]

Relative velocity with matched translation and nonzero body rotation:
[ 0.    0.    0.    0.2  -0.1   0.15]

Notebook 06 production relative velocity:
[ 0.7  -0.15  0.2 ]
Notebook 07 identity-orientation reduction:
[ 0.7  -0.15  0.2 ]

Damping wrench when vehicle moves exactly with water and does not rotate:
[-0. -0. -0. -0. -0. -0.]

Invalid state/current cases rejected: 6/6

PASS:

## Step 9 - Generalized Actuator Wrench From a Body-Fixed Thruster

Notebook 05 validated how a normalized actuator command becomes a body-frame thrust force.

Notebook 07 now needs to embed that validated three-dimensional force inside the six-dimensional marine dynamics.

The generalized actuator wrench is:

$$
\tau
=
\begin{bmatrix}
X\\
Y\\
Z\\
K\\
M\\
N
\end{bmatrix}
$$

where:

- $X,Y,Z$ are body-frame forces;
- $K,M,N$ are body-frame moments.

The objective of this step is deliberately narrow:

> Convert one already validated body-frame thruster force into a complete 6-DOF body wrench using its body-fixed application point.

We will **not** build a multi-thruster allocation system yet.

---

## Inherited Notebook 05 Thruster Model

Notebook 05 established the production actuator mapping:

```text
thruster_force_body(
    command,
    max_thrust,
    direction_body
)
```

For normalized command:

$$
u
$$

the saturated command is:

$$
u_{sat}
=
\operatorname{clip}(u,-1,1)
$$

and thrust magnitude is:

$$
F_T
=
u_{sat}F_{max}
$$

For a body-frame unit direction:

$$
\hat{\mathbf{d}}_B
$$

the generated force is:

$$
\boxed{
\mathbf{F}_{T,B}
=
F_T\hat{\mathbf{d}}_B
}
$$

Notebook 07 will reuse the production:

```text
thruster_force_body()
```

directly.

Command saturation, reverse thrust, maximum-thrust validation, and direction validation therefore remain inherited from Notebook 05 rather than being rewritten.

---

## Force Application Point

Let the thruster force act at body-fixed location:

$$
\mathbf{r}_T
=
\begin{bmatrix}
r_x\\
r_y\\
r_z
\end{bmatrix}
$$

measured from the body dynamics reference point.

The moment produced by that force is:

$$
\boxed{
\mathbf{M}_{T,B}
=
\mathbf{r}_T
\times
\mathbf{F}_{T,B}
}
$$

Therefore the complete actuator wrench is:

$$
\boxed{
\tau_T
=
\begin{bmatrix}
\mathbf{F}_{T,B}\\
\mathbf{r}_T\times\mathbf{F}_{T,B}
\end{bmatrix}
}
$$

or explicitly:

$$
\tau_T
=
\begin{bmatrix}
X_T\\
Y_T\\
Z_T\\
K_T\\
M_T\\
N_T
\end{bmatrix}
$$

---

## Thruster Through the Reference Point

If:

$$
\mathbf{r}_T
=
\mathbf{0}
$$

then:

$$
\mathbf{M}_{T,B}
=
\mathbf{0}\times\mathbf{F}_{T,B}
=
\mathbf{0}
$$

and therefore:

$$
\boxed{
\tau_T
=
\begin{bmatrix}
\mathbf{F}_{T,B}\\
\mathbf{0}
\end{bmatrix}
}
$$

This is the direct reduction to Notebook 05.

The previously validated force enters the 6-DOF model unchanged and produces no artificial moment.

---

## Offset-Thruster Example

Consider a forward-facing thruster:

$$
\hat{\mathbf{d}}_B
=
\begin{bmatrix}
1\\
0\\
0
\end{bmatrix}
$$

producing:

$$
\mathbf{F}_{T,B}
=
\begin{bmatrix}
20\\
0\\
0
\end{bmatrix}
\text{ N}
$$

Suppose it is mounted:

$$
0.25\text{ m}
$$

above the reference point:

$$
\mathbf{r}_T
=
\begin{bmatrix}
0\\
0\\
0.25
\end{bmatrix}
\text{ m}
$$

Then:

$$
\mathbf{M}_{T,B}
=
\begin{bmatrix}
0\\
0\\
0.25
\end{bmatrix}
\times
\begin{bmatrix}
20\\
0\\
0
\end{bmatrix}
$$

which gives:

$$
\boxed{
\mathbf{M}_{T,B}
=
\begin{bmatrix}
0\\
5\\
0
\end{bmatrix}
\text{ N m}
}
$$

Thus the generalized actuator wrench becomes:

$$
\boxed{
\tau_T
=
\begin{bmatrix}
20\\
0\\
0\\
0\\
5\\
0
\end{bmatrix}
}
$$

The same forward thrust therefore creates both surge force and pitch moment when applied away from the reference point.

---

## Offset Below the Reference Point

Reversing the vertical mounting offset:

$$
\mathbf{r}_T
=
\begin{bmatrix}
0\\
0\\
-0.25
\end{bmatrix}
$$

while retaining the same force reverses the pitch moment:

$$
\mathbf{M}_{T,B}
=
\begin{bmatrix}
0\\
-5\\
0
\end{bmatrix}
$$

This provides a clean sign-validation case.

---

## Force Parallel to Application-Point Vector

If:

$$
\mathbf{r}_T
\parallel
\mathbf{F}_{T,B}
$$

then:

$$
\mathbf{r}_T
\times
\mathbf{F}_{T,B}
=
0
$$

even though:

$$
\mathbf{r}_T\neq0
$$

An offset alone therefore does not guarantee a moment.

Only the perpendicular lever-arm component contributes.

---

## Zero Command

For:

$$
u=0
$$

Notebook 05 gives:

$$
\mathbf{F}_{T,B}=0
$$

and therefore automatically:

$$
\mathbf{M}_{T,B}=0
$$

so:

$$
\boxed{
\tau_T=0
}
$$

---

## Saturation Must Survive the 6-DOF Extension

The new wrench layer must not bypass or alter Notebook 05 saturation.

If:

$$
F_{max}=40\text{ N}
$$

and:

$$
u=2
$$

the production thruster model must still produce:

$$
F_T=40\text{ N}
$$

rather than:

$$
80\text{ N}
$$

The corresponding moment must therefore also be calculated from the **saturated physical force**.

For a lever arm magnitude perpendicular to thrust:

$$
d
$$

the maximum corresponding moment is:

$$
M_{max}=dF_{max}
$$

for that geometry.

---

## Reverse Thrust

A negative command reverses:

$$
\mathbf{F}_{T,B}
$$

Therefore:

$$
\mathbf{M}_{T,B}
=
\mathbf{r}_T
\times
\mathbf{F}_{T,B}
$$

must also reverse.

Thus:

$$
\boxed{
\tau_T(-u)
=
-\tau_T(u)
}
$$

for unsaturated symmetric command magnitude, and likewise for equally saturated positive and negative commands.

---

## Frame Convention

The generalized actuator wrench produced in this step is entirely **body-frame**:

$$
\tau_T
=
\tau_{T,B}
$$

This is intentional.

The 6-DOF dynamic equation is formulated in body coordinates, so no body-to-world transformation is required before applying the actuator wrench.

This differs from Notebook 05's:

```text
thruster_force_world()
```

which remains useful when a world-frame force representation is explicitly needed.

Notebook 07's dynamics should not unnecessarily transform a body-force into the world frame only to transform it back again.

---

## Multiple Thrusters Later

For several independent thrusters:

$$
i=1,\ldots,n
$$

their generalized wrenches would simply sum:

$$
\tau
=
\sum_{i=1}^{n}
\tau_{T,i}
$$

before any higher-level allocation logic is introduced.

However, constructing a thruster-allocation matrix is outside this step.

Notebook 07 first needs a correct primitive:

```text
one physical thruster
        ↓
body force
        ↓
force-at-offset moment
        ↓
6-DOF wrench
```

---

## Prototype Function

This step introduces the notebook-local function:

```text
thruster_wrench_body()
```

which:

1. calls the existing production `thruster_force_body()`;
2. validates the body-frame application point;
3. calculates:

$$
\mathbf{r}_T\times\mathbf{F}_{T,B}
$$

4. returns the complete six-component wrench.

Nothing is promoted into production code yet.

---

## Validation Criteria

This step passes only if:

1. the returned wrench has shape $(6,)$;

2. all values are finite;

3. the first three components exactly equal the existing Notebook 05 production body force;

4. a thruster acting through the reference point produces zero moment;

5. the canonical offset-thruster case produces:

$$
\tau_T
=
\begin{bmatrix}
20\\
0\\
0\\
0\\
5\\
0
\end{bmatrix}
$$

6. reversing the mounting offset reverses the corresponding moment;

7. reversing thrust reverses both force and moment;

8. zero command produces a zero generalized wrench;

9. command saturation from Notebook 05 remains intact;

10. a force whose line of action is parallel to its position vector produces zero moment;

11. the numerical moment exactly matches an independent:

```text
np.cross(r_T, F_T)
```

calculation;

12. malformed or non-finite application-point vectors are rejected.

If these checks pass, Notebook 07 will possess a validated bridge between the existing thruster model and the six-degree-of-freedom vehicle dynamics.

In [9]:
import numpy as np

from hydrolab3d.dynamics.thrusters import (
    thruster_force_body,
)


def thruster_wrench_body(
    command,
    max_thrust,
    direction_body,
    application_point_body,
):
    """
    Convert a validated Notebook 05 thruster force into a
    six-component body-frame actuator wrench.

    tau_T = [F_T,B, r_T x F_T,B]

    Parameters
    ----------
    command : float
        Normalized requested thruster command.

    max_thrust : float
        Maximum thrust magnitude [N].

    direction_body : array_like, shape (3,)
        Body-frame unit direction of positive thrust.

    application_point_body : array_like, shape (3,)
        Thruster force application point relative to the
        body dynamics reference point [m].

    Returns
    -------
    numpy.ndarray
        Body-frame generalized actuator wrench:
        [X, Y, Z, K, M, N].
    """
    application_point_body = np.asarray(
        application_point_body,
        dtype=float,
    )

    if application_point_body.shape != (3,):
        raise ValueError(
            "application_point_body must have shape (3,)"
        )

    if not np.all(
        np.isfinite(application_point_body)
    ):
        raise ValueError(
            "application_point_body must contain only finite values"
        )

    # Reuse the validated Notebook 05 production actuator mapping.
    force_body = thruster_force_body(
        command,
        max_thrust,
        direction_body,
    )

    moment_body = np.cross(
        application_point_body,
        force_body,
    )

    return np.concatenate(
        [
            force_body,
            moment_body,
        ]
    )


# ---------------------------------------------------------------------
# Canonical Notebook 05-compatible actuator
# ---------------------------------------------------------------------

max_thrust_t = 40.0

forward_body_t = np.array(
    [
        1.0,
        0.0,
        0.0,
    ]
)

command_t = 0.5

force_notebook05 = thruster_force_body(
    command_t,
    max_thrust_t,
    forward_body_t,
)

np.testing.assert_allclose(
    force_notebook05,
    np.array(
        [
            20.0,
            0.0,
            0.0,
        ]
    ),
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Through-reference-point reduction
# ---------------------------------------------------------------------

r_origin_t = np.zeros(3)

wrench_through_origin = thruster_wrench_body(
    command_t,
    max_thrust_t,
    forward_body_t,
    r_origin_t,
)

np.testing.assert_allclose(
    wrench_through_origin[:3],
    force_notebook05,
    atol=1e-15,
)

np.testing.assert_allclose(
    wrench_through_origin[3:],
    np.zeros(3),
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Canonical positive vertical offset
# ---------------------------------------------------------------------

r_above_t = np.array(
    [
        0.0,
        0.0,
        0.25,
    ]
)

wrench_above = thruster_wrench_body(
    command_t,
    max_thrust_t,
    forward_body_t,
    r_above_t,
)

expected_wrench_above = np.array(
    [
        20.0,
        0.0,
        0.0,
        0.0,
        5.0,
        0.0,
    ]
)

np.testing.assert_allclose(
    wrench_above,
    expected_wrench_above,
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Negative vertical offset
# ---------------------------------------------------------------------

r_below_t = np.array(
    [
        0.0,
        0.0,
        -0.25,
    ]
)

wrench_below = thruster_wrench_body(
    command_t,
    max_thrust_t,
    forward_body_t,
    r_below_t,
)

expected_wrench_below = np.array(
    [
        20.0,
        0.0,
        0.0,
        0.0,
        -5.0,
        0.0,
    ]
)

np.testing.assert_allclose(
    wrench_below,
    expected_wrench_below,
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Independent cross-product validation
# ---------------------------------------------------------------------

independent_moment = np.cross(
    r_above_t,
    force_notebook05,
)

np.testing.assert_allclose(
    wrench_above[3:],
    independent_moment,
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Reverse thrust
# ---------------------------------------------------------------------

wrench_reverse = thruster_wrench_body(
    -command_t,
    max_thrust_t,
    forward_body_t,
    r_above_t,
)

np.testing.assert_allclose(
    wrench_reverse,
    -wrench_above,
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Zero command
# ---------------------------------------------------------------------

wrench_zero_command = thruster_wrench_body(
    0.0,
    max_thrust_t,
    forward_body_t,
    r_above_t,
)

np.testing.assert_allclose(
    wrench_zero_command,
    np.zeros(6),
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Notebook 05 saturation survives wrench construction
# ---------------------------------------------------------------------

wrench_saturated_positive = thruster_wrench_body(
    2.0,
    max_thrust_t,
    forward_body_t,
    r_above_t,
)

wrench_saturated_negative = thruster_wrench_body(
    -2.0,
    max_thrust_t,
    forward_body_t,
    r_above_t,
)

expected_saturated_positive = np.array(
    [
        40.0,
        0.0,
        0.0,
        0.0,
        10.0,
        0.0,
    ]
)

np.testing.assert_allclose(
    wrench_saturated_positive,
    expected_saturated_positive,
    atol=1e-15,
)

np.testing.assert_allclose(
    wrench_saturated_negative,
    -expected_saturated_positive,
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Position vector parallel to force -> zero moment
# ---------------------------------------------------------------------

r_parallel_t = np.array(
    [
        0.30,
        0.0,
        0.0,
    ]
)

wrench_parallel = thruster_wrench_body(
    command_t,
    max_thrust_t,
    forward_body_t,
    r_parallel_t,
)

np.testing.assert_allclose(
    wrench_parallel[:3],
    force_notebook05,
    atol=1e-15,
)

np.testing.assert_allclose(
    wrench_parallel[3:],
    np.zeros(3),
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Arbitrary geometry cross-product validation
# ---------------------------------------------------------------------

direction_arbitrary = np.array(
    [
        0.0,
        1.0,
        0.0,
    ]
)

r_arbitrary = np.array(
    [
        0.20,
        -0.10,
        0.30,
    ]
)

wrench_arbitrary = thruster_wrench_body(
    0.4,
    50.0,
    direction_arbitrary,
    r_arbitrary,
)

force_arbitrary = thruster_force_body(
    0.4,
    50.0,
    direction_arbitrary,
)

moment_arbitrary_expected = np.cross(
    r_arbitrary,
    force_arbitrary,
)

np.testing.assert_allclose(
    wrench_arbitrary[:3],
    force_arbitrary,
    atol=1e-15,
)

np.testing.assert_allclose(
    wrench_arbitrary[3:],
    moment_arbitrary_expected,
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Structural / finite checks
# ---------------------------------------------------------------------

validated_wrenches = [
    wrench_through_origin,
    wrench_above,
    wrench_below,
    wrench_reverse,
    wrench_zero_command,
    wrench_saturated_positive,
    wrench_saturated_negative,
    wrench_parallel,
    wrench_arbitrary,
]

for wrench in validated_wrenches:
    assert wrench.shape == (6,)
    assert np.all(np.isfinite(wrench))


# ---------------------------------------------------------------------
# Invalid application-point validation
#
# Other actuator-parameter validation remains inherited directly
# from Notebook 05's thruster_force_body().
# ---------------------------------------------------------------------

invalid_application_calls = [
    lambda: thruster_wrench_body(
        command_t,
        max_thrust_t,
        forward_body_t,
        [0.0, 0.0],
    ),
    lambda: thruster_wrench_body(
        command_t,
        max_thrust_t,
        forward_body_t,
        [0.0, np.nan, 0.0],
    ),
    lambda: thruster_wrench_body(
        command_t,
        max_thrust_t,
        forward_body_t,
        [0.0, 0.0, np.inf],
    ),
]

invalid_application_rejected = 0

for call in invalid_application_calls:
    try:
        call()
    except ValueError:
        invalid_application_rejected += 1

assert invalid_application_rejected == len(
    invalid_application_calls
)


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("Notebook 05 production body force:")
print(force_notebook05)

print("\nThruster through reference point:")
print(wrench_through_origin)

print("\nThruster 0.25 m above reference point:")
print(wrench_above)

print("\nExpected offset wrench:")
print(expected_wrench_above)

print("\nThruster 0.25 m below reference point:")
print(wrench_below)

print("\nIndependent r x F moment:")
print(independent_moment)

print("\nReverse-thrust wrench:")
print(wrench_reverse)

print("\nZero-command wrench:")
print(wrench_zero_command)

print("\nPositive saturated wrench:")
print(wrench_saturated_positive)

print("\nNegative saturated wrench:")
print(wrench_saturated_negative)

print("\nParallel line-of-action wrench:")
print(wrench_parallel)

print("\nArbitrary geometry wrench:")
print(wrench_arbitrary)

print("Arbitrary expected moment:")
print(moment_arbitrary_expected)

print(
    "\nInvalid application-point cases rejected:",
    f"{invalid_application_rejected}/"
    f"{len(invalid_application_calls)}",
)

print(
    "\nPASS: Notebook 05 production thrust force "
    "is preserved exactly."
)

print(
    "PASS: force through the reference point "
    "produces zero actuator moment."
)

print(
    "PASS: offset thrust produces the expected "
    "r x F body moment."
)

print(
    "PASS: reversing mounting offset reverses "
    "the corresponding moment."
)

print(
    "PASS: reverse thrust reverses both force "
    "and moment."
)

print(
    "PASS: zero command produces zero generalized wrench."
)

print(
    "PASS: Notebook 05 command saturation remains intact "
    "through the 6-DOF wrench mapping."
)

print(
    "PASS: a force parallel to its application-point "
    "vector produces zero moment."
)

print(
    "PASS: malformed and non-finite application points "
    "are rejected cleanly."
)

Notebook 05 production body force:
[20.  0.  0.]

Thruster through reference point:
[20.  0.  0.  0.  0.  0.]

Thruster 0.25 m above reference point:
[20.  0.  0.  0.  5.  0.]

Expected offset wrench:
[20.  0.  0.  0.  5.  0.]

Thruster 0.25 m below reference point:
[20.  0.  0.  0. -5.  0.]

Independent r x F moment:
[0. 5. 0.]

Reverse-thrust wrench:
[-20.  -0.  -0.   0.  -5.   0.]

Zero-command wrench:
[0. 0. 0. 0. 0. 0.]

Positive saturated wrench:
[40.  0.  0.  0. 10.  0.]

Negative saturated wrench:
[-40.  -0.  -0.   0. -10.   0.]

Parallel line-of-action wrench:
[20.  0.  0.  0.  0.  0.]

Arbitrary geometry wrench:
[ 0. 20.  0. -6.  0.  4.]
Arbitrary expected moment:
[-6.  0.  4.]

Invalid application-point cases rejected: 3/3

PASS: Notebook 05 production thrust force is preserved exactly.
PASS: force through the reference point produces zero actuator moment.
PASS: offset thrust produces the expected r x F body moment.
PASS: reversing mounting offset reverses the corresponding 

## Step 10 - Assemble the Complete Current-Aware 6-DOF State Derivative

The individual components required by the integrated UUV dynamics have now been validated:

```text
6-DOF kinematics
        +
rigid-body inertia
        +
added mass
        +
rigid-body Coriolis
        +
added-mass Coriolis
        +
six-axis damping
        +
gravity / buoyancy restoring wrench
        +
world-frame ocean current
        +
water-relative velocity
        +
generalized actuator wrench
```

This step assembles those pieces into the first complete Notebook 07 state derivative.

No timestep integration is performed yet.

The objective is only to compute:

$$
\boxed{
\dot{\eta}
}
$$

and:

$$
\boxed{
\dot{\nu}
}
$$

from a supplied instantaneous state and actuator wrench.

---

## Kinematic Equation

The pose derivative remains the already validated relation:

$$
\boxed{
\dot{\eta}
=
J(\eta)\nu
}
$$

where:

$$
\eta
=
\begin{bmatrix}
x &
y &
z &
\phi &
\theta &
\psi
\end{bmatrix}^{T}
$$

and:

$$
\nu
=
\begin{bmatrix}
u &
v &
w &
p &
q &
r
\end{bmatrix}^{T}
$$

No dynamic force term enters this equation directly.

---

# Dynamic Equation

For the current-aware added-mass formulation, it is useful to begin from the relative-acceleration form:

$$
M_{RB}\dot{\nu}
+
C_{RB}(\nu)\nu
+
M_A\dot{\nu}_r
+
C_A(\nu_r)\nu_r
=
\tau_{act}
+
\tau_H
+
\tau_D
+
\tau_{env}
$$

where:

- $M_{RB}$ is rigid-body inertia;
- $M_A$ is added mass;
- $\tau_{act}$ is actuator wrench;
- $\tau_H$ is the physical gravity/buoyancy wrench;
- $\tau_D$ is the physical hydrodynamic damping wrench;
- $\tau_{env}$ is any additional external generalized disturbance;
- $\nu_r$ is water-relative generalized velocity.

Because:

$$
\nu_r
=
\nu-\nu_c
$$

its derivative is:

$$
\dot{\nu}_r
=
\dot{\nu}
-
\dot{\nu}_c
$$

Substituting gives:

$$
M_{RB}\dot{\nu}
+
M_A
\left(
\dot{\nu}-\dot{\nu}_c
\right)
+
C_{RB}(\nu)\nu
+
C_A(\nu_r)\nu_r
=
\tau_{act}
+
\tau_H
+
\tau_D
+
\tau_{env}
$$

Collecting the vehicle acceleration terms:

$$
\left(
M_{RB}+M_A
\right)
\dot{\nu}
=
\tau_{act}
+
\tau_H
+
\tau_D
+
\tau_{env}
-
C_{RB}(\nu)\nu
-
C_A(\nu_r)\nu_r
+
M_A\dot{\nu}_c
$$

Using:

$$
M
=
M_{RB}+M_A
$$

the implemented acceleration equation becomes:

$$
\boxed{
\dot{\nu}
=
M^{-1}
\left[
\tau_{act}
+
\tau_H
+
\tau_D
+
\tau_{env}
-
C_{RB}(\nu)\nu
-
C_A(\nu_r)\nu_r
+
M_A\dot{\nu}_c
\right]
}
$$

This is the current-aware formulation selected for Notebook 07.

---

# Why the $\dot{\nu}_c$ Term Matters

Notebook 06 supplies a current that is constant in the **world frame**.

Notebook 07 converts it into the rotating body frame:

$$
\mathbf{v}_{c,B}
=
(R_B^W)^T
\mathbf{v}_{c,W}
$$

Even when:

$$
\dot{\mathbf{v}}_{c,W}=0
$$

a rotating vehicle generally observes a changing body-frame current vector.

For body angular velocity:

$$
\boldsymbol{\omega}
=
\begin{bmatrix}
p\\
q\\
r
\end{bmatrix}
$$

the body-frame derivative of a spatially uniform, world-fixed current is:

$$
\boxed{
\dot{\mathbf{v}}_{c,B}
=
-
\boldsymbol{\omega}
\times
\mathbf{v}_{c,B}
}
$$

or:

$$
\dot{\mathbf{v}}_{c,B}
=
-S(\boldsymbol{\omega})
\mathbf{v}_{c,B}
$$

The initial HydroLab-3D current model contains no ambient angular fluid velocity, so:

$$
\boxed{
\dot{\nu}_c
=
\begin{bmatrix}
-\boldsymbol{\omega}\times\mathbf{v}_{c,B}\\
0\\
0\\
0
\end{bmatrix}
}
$$

This term vanishes automatically when:

- current is zero;
- body angular velocity is zero;
- or the current happens to be parallel to the angular-velocity axis.

---

## Selective-Fidelity Boundary

This treatment assumes a **spatially uniform and time-invariant world-frame current** during this notebook.

Therefore it captures the body-frame time variation caused by vehicle rotation but does not yet include:

$$
\frac{\partial \mathbf{v}_c}{\partial t}
$$

or spatial current gradients such as:

$$
\nabla\mathbf{v}_c
$$

Those effects belong to future nonuniform or time-varying current models.

For Notebook 07:

$$
\boxed{
\text{uniform inertial current + rotating body-frame transformation}
}
$$

is the selected current fidelity.

---

# Physical Wrench Sign Convention

The implementation uses physical wrenches directly.

The actuator term:

$$
\tau_{act}
$$

contains the actual actuator force and moment.

The hydrostatic function returns:

$$
\tau_H
$$

with gravity and buoyancy already carrying their physical signs.

The damping function returns:

$$
\tau_D
$$

which already opposes relative motion.

Therefore these quantities appear as:

$$
+\tau_{act}
+
\tau_H
+
\tau_D
+
\tau_{env}
$$

The Coriolis terms are inertial transport terms and therefore appear as:

$$
-
C_{RB}(\nu)\nu
-
C_A(\nu_r)\nu_r
$$

when solving explicitly for acceleration.

This avoids hidden sign inversions between individually validated functions and the assembled dynamics.

---

# Complete State Derivative

The final instantaneous state derivative produced in this step is:

$$
\boxed{
\dot{x}_{state}
=
\begin{bmatrix}
\dot{\eta}\\
\dot{\nu}
\end{bmatrix}
}
$$

with twelve components.

Conceptually:

```text
pose eta
body velocity nu
world current
vehicle parameters
actuator wrench
environmental wrench
        │
        ├── J(eta)
        │       ↓
        │    eta_dot
        │
        ├── transform current W -> B
        │       ↓
        │     nu_c
        │       ↓
        │     nu_r
        │
        ├── M_RB + M_A
        ├── C_RB(nu)
        ├── C_A(nu_r)
        ├── damping(nu_r)
        ├── hydrostatics(eta)
        └── current-frame acceleration
                ↓
              nu_dot
```

---

# Reduction to Notebook 02

Disable:

- added mass;
- damping;
- current;
- hydrostatic imbalance;
- Coriolis coupling through a zero-velocity axis-aligned initial case.

For pure body-$x$ force:

$$
\tau_{act}
=
\begin{bmatrix}
F_x\\
0\\
0\\
0\\
0\\
0
\end{bmatrix}
$$

we must recover:

$$
\boxed{
\dot{u}
=
\frac{F_x}{m}
}
$$

For:

$$
F_x=12\text{ N}
$$

and:

$$
m=20\text{ kg}
$$

the expected acceleration is:

$$
\boxed{
\dot{u}=0.6\text{ m/s}^2
}
$$

This directly validates the Notebook 02 lineage.

---

# Pure Rotational Reduction

With zero body velocity, zero current, neutral hydrostatics, and no damping, apply only:

$$
\tau_{act}
=
\begin{bmatrix}
0\\
0\\
0\\
0\\
M_y\\
0
\end{bmatrix}
$$

With added rotational inertia disabled:

$$
\dot{q}
=
\frac{M_y}{I_y}
$$

For:

$$
M_y=3.2\text{ N m}
$$

and:

$$
I_y=1.6\text{ kg m}^2
$$

we expect:

$$
\boxed{
\dot{q}=2.0\text{ rad/s}^2
}
$$

This proves that pure generalized moments produce the expected rotational acceleration.

---

# Neutral Zero-Input Equilibrium

For:

- zero body velocity;
- zero current;
- exact neutral buoyancy;
- coincident CG/CB;
- zero actuator wrench;
- zero environmental wrench;

every dynamic contribution should vanish.

Therefore:

$$
\boxed{
\dot{\nu}=0
}
$$

and because:

$$
\nu=0
$$

we also require:

$$
\boxed{
\dot{\eta}=0
}
$$

Thus the complete 12-component derivative must be exactly zero within numerical tolerance.

---

# Vehicle Moving With Uniform Water

For a fixed-orientation vehicle whose translational velocity equals the surrounding current:

$$
\mathbf{v}_B
=
\mathbf{v}_{c,B}
$$

with:

$$
\boldsymbol{\omega}=0
$$

we have:

$$
\nu_r=0
$$

Therefore:

$$
\tau_D=0
$$

$$
C_A(\nu_r)\nu_r=0
$$

and:

$$
\dot{\nu}_c=0
$$

For neutral hydrostatics and zero actuation, the vehicle should therefore have:

$$
\boxed{
\dot{\nu}=0
}
$$

It continues moving with the water rather than being artificially slowed relative to the world.

This is the 6-DOF preservation of Notebook 06's central physical result.

---

# Stationary Vehicle in Moving Water

A stationary vehicle in nonzero current has:

$$
\nu=0
$$

but:

$$
\nu_r
=
-\nu_c
$$

Hydrodynamic damping therefore becomes nonzero.

The resulting translational acceleration should point generally toward the local water motion.

This confirms that the assembled dynamics use water-relative rather than world-relative hydrodynamics.

---

# Additional Environmental Wrench

The function will accept:

$$
\tau_{env}
$$

as an optional six-component generalized disturbance.

For Notebook 07 it defaults to:

$$
\tau_{env}=0
$$

This keeps explicit room for later disturbances without artificially routing every environmental phenomenon through the ocean-current model.

No specific disturbance model is introduced yet.

---

# Prototype Functions

This step introduces:

```text
current_generalized_acceleration()
six_dof_state_derivative()
```

as notebook-local functions.

The derivative function will reuse all previously validated Notebook 07 prototypes rather than reimplementing their equations internally.

This is important: assembly should connect validated components, not silently fork them.

---

# Validation Criteria

This step passes only if:

1. $\dot{\eta}$ has shape $(6,)$;

2. $\dot{\nu}$ has shape $(6,)$;

3. the complete state derivative has shape $(12,)$;

4. all derivative values remain finite;

5. neutral zero-input equilibrium produces a zero 12-state derivative;

6. the no-added-mass pure-force case reproduces:

$$
\dot{u}=\frac{F_x}{m}
$$

exactly within numerical tolerance;

7. the pure-moment case reproduces:

$$
\dot{q}=\frac{M_y}{I_y}
$$

8. moving exactly with uniform water at fixed orientation produces zero dynamic acceleration when no other wrench acts;

9. a stationary vehicle in moving water experiences acceleration caused by the previously validated relative-velocity damping;

10. zero angular velocity gives:

$$
\dot{\nu}_c=0
$$

for a constant world current;

11. a rotating body in constant world current produces the analytically expected:

$$
\dot{\mathbf{v}}_{c,B}
=
-\boldsymbol{\omega}\times\mathbf{v}_{c,B}
$$

12. the assembled net generalized wrench equals the explicit sum of its individually validated terms;

13. malformed actuator or environmental wrenches are rejected cleanly.

If these tests pass, HydroLab-3D will possess its first complete validated instantaneous 6-DOF UUV dynamics model.

The next step will integrate this derivative deterministically in time.

In [10]:
import numpy as np


def current_generalized_acceleration(
    pose,
    body_velocity,
    current_velocity_world,
):
    """
    Compute the generalized body-frame time derivative of a
    spatially uniform, time-invariant world-frame current.

    For:

        v_c,B = R_B^W.T @ v_c,W

    and body angular velocity omega:

        d(v_c,B)/dt = -omega x v_c,B

    The current model contains no ambient angular velocity.
    """
    pose = np.asarray(
        pose,
        dtype=float,
    )

    body_velocity = np.asarray(
        body_velocity,
        dtype=float,
    )

    current_velocity_world = np.asarray(
        current_velocity_world,
        dtype=float,
    )

    if pose.shape != (6,):
        raise ValueError(
            "pose must have shape (6,)"
        )

    if body_velocity.shape != (6,):
        raise ValueError(
            "body_velocity must have shape (6,)"
        )

    if current_velocity_world.shape != (3,):
        raise ValueError(
            "current_velocity_world must have shape (3,)"
        )

    if not np.all(np.isfinite(pose)):
        raise ValueError(
            "pose must contain only finite values"
        )

    if not np.all(np.isfinite(body_velocity)):
        raise ValueError(
            "body_velocity must contain only finite values"
        )

    if not np.all(np.isfinite(current_velocity_world)):
        raise ValueError(
            "current_velocity_world must contain only finite values"
        )

    v_c_body = current_velocity_body(
        pose,
        current_velocity_world,
    )

    omega_body = body_velocity[3:]

    v_c_body_dot = -np.cross(
        omega_body,
        v_c_body,
    )

    nu_c_dot = np.zeros(6, dtype=float)
    nu_c_dot[:3] = v_c_body_dot

    return nu_c_dot


def six_dof_state_derivative(
    pose,
    body_velocity,
    *,
    mass,
    inertia,
    added_inertia,
    linear_damping,
    quadratic_damping,
    fluid_density,
    displaced_volume,
    center_of_gravity,
    center_of_buoyancy,
    current_velocity_world,
    actuator_wrench,
    environmental_wrench=None,
    gravity=9.81,
):
    """
    Compute the complete instantaneous HydroLab-3D 6-DOF
    state derivative.

    Returns
    -------
    eta_dot : ndarray, shape (6,)
        Pose derivative.

    nu_dot : ndarray, shape (6,)
        Body generalized acceleration.

    diagnostics : dict
        Individually resolved dynamic terms for validation.
    """
    pose = np.asarray(
        pose,
        dtype=float,
    )

    body_velocity = np.asarray(
        body_velocity,
        dtype=float,
    )

    actuator_wrench = np.asarray(
        actuator_wrench,
        dtype=float,
    )

    current_velocity_world = np.asarray(
        current_velocity_world,
        dtype=float,
    )

    if environmental_wrench is None:
        environmental_wrench = np.zeros(6)
    else:
        environmental_wrench = np.asarray(
            environmental_wrench,
            dtype=float,
        )

    if pose.shape != (6,):
        raise ValueError(
            "pose must have shape (6,)"
        )

    if body_velocity.shape != (6,):
        raise ValueError(
            "body_velocity must have shape (6,)"
        )

    if actuator_wrench.shape != (6,):
        raise ValueError(
            "actuator_wrench must have shape (6,)"
        )

    if environmental_wrench.shape != (6,):
        raise ValueError(
            "environmental_wrench must have shape (6,)"
        )

    if current_velocity_world.shape != (3,):
        raise ValueError(
            "current_velocity_world must have shape (3,)"
        )

    for name, value in [
        ("pose", pose),
        ("body_velocity", body_velocity),
        ("actuator_wrench", actuator_wrench),
        ("environmental_wrench", environmental_wrench),
        ("current_velocity_world", current_velocity_world),
    ]:
        if not np.all(np.isfinite(value)):
            raise ValueError(
                f"{name} must contain only finite values"
            )

    # -------------------------------------------------------------
    # Kinematics
    # -------------------------------------------------------------

    eta_dot = pose_rate_6dof(
        pose,
        body_velocity,
    )

    # -------------------------------------------------------------
    # Inertia
    # -------------------------------------------------------------

    M_rb_local = rigid_body_mass_matrix(
        mass,
        inertia,
    )

    M_a_local = added_mass_matrix(
        added_inertia,
    )

    M_total_local = (
        M_rb_local + M_a_local
    )

    # -------------------------------------------------------------
    # Current-relative motion
    # -------------------------------------------------------------

    nu_r_local = relative_velocity_6dof(
        pose,
        body_velocity,
        current_velocity_world,
    )

    nu_c_dot_local = (
        current_generalized_acceleration(
            pose,
            body_velocity,
            current_velocity_world,
        )
    )

    # -------------------------------------------------------------
    # Coriolis / centripetal terms
    # -------------------------------------------------------------

    C_rb_local = rigid_body_coriolis_matrix(
        mass,
        inertia,
        body_velocity,
    )

    C_a_local = added_mass_coriolis_matrix(
        added_inertia,
        nu_r_local,
    )

    coriolis_rb_wrench = (
        C_rb_local @ body_velocity
    )

    coriolis_a_wrench = (
        C_a_local @ nu_r_local
    )

    # -------------------------------------------------------------
    # Damping
    # -------------------------------------------------------------

    damping_wrench = hydrodynamic_damping_wrench(
        nu_r_local,
        linear_damping,
        quadratic_damping,
    )

    # -------------------------------------------------------------
    # Hydrostatics
    # -------------------------------------------------------------

    hydrostatic_wrench_local = hydrostatic_wrench(
        pose,
        mass,
        fluid_density,
        displaced_volume,
        center_of_gravity,
        center_of_buoyancy,
        gravity,
    )

    # -------------------------------------------------------------
    # Current-frame added-mass correction
    # -------------------------------------------------------------

    added_mass_current_correction = (
        M_a_local @ nu_c_dot_local
    )

    # -------------------------------------------------------------
    # Net physical / inertial generalized wrench
    # -------------------------------------------------------------

    net_wrench = (
        actuator_wrench
        + environmental_wrench
        + hydrostatic_wrench_local
        + damping_wrench
        - coriolis_rb_wrench
        - coriolis_a_wrench
        + added_mass_current_correction
    )

    # Solve rather than explicitly forming M^{-1}.
    nu_dot = np.linalg.solve(
        M_total_local,
        net_wrench,
    )

    if not np.all(np.isfinite(eta_dot)):
        raise FloatingPointError(
            "eta_dot contains non-finite values"
        )

    if not np.all(np.isfinite(nu_dot)):
        raise FloatingPointError(
            "nu_dot contains non-finite values"
        )

    diagnostics = {
        "M_RB": M_rb_local,
        "M_A": M_a_local,
        "M": M_total_local,
        "nu_r": nu_r_local,
        "nu_c_dot": nu_c_dot_local,
        "C_RB": C_rb_local,
        "C_A": C_a_local,
        "coriolis_rb_wrench": coriolis_rb_wrench,
        "coriolis_a_wrench": coriolis_a_wrench,
        "damping_wrench": damping_wrench,
        "hydrostatic_wrench": hydrostatic_wrench_local,
        "actuator_wrench": actuator_wrench.copy(),
        "environmental_wrench": environmental_wrench.copy(),
        "added_mass_current_correction": (
            added_mass_current_correction
        ),
        "net_wrench": net_wrench,
    }

    return eta_dot, nu_dot, diagnostics


# =====================================================================
# Canonical common parameters
# =====================================================================

pose_zero_d = np.zeros(6)
nu_zero_d = np.zeros(6)

mass_d = 20.0

inertia_d = np.array(
    [
        1.2,
        1.6,
        2.0,
    ]
)

added_d = np.array(
    [
        5.0,
        8.0,
        10.0,
        0.20,
        0.30,
        0.40,
    ]
)

linear_d = np.array(
    [
        2.0,
        3.0,
        4.0,
        0.10,
        0.15,
        0.20,
    ]
)

quadratic_d = np.array(
    [
        5.0,
        6.0,
        7.0,
        0.30,
        0.40,
        0.50,
    ]
)

rho_d = 1000.0
volume_neutral_d = mass_d / rho_d

r_g_d = np.zeros(3)
r_b_d = np.zeros(3)

current_zero_d = np.zeros(3)
wrench_zero_d = np.zeros(6)


# =====================================================================
# Test 1 - Neutral zero-input equilibrium
# =====================================================================

eta_dot_eq, nu_dot_eq, diag_eq = (
    six_dof_state_derivative(
        pose_zero_d,
        nu_zero_d,
        mass=mass_d,
        inertia=inertia_d,
        added_inertia=added_d,
        linear_damping=linear_d,
        quadratic_damping=quadratic_d,
        fluid_density=rho_d,
        displaced_volume=volume_neutral_d,
        center_of_gravity=r_g_d,
        center_of_buoyancy=r_b_d,
        current_velocity_world=current_zero_d,
        actuator_wrench=wrench_zero_d,
    )
)

np.testing.assert_allclose(
    eta_dot_eq,
    np.zeros(6),
    atol=1e-14,
)

np.testing.assert_allclose(
    nu_dot_eq,
    np.zeros(6),
    atol=1e-14,
)

np.testing.assert_allclose(
    diag_eq["net_wrench"],
    np.zeros(6),
    atol=1e-14,
)


# =====================================================================
# Test 2 - Reduction to Notebook 02: F / m
# =====================================================================

pure_force = np.array(
    [
        12.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
    ]
)

eta_dot_force, nu_dot_force, diag_force = (
    six_dof_state_derivative(
        pose_zero_d,
        nu_zero_d,
        mass=mass_d,
        inertia=inertia_d,
        added_inertia=np.zeros(6),
        linear_damping=np.zeros(6),
        quadratic_damping=np.zeros(6),
        fluid_density=rho_d,
        displaced_volume=volume_neutral_d,
        center_of_gravity=r_g_d,
        center_of_buoyancy=r_b_d,
        current_velocity_world=current_zero_d,
        actuator_wrench=pure_force,
    )
)

expected_force_acceleration = np.array(
    [
        12.0 / mass_d,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
    ]
)

np.testing.assert_allclose(
    nu_dot_force,
    expected_force_acceleration,
    atol=1e-14,
)


# =====================================================================
# Test 3 - Pure moment / inertia
# =====================================================================

pure_pitch_moment = np.array(
    [
        0.0,
        0.0,
        0.0,
        0.0,
        3.2,
        0.0,
    ]
)

_, nu_dot_moment, diag_moment = (
    six_dof_state_derivative(
        pose_zero_d,
        nu_zero_d,
        mass=mass_d,
        inertia=inertia_d,
        added_inertia=np.zeros(6),
        linear_damping=np.zeros(6),
        quadratic_damping=np.zeros(6),
        fluid_density=rho_d,
        displaced_volume=volume_neutral_d,
        center_of_gravity=r_g_d,
        center_of_buoyancy=r_b_d,
        current_velocity_world=current_zero_d,
        actuator_wrench=pure_pitch_moment,
    )
)

expected_moment_acceleration = np.array(
    [
        0.0,
        0.0,
        0.0,
        0.0,
        3.2 / inertia_d[1],
        0.0,
    ]
)

np.testing.assert_allclose(
    nu_dot_moment,
    expected_moment_acceleration,
    atol=1e-14,
)


# =====================================================================
# Test 4 - Vehicle moving exactly with uniform water
# =====================================================================

current_following = np.array(
    [
        0.50,
        -0.25,
        0.10,
    ]
)

nu_following = np.array(
    [
        0.50,
        -0.25,
        0.10,
        0.0,
        0.0,
        0.0,
    ]
)

_, nu_dot_following, diag_following = (
    six_dof_state_derivative(
        pose_zero_d,
        nu_following,
        mass=mass_d,
        inertia=inertia_d,
        added_inertia=added_d,
        linear_damping=linear_d,
        quadratic_damping=quadratic_d,
        fluid_density=rho_d,
        displaced_volume=volume_neutral_d,
        center_of_gravity=r_g_d,
        center_of_buoyancy=r_b_d,
        current_velocity_world=current_following,
        actuator_wrench=wrench_zero_d,
    )
)

np.testing.assert_allclose(
    diag_following["nu_r"],
    np.zeros(6),
    atol=1e-14,
)

np.testing.assert_allclose(
    diag_following["damping_wrench"],
    np.zeros(6),
    atol=1e-14,
)

np.testing.assert_allclose(
    nu_dot_following,
    np.zeros(6),
    atol=1e-14,
)


# =====================================================================
# Test 5 - Stationary vehicle in moving water
# =====================================================================

_, nu_dot_stationary_current, diag_stationary_current = (
    six_dof_state_derivative(
        pose_zero_d,
        nu_zero_d,
        mass=mass_d,
        inertia=inertia_d,
        added_inertia=added_d,
        linear_damping=linear_d,
        quadratic_damping=quadratic_d,
        fluid_density=rho_d,
        displaced_volume=volume_neutral_d,
        center_of_gravity=r_g_d,
        center_of_buoyancy=r_b_d,
        current_velocity_world=current_following,
        actuator_wrench=wrench_zero_d,
    )
)

# At zero angular velocity the body-frame current is momentarily fixed.
np.testing.assert_allclose(
    diag_stationary_current["nu_c_dot"],
    np.zeros(6),
    atol=1e-14,
)

# The resulting acceleration should have positive projection
# onto the current direction.
assert (
    nu_dot_stationary_current[:3]
    @ current_following
) > 0.0


# =====================================================================
# Test 6 - Rotating-body current derivative
# =====================================================================

pose_rot_current = np.array(
    [
        0.0,
        0.0,
        0.0,
        np.deg2rad(15.0),
        np.deg2rad(-10.0),
        np.deg2rad(25.0),
    ]
)

nu_rot_current = np.array(
    [
        0.2,
        -0.1,
        0.05,
        0.30,
        -0.20,
        0.10,
    ]
)

v_c_body_rot = current_velocity_body(
    pose_rot_current,
    current_following,
)

nu_c_dot_rot = current_generalized_acceleration(
    pose_rot_current,
    nu_rot_current,
    current_following,
)

expected_v_c_dot_rot = -np.cross(
    nu_rot_current[3:],
    v_c_body_rot,
)

np.testing.assert_allclose(
    nu_c_dot_rot[:3],
    expected_v_c_dot_rot,
    atol=1e-14,
)

np.testing.assert_array_equal(
    nu_c_dot_rot[3:],
    np.zeros(3),
)


# =====================================================================
# Test 7 - Nontrivial assembled derivative and wrench reconstruction
# =====================================================================

pose_general = np.array(
    [
        1.0,
        -2.0,
        3.0,
        np.deg2rad(12.0),
        np.deg2rad(-8.0),
        np.deg2rad(20.0),
    ]
)

nu_general = np.array(
    [
        0.8,
        -0.3,
        0.2,
        0.15,
        -0.08,
        0.12,
    ]
)

actuator_general = np.array(
    [
        18.0,
        2.0,
        -1.0,
        0.3,
        1.5,
        -0.4,
    ]
)

environment_general = np.array(
    [
        0.5,
        -0.2,
        0.1,
        0.02,
        0.00,
        -0.01,
    ]
)

eta_dot_general, nu_dot_general, diag_general = (
    six_dof_state_derivative(
        pose_general,
        nu_general,
        mass=mass_d,
        inertia=inertia_d,
        added_inertia=added_d,
        linear_damping=linear_d,
        quadratic_damping=quadratic_d,
        fluid_density=rho_d,
        displaced_volume=volume_neutral_d,
        center_of_gravity=r_g_d,
        center_of_buoyancy=np.array(
            [0.0, 0.0, 0.05]
        ),
        current_velocity_world=current_following,
        actuator_wrench=actuator_general,
        environmental_wrench=environment_general,
    )
)

explicit_net_wrench = (
    diag_general["actuator_wrench"]
    + diag_general["environmental_wrench"]
    + diag_general["hydrostatic_wrench"]
    + diag_general["damping_wrench"]
    - diag_general["coriolis_rb_wrench"]
    - diag_general["coriolis_a_wrench"]
    + diag_general["added_mass_current_correction"]
)

np.testing.assert_allclose(
    diag_general["net_wrench"],
    explicit_net_wrench,
    atol=1e-13,
)

np.testing.assert_allclose(
    diag_general["M"] @ nu_dot_general,
    diag_general["net_wrench"],
    atol=1e-12,
)

assert eta_dot_general.shape == (6,)
assert nu_dot_general.shape == (6,)

state_derivative_general = np.concatenate(
    [
        eta_dot_general,
        nu_dot_general,
    ]
)

assert state_derivative_general.shape == (12,)
assert np.all(np.isfinite(state_derivative_general))


# =====================================================================
# Invalid-wrench checks
# =====================================================================

invalid_derivative_calls = [
    lambda: six_dof_state_derivative(
        pose_zero_d,
        nu_zero_d,
        mass=mass_d,
        inertia=inertia_d,
        added_inertia=added_d,
        linear_damping=linear_d,
        quadratic_damping=quadratic_d,
        fluid_density=rho_d,
        displaced_volume=volume_neutral_d,
        center_of_gravity=r_g_d,
        center_of_buoyancy=r_b_d,
        current_velocity_world=current_zero_d,
        actuator_wrench=[1.0, 2.0],
    ),
    lambda: six_dof_state_derivative(
        pose_zero_d,
        nu_zero_d,
        mass=mass_d,
        inertia=inertia_d,
        added_inertia=added_d,
        linear_damping=linear_d,
        quadratic_damping=quadratic_d,
        fluid_density=rho_d,
        displaced_volume=volume_neutral_d,
        center_of_gravity=r_g_d,
        center_of_buoyancy=r_b_d,
        current_velocity_world=current_zero_d,
        actuator_wrench=wrench_zero_d,
        environmental_wrench=[
            0.0,
            0.0,
            0.0,
            np.nan,
            0.0,
            0.0,
        ],
    ),
]

invalid_derivative_rejected = 0

for call in invalid_derivative_calls:
    try:
        call()
    except ValueError:
        invalid_derivative_rejected += 1

assert invalid_derivative_rejected == len(
    invalid_derivative_calls
)


# =====================================================================
# Report
# =====================================================================

print("Neutral zero-input eta_dot:")
print(eta_dot_eq)

print("Neutral zero-input nu_dot:")
print(nu_dot_eq)

print("\nPure 12 N surge-force acceleration:")
print(nu_dot_force)

print("Expected Notebook 02 reduction:")
print(expected_force_acceleration)

print("\nPure 3.2 N m pitch-moment acceleration:")
print(nu_dot_moment)

print("Expected pure-moment acceleration:")
print(expected_moment_acceleration)

print("\nMoving-with-water relative velocity:")
print(diag_following["nu_r"])

print("Moving-with-water dynamic acceleration:")
print(nu_dot_following)

print("\nStationary-in-current relative velocity:")
print(diag_stationary_current["nu_r"])

print("Stationary-in-current damping wrench:")
print(diag_stationary_current["damping_wrench"])

print("Stationary-in-current acceleration:")
print(nu_dot_stationary_current)

print("\nRotating-body body-frame current:")
print(v_c_body_rot)

print("Rotating-body nu_c_dot:")
print(nu_c_dot_rot)

print("Expected translational nu_c_dot:")
print(expected_v_c_dot_rot)

print("\nNontrivial eta_dot:")
print(eta_dot_general)

print("\nNontrivial nu_dot:")
print(nu_dot_general)

print("\nNontrivial net generalized wrench:")
print(diag_general["net_wrench"])

print("\nM @ nu_dot reconstruction:")
print(diag_general["M"] @ nu_dot_general)

print(
    "\nInvalid derivative-input cases rejected:",
    f"{invalid_derivative_rejected}/"
    f"{len(invalid_derivative_calls)}",
)

print(
    "\nPASS: neutral zero-input state is an exact "
    "6-DOF equilibrium."
)

print(
    "PASS: pure translational force reduces exactly "
    "to the Notebook 02 F/m model."
)

print(
    "PASS: pure body moment produces the expected "
    "angular acceleration."
)

print(
    "PASS: motion exactly with uniform water produces "
    "zero hydrodynamic acceleration."
)

print(
    "PASS: a stationary vehicle in moving water "
    "accelerates with the current-induced damping force."
)

print(
    "PASS: rotating-body current derivative matches "
    "-omega x v_c,B."
)

print(
    "PASS: complete generalized wrench decomposition "
    "reconstructs M @ nu_dot."
)

print(
    "PASS: complete 12-component state derivative "
    "is finite and structurally valid."
)

Neutral zero-input eta_dot:
[0. 0. 0. 0. 0. 0.]
Neutral zero-input nu_dot:
[0. 0. 0. 0. 0. 0.]

Pure 12 N surge-force acceleration:
[0.6 0.  0.  0.  0.  0. ]
Expected Notebook 02 reduction:
[0.6 0.  0.  0.  0.  0. ]

Pure 3.2 N m pitch-moment acceleration:
[0. 0. 0. 0. 2. 0.]
Expected pure-moment acceleration:
[0. 0. 0. 0. 2. 0.]

Moving-with-water relative velocity:
[0. 0. 0. 0. 0. 0.]
Moving-with-water dynamic acceleration:
[0. 0. 0. 0. 0. 0.]

Stationary-in-current relative velocity:
[-0.5   0.25 -0.1   0.    0.    0.  ]
Stationary-in-current damping wrench:
[ 2.25  -1.125  0.47  -0.    -0.    -0.   ]
Stationary-in-current acceleration:
[ 0.09       -0.04017857  0.01566667  0.03571429  0.13157895  0.15625   ]

Rotating-body body-frame current:
[ 0.35958485 -0.41309459  0.15017188]
Rotating-body nu_c_dot:
[-0.01127508  0.00909308  0.05201141  0.          0.          0.        ]
Expected translational nu_c_dot:
[-0.01127508  0.00909308  0.05201141]

Nontrivial eta_dot:
[ 0.84159672 -0

## Step 11 - Deterministic Explicit-Euler 6-DOF State Integration

Step 10 produced a validated instantaneous state derivative:

$$
\dot{\eta}
$$

and:

$$
\dot{\nu}
$$

for the complete current-aware 6-DOF UUV model.

We now introduce deterministic timestep propagation.

This step does not yet run a long canonical simulation.

Its purpose is to establish and validate the **single-step numerical update rule** that all later Notebook 07 trajectories will use.

---

## Inherited Numerical Philosophy

Notebook 00 validated explicit position propagation:

$$
\mathbf{p}_{k+1}
=
\mathbf{p}_k
+
\mathbf{v}_k\Delta t
$$

Notebook 02 validated explicit velocity propagation:

$$
\mathbf{v}_{k+1}
=
\mathbf{v}_k
+
\mathbf{a}_k\Delta t
$$

and established the update ordering:

```text
evaluate beginning-of-step state
        ↓
propagate position
        ↓
propagate velocity
```

Notebook 07 preserves that same numerical philosophy.

---

## 6-DOF Explicit-Euler Update

At timestep:

$$
k
$$

the state is:

$$
\eta_k
$$

and:

$$
\nu_k
$$

The instantaneous derivative is evaluated entirely from this beginning-of-step state:

$$
\dot{\eta}_k
=
J(\eta_k)\nu_k
$$

and:

$$
\dot{\nu}_k
=
f
\left(
\eta_k,
\nu_k,
\tau_k,
\mathbf{v}_{c,W,k}
\right)
$$

where:

$$
f(\cdot)
$$

is the complete dynamic model validated in Step 10.

The state is then advanced using:

$$
\boxed{
\eta_{k+1}
=
\eta_k
+
\dot{\eta}_k\Delta t
}
$$

and:

$$
\boxed{
\nu_{k+1}
=
\nu_k
+
\dot{\nu}_k\Delta t
}
$$

Both updates use derivatives computed from the same beginning-of-step state.

The newly calculated:

$$
\nu_{k+1}
$$

is **not** used retroactively when computing:

$$
\eta_{k+1}
$$

during the same timestep.

This preserves the ordering established in the earlier HydroLab-3D notebooks.

---

## Complete Discrete State

Define the twelve-component state:

$$
x_k
=
\begin{bmatrix}
\eta_k\\
\nu_k
\end{bmatrix}
$$

Then explicit Euler may be written compactly as:

$$
\boxed{
x_{k+1}
=
x_k
+
\dot{x}_k\Delta t
}
$$

where:

$$
\dot{x}_k
=
\begin{bmatrix}
\dot{\eta}_k\\
\dot{\nu}_k
\end{bmatrix}
$$

---

## Why We Keep Explicit Euler

Notebook 07 could use a higher-order integrator, but that would introduce a numerical-method change at the same time that the physics model becomes substantially more complex.

That would make debugging harder because an unexpected trajectory could originate from either:

```text
new physics
```

or:

```text
new integration
```

HydroLab-3D instead changes one layer at a time.

Explicit Euler remains appropriate here because:

- it matches the established simulator lineage;
- it is deterministic;
- it is transparent;
- it is inexpensive;
- timestep sensitivity can be measured later;
- the eventual simulator kernel benefits from a simple fixed-step update.

A later notebook or future simulator version may introduce a higher-order integration option without invalidating this baseline.

---

## Euler-Angle Domain Check

The pose update includes:

$$
\phi,\theta,\psi
$$

and the kinematic mapping becomes singular near:

$$
\theta
=
\pm\frac{\pi}{2}
$$

The existing `euler_rate_matrix()` guard prevents evaluation directly at that singularity.

After an explicit timestep, however, the updated pitch must also remain in the supported Euler-angle domain.

For Notebook 07 the propagator will therefore verify the new pose before accepting the state.

This does not remove the mathematical singularity.

It prevents the integrator from silently returning a state that the next derivative evaluation cannot safely use.

---

## One-Step Reduction to Notebook 00

For identity orientation:

$$
R_B^W=I
$$

and constant translational body velocity:

$$
\nu
=
\begin{bmatrix}
u\\
v\\
w\\
0\\
0\\
0
\end{bmatrix}
$$

with zero dynamic acceleration:

$$
\dot{\nu}=0
$$

we obtain:

$$
\eta_{k+1}
=
\eta_k
+
\begin{bmatrix}
u\\
v\\
w\\
0\\
0\\
0
\end{bmatrix}
\Delta t
$$

Thus the translational part reduces to the Notebook 00 update:

$$
\boxed{
\mathbf{p}_{k+1}
=
\mathbf{p}_k
+
\mathbf{v}_k\Delta t
}
$$

---

## One-Step Reduction to Notebook 02

Consider:

$$
m=20\text{ kg}
$$

with:

$$
F_x=12\text{ N}
$$

and all additional effects disabled.

Step 10 already established:

$$
\dot{u}
=
0.6\text{ m/s}^2
$$

For:

$$
\Delta t=0.1\text{ s}
$$

and:

$$
u_k=0
$$

the velocity update must therefore give:

$$
u_{k+1}
=
0
+
0.6(0.1)
$$

so:

$$
\boxed{
u_{k+1}
=
0.06\text{ m/s}
}
$$

Because explicit Euler uses the beginning-of-step velocity for pose propagation and:

$$
u_k=0
$$

the position during this first timestep remains:

$$
\boxed{
x_{k+1}=x_k
}
$$

This is exactly the numerical ordering used previously.

Only the following timestep begins accumulating position from the newly acquired velocity.

---

## Zero-Input Equilibrium

If:

$$
\dot{\eta}_k=0
$$

and:

$$
\dot{\nu}_k=0
$$

then:

$$
\eta_{k+1}=\eta_k
$$

and:

$$
\nu_{k+1}=\nu_k
$$

exactly.

Therefore the neutral, zero-input equilibrium validated in Step 10 must remain unchanged after propagation.

---

## Motion With the Water

For a fixed-orientation neutrally buoyant vehicle moving exactly with uniform current:

$$
\nu_r=0
$$

and Step 10 established:

$$
\dot{\nu}=0
$$

The body velocity must therefore remain constant after integration.

The pose should continue translating according to:

$$
\dot{\mathbf{p}}_W
=
R_B^W\mathbf{v}_B
$$

This distinction is important:

$$
\boxed{
\dot{\nu}=0
\not\Rightarrow
\dot{\eta}=0
}
$$

Zero acceleration means constant velocity, not stationary position.

---

## Determinism

Given identical:

- initial pose;
- initial body velocity;
- physical parameters;
- current;
- actuator wrench;
- environmental wrench;
- timestep;

the state update must return exactly the same numerical result on repeated calls.

No randomness belongs in the vehicle dynamics layer.

Sensor noise and stochastic effects belong to later notebooks.

---

## Prototype Function

This step introduces:

```text
integrate_six_dof_euler()
```

as a notebook-local propagator.

It calls:

```text
six_dof_state_derivative()
```

rather than reproducing any dynamic equation itself.

This maintains a single source of truth for the notebook-local dynamics.

---

## Validation Criteria

This step passes only if:

1. the updated pose has shape $(6,)$;

2. the updated body velocity has shape $(6,)$;

3. all state values remain finite;

4. timestep must be finite and strictly positive;

5. neutral zero-input equilibrium remains exactly unchanged;

6. the pure-force one-step case gives:

$$
u_{k+1}=0.06\text{ m/s}
$$

for the canonical:

$$
F_x=12\text{ N},
\quad
m=20\text{ kg},
\quad
\Delta t=0.1\text{ s}
$$

7. the same first force-driven step leaves position unchanged because the beginning-of-step velocity is zero;

8. a second identical step begins translating using the first-step velocity;

9. constant-velocity motion with zero acceleration reduces to the Notebook 00 explicit position update;

10. a vehicle moving exactly with uniform water retains its body velocity while its pose advances;

11. repeated identical updates produce identical output;

12. malformed or nonpositive timesteps are rejected.

If these checks pass, Notebook 07 will possess a deterministic single-step propagation rule ready for canonical multi-step experiments.

In [11]:
import numpy as np


def integrate_six_dof_euler(
    pose,
    body_velocity,
    dt,
    **dynamics_kwargs,
):
    """
    Advance the HydroLab-3D 6-DOF state by one explicit-Euler step.

    Ordering
    --------
    1. Evaluate eta_dot and nu_dot from (eta_k, nu_k).
    2. eta_{k+1} = eta_k + eta_dot_k * dt
    3. nu_{k+1}  = nu_k  + nu_dot_k  * dt

    Returns
    -------
    next_pose : ndarray, shape (6,)
    next_body_velocity : ndarray, shape (6,)
    diagnostics : dict
        Includes beginning-of-step derivatives and dynamic terms.
    """
    pose = np.asarray(
        pose,
        dtype=float,
    )

    body_velocity = np.asarray(
        body_velocity,
        dtype=float,
    )

    dt = float(dt)

    if pose.shape != (6,):
        raise ValueError(
            "pose must have shape (6,)"
        )

    if body_velocity.shape != (6,):
        raise ValueError(
            "body_velocity must have shape (6,)"
        )

    if not np.all(np.isfinite(pose)):
        raise ValueError(
            "pose must contain only finite values"
        )

    if not np.all(np.isfinite(body_velocity)):
        raise ValueError(
            "body_velocity must contain only finite values"
        )

    if not np.isfinite(dt) or dt <= 0.0:
        raise ValueError(
            "dt must be finite and strictly positive"
        )

    eta_dot, nu_dot, dynamic_diagnostics = (
        six_dof_state_derivative(
            pose,
            body_velocity,
            **dynamics_kwargs,
        )
    )

    next_pose = (
        pose
        + eta_dot * dt
    )

    next_body_velocity = (
        body_velocity
        + nu_dot * dt
    )

    if not np.all(np.isfinite(next_pose)):
        raise FloatingPointError(
            "integrated pose contains non-finite values"
        )

    if not np.all(
        np.isfinite(next_body_velocity)
    ):
        raise FloatingPointError(
            "integrated body velocity contains non-finite values"
        )

    # Verify that the updated Euler orientation remains inside
    # the supported kinematic domain.
    euler_rate_matrix(
        next_pose[3],
        next_pose[4],
    )

    diagnostics = dict(
        dynamic_diagnostics
    )

    diagnostics["eta_dot"] = eta_dot
    diagnostics["nu_dot"] = nu_dot
    diagnostics["dt"] = dt

    return (
        next_pose,
        next_body_velocity,
        diagnostics,
    )


# =====================================================================
# Common canonical arguments
# =====================================================================

common_args = dict(
    mass=mass_d,
    inertia=inertia_d,
    added_inertia=added_d,
    linear_damping=linear_d,
    quadratic_damping=quadratic_d,
    fluid_density=rho_d,
    displaced_volume=volume_neutral_d,
    center_of_gravity=r_g_d,
    center_of_buoyancy=r_b_d,
    current_velocity_world=current_zero_d,
    actuator_wrench=wrench_zero_d,
)


# =====================================================================
# Test 1 - Neutral zero-input equilibrium
# =====================================================================

pose_eq_next, nu_eq_next, diag_int_eq = (
    integrate_six_dof_euler(
        pose_zero_d,
        nu_zero_d,
        0.1,
        **common_args,
    )
)

np.testing.assert_allclose(
    pose_eq_next,
    pose_zero_d,
    atol=1e-15,
)

np.testing.assert_allclose(
    nu_eq_next,
    nu_zero_d,
    atol=1e-15,
)


# =====================================================================
# Test 2 - One-step Notebook 02 reduction
# =====================================================================

force_args = dict(
    mass=mass_d,
    inertia=inertia_d,
    added_inertia=np.zeros(6),
    linear_damping=np.zeros(6),
    quadratic_damping=np.zeros(6),
    fluid_density=rho_d,
    displaced_volume=volume_neutral_d,
    center_of_gravity=r_g_d,
    center_of_buoyancy=r_b_d,
    current_velocity_world=current_zero_d,
    actuator_wrench=pure_force,
)

dt_force = 0.1

pose_force_1, nu_force_1, diag_force_1 = (
    integrate_six_dof_euler(
        pose_zero_d,
        nu_zero_d,
        dt_force,
        **force_args,
    )
)

expected_nu_force_1 = np.array(
    [
        0.06,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
    ]
)

np.testing.assert_allclose(
    nu_force_1,
    expected_nu_force_1,
    atol=1e-15,
)

# Beginning-of-step velocity was zero.
np.testing.assert_allclose(
    pose_force_1,
    pose_zero_d,
    atol=1e-15,
)


# =====================================================================
# Test 3 - Second force-driven step confirms update ordering
# =====================================================================

pose_force_2, nu_force_2, diag_force_2 = (
    integrate_six_dof_euler(
        pose_force_1,
        nu_force_1,
        dt_force,
        **force_args,
    )
)

# Position uses beginning-of-second-step u = 0.06 m/s.
expected_x_after_step_2 = (
    0.06 * dt_force
)

np.testing.assert_allclose(
    pose_force_2[0],
    expected_x_after_step_2,
    atol=1e-15,
)

# Constant acceleration remains 0.6 m/s^2.
np.testing.assert_allclose(
    nu_force_2[0],
    0.12,
    atol=1e-15,
)

np.testing.assert_allclose(
    pose_force_2[1:],
    np.zeros(5),
    atol=1e-15,
)

np.testing.assert_allclose(
    nu_force_2[1:],
    np.zeros(5),
    atol=1e-15,
)


# =====================================================================
# Test 4 - Constant velocity reduction to Notebook 00
#
# We choose zero added mass, zero damping, neutral hydrostatics,
# zero current, and zero actuator wrench.
# =====================================================================

nu_constant = np.array(
    [
        1.2,
        -0.7,
        0.35,
        0.0,
        0.0,
        0.0,
    ]
)

constant_velocity_args = dict(
    mass=mass_d,
    inertia=inertia_d,
    added_inertia=np.zeros(6),
    linear_damping=np.zeros(6),
    quadratic_damping=np.zeros(6),
    fluid_density=rho_d,
    displaced_volume=volume_neutral_d,
    center_of_gravity=r_g_d,
    center_of_buoyancy=r_b_d,
    current_velocity_world=current_zero_d,
    actuator_wrench=wrench_zero_d,
)

dt_constant = 0.2

pose_constant_next, nu_constant_next, _ = (
    integrate_six_dof_euler(
        pose_zero_d,
        nu_constant,
        dt_constant,
        **constant_velocity_args,
    )
)

expected_position_constant = (
    nu_constant[:3] * dt_constant
)

np.testing.assert_allclose(
    pose_constant_next[:3],
    expected_position_constant,
    atol=1e-15,
)

np.testing.assert_allclose(
    pose_constant_next[3:],
    np.zeros(3),
    atol=1e-15,
)

np.testing.assert_allclose(
    nu_constant_next,
    nu_constant,
    atol=1e-15,
)


# =====================================================================
# Test 5 - Motion exactly with uniform water
# =====================================================================

following_args = dict(
    mass=mass_d,
    inertia=inertia_d,
    added_inertia=added_d,
    linear_damping=linear_d,
    quadratic_damping=quadratic_d,
    fluid_density=rho_d,
    displaced_volume=volume_neutral_d,
    center_of_gravity=r_g_d,
    center_of_buoyancy=r_b_d,
    current_velocity_world=current_following,
    actuator_wrench=wrench_zero_d,
)

dt_following = 0.2

pose_following_next, nu_following_next, diag_follow_int = (
    integrate_six_dof_euler(
        pose_zero_d,
        nu_following,
        dt_following,
        **following_args,
    )
)

expected_following_position = (
    current_following * dt_following
)

np.testing.assert_allclose(
    pose_following_next[:3],
    expected_following_position,
    atol=1e-15,
)

np.testing.assert_allclose(
    pose_following_next[3:],
    np.zeros(3),
    atol=1e-15,
)

np.testing.assert_allclose(
    nu_following_next,
    nu_following,
    atol=1e-15,
)

np.testing.assert_allclose(
    diag_follow_int["nu_dot"],
    np.zeros(6),
    atol=1e-14,
)


# =====================================================================
# Test 6 - Deterministic repeated update
# =====================================================================

result_a = integrate_six_dof_euler(
    pose_general,
    nu_general,
    0.01,
    mass=mass_d,
    inertia=inertia_d,
    added_inertia=added_d,
    linear_damping=linear_d,
    quadratic_damping=quadratic_d,
    fluid_density=rho_d,
    displaced_volume=volume_neutral_d,
    center_of_gravity=r_g_d,
    center_of_buoyancy=np.array(
        [0.0, 0.0, 0.05]
    ),
    current_velocity_world=current_following,
    actuator_wrench=actuator_general,
    environmental_wrench=environment_general,
)

result_b = integrate_six_dof_euler(
    pose_general,
    nu_general,
    0.01,
    mass=mass_d,
    inertia=inertia_d,
    added_inertia=added_d,
    linear_damping=linear_d,
    quadratic_damping=quadratic_d,
    fluid_density=rho_d,
    displaced_volume=volume_neutral_d,
    center_of_gravity=r_g_d,
    center_of_buoyancy=np.array(
        [0.0, 0.0, 0.05]
    ),
    current_velocity_world=current_following,
    actuator_wrench=actuator_general,
    environmental_wrench=environment_general,
)

pose_det_a, nu_det_a, _ = result_a
pose_det_b, nu_det_b, _ = result_b

np.testing.assert_array_equal(
    pose_det_a,
    pose_det_b,
)

np.testing.assert_array_equal(
    nu_det_a,
    nu_det_b,
)


# =====================================================================
# Invalid timestep validation
# =====================================================================

invalid_dt_calls = [
    lambda: integrate_six_dof_euler(
        pose_zero_d,
        nu_zero_d,
        0.0,
        **common_args,
    ),
    lambda: integrate_six_dof_euler(
        pose_zero_d,
        nu_zero_d,
        -0.1,
        **common_args,
    ),
    lambda: integrate_six_dof_euler(
        pose_zero_d,
        nu_zero_d,
        np.nan,
        **common_args,
    ),
    lambda: integrate_six_dof_euler(
        pose_zero_d,
        nu_zero_d,
        np.inf,
        **common_args,
    ),
]

invalid_dt_rejected = 0

for call in invalid_dt_calls:
    try:
        call()
    except ValueError:
        invalid_dt_rejected += 1

assert invalid_dt_rejected == len(
    invalid_dt_calls
)


# =====================================================================
# Report
# =====================================================================

print("Neutral equilibrium next pose:")
print(pose_eq_next)

print("Neutral equilibrium next velocity:")
print(nu_eq_next)

print("\nPure-force first-step pose:")
print(pose_force_1)

print("Pure-force first-step velocity:")
print(nu_force_1)

print("\nPure-force second-step pose:")
print(pose_force_2)

print("Pure-force second-step velocity:")
print(nu_force_2)

print(
    "\nExpected x after second force-driven step:",
    expected_x_after_step_2,
)

print("\nConstant-velocity next pose:")
print(pose_constant_next)

print("Expected translational position:")
print(expected_position_constant)

print("\nMoving-with-water next pose:")
print(pose_following_next)

print("Moving-with-water next velocity:")
print(nu_following_next)

print(
    "\nRepeated nontrivial update deterministic:",
    np.array_equal(pose_det_a, pose_det_b)
    and np.array_equal(nu_det_a, nu_det_b),
)

print(
    "Invalid timestep cases rejected:",
    f"{invalid_dt_rejected}/{len(invalid_dt_calls)}",
)

print(
    "\nPASS: explicit-Euler 6-DOF propagation preserves "
    "neutral equilibrium."
)

print(
    "PASS: one-step force response preserves the "
    "Notebook 02 integration ordering."
)

print(
    "PASS: the second force-driven step uses the "
    "previously updated velocity for pose propagation."
)

print(
    "PASS: constant-velocity translation reduces to "
    "Notebook 00 kinematics."
)

print(
    "PASS: motion with uniform water preserves body velocity "
    "while world pose advances."
)

print(
    "PASS: repeated identical 6-DOF updates are "
    "numerically deterministic."
)

print(
    "PASS: invalid timesteps are rejected cleanly."
)

Neutral equilibrium next pose:
[0. 0. 0. 0. 0. 0.]
Neutral equilibrium next velocity:
[0. 0. 0. 0. 0. 0.]

Pure-force first-step pose:
[0. 0. 0. 0. 0. 0.]
Pure-force first-step velocity:
[0.06 0.   0.   0.   0.   0.  ]

Pure-force second-step pose:
[0.006 0.    0.    0.    0.    0.   ]
Pure-force second-step velocity:
[0.12 0.   0.   0.   0.   0.  ]

Expected x after second force-driven step: 0.006

Constant-velocity next pose:
[ 0.24 -0.14  0.07  0.    0.    0.  ]
Expected translational position:
[ 0.24 -0.14  0.07]

Moving-with-water next pose:
[ 0.1  -0.05  0.02  0.    0.    0.  ]
Moving-with-water next velocity:
[ 0.5  -0.25  0.1   0.    0.    0.  ]

Repeated nontrivial update deterministic: True
Invalid timestep cases rejected: 4/4

PASS: explicit-Euler 6-DOF propagation preserves neutral equilibrium.
PASS: one-step force response preserves the Notebook 02 integration ordering.
PASS: the second force-driven step uses the previously updated velocity for pose propagation.
PASS: cons

## Step 12 - Canonical Multi-Step 6-DOF UUV Experiments

The complete instantaneous dynamics and explicit-Euler propagator are now validated.

We can therefore move from isolated one-step tests to **multi-step trajectories**.

This is the first point in Notebook 07 where the complete model is allowed to evolve continuously over time.

The purpose is not yet to generate final project artifacts or promote code into `src/`.

Instead, this step asks:

> When the individually validated terms interact repeatedly over many timesteps, does the resulting UUV behavior remain physically coherent?

Three canonical experiments are used:

1. forward thrust against hydrodynamic damping;
2. passive hydrostatic roll recovery;
3. passive entrainment by uniform ocean current.

---

# Experiment A - Neutral Forward-Thrust Response

Use:

- neutral buoyancy;
- coincident CG and CB;
- zero ocean current;
- zero initial velocity;
- zero initial orientation;
- one forward thruster acting through the body reference point.

The thruster command is:

$$
u_T=0.5
$$

with:

$$
F_{max}=40\text{ N}
$$

so the applied surge force is:

$$
X_T=20\text{ N}
$$

and:

$$
K_T=M_T=N_T=0
$$

As surge velocity increases, the validated damping law produces:

$$
X_D
=
-d_{L,u}u
-
d_{Q,u}|u|u
$$

Eventually:

$$
\dot{u}\rightarrow0
$$

and the steady force balance approaches:

$$
X_T+X_D=0
$$

For the canonical coefficients:

$$
d_{L,u}=2
$$

and:

$$
d_{Q,u}=5
$$

the positive steady-surge condition is:

$$
20-2u-5u^2=0
$$

so:

$$
\boxed{
u_{\infty}
=
\frac{-2+\sqrt{404}}{10}
}
$$

which is approximately:

$$
1.81\text{ m/s}
$$

Added mass changes the transient acceleration but not this final static force balance.

The simulation should therefore approach this analytical velocity while remaining purely surge-dominated.

---

# Experiment B - Passive Roll Recovery

Use:

- exact neutral buoyancy;
- zero current;
- zero actuator wrench;
- zero initial translational velocity;
- zero initial angular velocity;
- center of gravity at the body reference point;
- center of buoyancy $0.05$ m above the reference point.

Initialize:

$$
\phi_0=15^\circ
$$

with:

$$
\theta_0=\psi_0=0
$$

The initial hydrostatic moment must satisfy:

$$
K_H<0
$$

because positive roll must generate an opposing restoring moment.

The resulting roll motion is expected to be a **lightly damped oscillator**.

This is important.

The canonical roll damping coefficients are intentionally modest:

$$
d_{L,p}=0.10
$$

$$
d_{Q,p}=0.30
$$

Therefore the correct physical expectation is **not**:

> roll must be nearly zero after an arbitrarily selected number of seconds.

Instead, the meaningful validation criteria are:

- restoring acceleration initially points toward equilibrium;
- roll remains bounded;
- successive oscillation amplitudes decrease;
- rotational mechanical energy decreases;
- no unrelated pitch/yaw instability develops.

---

## Roll Mechanical Energy

For the neutral vehicle with:

$$
\mathbf{r}_G=0
$$

and:

$$
\mathbf{r}_B
=
\begin{bmatrix}
0\\
0\\
d
\end{bmatrix}
$$

the roll restoring moment is:

$$
K_H
=
-Bd\sin\phi
$$

where:

$$
B=\rho gV
$$

The corresponding hydrostatic potential relative to upright equilibrium is:

$$
\boxed{
U_H(\phi)
=
Bd(1-\cos\phi)
}
$$

The effective roll kinetic energy is:

$$
\boxed{
T_{\phi}
=
\frac{1}{2}
I_{x,eff}p^2
}
$$

where:

$$
I_{x,eff}
=
I_x+m_{A,p}
$$

Therefore the roll-mode mechanical energy diagnostic is:

$$
\boxed{
E_{\phi}
=
\frac{1}{2}
I_{x,eff}p^2
+
Bd(1-\cos\phi)
}
$$

Because damping performs negative generalized power, the oscillation energy should decrease over time.

This is a physically stronger validation than checking the roll angle at one arbitrary final phase of the oscillation.

---

# Oscillation-Amplitude Validation

A lightly damped oscillator may happen to be near either:

- zero crossing;
- positive peak;
- negative peak;

at the exact final timestamp.

Therefore the final single roll sample is not a robust damping metric.

Instead, divide the trajectory into an early and late window.

Define:

$$
A_{early}
=
\max |\phi(t)|
$$

over an early post-transient interval and:

$$
A_{late}
=
\max |\phi(t)|
$$

over the final quarter of the trajectory.

Successful damping requires:

$$
\boxed{
A_{late}<A_{early}
}
$$

Likewise, the mean roll mechanical energy in the final quarter must be lower than in the first quarter.

---

# Experiment C - Passive Current Capture

Use:

- exact neutral buoyancy;
- coincident CG and CB;
- zero actuator wrench;
- zero initial velocity;
- zero initial orientation;
- uniform world-frame current:

$$
\mathbf{v}_{c,W}
=
\begin{bmatrix}
0.50\\
-0.25\\
0.10
\end{bmatrix}
\text{ m/s}
$$

Initially:

$$
\nu=0
$$

so:

$$
\nu_r=-\nu_c
$$

and hydrodynamic forces act immediately.

Over time the vehicle should be accelerated by the moving fluid.

As the vehicle velocity approaches the surrounding water velocity:

$$
\mathbf{v}_{r,B}\rightarrow0
$$

and the hydrodynamic damping force decreases.

The important quantity is therefore:

$$
\boxed{
V_r
=
\left\|
\mathbf{v}_{r,B}
\right\|
}
$$

The final relative-water speed should be substantially below its initial value.

---

# Simulation Horizon

Use:

$$
\Delta t=0.01\text{ s}
$$

for all experiments.

The horizons are:

### Experiment A

$$
T_A=20\text{ s}
$$

### Experiment B

$$
T_B=40\text{ s}
$$

The longer roll horizon is intentional because the validated canonical roll damping is light.

### Experiment C

$$
T_C=30\text{ s}
$$

This gives:

$$
N_A=2000
$$

$$
N_B=4000
$$

$$
N_C=3000
$$

integration steps.

---

# Numerical Integrity

Every stored state must remain finite.

The simulations must remain away from the Z-Y-X Euler singularity.

The model must also remain deterministic:

identical initial conditions and parameters must generate identical trajectories.

---

# Validation Criteria

This step passes only if:

## Experiment A

1. forward thrust generates positive surge velocity;

2. surge remains the dominant vehicle motion;

3. the final surge velocity approaches the analytical force-balance value;

4. final surge acceleration is small relative to its initial value;

5. no unintended rotational motion appears.

## Experiment B

6. the initial positive roll produces a negative hydrostatic roll moment;

7. the initial roll acceleration points toward upright equilibrium;

8. roll remains bounded;

9. the late oscillation amplitude is smaller than the early oscillation amplitude;

10. roll-mode mechanical energy decreases significantly;

11. the final-quarter mean energy is below the first-quarter mean energy;

12. pitch and yaw remain bounded and far from the Euler singularity.

No arbitrary requirement is imposed on the exact final roll sample because a damped oscillator may be observed at any phase.

## Experiment C

13. the stationary UUV initially experiences hydrodynamic acceleration from the moving water;

14. translational water-relative speed decreases substantially;

15. final world-frame vehicle velocity moves toward the imposed world current;

16. no non-finite state develops.

## Overall

17. every simulation contains the expected number of samples;

18. every logged state remains finite;

19. rerunning the same canonical experiment gives an identical trajectory.

If these tests pass, the complete notebook-local 6-DOF UUV model has demonstrated physically coherent multi-step behavior and is ready for formal cross-notebook regression tests.

In [13]:
import numpy as np


def simulate_six_dof(
    initial_pose,
    initial_body_velocity,
    duration,
    dt,
    dynamics_kwargs,
):
    """
    Run a deterministic fixed-step HydroLab-3D 6-DOF simulation.

    Returns
    -------
    times : ndarray, shape (N+1,)
    poses : ndarray, shape (N+1, 6)
    velocities : ndarray, shape (N+1, 6)
    """
    initial_pose = np.asarray(
        initial_pose,
        dtype=float,
    )

    initial_body_velocity = np.asarray(
        initial_body_velocity,
        dtype=float,
    )

    duration = float(duration)
    dt = float(dt)

    if initial_pose.shape != (6,):
        raise ValueError(
            "initial_pose must have shape (6,)"
        )

    if initial_body_velocity.shape != (6,):
        raise ValueError(
            "initial_body_velocity must have shape (6,)"
        )

    if not np.all(np.isfinite(initial_pose)):
        raise ValueError(
            "initial_pose must contain only finite values"
        )

    if not np.all(
        np.isfinite(initial_body_velocity)
    ):
        raise ValueError(
            "initial_body_velocity must contain only finite values"
        )

    if not np.isfinite(duration) or duration <= 0.0:
        raise ValueError(
            "duration must be finite and strictly positive"
        )

    if not np.isfinite(dt) or dt <= 0.0:
        raise ValueError(
            "dt must be finite and strictly positive"
        )

    n_steps_float = duration / dt
    n_steps = int(round(n_steps_float))

    if not np.isclose(
        n_steps * dt,
        duration,
        atol=1e-12,
        rtol=0.0,
    ):
        raise ValueError(
            "duration must be an integer multiple of dt"
        )

    times = (
        np.arange(
            n_steps + 1,
            dtype=float,
        )
        * dt
    )

    poses = np.empty(
        (n_steps + 1, 6),
        dtype=float,
    )

    velocities = np.empty(
        (n_steps + 1, 6),
        dtype=float,
    )

    poses[0] = initial_pose
    velocities[0] = initial_body_velocity

    pose = initial_pose.copy()
    velocity = initial_body_velocity.copy()

    for k in range(n_steps):

        pose, velocity, _ = integrate_six_dof_euler(
            pose,
            velocity,
            dt,
            **dynamics_kwargs,
        )

        poses[k + 1] = pose
        velocities[k + 1] = velocity

    if not np.all(np.isfinite(poses)):
        raise FloatingPointError(
            "simulation produced non-finite pose values"
        )

    if not np.all(np.isfinite(velocities)):
        raise FloatingPointError(
            "simulation produced non-finite velocity values"
        )

    return times, poses, velocities


# =====================================================================
# Shared canonical parameters
# =====================================================================

dt_sim = 0.01

base_vehicle_parameters = dict(
    mass=mass_d,
    inertia=inertia_d,
    added_inertia=added_d,
    linear_damping=linear_d,
    quadratic_damping=quadratic_d,
    fluid_density=rho_d,
    displaced_volume=volume_neutral_d,
    center_of_gravity=np.zeros(3),
    center_of_buoyancy=np.zeros(3),
    current_velocity_world=np.zeros(3),
    actuator_wrench=np.zeros(6),
)


# =====================================================================
# Experiment A - Neutral forward thrust
# =====================================================================

forward_thruster_wrench = thruster_wrench_body(
    command=0.5,
    max_thrust=40.0,
    direction_body=np.array(
        [
            1.0,
            0.0,
            0.0,
        ]
    ),
    application_point_body=np.zeros(3),
)

forward_args = dict(
    base_vehicle_parameters
)

forward_args["actuator_wrench"] = (
    forward_thruster_wrench
)

t_forward, pose_forward, nu_forward_history = (
    simulate_six_dof(
        initial_pose=np.zeros(6),
        initial_body_velocity=np.zeros(6),
        duration=20.0,
        dt=dt_sim,
        dynamics_kwargs=forward_args,
    )
)


# Analytical positive root:
#
# 20 - 2u - 5u^2 = 0
#
u_terminal_analytical = (
    -2.0 + np.sqrt(404.0)
) / 10.0


_, final_forward_acceleration, final_forward_diag = (
    six_dof_state_derivative(
        pose_forward[-1],
        nu_forward_history[-1],
        **forward_args,
    )
)

initial_forward_acceleration = (
    forward_thruster_wrench[0]
    / (mass_d + added_d[0])
)

final_forward_u = (
    nu_forward_history[-1, 0]
)

forward_terminal_error = abs(
    final_forward_u
    - u_terminal_analytical
)


assert final_forward_u > 0.0

assert forward_terminal_error < 0.03

assert (
    abs(final_forward_acceleration[0])
    <
    0.02 * abs(initial_forward_acceleration)
)


# No unintended sway / heave.
np.testing.assert_allclose(
    nu_forward_history[:, 1:3],
    np.zeros_like(
        nu_forward_history[:, 1:3]
    ),
    atol=1e-12,
)

# No unintended rotation.
np.testing.assert_allclose(
    nu_forward_history[:, 3:],
    np.zeros_like(
        nu_forward_history[:, 3:]
    ),
    atol=1e-12,
)

np.testing.assert_allclose(
    pose_forward[:, 3:],
    np.zeros_like(
        pose_forward[:, 3:]
    ),
    atol=1e-12,
)


# =====================================================================
# Experiment B - Passive roll recovery
# =====================================================================

roll_initial = np.deg2rad(15.0)

pose_roll_initial = np.array(
    [
        0.0,
        0.0,
        0.0,
        roll_initial,
        0.0,
        0.0,
    ]
)

roll_args = dict(
    base_vehicle_parameters
)

roll_args["center_of_buoyancy"] = np.array(
    [
        0.0,
        0.0,
        0.05,
    ]
)


initial_roll_hydrostatic = hydrostatic_wrench(
    pose_roll_initial,
    mass_d,
    rho_d,
    volume_neutral_d,
    np.zeros(3),
    roll_args["center_of_buoyancy"],
)

assert initial_roll_hydrostatic[3] < 0.0


# Verify the initial dynamic angular acceleration also points
# toward upright equilibrium.
_, initial_roll_nu_dot, initial_roll_diag = (
    six_dof_state_derivative(
        pose_roll_initial,
        np.zeros(6),
        **roll_args,
    )
)

assert initial_roll_nu_dot[3] < 0.0


# The canonical roll mode is intentionally lightly damped,
# so use a longer horizon than the forward-thrust case.
t_roll, pose_roll, nu_roll_history = (
    simulate_six_dof(
        initial_pose=pose_roll_initial,
        initial_body_velocity=np.zeros(6),
        duration=40.0,
        dt=dt_sim,
        dynamics_kwargs=roll_args,
    )
)

roll_history = pose_roll[:, 3]
pitch_history = pose_roll[:, 4]
yaw_history = pose_roll[:, 5]

roll_rate_history = nu_roll_history[:, 3]


# ---------------------------------------------------------------------
# Roll-mode mechanical energy
# ---------------------------------------------------------------------

B_roll = (
    rho_d
    * 9.81
    * volume_neutral_d
)

cb_vertical_offset = (
    roll_args["center_of_buoyancy"][2]
)

effective_roll_inertia = (
    inertia_d[0]
    + added_d[3]
)

roll_kinetic_energy = (
    0.5
    * effective_roll_inertia
    * roll_rate_history**2
)

roll_hydrostatic_potential = (
    B_roll
    * cb_vertical_offset
    * (
        1.0
        - np.cos(roll_history)
    )
)

roll_mechanical_energy = (
    roll_kinetic_energy
    + roll_hydrostatic_potential
)


# ---------------------------------------------------------------------
# Early / late envelope comparisons
# ---------------------------------------------------------------------

n_roll_samples = len(t_roll)

quarter_roll = (
    n_roll_samples // 4
)

# Ignore the exact initial sample when forming the early
# post-release oscillation envelope.
early_roll_slice = slice(
    1,
    quarter_roll + 1,
)

late_roll_slice = slice(
    3 * quarter_roll,
    n_roll_samples,
)

early_roll_amplitude = np.max(
    np.abs(
        roll_history[
            early_roll_slice
        ]
    )
)

late_roll_amplitude = np.max(
    np.abs(
        roll_history[
            late_roll_slice
        ]
    )
)

early_mean_roll_energy = np.mean(
    roll_mechanical_energy[
        :quarter_roll
    ]
)

late_mean_roll_energy = np.mean(
    roll_mechanical_energy[
        late_roll_slice
    ]
)

initial_roll_energy = (
    roll_mechanical_energy[0]
)

final_roll_energy = (
    roll_mechanical_energy[-1]
)

max_roll_abs = np.max(
    np.abs(roll_history)
)

max_pitch_abs = np.max(
    np.abs(pitch_history)
)

max_yaw_abs = np.max(
    np.abs(yaw_history)
)


# Must remain bounded near the original disturbance magnitude.
assert max_roll_abs < np.deg2rad(20.0)

# The late oscillation envelope must be smaller.
assert (
    late_roll_amplitude
    <
    early_roll_amplitude
)

# Mean mechanical energy must decay.
assert (
    late_mean_roll_energy
    <
    early_mean_roll_energy
)

# End-state mechanical energy must be below initial energy.
assert (
    final_roll_energy
    <
    initial_roll_energy
)

# Require meaningful, not merely machine-level, energy decay.
assert (
    late_mean_roll_energy
    <
    0.75 * early_mean_roll_energy
)

# Pure roll configuration should not create meaningful
# pitch/yaw motion in this symmetric canonical case.
assert max_pitch_abs < np.deg2rad(1.0)
assert max_yaw_abs < np.deg2rad(1.0)

# Stay very far from Euler pitch singularity.
assert max_pitch_abs < np.deg2rad(30.0)


# =====================================================================
# Experiment C - Passive uniform-current capture
# =====================================================================

current_capture_world = np.array(
    [
        0.50,
        -0.25,
        0.10,
    ]
)

current_args = dict(
    base_vehicle_parameters
)

current_args["current_velocity_world"] = (
    current_capture_world
)


# Confirm the initial state is genuinely driven by current.
_, initial_current_acceleration, initial_current_diag = (
    six_dof_state_derivative(
        np.zeros(6),
        np.zeros(6),
        **current_args,
    )
)

assert (
    initial_current_acceleration[:3]
    @ current_capture_world
) > 0.0


t_current, pose_current, nu_current_history = (
    simulate_six_dof(
        initial_pose=np.zeros(6),
        initial_body_velocity=np.zeros(6),
        duration=30.0,
        dt=dt_sim,
        dynamics_kwargs=current_args,
    )
)


relative_speed_history = np.empty(
    len(t_current),
    dtype=float,
)

world_velocity_history = np.empty(
    (len(t_current), 3),
    dtype=float,
)


for k in range(len(t_current)):

    nu_r_k = relative_velocity_6dof(
        pose_current[k],
        nu_current_history[k],
        current_capture_world,
    )

    relative_speed_history[k] = np.linalg.norm(
        nu_r_k[:3]
    )

    R_bw_k = rotation_body_to_world(
        pose_current[k, 3],
        pose_current[k, 4],
        pose_current[k, 5],
    )

    world_velocity_history[k] = (
        R_bw_k
        @ nu_current_history[k, :3]
    )


initial_relative_speed = (
    relative_speed_history[0]
)

final_relative_speed = (
    relative_speed_history[-1]
)

final_world_velocity = (
    world_velocity_history[-1]
)

current_velocity_error = np.linalg.norm(
    final_world_velocity
    - current_capture_world
)


assert (
    final_relative_speed
    <
    initial_relative_speed
)

assert (
    final_relative_speed
    <
    0.20 * initial_relative_speed
)

assert (
    current_velocity_error
    <
    0.20
    * np.linalg.norm(
        current_capture_world
    )
)


# =====================================================================
# Sample-count and finite-state checks
# =====================================================================

assert len(t_forward) == 2001
assert len(t_roll) == 4001
assert len(t_current) == 3001


for array in [
    pose_forward,
    nu_forward_history,
    pose_roll,
    nu_roll_history,
    roll_mechanical_energy,
    pose_current,
    nu_current_history,
    relative_speed_history,
    world_velocity_history,
]:
    assert np.all(np.isfinite(array))


# =====================================================================
# Determinism check
# =====================================================================

(
    t_forward_repeat,
    pose_forward_repeat,
    nu_forward_repeat,
) = simulate_six_dof(
    initial_pose=np.zeros(6),
    initial_body_velocity=np.zeros(6),
    duration=20.0,
    dt=dt_sim,
    dynamics_kwargs=forward_args,
)


np.testing.assert_array_equal(
    t_forward_repeat,
    t_forward,
)

np.testing.assert_array_equal(
    pose_forward_repeat,
    pose_forward,
)

np.testing.assert_array_equal(
    nu_forward_repeat,
    nu_forward_history,
)


# =====================================================================
# Report
# =====================================================================

print("=" * 72)
print("EXPERIMENT A - NEUTRAL FORWARD THRUST")
print("=" * 72)

print(
    "Analytical terminal surge velocity :",
    u_terminal_analytical,
)

print(
    "Simulated final surge velocity      :",
    final_forward_u,
)

print(
    "Terminal surge-velocity error       :",
    forward_terminal_error,
)

print(
    "Initial surge acceleration          :",
    initial_forward_acceleration,
)

print(
    "Final surge acceleration            :",
    final_forward_acceleration[0],
)

print(
    "Final surge damping force [N]       :",
    final_forward_diag[
        "damping_wrench"
    ][0],
)

print(
    "Final world position                :",
    pose_forward[-1, :3],
)

print(
    "Maximum |sway velocity|             :",
    np.max(
        np.abs(
            nu_forward_history[:, 1]
        )
    ),
)

print(
    "Maximum |heave velocity|            :",
    np.max(
        np.abs(
            nu_forward_history[:, 2]
        )
    ),
)

print(
    "Maximum orientation magnitude [deg] :",
    np.rad2deg(
        np.max(
            np.abs(
                pose_forward[:, 3:]
            )
        )
    ),
)


print("\n" + "=" * 72)
print("EXPERIMENT B - PASSIVE ROLL RECOVERY")
print("=" * 72)

print(
    "Initial roll [deg]                  :",
    np.rad2deg(
        roll_history[0]
    ),
)

print(
    "Initial restoring roll moment [N m] :",
    initial_roll_hydrostatic[3],
)

print(
    "Initial roll acceleration [rad/s^2] :",
    initial_roll_nu_dot[3],
)

print(
    "Minimum roll [deg]                  :",
    np.rad2deg(
        np.min(
            roll_history
        )
    ),
)

print(
    "Maximum roll [deg]                  :",
    np.rad2deg(
        np.max(
            roll_history
        )
    ),
)

print(
    "Final roll [deg]                    :",
    np.rad2deg(
        roll_history[-1]
    ),
)

print(
    "Final roll rate [rad/s]             :",
    roll_rate_history[-1],
)

print(
    "Early roll envelope [deg]           :",
    np.rad2deg(
        early_roll_amplitude
    ),
)

print(
    "Late roll envelope [deg]            :",
    np.rad2deg(
        late_roll_amplitude
    ),
)

print(
    "Initial roll mechanical energy [J]  :",
    initial_roll_energy,
)

print(
    "Final roll mechanical energy [J]    :",
    final_roll_energy,
)

print(
    "First-quarter mean roll energy [J]  :",
    early_mean_roll_energy,
)

print(
    "Final-quarter mean roll energy [J]  :",
    late_mean_roll_energy,
)

print(
    "Late / early mean-energy ratio      :",
    (
        late_mean_roll_energy
        / early_mean_roll_energy
    ),
)

print(
    "Maximum |pitch| [deg]               :",
    np.rad2deg(
        max_pitch_abs
    ),
)

print(
    "Maximum |yaw| [deg]                 :",
    np.rad2deg(
        max_yaw_abs
    ),
)


print("\n" + "=" * 72)
print("EXPERIMENT C - PASSIVE CURRENT CAPTURE")
print("=" * 72)

print(
    "World current                       :",
    current_capture_world,
)

print(
    "Initial current-driven acceleration :",
    initial_current_acceleration[:3],
)

print(
    "Initial relative-water speed        :",
    initial_relative_speed,
)

print(
    "Final relative-water speed          :",
    final_relative_speed,
)

print(
    "Relative-speed reduction [%]        :",
    100.0
    * (
        1.0
        - final_relative_speed
        / initial_relative_speed
    ),
)

print(
    "Final vehicle world velocity        :",
    final_world_velocity,
)

print(
    "Final world-current velocity error  :",
    current_velocity_error,
)

print(
    "Final world position                :",
    pose_current[-1, :3],
)


print("\n" + "=" * 72)
print("STEP 12 VALIDATION RESULT")
print("=" * 72)

print(
    "Forward trajectory samples:",
    len(t_forward),
)

print(
    "Roll trajectory samples   :",
    len(t_roll),
)

print(
    "Current trajectory samples:",
    len(t_current),
)

print(
    "Repeated forward simulation deterministic:",
    (
        np.array_equal(
            pose_forward_repeat,
            pose_forward,
        )
        and
        np.array_equal(
            nu_forward_repeat,
            nu_forward_history,
        )
    ),
)

print(
    "\nPASS: forward thrust converges toward the "
    "analytical surge force-balance velocity."
)

print(
    "PASS: forward motion remains axis-consistent "
    "without unintended rotation."
)

print(
    "PASS: positive roll produces an immediate "
    "restoring angular acceleration."
)

print(
    "PASS: passive roll motion remains bounded."
)

print(
    "PASS: the late roll oscillation envelope is "
    "smaller than the early envelope."
)

print(
    "PASS: roll-mode mechanical energy decreases "
    "under rotational damping."
)

print(
    "PASS: pure-roll recovery does not generate "
    "meaningful pitch/yaw instability."
)

print(
    "PASS: a stationary UUV is accelerated by "
    "the imposed uniform current."
)

print(
    "PASS: water-relative translational speed decreases "
    "substantially during passive current capture."
)

print(
    "PASS: all canonical multi-step trajectories "
    "remain finite."
)

print(
    "PASS: canonical multi-step simulation is deterministic."
)

EXPERIMENT A - NEUTRAL FORWARD THRUST
Analytical terminal surge velocity : 1.809975124224178
Simulated final surge velocity      : 1.8099748021049136
Terminal surge-velocity error       : 3.221192643731996e-07
Initial surge acceleration          : 0.8
Final surge acceleration            : 2.589806626929203e-07
Final surge damping force [N]       : -19.999993525483433
Final world position                : [33.20507904  0.          0.        ]
Maximum |sway velocity|             : 0.0
Maximum |heave velocity|            : 0.0
Maximum orientation magnitude [deg] : 0.0

EXPERIMENT B - PASSIVE ROLL RECOVERY
Initial roll [deg]                  : 14.999999999999998
Initial restoring roll moment [N m] : -2.539014832455729
Initial roll acceleration [rad/s^2] : -1.8135820231826638
Minimum roll [deg]                  : -13.935822856611642
Maximum roll [deg]                  : 14.999999999999998
Final roll [deg]                    : 1.9969028826625832
Final roll rate [rad/s]             : 0.167427

## Step 13 - Cross-Notebook Reduction and Regression Validation

The notebook-local 6-DOF model has now passed:

- analytical kinematic tests;
- generalized-inertia checks;
- Coriolis energy checks;
- six-axis damping validation;
- hydrostatic restoring validation;
- current-frame validation;
- actuator-wrench validation;
- complete instantaneous dynamics validation;
- explicit-Euler propagation;
- multi-step trajectory experiments.

Before any Notebook 07 code is promoted into `src/`, we must verify that the new 6-DOF formulation reduces correctly to the validated behavior inherited from Notebooks 00–06.

This is a crucial engineering gate.

A more general model is only trustworthy if its reduced cases recover the simpler models it claims to extend.

The regression chain is:

```text
Notebook 00  -> 3D kinematics
Notebook 01  -> orientation transform
Notebook 02  -> F = ma translational dynamics
Notebook 03  -> quadratic drag
Notebook 04  -> gravity + buoyancy
Notebook 05  -> thruster force
Notebook 06  -> current-relative water velocity
Notebook 07  -> complete 6-DOF UUV model
```

The goal of this step is therefore:

> Disable the additional Notebook 07 physics selectively and verify that the resulting equations reproduce the earlier production APIs.

No production files are modified in this step.

---

# Regression A - Notebook 00 Kinematics

At identity orientation:

$$
R_B^W=I
$$

and with:

$$
p=q=r=0
$$

the translational part of the 6-DOF kinematic equation becomes:

$$
\dot{\mathbf{p}}
=
\mathbf{v}
$$

For one explicit timestep:

$$
\mathbf{p}_{k+1}
=
\mathbf{p}_k
+
\mathbf{v}_k\Delta t
$$

This must exactly reproduce the production function:

```text
integrate_position()
```

from Notebook 00.

---

# Regression B - Notebook 01 Orientation Transform

The upper-left block of:

$$
J(\eta)
$$

must remain exactly:

$$
R_B^W
$$

from the production:

```text
rotation_body_to_world()
```

For an arbitrary nontrivial orientation and body velocity:

$$
\dot{\mathbf{p}}_W
=
R_B^W\mathbf{v}_B
$$

must match the production transform directly.

This verifies that Notebook 07 has not silently introduced a different rotation order or sign convention.

---

# Regression C - Notebook 02 Translational Dynamics

Disable:

- added mass;
- damping;
- hydrostatic imbalance;
- current;
- angular motion.

Then:

$$
M_{RB}
=
mI
$$

for translation and:

$$
\dot{\mathbf{v}}
=
\frac{\mathbf{F}}{m}
$$

The Notebook 07 acceleration must match:

```text
acceleration_from_force()
```

exactly.

After one timestep, velocity must also match:

```text
integrate_velocity()
```

This checks both the force law and the inherited explicit-Euler integration rule.

---

# Regression D - Notebook 03 Quadratic Drag

Notebook 03 uses:

$$
\mathbf{F}_D
=
-C_D
\|\mathbf{v}\|
\mathbf{v}
$$

Notebook 07 uses diagonal axis-wise quadratic damping.

These models are not globally identical for arbitrary simultaneous 3D velocity.

The valid reduction is therefore a **pure-axis translational case**.

For:

$$
\mathbf{v}
=
\begin{bmatrix}
u\\
0\\
0
\end{bmatrix}
$$

and:

$$
d_{L,u}=0
$$

$$
d_{Q,u}=C_D
$$

the Notebook 07 surge damping force must exactly equal the production:

```text
quadratic_drag_force()
```

The same regression is evaluated for both positive and negative surge velocity.

---

# Regression E - Notebook 04 Gravity and Buoyancy

At identity orientation and with:

$$
\mathbf{r}_G
=
\mathbf{r}_B
=
0
$$

the first three components of the Notebook 07 hydrostatic wrench must equal:

$$
\mathbf{F}_G+\mathbf{F}_B
$$

from:

```text
gravity_force()
buoyancy_force()
```

The rotational components must be zero.

We test:

- positive buoyancy;
- neutral buoyancy;
- negative buoyancy.

---

# Regression F - Notebook 05 Thruster Model

For a thruster acting through the body reference point:

$$
\mathbf{r}_T=0
$$

the Notebook 07 generalized actuator wrench must satisfy:

$$
\tau_T
=
\begin{bmatrix}
\mathbf{F}_{T,B}\\
0\\
0\\
0
\end{bmatrix}
$$

where:

$$
\mathbf{F}_{T,B}
$$

must exactly equal:

```text
thruster_force_body()
```

from Notebook 05.

Command saturation is also checked by supplying a command outside:

$$
[-1,1]
$$

and confirming the generalized wrench preserves the production actuator saturation behavior.

---

# Regression G - Notebook 06 Ocean Current

At identity orientation:

$$
R_B^W=I
$$

so:

$$
\mathbf{v}_{c,B}
=
\mathbf{v}_{c,W}
$$

The Notebook 07 translational relative velocity must therefore reduce exactly to:

$$
\mathbf{v}_{rel}
=
\mathbf{v}
-
\mathbf{v}_c
$$

from:

```text
relative_water_velocity()
```

The generalized angular relative-velocity components remain equal to the vehicle's angular velocity because the Notebook 06 current model contains no ambient fluid rotation.

---

# Regression H - Complete Legacy Translational Chain

The strongest reduction combines several inherited components in one simulation.

Construct a purely translational, identity-orientation case with:

- zero angular velocity;
- zero actuator offset;
- coincident CG and CB;
- zero added mass;
- zero rotational damping;
- world current aligned with a body axis;
- force terms evaluated using the production Notebook 03–06 functions.

Then independently construct the equivalent Notebook 07 net translational force.

The two translational accelerations must agree.

This verifies that the complete generalized model preserves the earlier force accounting rather than only matching each primitive in isolation.

---

# Production Namespace Integrity

Notebook 07 has not yet promoted any implementation.

Therefore this step must also verify that the existing package exports inherited from Notebooks 00–06 remain callable.

The regression should import the existing production functions directly and execute them successfully.

No monkey-patching or notebook-local replacement is allowed for the legacy side of these comparisons.

---

# Numerical Tolerance

These reductions compare expressions that should be algebraically identical.

Use tight tolerances:

$$
10^{-12}
$$

or smaller where practical.

The goal is not approximate qualitative similarity.

The goal is to demonstrate that the general 6-DOF formulation **actually contains the earlier reduced models**.

---

# Validation Criteria

This step passes only if:

1. Notebook 07 explicit translation matches Notebook 00 `integrate_position()`;

2. Notebook 07 body/world translation matches Notebook 01 `rotation_body_to_world()`;

3. Notebook 07 force acceleration matches Notebook 02 `acceleration_from_force()`;

4. Notebook 07 velocity integration matches Notebook 02 `integrate_velocity()`;

5. pure-axis quadratic damping matches Notebook 03 for positive velocity;

6. pure-axis quadratic damping matches Notebook 03 for negative velocity;

7. hydrostatic force matches Notebook 04 for positive, neutral, and negative buoyancy;

8. through-origin thruster wrench exactly preserves Notebook 05 body thrust;

9. saturated thruster wrench exactly preserves Notebook 05 saturation;

10. identity-orientation current-relative velocity matches Notebook 06;

11. the combined legacy translational force chain matches the Notebook 07 generalized translational acceleration;

12. all inherited production functions remain importable and callable;

13. every comparison remains finite.

If these checks pass, Notebook 07 will have demonstrated formal backward compatibility with the validated physical lineage of Notebooks 00–06.

Only then should the validated 6-DOF implementation be promoted into production code.

In [14]:
import numpy as np

from hydrolab3d.dynamics.kinematics import (
    integrate_position,
)
from hydrolab3d.utils.transforms import (
    rotation_body_to_world,
)
from hydrolab3d.dynamics.rigid_body import (
    acceleration_from_force,
    integrate_velocity,
)
from hydrolab3d.dynamics.drag import (
    quadratic_drag_force,
)
from hydrolab3d.dynamics.buoyancy import (
    gravity_force,
    buoyancy_force,
)
from hydrolab3d.dynamics.thrusters import (
    thruster_force_body,
)
from hydrolab3d.world.currents import (
    relative_water_velocity,
    uniform_current_velocity,
)


# =====================================================================
# Regression A - Notebook 00 kinematics
# =====================================================================

position_00 = np.array(
    [
        2.0,
        -3.0,
        1.5,
    ]
)

velocity_00 = np.array(
    [
        1.2,
        -0.4,
        0.3,
    ]
)

dt_00 = 0.05

position_00_production = integrate_position(
    position_00,
    velocity_00,
    dt_00,
)

pose_07 = np.zeros(6)
pose_07[:3] = position_00

nu_07 = np.zeros(6)
nu_07[:3] = velocity_00

eta_dot_07 = pose_rate_6dof(
    pose_07,
    nu_07,
)

position_00_from_07 = (
    pose_07[:3]
    + eta_dot_07[:3] * dt_00
)

np.testing.assert_allclose(
    position_00_from_07,
    position_00_production,
    atol=1e-15,
    rtol=0.0,
)


# =====================================================================
# Regression B - Notebook 01 orientation
# =====================================================================

roll_01 = np.deg2rad(20.0)
pitch_01 = np.deg2rad(-12.0)
yaw_01 = np.deg2rad(35.0)

body_velocity_01 = np.array(
    [
        0.8,
        -0.3,
        0.25,
    ]
)

R_01 = rotation_body_to_world(
    roll_01,
    pitch_01,
    yaw_01,
)

world_velocity_01_production = (
    R_01 @ body_velocity_01
)

pose_orientation_07 = np.array(
    [
        0.0,
        0.0,
        0.0,
        roll_01,
        pitch_01,
        yaw_01,
    ]
)

nu_orientation_07 = np.zeros(6)
nu_orientation_07[:3] = body_velocity_01

world_velocity_01_from_07 = (
    pose_rate_6dof(
        pose_orientation_07,
        nu_orientation_07,
    )[:3]
)

np.testing.assert_allclose(
    world_velocity_01_from_07,
    world_velocity_01_production,
    atol=1e-14,
    rtol=0.0,
)


# =====================================================================
# Regression C - Notebook 02 dynamics
# =====================================================================

mass_02 = 20.0

force_02 = np.array(
    [
        12.0,
        -5.0,
        3.0,
    ]
)

velocity_02 = np.array(
    [
        0.4,
        -0.2,
        0.1,
    ]
)

dt_02 = 0.1

acceleration_02_production = (
    acceleration_from_force(
        force_02,
        mass_02,
    )
)

velocity_02_production = integrate_velocity(
    velocity_02,
    acceleration_02_production,
    dt_02,
)

M_02 = rigid_body_mass_matrix(
    mass_02,
    inertia_d,
)

generalized_force_02 = np.zeros(6)
generalized_force_02[:3] = force_02

acceleration_02_from_07 = np.linalg.solve(
    M_02,
    generalized_force_02,
)[:3]

velocity_02_from_07 = (
    velocity_02
    + acceleration_02_from_07 * dt_02
)

np.testing.assert_allclose(
    acceleration_02_from_07,
    acceleration_02_production,
    atol=1e-15,
    rtol=0.0,
)

np.testing.assert_allclose(
    velocity_02_from_07,
    velocity_02_production,
    atol=1e-15,
    rtol=0.0,
)


# =====================================================================
# Regression D - Notebook 03 quadratic drag
# =====================================================================

drag_coefficient_03 = 5.0

linear_zero_03 = np.zeros(6)

quadratic_07_for_03 = np.array(
    [
        drag_coefficient_03,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
    ]
)


def compare_notebook03_reduction(
    surge_velocity,
):
    velocity_03 = np.array(
        [
            surge_velocity,
            0.0,
            0.0,
        ]
    )

    drag_03 = quadratic_drag_force(
        velocity_03,
        drag_coefficient_03,
    )

    nu_r_07 = np.array(
        [
            surge_velocity,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
        ]
    )

    damping_07 = hydrodynamic_damping_wrench(
        nu_r_07,
        linear_zero_03,
        quadratic_07_for_03,
    )

    np.testing.assert_allclose(
        damping_07[:3],
        drag_03,
        atol=1e-15,
        rtol=0.0,
    )

    np.testing.assert_allclose(
        damping_07[3:],
        np.zeros(3),
        atol=1e-15,
        rtol=0.0,
    )

    return drag_03, damping_07[:3]


drag_positive_03, drag_positive_07 = (
    compare_notebook03_reduction(
        1.7
    )
)

drag_negative_03, drag_negative_07 = (
    compare_notebook03_reduction(
        -1.7
    )
)


# =====================================================================
# Regression E - Notebook 04 gravity and buoyancy
# =====================================================================

rho_04 = 1000.0
g_04 = 9.81
mass_04 = 20.0

volumes_04 = {
    "positive": 0.022,
    "neutral": 0.020,
    "negative": 0.018,
}

hydrostatic_regression = {}

for label, volume_04 in volumes_04.items():

    force_04_production = (
        gravity_force(
            mass_04,
            g_04,
        )
        +
        buoyancy_force(
            rho_04,
            volume_04,
            g_04,
        )
    )

    wrench_07 = hydrostatic_wrench(
        np.zeros(6),
        mass_04,
        rho_04,
        volume_04,
        np.zeros(3),
        np.zeros(3),
        g_04,
    )

    np.testing.assert_allclose(
        wrench_07[:3],
        force_04_production,
        atol=1e-12,
        rtol=0.0,
    )

    np.testing.assert_allclose(
        wrench_07[3:],
        np.zeros(3),
        atol=1e-15,
        rtol=0.0,
    )

    hydrostatic_regression[label] = (
        force_04_production.copy(),
        wrench_07.copy(),
    )


# =====================================================================
# Regression F - Notebook 05 thruster model
# =====================================================================

direction_05 = np.array(
    [
        1.0,
        0.0,
        0.0,
    ]
)

force_05 = thruster_force_body(
    0.6,
    40.0,
    direction_05,
)

wrench_07_origin = thruster_wrench_body(
    0.6,
    40.0,
    direction_05,
    np.zeros(3),
)

np.testing.assert_allclose(
    wrench_07_origin[:3],
    force_05,
    atol=1e-15,
    rtol=0.0,
)

np.testing.assert_allclose(
    wrench_07_origin[3:],
    np.zeros(3),
    atol=1e-15,
    rtol=0.0,
)


force_05_saturated = thruster_force_body(
    2.0,
    40.0,
    direction_05,
)

wrench_07_saturated = thruster_wrench_body(
    2.0,
    40.0,
    direction_05,
    np.zeros(3),
)

np.testing.assert_allclose(
    wrench_07_saturated[:3],
    force_05_saturated,
    atol=1e-15,
    rtol=0.0,
)


# =====================================================================
# Regression G - Notebook 06 current semantics
# =====================================================================

current_06 = uniform_current_velocity(
    np.array(
        [
            0.5,
            -0.25,
            0.1,
        ]
    )
)

vehicle_velocity_06 = np.array(
    [
        1.1,
        -0.3,
        0.4,
    ]
)

relative_06_production = relative_water_velocity(
    vehicle_velocity_06,
    current_06,
)

nu_vehicle_07 = np.array(
    [
        vehicle_velocity_06[0],
        vehicle_velocity_06[1],
        vehicle_velocity_06[2],
        0.2,
        -0.1,
        0.05,
    ]
)

relative_07 = relative_velocity_6dof(
    np.zeros(6),
    nu_vehicle_07,
    current_06,
)

np.testing.assert_allclose(
    relative_07[:3],
    relative_06_production,
    atol=1e-15,
    rtol=0.0,
)

np.testing.assert_allclose(
    relative_07[3:],
    nu_vehicle_07[3:],
    atol=1e-15,
    rtol=0.0,
)


# =====================================================================
# Regression H - Complete legacy translational force chain
# =====================================================================

mass_chain = 20.0
rho_chain = 1000.0
volume_chain = 0.021
gravity_chain = 9.81

velocity_chain = np.array(
    [
        1.3,
        0.0,
        0.0,
    ]
)

current_chain = np.array(
    [
        0.4,
        0.0,
        0.0,
    ]
)

relative_chain = relative_water_velocity(
    velocity_chain,
    current_chain,
)

drag_coefficient_chain = 5.0

drag_chain_production = quadratic_drag_force(
    relative_chain,
    drag_coefficient_chain,
)

gravity_chain_force = gravity_force(
    mass_chain,
    gravity_chain,
)

buoyancy_chain_force = buoyancy_force(
    rho_chain,
    volume_chain,
    gravity_chain,
)

thruster_chain_force = thruster_force_body(
    0.5,
    40.0,
    np.array(
        [
            1.0,
            0.0,
            0.0,
        ]
    ),
)

legacy_net_force = (
    thruster_chain_force
    + drag_chain_production
    + gravity_chain_force
    + buoyancy_chain_force
)

legacy_acceleration = acceleration_from_force(
    legacy_net_force,
    mass_chain,
)


# Notebook 07 equivalent:
#
# - identity orientation
# - zero angular velocity
# - zero added mass
# - pure surge quadratic damping matching Notebook 03
# - coincident CG/CB
# - same current
# - same through-origin thruster
#
pose_chain_07 = np.zeros(6)

nu_chain_07 = np.array(
    [
        velocity_chain[0],
        velocity_chain[1],
        velocity_chain[2],
        0.0,
        0.0,
        0.0,
    ]
)

linear_chain_07 = np.zeros(6)

quadratic_chain_07 = np.array(
    [
        drag_coefficient_chain,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
    ]
)

actuator_chain_07 = thruster_wrench_body(
    0.5,
    40.0,
    np.array(
        [
            1.0,
            0.0,
            0.0,
        ]
    ),
    np.zeros(3),
)

_, acceleration_chain_07, diag_chain_07 = (
    six_dof_state_derivative(
        pose_chain_07,
        nu_chain_07,
        mass=mass_chain,
        inertia=inertia_d,
        added_inertia=np.zeros(6),
        linear_damping=linear_chain_07,
        quadratic_damping=quadratic_chain_07,
        fluid_density=rho_chain,
        displaced_volume=volume_chain,
        center_of_gravity=np.zeros(3),
        center_of_buoyancy=np.zeros(3),
        current_velocity_world=current_chain,
        actuator_wrench=actuator_chain_07,
        gravity=gravity_chain,
    )
)

np.testing.assert_allclose(
    acceleration_chain_07[:3],
    legacy_acceleration,
    atol=1e-12,
    rtol=0.0,
)

np.testing.assert_allclose(
    acceleration_chain_07[3:],
    np.zeros(3),
    atol=1e-12,
    rtol=0.0,
)


# =====================================================================
# Finite-value / callability checks
# =====================================================================

legacy_outputs = [
    position_00_production,
    world_velocity_01_production,
    acceleration_02_production,
    velocity_02_production,
    drag_positive_03,
    drag_negative_03,
    force_05,
    force_05_saturated,
    relative_06_production,
    legacy_net_force,
    legacy_acceleration,
    acceleration_chain_07,
]

for value in legacy_outputs:
    assert np.all(np.isfinite(value))


# =====================================================================
# Report
# =====================================================================

print("=" * 72)
print("NOTEBOOK 00 - KINEMATICS REDUCTION")
print("=" * 72)

print("Production integrate_position:")
print(position_00_production)

print("Notebook 07 reduction:")
print(position_00_from_07)


print("\n" + "=" * 72)
print("NOTEBOOK 01 - ORIENTATION REDUCTION")
print("=" * 72)

print("Production world velocity:")
print(world_velocity_01_production)

print("Notebook 07 world velocity:")
print(world_velocity_01_from_07)


print("\n" + "=" * 72)
print("NOTEBOOK 02 - DYNAMICS REDUCTION")
print("=" * 72)

print("Production acceleration:")
print(acceleration_02_production)

print("Notebook 07 acceleration:")
print(acceleration_02_from_07)

print("Production integrated velocity:")
print(velocity_02_production)

print("Notebook 07 integrated velocity:")
print(velocity_02_from_07)


print("\n" + "=" * 72)
print("NOTEBOOK 03 - DRAG REDUCTION")
print("=" * 72)

print("Positive production drag:")
print(drag_positive_03)

print("Positive Notebook 07 damping:")
print(drag_positive_07)

print("Negative production drag:")
print(drag_negative_03)

print("Negative Notebook 07 damping:")
print(drag_negative_07)


print("\n" + "=" * 72)
print("NOTEBOOK 04 - HYDROSTATIC REDUCTION")
print("=" * 72)

for label in [
    "positive",
    "neutral",
    "negative",
]:
    force_prod, wrench_07 = (
        hydrostatic_regression[label]
    )

    print(f"{label.capitalize()} buoyancy:")
    print("  Production force :", force_prod)
    print("  Notebook 07 force:", wrench_07[:3])
    print("  Notebook 07 moment:", wrench_07[3:])


print("\n" + "=" * 72)
print("NOTEBOOK 05 - THRUSTER REDUCTION")
print("=" * 72)

print("Production body force:")
print(force_05)

print("Notebook 07 through-origin wrench:")
print(wrench_07_origin)

print("Production saturated body force:")
print(force_05_saturated)

print("Notebook 07 saturated wrench:")
print(wrench_07_saturated)


print("\n" + "=" * 72)
print("NOTEBOOK 06 - CURRENT REDUCTION")
print("=" * 72)

print("Production relative velocity:")
print(relative_06_production)

print("Notebook 07 generalized relative velocity:")
print(relative_07)


print("\n" + "=" * 72)
print("COMPLETE LEGACY TRANSLATIONAL CHAIN")
print("=" * 72)

print("Legacy production net force:")
print(legacy_net_force)

print("Legacy production acceleration:")
print(legacy_acceleration)

print("Notebook 07 net wrench:")
print(diag_chain_07["net_wrench"])

print("Notebook 07 acceleration:")
print(acceleration_chain_07)


print("\n" + "=" * 72)
print("STEP 13 REGRESSION RESULT")
print("=" * 72)

print(
    "PASS: Notebook 00 position integration is preserved."
)

print(
    "PASS: Notebook 01 body-to-world rotation is preserved."
)

print(
    "PASS: Notebook 02 F/m acceleration and Euler velocity "
    "integration are preserved."
)

print(
    "PASS: Notebook 03 pure-axis quadratic drag is preserved "
    "for positive and negative velocity."
)

print(
    "PASS: Notebook 04 gravity/buoyancy force behavior is "
    "preserved for positive, neutral, and negative buoyancy."
)

print(
    "PASS: Notebook 05 body thrust and saturation behavior "
    "are preserved by the generalized wrench mapping."
)

print(
    "PASS: Notebook 06 water-relative velocity semantics are "
    "preserved at identity orientation."
)

print(
    "PASS: complete legacy translational force accounting "
    "reduces exactly from the Notebook 07 generalized model."
)

print(
    "\nNotebook 07 is backward-compatible with the validated "
    "Notebook 00-06 physical lineage."
)

NOTEBOOK 00 - KINEMATICS REDUCTION
Production integrate_position:
[ 2.06  -3.02   1.515]
Notebook 07 reduction:
[ 2.06  -3.02   1.515]

NOTEBOOK 01 - ORIENTATION REDUCTION
Production world velocity:
[0.82920552 0.13208772 0.29575502]
Notebook 07 world velocity:
[0.82920552 0.13208772 0.29575502]

NOTEBOOK 02 - DYNAMICS REDUCTION
Production acceleration:
[ 0.6  -0.25  0.15]
Notebook 07 acceleration:
[ 0.6  -0.25  0.15]
Production integrated velocity:
[ 0.46  -0.225  0.115]
Notebook 07 integrated velocity:
[ 0.46  -0.225  0.115]

NOTEBOOK 03 - DRAG REDUCTION
Positive production drag:
[-14.45  -0.    -0.  ]
Positive Notebook 07 damping:
[-14.45  -0.    -0.  ]
Negative production drag:
[14.45 -0.   -0.  ]
Negative Notebook 07 damping:
[14.45 -0.   -0.  ]

NOTEBOOK 04 - HYDROSTATIC REDUCTION
Positive buoyancy:
  Production force : [ 0.    0.   19.62]
  Notebook 07 force: [ 0.    0.   19.62]
  Notebook 07 moment: [0. 0. 0.]
Neutral buoyancy:
  Production force : [0. 0. 0.]
  Notebook 07 forc

## Step 14 - Promote the Validated 6-DOF Model Into Production Modules

Notebook-local development is now complete.

The complete 6-DOF formulation has passed analytical tests, trajectory-level validation, and formal reductions to every previously validated HydroLab-3D physics notebook.

The implementation can therefore be promoted into the reusable package.

The promotion follows the existing modular ownership of the codebase rather than placing the complete model into one monolithic file.

### Production placement

```text
src/hydrolab3d/dynamics/kinematics.py
    euler_rate_matrix()
    kinematic_matrix_6dof()
    pose_rate_6dof()

src/hydrolab3d/dynamics/hydrodynamics.py
    skew()
    rigid_body_mass_matrix()
    added_mass_matrix()
    total_mass_matrix()
    coriolis_from_block_diagonal_mass()
    rigid_body_coriolis_matrix()
    added_mass_coriolis_matrix()
    hydrodynamic_damping_wrench()
    hydrostatic_wrench()

src/hydrolab3d/dynamics/thrusters.py
    thruster_wrench_body()

src/hydrolab3d/world/currents.py
    current_velocity_body()
    generalized_current_velocity()
    relative_velocity_6dof()

src/hydrolab3d/vehicles/uuv.py
    current_generalized_acceleration()
    six_dof_state_derivative()
    integrate_six_dof_euler()
    simulate_six_dof()
```

The existing production functions from Notebooks 00–06 remain untouched in behavior.

`core/state.py` is deliberately left unused for now.

A dedicated state abstraction should only be introduced when a later simulator-kernel requirement actually benefits from one.

---

### Promotion Policy

This step will:

1. preserve the existing source files;
2. insert the validated Notebook 07 implementation into the appropriate modules;
3. add package exports;
4. avoid introducing new numerical methods or physics;
5. avoid running validation against the promoted implementation yet.

That separation is intentional.

The next step will restart the import path and verify that the production implementation, rather than notebook-local functions, reproduces the validated behavior.

---

### Expected Result

The promotion script should report the production files written or updated and finish with:

```text
PASS: Notebook 07 validated implementation promoted to production modules.
```

No physics result is considered validated merely because this write succeeds.

Production validation occurs only after a clean re-import in the next step.

In [15]:
from pathlib import Path
import textwrap


# ---------------------------------------------------------------------
# Locate repository
# ---------------------------------------------------------------------

start = Path.cwd().resolve()

repo_root = None

for candidate in [start, *start.parents]:
    if (
        (candidate / "README.md").exists()
        and (candidate / "src").exists()
        and (candidate / "notebooks").exists()
    ):
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate HydroLab-3D repository root."
    )


package_root = (
    repo_root
    / "src"
    / "hydrolab3d"
)

dynamics_dir = package_root / "dynamics"
world_dir = package_root / "world"
vehicles_dir = package_root / "vehicles"

vehicles_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# =====================================================================
# 1. Extend dynamics/kinematics.py
# =====================================================================

kinematics_path = (
    dynamics_dir / "kinematics.py"
)

kinematics_existing = (
    kinematics_path.read_text()
)

kinematics_addition = r'''

# =====================================================================
# 6-DOF UUV kinematics
# =====================================================================


def euler_rate_matrix(
    roll,
    pitch,
    singularity_tolerance=1e-9,
):
    """Map body angular velocity to Z-Y-X Euler-angle rates."""
    import numpy as np

    roll = float(roll)
    pitch = float(pitch)

    if not np.isfinite(roll) or not np.isfinite(pitch):
        raise ValueError(
            "roll and pitch must be finite"
        )

    singularity_tolerance = float(
        singularity_tolerance
    )

    if (
        not np.isfinite(singularity_tolerance)
        or singularity_tolerance <= 0.0
    ):
        raise ValueError(
            "singularity_tolerance must be finite and positive"
        )

    c_phi = np.cos(roll)
    s_phi = np.sin(roll)
    c_theta = np.cos(pitch)

    if abs(c_theta) <= singularity_tolerance:
        raise ValueError(
            "Euler-rate transformation is singular "
            "or too close to singular at this pitch angle"
        )

    t_theta = np.tan(pitch)

    return np.array(
        [
            [
                1.0,
                s_phi * t_theta,
                c_phi * t_theta,
            ],
            [
                0.0,
                c_phi,
                -s_phi,
            ],
            [
                0.0,
                s_phi / c_theta,
                c_phi / c_theta,
            ],
        ],
        dtype=float,
    )


def kinematic_matrix_6dof(pose):
    """Construct J(eta) for eta_dot = J(eta) @ nu."""
    import numpy as np

    from hydrolab3d.utils.transforms import (
        rotation_body_to_world,
    )

    pose = np.asarray(
        pose,
        dtype=float,
    )

    if pose.shape != (6,):
        raise ValueError(
            "pose must have shape (6,)"
        )

    if not np.all(np.isfinite(pose)):
        raise ValueError(
            "pose must contain only finite values"
        )

    roll, pitch, yaw = pose[3:]

    R_bw = rotation_body_to_world(
        roll,
        pitch,
        yaw,
    )

    T_euler = euler_rate_matrix(
        roll,
        pitch,
    )

    J = np.zeros(
        (6, 6),
        dtype=float,
    )

    J[:3, :3] = R_bw
    J[3:, 3:] = T_euler

    return J


def pose_rate_6dof(
    pose,
    body_velocity,
):
    """Compute the six-component pose derivative."""
    import numpy as np

    body_velocity = np.asarray(
        body_velocity,
        dtype=float,
    )

    if body_velocity.shape != (6,):
        raise ValueError(
            "body_velocity must have shape (6,)"
        )

    if not np.all(
        np.isfinite(body_velocity)
    ):
        raise ValueError(
            "body_velocity must contain only finite values"
        )

    return (
        kinematic_matrix_6dof(pose)
        @ body_velocity
    )
'''

if "def euler_rate_matrix(" not in kinematics_existing:
    kinematics_path.write_text(
        kinematics_existing.rstrip()
        + "\n"
        + textwrap.dedent(
            kinematics_addition
        ).lstrip()
    )


# =====================================================================
# 2. Write dynamics/hydrodynamics.py
# =====================================================================

hydrodynamics_path = (
    dynamics_dir
    / "hydrodynamics.py"
)

hydrodynamics_source = r'''
"""Six-degree-of-freedom marine hydrodynamics utilities."""

import numpy as np

from hydrolab3d.dynamics.buoyancy import (
    buoyancy_force,
    gravity_force,
)
from hydrolab3d.utils.transforms import (
    rotation_body_to_world,
)


def skew(vector):
    """Return S(a) such that S(a) @ b == cross(a, b)."""
    vector = np.asarray(
        vector,
        dtype=float,
    )

    if vector.shape != (3,):
        raise ValueError(
            "vector must have shape (3,)"
        )

    if not np.all(np.isfinite(vector)):
        raise ValueError(
            "vector must contain only finite values"
        )

    x, y, z = vector

    return np.array(
        [
            [0.0, -z, y],
            [z, 0.0, -x],
            [-y, x, 0.0],
        ],
        dtype=float,
    )


def rigid_body_mass_matrix(
    mass,
    inertia,
):
    """Construct CG-centered diagonal rigid-body inertia."""
    mass = float(mass)

    inertia = np.asarray(
        inertia,
        dtype=float,
    )

    if not np.isfinite(mass) or mass <= 0.0:
        raise ValueError(
            "mass must be finite and strictly positive"
        )

    if inertia.shape != (3,):
        raise ValueError(
            "inertia must have shape (3,)"
        )

    if not np.all(np.isfinite(inertia)):
        raise ValueError(
            "inertia must contain only finite values"
        )

    if np.any(inertia <= 0.0):
        raise ValueError(
            "principal inertias must be strictly positive"
        )

    M_rb = np.zeros(
        (6, 6),
        dtype=float,
    )

    M_rb[:3, :3] = (
        mass * np.eye(3)
    )

    M_rb[3:, 3:] = np.diag(
        inertia
    )

    return M_rb


def added_mass_matrix(
    added_inertia,
):
    """Construct diagonal positive-effective added inertia."""
    added_inertia = np.asarray(
        added_inertia,
        dtype=float,
    )

    if added_inertia.shape != (6,):
        raise ValueError(
            "added_inertia must have shape (6,)"
        )

    if not np.all(
        np.isfinite(added_inertia)
    ):
        raise ValueError(
            "added_inertia must contain only finite values"
        )

    if np.any(added_inertia < 0.0):
        raise ValueError(
            "added-inertia magnitudes must be nonnegative"
        )

    return np.diag(
        added_inertia
    )


def total_mass_matrix(
    mass,
    inertia,
    added_inertia,
):
    """Return M_RB + M_A."""
    return (
        rigid_body_mass_matrix(
            mass,
            inertia,
        )
        +
        added_mass_matrix(
            added_inertia,
        )
    )


def coriolis_from_block_diagonal_mass(
    mass_matrix,
    generalized_velocity,
):
    """Construct the selected skew-symmetric 6-DOF Coriolis matrix."""
    mass_matrix = np.asarray(
        mass_matrix,
        dtype=float,
    )

    generalized_velocity = np.asarray(
        generalized_velocity,
        dtype=float,
    )

    if mass_matrix.shape != (6, 6):
        raise ValueError(
            "mass_matrix must have shape (6, 6)"
        )

    if generalized_velocity.shape != (6,):
        raise ValueError(
            "generalized_velocity must have shape (6,)"
        )

    if not np.all(np.isfinite(mass_matrix)):
        raise ValueError(
            "mass_matrix must contain only finite values"
        )

    if not np.all(
        np.isfinite(generalized_velocity)
    ):
        raise ValueError(
            "generalized_velocity must contain only finite values"
        )

    if not np.allclose(
        mass_matrix[:3, 3:],
        0.0,
        atol=1e-15,
        rtol=0.0,
    ):
        raise ValueError(
            "mass_matrix must use the Notebook 07 "
            "block-diagonal representation"
        )

    if not np.allclose(
        mass_matrix[3:, :3],
        0.0,
        atol=1e-15,
        rtol=0.0,
    ):
        raise ValueError(
            "mass_matrix must use the Notebook 07 "
            "block-diagonal representation"
        )

    momentum = (
        mass_matrix
        @ generalized_velocity
    )

    linear_momentum = momentum[:3]
    angular_momentum = momentum[3:]

    C = np.zeros(
        (6, 6),
        dtype=float,
    )

    C[:3, 3:] = -skew(
        linear_momentum
    )

    C[3:, :3] = -skew(
        linear_momentum
    )

    C[3:, 3:] = -skew(
        angular_momentum
    )

    return C


def rigid_body_coriolis_matrix(
    mass,
    inertia,
    body_velocity,
):
    """Construct C_RB(nu)."""
    return coriolis_from_block_diagonal_mass(
        rigid_body_mass_matrix(
            mass,
            inertia,
        ),
        body_velocity,
    )


def added_mass_coriolis_matrix(
    added_inertia,
    relative_velocity,
):
    """Construct C_A(nu_r)."""
    return coriolis_from_block_diagonal_mass(
        added_mass_matrix(
            added_inertia,
        ),
        relative_velocity,
    )


def hydrodynamic_damping_wrench(
    relative_velocity,
    linear_damping,
    quadratic_damping,
):
    """Return diagonal linear + quadratic hydrodynamic damping."""
    relative_velocity = np.asarray(
        relative_velocity,
        dtype=float,
    )

    linear_damping = np.asarray(
        linear_damping,
        dtype=float,
    )

    quadratic_damping = np.asarray(
        quadratic_damping,
        dtype=float,
    )

    if relative_velocity.shape != (6,):
        raise ValueError(
            "relative_velocity must have shape (6,)"
        )

    if linear_damping.shape != (6,):
        raise ValueError(
            "linear_damping must have shape (6,)"
        )

    if quadratic_damping.shape != (6,):
        raise ValueError(
            "quadratic_damping must have shape (6,)"
        )

    if not np.all(
        np.isfinite(relative_velocity)
    ):
        raise ValueError(
            "relative_velocity must contain only finite values"
        )

    if not np.all(
        np.isfinite(linear_damping)
    ):
        raise ValueError(
            "linear_damping must contain only finite values"
        )

    if not np.all(
        np.isfinite(quadratic_damping)
    ):
        raise ValueError(
            "quadratic_damping must contain only finite values"
        )

    if np.any(linear_damping < 0.0):
        raise ValueError(
            "linear damping coefficients must be nonnegative"
        )

    if np.any(quadratic_damping < 0.0):
        raise ValueError(
            "quadratic damping coefficients must be nonnegative"
        )

    return (
        -linear_damping
        * relative_velocity
        -
        quadratic_damping
        * np.abs(relative_velocity)
        * relative_velocity
    )


def hydrostatic_wrench(
    pose,
    mass,
    fluid_density,
    displaced_volume,
    center_of_gravity,
    center_of_buoyancy,
    gravity=9.81,
):
    """Return physical gravity + buoyancy wrench in body coordinates."""
    pose = np.asarray(
        pose,
        dtype=float,
    )

    center_of_gravity = np.asarray(
        center_of_gravity,
        dtype=float,
    )

    center_of_buoyancy = np.asarray(
        center_of_buoyancy,
        dtype=float,
    )

    if pose.shape != (6,):
        raise ValueError(
            "pose must have shape (6,)"
        )

    if center_of_gravity.shape != (3,):
        raise ValueError(
            "center_of_gravity must have shape (3,)"
        )

    if center_of_buoyancy.shape != (3,):
        raise ValueError(
            "center_of_buoyancy must have shape (3,)"
        )

    if not np.all(np.isfinite(pose)):
        raise ValueError(
            "pose must contain only finite values"
        )

    if not np.all(
        np.isfinite(center_of_gravity)
    ):
        raise ValueError(
            "center_of_gravity must contain only finite values"
        )

    if not np.all(
        np.isfinite(center_of_buoyancy)
    ):
        raise ValueError(
            "center_of_buoyancy must contain only finite values"
        )

    mass = float(mass)
    fluid_density = float(
        fluid_density
    )
    displaced_volume = float(
        displaced_volume
    )
    gravity = float(gravity)

    if not np.isfinite(mass) or mass <= 0.0:
        raise ValueError(
            "mass must be finite and strictly positive"
        )

    if (
        not np.isfinite(fluid_density)
        or fluid_density <= 0.0
    ):
        raise ValueError(
            "fluid_density must be finite and strictly positive"
        )

    if (
        not np.isfinite(displaced_volume)
        or displaced_volume <= 0.0
    ):
        raise ValueError(
            "displaced_volume must be finite and strictly positive"
        )

    if (
        not np.isfinite(gravity)
        or gravity <= 0.0
    ):
        raise ValueError(
            "gravity must be finite and strictly positive"
        )

    roll, pitch, yaw = pose[3:]

    R_bw = rotation_body_to_world(
        roll,
        pitch,
        yaw,
    )

    F_g_world = gravity_force(
        mass,
        gravity,
    )

    F_b_world = buoyancy_force(
        fluid_density,
        displaced_volume,
        gravity,
    )

    F_g_body = (
        R_bw.T @ F_g_world
    )

    F_b_body = (
        R_bw.T @ F_b_world
    )

    force_body = (
        F_g_body
        + F_b_body
    )

    moment_body = (
        np.cross(
            center_of_gravity,
            F_g_body,
        )
        +
        np.cross(
            center_of_buoyancy,
            F_b_body,
        )
    )

    return np.concatenate(
        [
            force_body,
            moment_body,
        ]
    )
'''

hydrodynamics_path.write_text(
    textwrap.dedent(
        hydrodynamics_source
    ).lstrip()
)


# =====================================================================
# 3. Extend dynamics/thrusters.py
# =====================================================================

thrusters_path = (
    dynamics_dir
    / "thrusters.py"
)

thrusters_existing = (
    thrusters_path.read_text()
)

thrusters_addition = r'''

# =====================================================================
# 6-DOF generalized actuator wrench
# =====================================================================


def thruster_wrench_body(
    command,
    max_thrust,
    direction_body,
    application_point_body,
):
    """Convert body thrust into [force, r x F] generalized wrench."""
    import numpy as np

    application_point_body = np.asarray(
        application_point_body,
        dtype=float,
    )

    if application_point_body.shape != (3,):
        raise ValueError(
            "application_point_body must have shape (3,)"
        )

    if not np.all(
        np.isfinite(application_point_body)
    ):
        raise ValueError(
            "application_point_body must contain only finite values"
        )

    force_body = thruster_force_body(
        command,
        max_thrust,
        direction_body,
    )

    moment_body = np.cross(
        application_point_body,
        force_body,
    )

    return np.concatenate(
        [
            force_body,
            moment_body,
        ]
    )
'''

if "def thruster_wrench_body(" not in thrusters_existing:
    thrusters_path.write_text(
        thrusters_existing.rstrip()
        + "\n"
        + textwrap.dedent(
            thrusters_addition
        ).lstrip()
    )


# =====================================================================
# 4. Extend world/currents.py
# =====================================================================

currents_path = (
    world_dir
    / "currents.py"
)

currents_existing = (
    currents_path.read_text()
)

currents_addition = r'''

# =====================================================================
# 6-DOF body-frame current utilities
# =====================================================================


def current_velocity_body(
    pose,
    current_velocity_world,
):
    """Transform world-frame translational current into body coordinates."""
    import numpy as np

    from hydrolab3d.utils.transforms import (
        rotation_body_to_world,
    )

    pose = np.asarray(
        pose,
        dtype=float,
    )

    current_velocity_world = np.asarray(
        current_velocity_world,
        dtype=float,
    )

    if pose.shape != (6,):
        raise ValueError(
            "pose must have shape (6,)"
        )

    if current_velocity_world.shape != (3,):
        raise ValueError(
            "current_velocity_world must have shape (3,)"
        )

    if not np.all(np.isfinite(pose)):
        raise ValueError(
            "pose must contain only finite values"
        )

    if not np.all(
        np.isfinite(current_velocity_world)
    ):
        raise ValueError(
            "current_velocity_world must contain only finite values"
        )

    roll, pitch, yaw = pose[3:]

    R_bw = rotation_body_to_world(
        roll,
        pitch,
        yaw,
    )

    return (
        R_bw.T
        @ current_velocity_world
    )


def generalized_current_velocity(
    pose,
    current_velocity_world,
):
    """Return [v_c,B, 0, 0, 0]."""
    import numpy as np

    nu_c = np.zeros(
        6,
        dtype=float,
    )

    nu_c[:3] = current_velocity_body(
        pose,
        current_velocity_world,
    )

    return nu_c


def relative_velocity_6dof(
    pose,
    body_velocity,
    current_velocity_world,
):
    """Return generalized water-relative body velocity."""
    import numpy as np

    body_velocity = np.asarray(
        body_velocity,
        dtype=float,
    )

    if body_velocity.shape != (6,):
        raise ValueError(
            "body_velocity must have shape (6,)"
        )

    if not np.all(
        np.isfinite(body_velocity)
    ):
        raise ValueError(
            "body_velocity must contain only finite values"
        )

    return (
        body_velocity
        -
        generalized_current_velocity(
            pose,
            current_velocity_world,
        )
    )
'''

if "def current_velocity_body(" not in currents_existing:
    currents_path.write_text(
        currents_existing.rstrip()
        + "\n"
        + textwrap.dedent(
            currents_addition
        ).lstrip()
    )


# =====================================================================
# 5. Write vehicles/uuv.py
# =====================================================================

uuv_path = (
    vehicles_dir
    / "uuv.py"
)

uuv_source = r'''
"""Complete lightweight six-degree-of-freedom UUV dynamics."""

import numpy as np

from hydrolab3d.dynamics.hydrodynamics import (
    added_mass_coriolis_matrix,
    added_mass_matrix,
    hydrodynamic_damping_wrench,
    hydrostatic_wrench,
    rigid_body_coriolis_matrix,
    rigid_body_mass_matrix,
)
from hydrolab3d.dynamics.kinematics import (
    euler_rate_matrix,
    pose_rate_6dof,
)
from hydrolab3d.world.currents import (
    current_velocity_body,
    relative_velocity_6dof,
)


def current_generalized_acceleration(
    pose,
    body_velocity,
    current_velocity_world,
):
    """Body derivative of a uniform world-fixed translational current."""
    pose = np.asarray(
        pose,
        dtype=float,
    )

    body_velocity = np.asarray(
        body_velocity,
        dtype=float,
    )

    current_velocity_world = np.asarray(
        current_velocity_world,
        dtype=float,
    )

    if pose.shape != (6,):
        raise ValueError(
            "pose must have shape (6,)"
        )

    if body_velocity.shape != (6,):
        raise ValueError(
            "body_velocity must have shape (6,)"
        )

    if current_velocity_world.shape != (3,):
        raise ValueError(
            "current_velocity_world must have shape (3,)"
        )

    if not np.all(np.isfinite(pose)):
        raise ValueError(
            "pose must contain only finite values"
        )

    if not np.all(
        np.isfinite(body_velocity)
    ):
        raise ValueError(
            "body_velocity must contain only finite values"
        )

    if not np.all(
        np.isfinite(current_velocity_world)
    ):
        raise ValueError(
            "current_velocity_world must contain only finite values"
        )

    v_c_body = current_velocity_body(
        pose,
        current_velocity_world,
    )

    omega_body = body_velocity[3:]

    nu_c_dot = np.zeros(
        6,
        dtype=float,
    )

    nu_c_dot[:3] = -np.cross(
        omega_body,
        v_c_body,
    )

    return nu_c_dot


def six_dof_state_derivative(
    pose,
    body_velocity,
    *,
    mass,
    inertia,
    added_inertia,
    linear_damping,
    quadratic_damping,
    fluid_density,
    displaced_volume,
    center_of_gravity,
    center_of_buoyancy,
    current_velocity_world,
    actuator_wrench,
    environmental_wrench=None,
    gravity=9.81,
):
    """Compute eta_dot and nu_dot for the complete UUV model."""
    pose = np.asarray(
        pose,
        dtype=float,
    )

    body_velocity = np.asarray(
        body_velocity,
        dtype=float,
    )

    actuator_wrench = np.asarray(
        actuator_wrench,
        dtype=float,
    )

    current_velocity_world = np.asarray(
        current_velocity_world,
        dtype=float,
    )

    if environmental_wrench is None:
        environmental_wrench = np.zeros(
            6,
            dtype=float,
        )
    else:
        environmental_wrench = np.asarray(
            environmental_wrench,
            dtype=float,
        )

    if pose.shape != (6,):
        raise ValueError(
            "pose must have shape (6,)"
        )

    if body_velocity.shape != (6,):
        raise ValueError(
            "body_velocity must have shape (6,)"
        )

    if actuator_wrench.shape != (6,):
        raise ValueError(
            "actuator_wrench must have shape (6,)"
        )

    if environmental_wrench.shape != (6,):
        raise ValueError(
            "environmental_wrench must have shape (6,)"
        )

    if current_velocity_world.shape != (3,):
        raise ValueError(
            "current_velocity_world must have shape (3,)"
        )

    for name, value in [
        ("pose", pose),
        ("body_velocity", body_velocity),
        ("actuator_wrench", actuator_wrench),
        (
            "environmental_wrench",
            environmental_wrench,
        ),
        (
            "current_velocity_world",
            current_velocity_world,
        ),
    ]:
        if not np.all(np.isfinite(value)):
            raise ValueError(
                f"{name} must contain only finite values"
            )

    eta_dot = pose_rate_6dof(
        pose,
        body_velocity,
    )

    M_rb = rigid_body_mass_matrix(
        mass,
        inertia,
    )

    M_a = added_mass_matrix(
        added_inertia,
    )

    M = M_rb + M_a

    nu_r = relative_velocity_6dof(
        pose,
        body_velocity,
        current_velocity_world,
    )

    nu_c_dot = current_generalized_acceleration(
        pose,
        body_velocity,
        current_velocity_world,
    )

    C_rb = rigid_body_coriolis_matrix(
        mass,
        inertia,
        body_velocity,
    )

    C_a = added_mass_coriolis_matrix(
        added_inertia,
        nu_r,
    )

    coriolis_rb_wrench = (
        C_rb @ body_velocity
    )

    coriolis_a_wrench = (
        C_a @ nu_r
    )

    damping_wrench = (
        hydrodynamic_damping_wrench(
            nu_r,
            linear_damping,
            quadratic_damping,
        )
    )

    hydrostatic = hydrostatic_wrench(
        pose,
        mass,
        fluid_density,
        displaced_volume,
        center_of_gravity,
        center_of_buoyancy,
        gravity,
    )

    added_mass_current_correction = (
        M_a @ nu_c_dot
    )

    net_wrench = (
        actuator_wrench
        + environmental_wrench
        + hydrostatic
        + damping_wrench
        - coriolis_rb_wrench
        - coriolis_a_wrench
        + added_mass_current_correction
    )

    nu_dot = np.linalg.solve(
        M,
        net_wrench,
    )

    if not np.all(np.isfinite(eta_dot)):
        raise FloatingPointError(
            "eta_dot contains non-finite values"
        )

    if not np.all(np.isfinite(nu_dot)):
        raise FloatingPointError(
            "nu_dot contains non-finite values"
        )

    diagnostics = {
        "M_RB": M_rb,
        "M_A": M_a,
        "M": M,
        "nu_r": nu_r,
        "nu_c_dot": nu_c_dot,
        "C_RB": C_rb,
        "C_A": C_a,
        "coriolis_rb_wrench": (
            coriolis_rb_wrench
        ),
        "coriolis_a_wrench": (
            coriolis_a_wrench
        ),
        "damping_wrench": (
            damping_wrench
        ),
        "hydrostatic_wrench": (
            hydrostatic
        ),
        "actuator_wrench": (
            actuator_wrench.copy()
        ),
        "environmental_wrench": (
            environmental_wrench.copy()
        ),
        "added_mass_current_correction": (
            added_mass_current_correction
        ),
        "net_wrench": net_wrench,
    }

    return (
        eta_dot,
        nu_dot,
        diagnostics,
    )


def integrate_six_dof_euler(
    pose,
    body_velocity,
    dt,
    **dynamics_kwargs,
):
    """Advance the UUV state by one explicit-Euler timestep."""
    pose = np.asarray(
        pose,
        dtype=float,
    )

    body_velocity = np.asarray(
        body_velocity,
        dtype=float,
    )

    dt = float(dt)

    if pose.shape != (6,):
        raise ValueError(
            "pose must have shape (6,)"
        )

    if body_velocity.shape != (6,):
        raise ValueError(
            "body_velocity must have shape (6,)"
        )

    if not np.all(np.isfinite(pose)):
        raise ValueError(
            "pose must contain only finite values"
        )

    if not np.all(
        np.isfinite(body_velocity)
    ):
        raise ValueError(
            "body_velocity must contain only finite values"
        )

    if not np.isfinite(dt) or dt <= 0.0:
        raise ValueError(
            "dt must be finite and strictly positive"
        )

    eta_dot, nu_dot, diagnostics = (
        six_dof_state_derivative(
            pose,
            body_velocity,
            **dynamics_kwargs,
        )
    )

    next_pose = (
        pose
        + eta_dot * dt
    )

    next_body_velocity = (
        body_velocity
        + nu_dot * dt
    )

    if not np.all(np.isfinite(next_pose)):
        raise FloatingPointError(
            "integrated pose contains non-finite values"
        )

    if not np.all(
        np.isfinite(next_body_velocity)
    ):
        raise FloatingPointError(
            "integrated body velocity contains non-finite values"
        )

    euler_rate_matrix(
        next_pose[3],
        next_pose[4],
    )

    diagnostics = dict(
        diagnostics
    )

    diagnostics["eta_dot"] = eta_dot
    diagnostics["nu_dot"] = nu_dot
    diagnostics["dt"] = dt

    return (
        next_pose,
        next_body_velocity,
        diagnostics,
    )


def simulate_six_dof(
    initial_pose,
    initial_body_velocity,
    duration,
    dt,
    dynamics_kwargs,
):
    """Run a deterministic fixed-step 6-DOF UUV simulation."""
    initial_pose = np.asarray(
        initial_pose,
        dtype=float,
    )

    initial_body_velocity = np.asarray(
        initial_body_velocity,
        dtype=float,
    )

    duration = float(duration)
    dt = float(dt)

    if initial_pose.shape != (6,):
        raise ValueError(
            "initial_pose must have shape (6,)"
        )

    if initial_body_velocity.shape != (6,):
        raise ValueError(
            "initial_body_velocity must have shape (6,)"
        )

    if not np.all(
        np.isfinite(initial_pose)
    ):
        raise ValueError(
            "initial_pose must contain only finite values"
        )

    if not np.all(
        np.isfinite(initial_body_velocity)
    ):
        raise ValueError(
            "initial_body_velocity must contain only finite values"
        )

    if not np.isfinite(duration) or duration <= 0.0:
        raise ValueError(
            "duration must be finite and strictly positive"
        )

    if not np.isfinite(dt) or dt <= 0.0:
        raise ValueError(
            "dt must be finite and strictly positive"
        )

    n_steps = int(
        round(
            duration / dt
        )
    )

    if not np.isclose(
        n_steps * dt,
        duration,
        atol=1e-12,
        rtol=0.0,
    ):
        raise ValueError(
            "duration must be an integer multiple of dt"
        )

    times = (
        np.arange(
            n_steps + 1,
            dtype=float,
        )
        * dt
    )

    poses = np.empty(
        (n_steps + 1, 6),
        dtype=float,
    )

    velocities = np.empty(
        (n_steps + 1, 6),
        dtype=float,
    )

    poses[0] = initial_pose
    velocities[0] = initial_body_velocity

    pose = initial_pose.copy()
    velocity = initial_body_velocity.copy()

    for k in range(n_steps):
        pose, velocity, _ = (
            integrate_six_dof_euler(
                pose,
                velocity,
                dt,
                **dynamics_kwargs,
            )
        )

        poses[k + 1] = pose
        velocities[k + 1] = velocity

    if not np.all(np.isfinite(poses)):
        raise FloatingPointError(
            "simulation produced non-finite pose values"
        )

    if not np.all(
        np.isfinite(velocities)
    ):
        raise FloatingPointError(
            "simulation produced non-finite velocity values"
        )

    return (
        times,
        poses,
        velocities,
    )
'''

uuv_path.write_text(
    textwrap.dedent(
        uuv_source
    ).lstrip()
)


# =====================================================================
# 6. Update dynamics package exports
# =====================================================================

dynamics_init_path = (
    dynamics_dir
    / "__init__.py"
)

dynamics_init = (
    dynamics_init_path.read_text()
)

extra_dynamics_imports = r'''
from .hydrodynamics import (
    added_mass_coriolis_matrix,
    added_mass_matrix,
    coriolis_from_block_diagonal_mass,
    hydrodynamic_damping_wrench,
    hydrostatic_wrench,
    rigid_body_coriolis_matrix,
    rigid_body_mass_matrix,
    skew,
    total_mass_matrix,
)
from .kinematics import (
    euler_rate_matrix,
    kinematic_matrix_6dof,
    pose_rate_6dof,
)
'''

if "from .hydrodynamics import (" not in dynamics_init:
    dynamics_init += (
        "\n"
        + textwrap.dedent(
            extra_dynamics_imports
        ).strip()
        + "\n"
    )

dynamics_init_path.write_text(
    dynamics_init
)


# =====================================================================
# 7. Update world package exports
# =====================================================================

world_init_path = (
    world_dir
    / "__init__.py"
)

world_init = (
    world_init_path.read_text()
)

extra_world_imports = r'''
from .currents import (
    current_velocity_body,
    generalized_current_velocity,
    relative_velocity_6dof,
)
'''

if "current_velocity_body" not in world_init:
    world_init += (
        "\n"
        + textwrap.dedent(
            extra_world_imports
        ).strip()
        + "\n"
    )

world_init_path.write_text(
    world_init
)


# =====================================================================
# 8. Create/update vehicles package exports
# =====================================================================

vehicles_init_path = (
    vehicles_dir
    / "__init__.py"
)

vehicles_init_source = '''\
"""Vehicle models for HydroLab-3D."""

from .uuv import (
    current_generalized_acceleration,
    integrate_six_dof_euler,
    simulate_six_dof,
    six_dof_state_derivative,
)

__all__ = [
    "current_generalized_acceleration",
    "integrate_six_dof_euler",
    "simulate_six_dof",
    "six_dof_state_derivative",
]
'''

vehicles_init_path.write_text(
    vehicles_init_source
)


# =====================================================================
# Promotion audit
# =====================================================================

promoted_files = [
    kinematics_path,
    hydrodynamics_path,
    thrusters_path,
    currents_path,
    uuv_path,
    dynamics_init_path,
    world_init_path,
    vehicles_init_path,
]

print(
    "=" * 72
)

print(
    "NOTEBOOK 07 PRODUCTION PROMOTION"
)

print(
    "=" * 72
)

for path in promoted_files:
    print(
        f"{path.relative_to(repo_root)} "
        f"-> {path.stat().st_size} bytes"
    )


# Confirm the intentionally unused tentative state module
# has not been populated accidentally.
state_path = (
    package_root
    / "core"
    / "state.py"
)

if state_path.exists():
    print(
        "\ncore/state.py retained without "
        "Notebook 07 state abstraction:"
    )
    print(
        f"  {state_path.relative_to(repo_root)} "
        f"-> {state_path.stat().st_size} bytes"
    )


print(
    "\nPASS: Notebook 07 validated implementation "
    "promoted to production modules."
)

print(
    "NOTE: production behavior has NOT yet been "
    "validated after clean re-import."
)

print(
    "Next gate: purge notebook-local module cache, "
    "re-import production code, and validate it independently."
)

NOTEBOOK 07 PRODUCTION PROMOTION
src/hydrolab3d/dynamics/kinematics.py -> 4079 bytes
src/hydrolab3d/dynamics/hydrodynamics.py -> 9605 bytes
src/hydrolab3d/dynamics/thrusters.py -> 4761 bytes
src/hydrolab3d/world/currents.py -> 4844 bytes
src/hydrolab3d/vehicles/uuv.py -> 11227 bytes
src/hydrolab3d/dynamics/__init__.py -> 908 bytes
src/hydrolab3d/world/__init__.py -> 337 bytes
src/hydrolab3d/vehicles/__init__.py -> 321 bytes

core/state.py retained without Notebook 07 state abstraction:
  src/hydrolab3d/core/state.py -> 0 bytes

PASS: Notebook 07 validated implementation promoted to production modules.
NOTE: production behavior has NOT yet been validated after clean re-import.
Next gate: purge notebook-local module cache, re-import production code, and validate it independently.


## Step 15 - Clean Production Re-Import and Fresh-Process Validation

Notebook 07 has now promoted its validated implementation into:

```text
src/hydrolab3d/
```

However, the current Jupyter kernel still contains notebook-local prototype functions with identical names.

Therefore simply calling:

```text
six_dof_state_derivative()
```

again would not prove that the production implementation works.

This step explicitly separates the two.

The validation has two layers:

1. **clean in-kernel production imports** after purging cached `hydrolab3d` modules;
2. **fresh Python-process validation** using a new interpreter with no Notebook 07 local namespace.

Only after both pass can the promoted implementation be considered production-valid.

---

# Layer A - Purge and Re-Import

Python caches imported modules inside:

```text
sys.modules
```

Because several HydroLab-3D modules were imported before their source files were modified, those cached objects may represent the old pre-promotion versions.

We therefore remove all loaded modules whose names begin with:

```text
hydrolab3d
```

and invalidate the import caches.

The production functions are then imported under explicit aliases such as:

```text
prod_pose_rate_6dof
prod_six_dof_state_derivative
```

This prevents confusion with notebook-local prototype names.

---

# Source-Location Requirement

Every promoted callable must resolve to a file inside:

```text
src/hydrolab3d/
```

For example:

```text
prod_six_dof_state_derivative
```

must resolve to:

```text
src/hydrolab3d/vehicles/uuv.py
```

and not to the notebook kernel.

---

# Production Numerical Validation

The production implementation will repeat several high-information validation cases.

### Kinematics

At identity orientation:

$$
J(0)=I_6
$$

### Generalized Inertia

For the canonical vehicle:

$$
M
=
\operatorname{diag}
\left(
25,\,
28,\,
30,\,
1.4,\,
1.9,\,
2.4
\right)
$$

### Neutral Equilibrium

With neutral buoyancy, zero current, zero velocity, and zero actuation:

$$
\dot{\eta}=0
$$

$$
\dot{\nu}=0
$$

### Notebook 02 Reduction

For:

$$
F_x=12\text{ N}
$$

$$
m=20\text{ kg}
$$

with added mass and other effects disabled:

$$
\dot{u}=0.6\text{ m/s}^2
$$

### Notebook 03 Reduction

Pure surge quadratic damping must still reproduce:

$$
-C_D|u|u
$$

from the existing production drag implementation.

### Notebook 06 Reduction

At identity orientation:

$$
\mathbf{v}_{r}
=
\mathbf{v}
-
\mathbf{v}_c
$$

must remain exact.

### Multi-Step Production Simulation

The promoted simulator will rerun the canonical forward-thrust experiment.

The final surge velocity must again approach:

$$
u_{\infty}
=
\frac{-2+\sqrt{404}}{10}
$$

This confirms that the promoted model behaves correctly over repeated production calls rather than only for isolated equations.

---

# Fresh-Process Validation

A new Python interpreter is then started through:

```text
subprocess
```

with the repository's `src/` directory placed on `PYTHONPATH`.

The fresh interpreter will:

1. import the promoted package;
2. verify production source locations;
3. evaluate neutral equilibrium;
4. evaluate the pure-force reduction;
5. run a short multi-step simulation;
6. exit with nonzero status if any assertion fails.

This is stronger than merely reloading modules in the notebook because the new interpreter has no access to notebook-local definitions.

---

# Validation Criteria

This step passes only if:

1. cached HydroLab-3D modules are purged successfully;

2. all promoted functions import successfully;

3. their source paths point into the expected production files;

4. production kinematics reproduce the validated identity transformation;

5. production generalized inertia reproduces the validated canonical matrix;

6. neutral zero-input equilibrium remains exact;

7. the Notebook 02 force reduction remains exact;

8. the Notebook 03 drag reduction remains exact;

9. the Notebook 06 current-relative velocity reduction remains exact;

10. the production multi-step forward-thrust simulation approaches the validated analytical terminal velocity;

11. the new Python interpreter imports the production implementation successfully;

12. all fresh-process assertions pass;

13. the fresh process exits with return code `0`.

If these checks pass, the promoted Notebook 07 implementation is independently validated as real package code rather than notebook-local state.

In [16]:
from pathlib import Path
import importlib
import inspect
import os
import subprocess
import sys

import numpy as np


# =====================================================================
# Locate repository
# =====================================================================

start = Path.cwd().resolve()

repo_root = None

for candidate in [start, *start.parents]:
    if (
        (candidate / "README.md").exists()
        and (candidate / "src").exists()
        and (candidate / "notebooks").exists()
    ):
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate HydroLab-3D repository root."
    )

src_path = (
    repo_root / "src"
).resolve()

if str(src_path) not in sys.path:
    sys.path.insert(
        0,
        str(src_path),
    )


# =====================================================================
# 1. Purge cached production modules
# =====================================================================

cached_hydrolab_modules = [
    module_name
    for module_name in list(sys.modules)
    if (
        module_name == "hydrolab3d"
        or module_name.startswith(
            "hydrolab3d."
        )
    )
]

for module_name in cached_hydrolab_modules:
    del sys.modules[module_name]

importlib.invalidate_caches()


# =====================================================================
# 2. Import promoted production implementation under explicit aliases
# =====================================================================

from hydrolab3d.dynamics.kinematics import (
    euler_rate_matrix as prod_euler_rate_matrix,
    kinematic_matrix_6dof as prod_kinematic_matrix_6dof,
    pose_rate_6dof as prod_pose_rate_6dof,
)

from hydrolab3d.dynamics.hydrodynamics import (
    added_mass_matrix as prod_added_mass_matrix,
    hydrodynamic_damping_wrench as prod_hydrodynamic_damping_wrench,
    hydrostatic_wrench as prod_hydrostatic_wrench,
    rigid_body_mass_matrix as prod_rigid_body_mass_matrix,
    total_mass_matrix as prod_total_mass_matrix,
)

from hydrolab3d.dynamics.drag import (
    quadratic_drag_force as prod_quadratic_drag_force,
)

from hydrolab3d.world.currents import (
    current_velocity_body as prod_current_velocity_body,
    relative_velocity_6dof as prod_relative_velocity_6dof,
    relative_water_velocity as prod_relative_water_velocity,
)

from hydrolab3d.dynamics.thrusters import (
    thruster_wrench_body as prod_thruster_wrench_body,
)

from hydrolab3d.vehicles.uuv import (
    current_generalized_acceleration as prod_current_generalized_acceleration,
    integrate_six_dof_euler as prod_integrate_six_dof_euler,
    simulate_six_dof as prod_simulate_six_dof,
    six_dof_state_derivative as prod_six_dof_state_derivative,
)


# =====================================================================
# 3. Verify source locations
# =====================================================================

production_callables = {
    "euler_rate_matrix":
        prod_euler_rate_matrix,
    "kinematic_matrix_6dof":
        prod_kinematic_matrix_6dof,
    "pose_rate_6dof":
        prod_pose_rate_6dof,
    "rigid_body_mass_matrix":
        prod_rigid_body_mass_matrix,
    "added_mass_matrix":
        prod_added_mass_matrix,
    "total_mass_matrix":
        prod_total_mass_matrix,
    "hydrodynamic_damping_wrench":
        prod_hydrodynamic_damping_wrench,
    "hydrostatic_wrench":
        prod_hydrostatic_wrench,
    "current_velocity_body":
        prod_current_velocity_body,
    "relative_velocity_6dof":
        prod_relative_velocity_6dof,
    "thruster_wrench_body":
        prod_thruster_wrench_body,
    "current_generalized_acceleration":
        prod_current_generalized_acceleration,
    "six_dof_state_derivative":
        prod_six_dof_state_derivative,
    "integrate_six_dof_euler":
        prod_integrate_six_dof_euler,
    "simulate_six_dof":
        prod_simulate_six_dof,
}

source_locations = {}

for name, function in production_callables.items():

    source = Path(
        inspect.getsourcefile(function)
    ).resolve()

    source_locations[name] = source

    assert src_path in source.parents, (
        f"{name} did not resolve inside production src/: "
        f"{source}"
    )


# =====================================================================
# 4. Production identity kinematics
# =====================================================================

prod_J_identity = (
    prod_kinematic_matrix_6dof(
        np.zeros(6)
    )
)

np.testing.assert_allclose(
    prod_J_identity,
    np.eye(6),
    atol=1e-15,
    rtol=0.0,
)


# =====================================================================
# 5. Production canonical mass matrix
# =====================================================================

mass_prod = 20.0

inertia_prod = np.array(
    [
        1.2,
        1.6,
        2.0,
    ]
)

added_prod = np.array(
    [
        5.0,
        8.0,
        10.0,
        0.20,
        0.30,
        0.40,
    ]
)

M_prod = prod_total_mass_matrix(
    mass_prod,
    inertia_prod,
    added_prod,
)

expected_M_prod = np.diag(
    [
        25.0,
        28.0,
        30.0,
        1.4,
        1.9,
        2.4,
    ]
)

np.testing.assert_allclose(
    M_prod,
    expected_M_prod,
    atol=1e-15,
    rtol=0.0,
)


# =====================================================================
# 6. Neutral zero-input production equilibrium
# =====================================================================

rho_prod = 1000.0

neutral_volume_prod = (
    mass_prod / rho_prod
)

zero_prod = np.zeros(6)

canonical_prod_args = dict(
    mass=mass_prod,
    inertia=inertia_prod,
    added_inertia=added_prod,
    linear_damping=np.array(
        [
            2.0,
            3.0,
            4.0,
            0.10,
            0.15,
            0.20,
        ]
    ),
    quadratic_damping=np.array(
        [
            5.0,
            6.0,
            7.0,
            0.30,
            0.40,
            0.50,
        ]
    ),
    fluid_density=rho_prod,
    displaced_volume=neutral_volume_prod,
    center_of_gravity=np.zeros(3),
    center_of_buoyancy=np.zeros(3),
    current_velocity_world=np.zeros(3),
    actuator_wrench=np.zeros(6),
)

(
    eta_dot_prod_eq,
    nu_dot_prod_eq,
    diag_prod_eq,
) = prod_six_dof_state_derivative(
    zero_prod,
    zero_prod,
    **canonical_prod_args,
)

np.testing.assert_allclose(
    eta_dot_prod_eq,
    np.zeros(6),
    atol=1e-14,
    rtol=0.0,
)

np.testing.assert_allclose(
    nu_dot_prod_eq,
    np.zeros(6),
    atol=1e-14,
    rtol=0.0,
)


# =====================================================================
# 7. Notebook 02 production reduction
# =====================================================================

pure_force_prod = np.array(
    [
        12.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
    ]
)

(
    _,
    nu_dot_prod_force,
    _,
) = prod_six_dof_state_derivative(
    np.zeros(6),
    np.zeros(6),
    mass=mass_prod,
    inertia=inertia_prod,
    added_inertia=np.zeros(6),
    linear_damping=np.zeros(6),
    quadratic_damping=np.zeros(6),
    fluid_density=rho_prod,
    displaced_volume=neutral_volume_prod,
    center_of_gravity=np.zeros(3),
    center_of_buoyancy=np.zeros(3),
    current_velocity_world=np.zeros(3),
    actuator_wrench=pure_force_prod,
)

np.testing.assert_allclose(
    nu_dot_prod_force,
    np.array(
        [
            0.6,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
        ]
    ),
    atol=1e-14,
    rtol=0.0,
)


# =====================================================================
# 8. Notebook 03 production reduction
# =====================================================================

drag_velocity_prod = np.array(
    [
        1.7,
        0.0,
        0.0,
    ]
)

drag_coefficient_prod = 5.0

legacy_drag_prod = (
    prod_quadratic_drag_force(
        drag_velocity_prod,
        drag_coefficient_prod,
    )
)

generalized_drag_prod = (
    prod_hydrodynamic_damping_wrench(
        np.array(
            [
                1.7,
                0.0,
                0.0,
                0.0,
                0.0,
                0.0,
            ]
        ),
        np.zeros(6),
        np.array(
            [
                5.0,
                0.0,
                0.0,
                0.0,
                0.0,
                0.0,
            ]
        ),
    )
)

np.testing.assert_allclose(
    generalized_drag_prod[:3],
    legacy_drag_prod,
    atol=1e-15,
    rtol=0.0,
)


# =====================================================================
# 9. Notebook 06 production reduction
# =====================================================================

vehicle_velocity_prod = np.array(
    [
        1.1,
        -0.3,
        0.4,
    ]
)

current_prod = np.array(
    [
        0.5,
        -0.25,
        0.1,
    ]
)

legacy_relative_prod = (
    prod_relative_water_velocity(
        vehicle_velocity_prod,
        current_prod,
    )
)

generalized_relative_prod = (
    prod_relative_velocity_6dof(
        np.zeros(6),
        np.array(
            [
                1.1,
                -0.3,
                0.4,
                0.2,
                -0.1,
                0.05,
            ]
        ),
        current_prod,
    )
)

np.testing.assert_allclose(
    generalized_relative_prod[:3],
    legacy_relative_prod,
    atol=1e-15,
    rtol=0.0,
)


# =====================================================================
# 10. Production canonical forward-thrust simulation
# =====================================================================

forward_wrench_prod = (
    prod_thruster_wrench_body(
        command=0.5,
        max_thrust=40.0,
        direction_body=np.array(
            [
                1.0,
                0.0,
                0.0,
            ]
        ),
        application_point_body=np.zeros(3),
    )
)

forward_prod_args = dict(
    canonical_prod_args
)

forward_prod_args[
    "actuator_wrench"
] = forward_wrench_prod

(
    t_forward_prod,
    pose_forward_prod,
    nu_forward_prod,
) = prod_simulate_six_dof(
    initial_pose=np.zeros(6),
    initial_body_velocity=np.zeros(6),
    duration=20.0,
    dt=0.01,
    dynamics_kwargs=forward_prod_args,
)

analytical_terminal_prod = (
    -2.0 + np.sqrt(404.0)
) / 10.0

final_u_prod = (
    nu_forward_prod[-1, 0]
)

terminal_error_prod = abs(
    final_u_prod
    - analytical_terminal_prod
)

assert terminal_error_prod < 0.03

assert len(t_forward_prod) == 2001

assert np.all(
    np.isfinite(
        pose_forward_prod
    )
)

assert np.all(
    np.isfinite(
        nu_forward_prod
    )
)


# =====================================================================
# 11. Fresh-process validation
# =====================================================================

fresh_process_script = r'''
import inspect
from pathlib import Path

import numpy as np

from hydrolab3d.dynamics.kinematics import (
    kinematic_matrix_6dof,
)

from hydrolab3d.dynamics.hydrodynamics import (
    total_mass_matrix,
)

from hydrolab3d.dynamics.thrusters import (
    thruster_wrench_body,
)

from hydrolab3d.vehicles.uuv import (
    six_dof_state_derivative,
    simulate_six_dof,
)


src_root = Path(
    __import__("hydrolab3d").__file__
).resolve().parents[1]

for fn in [
    kinematic_matrix_6dof,
    total_mass_matrix,
    thruster_wrench_body,
    six_dof_state_derivative,
    simulate_six_dof,
]:
    source = Path(
        inspect.getsourcefile(fn)
    ).resolve()

    assert src_root in source.parents


np.testing.assert_allclose(
    kinematic_matrix_6dof(
        np.zeros(6)
    ),
    np.eye(6),
    atol=1e-15,
    rtol=0.0,
)


mass = 20.0

inertia = np.array(
    [
        1.2,
        1.6,
        2.0,
    ]
)

added = np.array(
    [
        5.0,
        8.0,
        10.0,
        0.2,
        0.3,
        0.4,
    ]
)

linear = np.array(
    [
        2.0,
        3.0,
        4.0,
        0.10,
        0.15,
        0.20,
    ]
)

quadratic = np.array(
    [
        5.0,
        6.0,
        7.0,
        0.30,
        0.40,
        0.50,
    ]
)

rho = 1000.0
volume = mass / rho

zero = np.zeros(6)


eta_dot, nu_dot, _ = (
    six_dof_state_derivative(
        zero,
        zero,
        mass=mass,
        inertia=inertia,
        added_inertia=added,
        linear_damping=linear,
        quadratic_damping=quadratic,
        fluid_density=rho,
        displaced_volume=volume,
        center_of_gravity=np.zeros(3),
        center_of_buoyancy=np.zeros(3),
        current_velocity_world=np.zeros(3),
        actuator_wrench=np.zeros(6),
    )
)

np.testing.assert_allclose(
    eta_dot,
    np.zeros(6),
    atol=1e-14,
)

np.testing.assert_allclose(
    nu_dot,
    np.zeros(6),
    atol=1e-14,
)


_, force_acceleration, _ = (
    six_dof_state_derivative(
        zero,
        zero,
        mass=mass,
        inertia=inertia,
        added_inertia=np.zeros(6),
        linear_damping=np.zeros(6),
        quadratic_damping=np.zeros(6),
        fluid_density=rho,
        displaced_volume=volume,
        center_of_gravity=np.zeros(3),
        center_of_buoyancy=np.zeros(3),
        current_velocity_world=np.zeros(3),
        actuator_wrench=np.array(
            [
                12.0,
                0.0,
                0.0,
                0.0,
                0.0,
                0.0,
            ]
        ),
    )
)

np.testing.assert_allclose(
    force_acceleration,
    np.array(
        [
            0.6,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
        ]
    ),
    atol=1e-14,
)


thruster = thruster_wrench_body(
    0.5,
    40.0,
    np.array(
        [
            1.0,
            0.0,
            0.0,
        ]
    ),
    np.zeros(3),
)

args = dict(
    mass=mass,
    inertia=inertia,
    added_inertia=added,
    linear_damping=linear,
    quadratic_damping=quadratic,
    fluid_density=rho,
    displaced_volume=volume,
    center_of_gravity=np.zeros(3),
    center_of_buoyancy=np.zeros(3),
    current_velocity_world=np.zeros(3),
    actuator_wrench=thruster,
)

times, poses, velocities = simulate_six_dof(
    np.zeros(6),
    np.zeros(6),
    2.0,
    0.01,
    args,
)

assert len(times) == 201
assert np.all(np.isfinite(poses))
assert np.all(np.isfinite(velocities))
assert velocities[-1, 0] > 0.0

print("FRESH_PROCESS_PASS")
print("hydrolab3d:", __import__("hydrolab3d").__file__)
print(
    "six_dof_state_derivative:",
    inspect.getsourcefile(
        six_dof_state_derivative
    ),
)
print(
    "final_short_rollout_u:",
    velocities[-1, 0],
)
'''


fresh_env = os.environ.copy()

existing_pythonpath = (
    fresh_env.get(
        "PYTHONPATH",
        "",
    )
)

if existing_pythonpath:
    fresh_env["PYTHONPATH"] = (
        str(src_path)
        + os.pathsep
        + existing_pythonpath
    )
else:
    fresh_env["PYTHONPATH"] = (
        str(src_path)
    )


fresh_result = subprocess.run(
    [
        sys.executable,
        "-c",
        fresh_process_script,
    ],
    cwd=repo_root,
    env=fresh_env,
    capture_output=True,
    text=True,
)

if fresh_result.returncode != 0:
    print(
        "Fresh-process STDOUT:"
    )
    print(
        fresh_result.stdout
    )

    print(
        "Fresh-process STDERR:"
    )
    print(
        fresh_result.stderr
    )

    raise RuntimeError(
        "Fresh-process production validation failed."
    )

assert (
    "FRESH_PROCESS_PASS"
    in fresh_result.stdout
)


# =====================================================================
# Report
# =====================================================================

print(
    "=" * 72
)

print(
    "STEP 15 - CLEAN PRODUCTION VALIDATION"
)

print(
    "=" * 72
)


print(
    "\nPurged cached HydroLab-3D modules:",
    len(cached_hydrolab_modules),
)


print(
    "\nProduction source locations:"
)

for name, source in source_locations.items():
    print(
        f"  {name:<34} -> "
        f"{source.relative_to(repo_root)}"
    )


print(
    "\nProduction identity J:"
)

print(
    prod_J_identity
)


print(
    "\nProduction canonical total mass matrix:"
)

print(
    M_prod
)


print(
    "\nNeutral production eta_dot:"
)

print(
    eta_dot_prod_eq
)

print(
    "Neutral production nu_dot:"
)

print(
    nu_dot_prod_eq
)


print(
    "\nProduction pure-force acceleration:"
)

print(
    nu_dot_prod_force
)


print(
    "\nProduction Notebook 03 reduction:"
)

print(
    generalized_drag_prod[:3]
)


print(
    "\nProduction Notebook 06 reduction:"
)

print(
    generalized_relative_prod[:3]
)


print(
    "\nAnalytical terminal surge velocity:",
    analytical_terminal_prod,
)

print(
    "Production simulated final surge velocity:",
    final_u_prod,
)

print(
    "Production terminal velocity error:",
    terminal_error_prod,
)


print(
    "\nFresh-process return code:",
    fresh_result.returncode,
)

print(
    "Fresh-process output:"
)

print(
    fresh_result.stdout.strip()
)


print(
    "\nPASS: promoted functions resolve to "
    "production source files."
)

print(
    "PASS: cleanly re-imported production "
    "kinematics and inertia match validated results."
)

print(
    "PASS: production neutral equilibrium remains exact."
)

print(
    "PASS: production Notebook 02 reduction remains exact."
)

print(
    "PASS: production Notebook 03 reduction remains exact."
)

print(
    "PASS: production Notebook 06 reduction remains exact."
)

print(
    "PASS: production multi-step surge behavior matches "
    "the validated analytical equilibrium."
)

print(
    "PASS: a fresh Python interpreter independently "
    "imports and executes the promoted 6-DOF model."
)

print(
    "\nNotebook 07 production promotion is independently validated."
)

STEP 15 - CLEAN PRODUCTION VALIDATION

Purged cached HydroLab-3D modules: 11

Production source locations:
  euler_rate_matrix                  -> src/hydrolab3d/dynamics/kinematics.py
  kinematic_matrix_6dof              -> src/hydrolab3d/dynamics/kinematics.py
  pose_rate_6dof                     -> src/hydrolab3d/dynamics/kinematics.py
  rigid_body_mass_matrix             -> src/hydrolab3d/dynamics/hydrodynamics.py
  added_mass_matrix                  -> src/hydrolab3d/dynamics/hydrodynamics.py
  total_mass_matrix                  -> src/hydrolab3d/dynamics/hydrodynamics.py
  hydrodynamic_damping_wrench        -> src/hydrolab3d/dynamics/hydrodynamics.py
  hydrostatic_wrench                 -> src/hydrolab3d/dynamics/hydrodynamics.py
  current_velocity_body              -> src/hydrolab3d/world/currents.py
  relative_velocity_6dof             -> src/hydrolab3d/world/currents.py
  thruster_wrench_body               -> src/hydrolab3d/dynamics/thrusters.py
  current_generalized_accelerat

## Step 16 - Generate Canonical Notebook 07 Validation Artifacts

The promoted 6-DOF implementation has now been independently validated from a fresh Python interpreter.

The final technical task before repository-wide testing is to preserve a compact set of reproducible Notebook 07 artifacts.

These artifacts are not intended to become a large dataset.

They serve as:

- human-readable evidence of the canonical behaviors;
- lightweight regression references;
- visual confirmation of trajectory-level dynamics;
- documentation inputs for future simulator development.

All artifacts are generated using the **production implementation** from:

```text
src/hydrolab3d/
```

rather than notebook-local prototype functions.

---

# Artifact Directory

Notebook 07 artifacts are stored under:

```text
results/notebooks/07_six_dof_uuv/
```

The directory will contain:

```text
forward_thrust.csv
roll_recovery.csv
current_capture.csv

forward_thrust_response.png
roll_recovery_response.png
current_capture_response.png

summary_metrics.csv
```

No binary simulation checkpoints or large trajectory dumps are required.

---

# Artifact A - Forward-Thrust Response

The production model will rerun the validated canonical surge experiment:

$$
u_T=0.5
$$

$$
F_{max}=40\text{ N}
$$

with:

- neutral buoyancy;
- zero current;
- zero initial velocity;
- through-origin forward thrust.

The CSV records:

```text
time
x
y
z
roll
pitch
yaw
u
v
w
p
q
r
```

The corresponding figure shows:

1. surge velocity versus time;
2. analytical terminal surge velocity as a reference;
3. surge position versus time.

The key terminal reference remains:

$$
u_{\infty}
=
\frac{-2+\sqrt{404}}{10}
$$

---

# Artifact B - Passive Roll Recovery

The production model will rerun the canonical roll-recovery experiment:

$$
\phi_0=15^\circ
$$

with:

$$
\mathbf{r}_B
=
\begin{bmatrix}
0\\
0\\
0.05
\end{bmatrix}
$$

and:

$$
\mathbf{r}_G=0
$$

The CSV records the complete state together with:

```text
roll_deg
roll_rate
roll_mechanical_energy
```

The figure shows:

1. roll angle versus time;
2. roll rate versus time;
3. roll-mode mechanical energy versus time.

This preserves the important result that the system is a damped restoring oscillator rather than requiring an arbitrary final-angle threshold.

---

# Artifact C - Passive Current Capture

The production model will rerun the uniform-current experiment:

$$
\mathbf{v}_{c,W}
=
\begin{bmatrix}
0.50\\
-0.25\\
0.10
\end{bmatrix}
\text{ m/s}
$$

from a stationary initial vehicle.

The CSV additionally records:

```text
relative_water_speed
world_vx
world_vy
world_vz
```

The corresponding figure shows:

1. relative-water speed versus time;
2. each world-frame vehicle velocity component;
3. constant current-component references.

This preserves the Notebook 06 → Notebook 07 current-relative dynamics lineage.

---

# Summary Metrics

A compact:

```text
summary_metrics.csv
```

will contain the major canonical results, including:

- analytical terminal surge velocity;
- simulated terminal surge velocity;
- terminal surge error;
- final surge acceleration;
- initial and late roll-envelope amplitudes;
- initial and final roll mechanical energy;
- roll energy reduction;
- initial and final relative-water speed;
- relative-speed reduction;
- final current-velocity error.

This file provides a small machine-readable overview without requiring future readers to parse every trajectory.

---

# Production-Only Requirement

Before generating artifacts, the notebook will explicitly import:

```text
simulate_six_dof
six_dof_state_derivative
thruster_wrench_body
relative_velocity_6dof
rotation_body_to_world
```

from the production package.

The cell does not use the notebook-local:

```text
simulate_six_dof()
```

prototype created earlier.

Production aliases are used explicitly to make that distinction visible.

---

# Plotting Philosophy

The plots are engineering diagnostics rather than decorative graphics.

They should be:

- readable;
- compact;
- labeled with units;
- deterministic;
- directly tied to validated physical quantities.

No animation or 3D renderer is required.

HydroLab-3D remains headless-first.

---

# Validation Criteria

This step passes only if:

1. the Notebook 07 artifact directory exists;

2. all three production simulations complete successfully;

3. every trajectory remains finite;

4. all three CSV trajectory files are written;

5. all three PNG diagnostic figures are written;

6. `summary_metrics.csv` is written;

7. every artifact file is non-empty;

8. the regenerated forward-thrust result still matches the validated analytical equilibrium;

9. roll mechanical energy decreases;

10. relative-water speed decreases strongly under passive current capture;

11. artifact generation does not modify production source code.

If these checks pass, Notebook 07 will possess persistent validation evidence generated from the promoted production model.

In [17]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from hydrolab3d.dynamics.thrusters import (
    thruster_wrench_body as prod_thruster_wrench,
)

from hydrolab3d.utils.transforms import (
    rotation_body_to_world as prod_rotation_body_to_world,
)

from hydrolab3d.vehicles.uuv import (
    simulate_six_dof as prod_simulate,
    six_dof_state_derivative as prod_state_derivative,
)

from hydrolab3d.world.currents import (
    relative_velocity_6dof as prod_relative_velocity_6dof,
)


# =====================================================================
# Locate repository and artifact directory
# =====================================================================

start = Path.cwd().resolve()

repo_root = None

for candidate in [start, *start.parents]:
    if (
        (candidate / "README.md").exists()
        and (candidate / "src").exists()
        and (candidate / "notebooks").exists()
    ):
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate HydroLab-3D repository root."
    )


artifact_dir = (
    repo_root
    / "results"
    / "notebooks"
    / "07_six_dof_uuv"
)

artifact_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# =====================================================================
# Canonical production parameters
# =====================================================================

mass_art = 20.0

inertia_art = np.array(
    [
        1.2,
        1.6,
        2.0,
    ]
)

added_art = np.array(
    [
        5.0,
        8.0,
        10.0,
        0.20,
        0.30,
        0.40,
    ]
)

linear_art = np.array(
    [
        2.0,
        3.0,
        4.0,
        0.10,
        0.15,
        0.20,
    ]
)

quadratic_art = np.array(
    [
        5.0,
        6.0,
        7.0,
        0.30,
        0.40,
        0.50,
    ]
)

rho_art = 1000.0

volume_art = (
    mass_art / rho_art
)

dt_art = 0.01


base_art_args = dict(
    mass=mass_art,
    inertia=inertia_art,
    added_inertia=added_art,
    linear_damping=linear_art,
    quadratic_damping=quadratic_art,
    fluid_density=rho_art,
    displaced_volume=volume_art,
    center_of_gravity=np.zeros(3),
    center_of_buoyancy=np.zeros(3),
    current_velocity_world=np.zeros(3),
    actuator_wrench=np.zeros(6),
)


# =====================================================================
# Helper - trajectory dataframe
# =====================================================================

def trajectory_dataframe(
    times,
    poses,
    velocities,
):
    return pd.DataFrame(
        {
            "time": times,
            "x": poses[:, 0],
            "y": poses[:, 1],
            "z": poses[:, 2],
            "roll": poses[:, 3],
            "pitch": poses[:, 4],
            "yaw": poses[:, 5],
            "u": velocities[:, 0],
            "v": velocities[:, 1],
            "w": velocities[:, 2],
            "p": velocities[:, 3],
            "q": velocities[:, 4],
            "r": velocities[:, 5],
        }
    )


# =====================================================================
# Experiment A - Forward thrust
# =====================================================================

forward_wrench_art = prod_thruster_wrench(
    command=0.5,
    max_thrust=40.0,
    direction_body=np.array(
        [
            1.0,
            0.0,
            0.0,
        ]
    ),
    application_point_body=np.zeros(3),
)

forward_art_args = dict(
    base_art_args
)

forward_art_args[
    "actuator_wrench"
] = forward_wrench_art


(
    t_forward_art,
    pose_forward_art,
    nu_forward_art,
) = prod_simulate(
    initial_pose=np.zeros(6),
    initial_body_velocity=np.zeros(6),
    duration=20.0,
    dt=dt_art,
    dynamics_kwargs=forward_art_args,
)


forward_df = trajectory_dataframe(
    t_forward_art,
    pose_forward_art,
    nu_forward_art,
)

forward_csv = (
    artifact_dir
    / "forward_thrust.csv"
)

forward_df.to_csv(
    forward_csv,
    index=False,
)


u_terminal_art = (
    -2.0 + np.sqrt(404.0)
) / 10.0


(
    _,
    final_forward_accel_art,
    final_forward_diag_art,
) = prod_state_derivative(
    pose_forward_art[-1],
    nu_forward_art[-1],
    **forward_art_args,
)


forward_terminal_error_art = abs(
    nu_forward_art[-1, 0]
    - u_terminal_art
)


fig = plt.figure(
    figsize=(9, 6)
)

ax1 = fig.add_subplot(
    2,
    1,
    1,
)

ax1.plot(
    t_forward_art,
    nu_forward_art[:, 0],
    label="Simulated surge velocity",
)

ax1.axhline(
    u_terminal_art,
    linestyle="--",
    label="Analytical terminal velocity",
)

ax1.set_xlabel(
    "Time [s]"
)

ax1.set_ylabel(
    "Surge velocity u [m/s]"
)

ax1.set_title(
    "Notebook 07 - Forward-Thrust Response"
)

ax1.grid(
    True,
    alpha=0.3,
)

ax1.legend()


ax2 = fig.add_subplot(
    2,
    1,
    2,
)

ax2.plot(
    t_forward_art,
    pose_forward_art[:, 0],
)

ax2.set_xlabel(
    "Time [s]"
)

ax2.set_ylabel(
    "World x position [m]"
)

ax2.grid(
    True,
    alpha=0.3,
)

fig.tight_layout()

forward_png = (
    artifact_dir
    / "forward_thrust_response.png"
)

fig.savefig(
    forward_png,
    dpi=160,
    bbox_inches="tight",
)

plt.close(fig)


# =====================================================================
# Experiment B - Passive roll recovery
# =====================================================================

roll_initial_art = np.deg2rad(
    15.0
)

roll_pose_initial_art = np.array(
    [
        0.0,
        0.0,
        0.0,
        roll_initial_art,
        0.0,
        0.0,
    ]
)

roll_art_args = dict(
    base_art_args
)

roll_art_args[
    "center_of_buoyancy"
] = np.array(
    [
        0.0,
        0.0,
        0.05,
    ]
)


(
    t_roll_art,
    pose_roll_art,
    nu_roll_art,
) = prod_simulate(
    initial_pose=roll_pose_initial_art,
    initial_body_velocity=np.zeros(6),
    duration=40.0,
    dt=dt_art,
    dynamics_kwargs=roll_art_args,
)


B_art = (
    rho_art
    * 9.81
    * volume_art
)

d_art = (
    roll_art_args[
        "center_of_buoyancy"
    ][2]
)

I_roll_eff_art = (
    inertia_art[0]
    + added_art[3]
)


roll_energy_art = (
    0.5
    * I_roll_eff_art
    * nu_roll_art[:, 3] ** 2
    +
    B_art
    * d_art
    * (
        1.0
        - np.cos(
            pose_roll_art[:, 3]
        )
    )
)


roll_df = trajectory_dataframe(
    t_roll_art,
    pose_roll_art,
    nu_roll_art,
)

roll_df[
    "roll_deg"
] = np.rad2deg(
    pose_roll_art[:, 3]
)

roll_df[
    "roll_rate"
] = nu_roll_art[:, 3]

roll_df[
    "roll_mechanical_energy"
] = roll_energy_art


roll_csv = (
    artifact_dir
    / "roll_recovery.csv"
)

roll_df.to_csv(
    roll_csv,
    index=False,
)


fig = plt.figure(
    figsize=(9, 8)
)

ax1 = fig.add_subplot(
    3,
    1,
    1,
)

ax1.plot(
    t_roll_art,
    np.rad2deg(
        pose_roll_art[:, 3]
    ),
)

ax1.axhline(
    0.0,
    linewidth=1.0,
)

ax1.set_ylabel(
    "Roll [deg]"
)

ax1.set_title(
    "Notebook 07 - Passive Roll Recovery"
)

ax1.grid(
    True,
    alpha=0.3,
)


ax2 = fig.add_subplot(
    3,
    1,
    2,
)

ax2.plot(
    t_roll_art,
    nu_roll_art[:, 3],
)

ax2.set_ylabel(
    "Roll rate p [rad/s]"
)

ax2.grid(
    True,
    alpha=0.3,
)


ax3 = fig.add_subplot(
    3,
    1,
    3,
)

ax3.plot(
    t_roll_art,
    roll_energy_art,
)

ax3.set_xlabel(
    "Time [s]"
)

ax3.set_ylabel(
    "Roll energy [J]"
)

ax3.grid(
    True,
    alpha=0.3,
)

fig.tight_layout()

roll_png = (
    artifact_dir
    / "roll_recovery_response.png"
)

fig.savefig(
    roll_png,
    dpi=160,
    bbox_inches="tight",
)

plt.close(fig)


# =====================================================================
# Experiment C - Passive current capture
# =====================================================================

current_world_art = np.array(
    [
        0.50,
        -0.25,
        0.10,
    ]
)

current_art_args = dict(
    base_art_args
)

current_art_args[
    "current_velocity_world"
] = current_world_art


(
    t_current_art,
    pose_current_art,
    nu_current_art,
) = prod_simulate(
    initial_pose=np.zeros(6),
    initial_body_velocity=np.zeros(6),
    duration=30.0,
    dt=dt_art,
    dynamics_kwargs=current_art_args,
)


relative_speed_art = np.empty(
    len(t_current_art),
    dtype=float,
)

world_velocity_art = np.empty(
    (
        len(t_current_art),
        3,
    ),
    dtype=float,
)


for k in range(
    len(t_current_art)
):

    nu_r_k = prod_relative_velocity_6dof(
        pose_current_art[k],
        nu_current_art[k],
        current_world_art,
    )

    relative_speed_art[k] = (
        np.linalg.norm(
            nu_r_k[:3]
        )
    )

    R_bw_k = prod_rotation_body_to_world(
        pose_current_art[k, 3],
        pose_current_art[k, 4],
        pose_current_art[k, 5],
    )

    world_velocity_art[k] = (
        R_bw_k
        @ nu_current_art[k, :3]
    )


current_df = trajectory_dataframe(
    t_current_art,
    pose_current_art,
    nu_current_art,
)

current_df[
    "relative_water_speed"
] = relative_speed_art

current_df[
    "world_vx"
] = world_velocity_art[:, 0]

current_df[
    "world_vy"
] = world_velocity_art[:, 1]

current_df[
    "world_vz"
] = world_velocity_art[:, 2]


current_csv = (
    artifact_dir
    / "current_capture.csv"
)

current_df.to_csv(
    current_csv,
    index=False,
)


fig = plt.figure(
    figsize=(9, 8)
)

ax1 = fig.add_subplot(
    2,
    1,
    1,
)

ax1.plot(
    t_current_art,
    relative_speed_art,
)

ax1.set_ylabel(
    "Relative-water speed [m/s]"
)

ax1.set_title(
    "Notebook 07 - Passive Current Capture"
)

ax1.grid(
    True,
    alpha=0.3,
)


ax2 = fig.add_subplot(
    2,
    1,
    2,
)

labels = [
    "world vx",
    "world vy",
    "world vz",
]

for axis in range(3):

    ax2.plot(
        t_current_art,
        world_velocity_art[
            :,
            axis
        ],
        label=labels[axis],
    )

    ax2.axhline(
        current_world_art[axis],
        linestyle="--",
        alpha=0.7,
    )


ax2.set_xlabel(
    "Time [s]"
)

ax2.set_ylabel(
    "World velocity [m/s]"
)

ax2.grid(
    True,
    alpha=0.3,
)

ax2.legend()

fig.tight_layout()

current_png = (
    artifact_dir
    / "current_capture_response.png"
)

fig.savefig(
    current_png,
    dpi=160,
    bbox_inches="tight",
)

plt.close(fig)


# =====================================================================
# Summary metrics
# =====================================================================

n_roll_art = len(
    t_roll_art
)

quarter_art = (
    n_roll_art // 4
)

early_roll_amplitude_art = np.max(
    np.abs(
        pose_roll_art[
            1:quarter_art + 1,
            3,
        ]
    )
)

late_roll_amplitude_art = np.max(
    np.abs(
        pose_roll_art[
            3 * quarter_art:,
            3,
        ]
    )
)


initial_roll_energy_art = (
    roll_energy_art[0]
)

final_roll_energy_art = (
    roll_energy_art[-1]
)


initial_relative_speed_art = (
    relative_speed_art[0]
)

final_relative_speed_art = (
    relative_speed_art[-1]
)


final_current_error_art = (
    np.linalg.norm(
        world_velocity_art[-1]
        - current_world_art
    )
)


summary_rows = [
    (
        "analytical_terminal_surge_velocity_mps",
        u_terminal_art,
    ),
    (
        "simulated_terminal_surge_velocity_mps",
        nu_forward_art[-1, 0],
    ),
    (
        "terminal_surge_velocity_error_mps",
        forward_terminal_error_art,
    ),
    (
        "final_surge_acceleration_mps2",
        final_forward_accel_art[0],
    ),
    (
        "early_roll_envelope_deg",
        np.rad2deg(
            early_roll_amplitude_art
        ),
    ),
    (
        "late_roll_envelope_deg",
        np.rad2deg(
            late_roll_amplitude_art
        ),
    ),
    (
        "initial_roll_energy_j",
        initial_roll_energy_art,
    ),
    (
        "final_roll_energy_j",
        final_roll_energy_art,
    ),
    (
        "roll_energy_reduction_percent",
        100.0
        * (
            1.0
            - final_roll_energy_art
            / initial_roll_energy_art
        ),
    ),
    (
        "initial_relative_water_speed_mps",
        initial_relative_speed_art,
    ),
    (
        "final_relative_water_speed_mps",
        final_relative_speed_art,
    ),
    (
        "relative_water_speed_reduction_percent",
        100.0
        * (
            1.0
            - final_relative_speed_art
            / initial_relative_speed_art
        ),
    ),
    (
        "final_current_velocity_error_mps",
        final_current_error_art,
    ),
]


summary_df = pd.DataFrame(
    summary_rows,
    columns=[
        "metric",
        "value",
    ],
)


summary_csv = (
    artifact_dir
    / "summary_metrics.csv"
)

summary_df.to_csv(
    summary_csv,
    index=False,
)


# =====================================================================
# Artifact validation
# =====================================================================

artifact_files = [
    forward_csv,
    roll_csv,
    current_csv,
    forward_png,
    roll_png,
    current_png,
    summary_csv,
]


for artifact in artifact_files:

    assert artifact.exists()

    assert artifact.is_file()

    assert artifact.stat().st_size > 0


assert (
    forward_terminal_error_art
    <
    0.03
)

assert (
    final_roll_energy_art
    <
    initial_roll_energy_art
)

assert (
    late_roll_amplitude_art
    <
    early_roll_amplitude_art
)

assert (
    final_relative_speed_art
    <
    0.20
    * initial_relative_speed_art
)


# =====================================================================
# Report
# =====================================================================

print(
    "=" * 72
)

print(
    "STEP 16 - NOTEBOOK 07 ARTIFACT GENERATION"
)

print(
    "=" * 72
)


print(
    "\nArtifact directory:"
)

print(
    artifact_dir.relative_to(
        repo_root
    )
)


print(
    "\nGenerated artifacts:"
)

for artifact in artifact_files:

    print(
        f"  {artifact.name:<34} "
        f"{artifact.stat().st_size:>10} bytes"
    )


print(
    "\nCanonical summary metrics:"
)

print(
    summary_df.to_string(
        index=False
    )
)


print(
    "\nForward trajectory rows:",
    len(forward_df),
)

print(
    "Roll trajectory rows   :",
    len(roll_df),
)

print(
    "Current trajectory rows:",
    len(current_df),
)


print(
    "\nPASS: all Notebook 07 artifact files were generated."
)

print(
    "PASS: artifact trajectories were generated using "
    "the promoted production simulator."
)

print(
    "PASS: production surge trajectory retains the "
    "analytical terminal-velocity result."
)

print(
    "PASS: production roll artifact retains damping "
    "and mechanical-energy decay."
)

print(
    "PASS: production current-capture artifact retains "
    "strong reduction in water-relative speed."
)

print(
    "\nNotebook 07 persistent validation artifacts are ready."
)

STEP 16 - NOTEBOOK 07 ARTIFACT GENERATION

Artifact directory:
results/notebooks/07_six_dof_uuv

Generated artifacts:
  forward_thrust.csv                     169136 bytes
  roll_recovery.csv                      589336 bytes
  current_capture.csv                    971608 bytes
  forward_thrust_response.png             96852 bytes
  roll_recovery_response.png             203862 bytes
  current_capture_response.png           110838 bytes
  summary_metrics.csv                       665 bytes

Canonical summary metrics:
                                metric        value
analytical_terminal_surge_velocity_mps 1.809975e+00
 simulated_terminal_surge_velocity_mps 1.809975e+00
     terminal_surge_velocity_error_mps 3.221193e-07
         final_surge_acceleration_mps2 2.589807e-07
               early_roll_envelope_deg 1.500000e+01
                late_roll_envelope_deg 5.006386e+00
                 initial_roll_energy_j 3.342676e-01
                   final_roll_energy_j 2.557989e-02
        

## Step 17 - Production API Lock, Notebook 07 Tests, and Full Repository Regression

The Notebook 07 implementation is now:

- analytically validated;
- behaviorally validated;
- backward-compatible with Notebooks 00–06;
- promoted into `src/hydrolab3d/`;
- independently validated in a fresh Python process;
- exercised through persistent production-generated artifacts.

The final engineering gate is repository-wide automated testing.

Before running `pytest`, this step performs one small production-interface cleanup.

---

# Public Package API

Notebook 07 added reusable capabilities to:

```text
hydrolab3d.dynamics
hydrolab3d.world
hydrolab3d.vehicles
```

The module implementations are already production-valid.

However, the package-level `__all__` declarations inherited from earlier notebooks must now be updated so the public API accurately represents the new capabilities.

This is an interface change only.

No equations, parameters, or simulation behavior are modified.

---

## `hydrolab3d.dynamics`

The public dynamics namespace should expose the existing Notebook 00–05 functions together with the validated Notebook 07 primitives:

```text
acceleration_from_force
added_mass_coriolis_matrix
added_mass_matrix
buoyancy_force
coriolis_from_block_diagonal_mass
euler_rate_matrix
gravity_force
hydrodynamic_damping_wrench
hydrostatic_wrench
integrate_position
integrate_velocity
kinematic_matrix_6dof
pose_rate_6dof
quadratic_drag_force
rigid_body_coriolis_matrix
rigid_body_mass_matrix
skew
thruster_force_body
thruster_force_world
thruster_wrench_body
total_mass_matrix
```

---

## `hydrolab3d.world`

The public world namespace should expose:

```text
current_velocity_body
generalized_current_velocity
relative_velocity_6dof
relative_water_velocity
uniform_current_velocity
```

---

## `hydrolab3d.vehicles`

The production vehicle namespace already exposes:

```text
current_generalized_acceleration
integrate_six_dof_euler
simulate_six_dof
six_dof_state_derivative
```

This package will be checked but does not require a new abstraction.

---

# Notebook 07 Automated Tests

A dedicated production test module will be created:

```text
tests/test_six_dof_uuv.py
```

These tests deliberately focus on high-information invariants rather than copying every notebook assertion.

They cover:

1. identity 6-DOF kinematics;

2. Euler singularity rejection;

3. canonical total mass matrix;

4. positive-definite generalized inertia;

5. Coriolis skew symmetry;

6. zero Coriolis generalized power;

7. dissipative damping;

8. Notebook 03 one-axis drag reduction;

9. neutral hydrostatic equilibrium;

10. passive roll-restoring moment sign;

11. body-frame current transformation;

12. Notebook 06 relative-velocity reduction;

13. thruster force-at-offset moment;

14. Notebook 02 force/acceleration reduction;

15. neutral complete-state equilibrium;

16. explicit-Euler update ordering;

17. canonical forward-thrust multi-step behavior;

18. deterministic simulation.

These tests use only production imports.

They do not depend on notebook-local functions.

---

# Full Repository Regression

After writing the production test file, run:

```text
python -m pytest -q
```

from the repository root.

The entire repository must pass.

The historical pre-Notebook-07 baseline was:

```text
95 passed
```

but the final count will **not** be predicted.

Notebook 07 adds new tests, so the actual count must come directly from the real pytest execution.

---

# Why the Entire Suite Matters

Passing only:

```text
tests/test_six_dof_uuv.py
```

would show that the new implementation works in isolation.

It would not prove that promotion preserved:

- existing kinematics;
- rigid-body dynamics;
- drag;
- buoyancy;
- thrusters;
- currents.

The repository-wide test suite is therefore the final compatibility gate.

---

# Validation Criteria

This step passes only if:

1. package-level exports contain the intended Notebook 07 API;

2. legacy public functions remain exported;

3. `tests/test_six_dof_uuv.py` is created successfully;

4. the new tests import production code only;

5. the complete pytest command exits with return code `0`;

6. no existing test fails;

7. no new Notebook 07 test fails;

8. the reported test count comes from the actual pytest output;

9. the final repository test status is preserved for the Notebook 07 conclusion.

Only after this gate passes should Notebook 07 be declared complete.

In [18]:
from pathlib import Path
import subprocess
import sys
import textwrap


# =====================================================================
# Locate repository
# =====================================================================

start = Path.cwd().resolve()

repo_root = None

for candidate in [start, *start.parents]:
    if (
        (candidate / "README.md").exists()
        and (candidate / "src").exists()
        and (candidate / "tests").exists()
    ):
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate HydroLab-3D repository root."
    )


package_root = (
    repo_root
    / "src"
    / "hydrolab3d"
)

dynamics_init_path = (
    package_root
    / "dynamics"
    / "__init__.py"
)

world_init_path = (
    package_root
    / "world"
    / "__init__.py"
)

vehicles_init_path = (
    package_root
    / "vehicles"
    / "__init__.py"
)


# =====================================================================
# 1. Lock dynamics public API
# =====================================================================

dynamics_init_source = '''\
"""Dynamics utilities for HydroLab-3D."""

from .buoyancy import (
    buoyancy_force,
    gravity_force,
)
from .drag import quadratic_drag_force
from .hydrodynamics import (
    added_mass_coriolis_matrix,
    added_mass_matrix,
    coriolis_from_block_diagonal_mass,
    hydrodynamic_damping_wrench,
    hydrostatic_wrench,
    rigid_body_coriolis_matrix,
    rigid_body_mass_matrix,
    skew,
    total_mass_matrix,
)
from .kinematics import (
    euler_rate_matrix,
    integrate_position,
    kinematic_matrix_6dof,
    pose_rate_6dof,
)
from .rigid_body import (
    acceleration_from_force,
    integrate_velocity,
)
from .thrusters import (
    thruster_force_body,
    thruster_force_world,
    thruster_wrench_body,
)

__all__ = [
    "acceleration_from_force",
    "added_mass_coriolis_matrix",
    "added_mass_matrix",
    "buoyancy_force",
    "coriolis_from_block_diagonal_mass",
    "euler_rate_matrix",
    "gravity_force",
    "hydrodynamic_damping_wrench",
    "hydrostatic_wrench",
    "integrate_position",
    "integrate_velocity",
    "kinematic_matrix_6dof",
    "pose_rate_6dof",
    "quadratic_drag_force",
    "rigid_body_coriolis_matrix",
    "rigid_body_mass_matrix",
    "skew",
    "thruster_force_body",
    "thruster_force_world",
    "thruster_wrench_body",
    "total_mass_matrix",
]
'''

dynamics_init_path.write_text(
    dynamics_init_source
)


# =====================================================================
# 2. Lock world public API
# =====================================================================

world_init_source = '''\
"""World and environmental utilities for HydroLab-3D."""

from .currents import (
    current_velocity_body,
    generalized_current_velocity,
    relative_velocity_6dof,
    relative_water_velocity,
    uniform_current_velocity,
)

__all__ = [
    "current_velocity_body",
    "generalized_current_velocity",
    "relative_velocity_6dof",
    "relative_water_velocity",
    "uniform_current_velocity",
]
'''

world_init_path.write_text(
    world_init_source
)


# =====================================================================
# 3. Verify vehicles public API file exists
# =====================================================================

if not vehicles_init_path.exists():
    raise FileNotFoundError(
        "vehicles/__init__.py is missing"
    )


# =====================================================================
# 4. Write dedicated Notebook 07 production tests
# =====================================================================

test_path = (
    repo_root
    / "tests"
    / "test_six_dof_uuv.py"
)


test_source = r'''
import numpy as np
import pytest

from hydrolab3d.dynamics.drag import (
    quadratic_drag_force,
)
from hydrolab3d.dynamics.hydrodynamics import (
    added_mass_coriolis_matrix,
    hydrodynamic_damping_wrench,
    hydrostatic_wrench,
    rigid_body_coriolis_matrix,
    total_mass_matrix,
)
from hydrolab3d.dynamics.kinematics import (
    euler_rate_matrix,
    kinematic_matrix_6dof,
)
from hydrolab3d.dynamics.thrusters import (
    thruster_wrench_body,
)
from hydrolab3d.vehicles.uuv import (
    integrate_six_dof_euler,
    simulate_six_dof,
    six_dof_state_derivative,
)
from hydrolab3d.world.currents import (
    current_velocity_body,
    relative_velocity_6dof,
    relative_water_velocity,
)


MASS = 20.0

INERTIA = np.array(
    [
        1.2,
        1.6,
        2.0,
    ]
)

ADDED = np.array(
    [
        5.0,
        8.0,
        10.0,
        0.20,
        0.30,
        0.40,
    ]
)

LINEAR = np.array(
    [
        2.0,
        3.0,
        4.0,
        0.10,
        0.15,
        0.20,
    ]
)

QUADRATIC = np.array(
    [
        5.0,
        6.0,
        7.0,
        0.30,
        0.40,
        0.50,
    ]
)

RHO = 1000.0

NEUTRAL_VOLUME = (
    MASS / RHO
)


def canonical_args():
    return {
        "mass": MASS,
        "inertia": INERTIA,
        "added_inertia": ADDED,
        "linear_damping": LINEAR,
        "quadratic_damping": QUADRATIC,
        "fluid_density": RHO,
        "displaced_volume": NEUTRAL_VOLUME,
        "center_of_gravity": np.zeros(3),
        "center_of_buoyancy": np.zeros(3),
        "current_velocity_world": np.zeros(3),
        "actuator_wrench": np.zeros(6),
    }


def test_identity_kinematic_matrix_is_identity():
    np.testing.assert_allclose(
        kinematic_matrix_6dof(
            np.zeros(6)
        ),
        np.eye(6),
        atol=1e-15,
        rtol=0.0,
    )


def test_euler_rate_matrix_rejects_pitch_singularity():
    with pytest.raises(ValueError):
        euler_rate_matrix(
            0.0,
            np.pi / 2.0,
        )


def test_canonical_total_mass_matrix():
    expected = np.diag(
        [
            25.0,
            28.0,
            30.0,
            1.4,
            1.9,
            2.4,
        ]
    )

    actual = total_mass_matrix(
        MASS,
        INERTIA,
        ADDED,
    )

    np.testing.assert_allclose(
        actual,
        expected,
        atol=1e-15,
        rtol=0.0,
    )


def test_total_mass_matrix_is_positive_definite():
    M = total_mass_matrix(
        MASS,
        INERTIA,
        ADDED,
    )

    assert np.all(
        np.linalg.eigvalsh(M) > 0.0
    )


def test_rigid_body_coriolis_is_skew_symmetric_and_power_free():
    nu = np.array(
        [
            1.2,
            -0.4,
            0.3,
            0.2,
            -0.1,
            0.15,
        ]
    )

    C = rigid_body_coriolis_matrix(
        MASS,
        INERTIA,
        nu,
    )

    np.testing.assert_allclose(
        C.T,
        -C,
        atol=1e-14,
        rtol=0.0,
    )

    assert abs(
        nu @ C @ nu
    ) < 1e-12


def test_added_mass_coriolis_is_skew_symmetric_and_power_free():
    nu_r = np.array(
        [
            0.8,
            -0.5,
            0.25,
            0.12,
            -0.08,
            0.18,
        ]
    )

    C = added_mass_coriolis_matrix(
        ADDED,
        nu_r,
    )

    np.testing.assert_allclose(
        C.T,
        -C,
        atol=1e-14,
        rtol=0.0,
    )

    assert abs(
        nu_r @ C @ nu_r
    ) < 1e-12


def test_hydrodynamic_damping_is_dissipative():
    nu_r = np.array(
        [
            1.2,
            -0.5,
            0.3,
            0.2,
            -0.1,
            0.15,
        ]
    )

    tau_d = hydrodynamic_damping_wrench(
        nu_r,
        LINEAR,
        QUADRATIC,
    )

    assert (
        nu_r @ tau_d
    ) < 0.0


def test_quadratic_damping_reduces_to_notebook03_drag():
    speed = 1.7
    coefficient = 5.0

    legacy = quadratic_drag_force(
        np.array(
            [
                speed,
                0.0,
                0.0,
            ]
        ),
        coefficient,
    )

    generalized = hydrodynamic_damping_wrench(
        np.array(
            [
                speed,
                0.0,
                0.0,
                0.0,
                0.0,
                0.0,
            ]
        ),
        np.zeros(6),
        np.array(
            [
                coefficient,
                0.0,
                0.0,
                0.0,
                0.0,
                0.0,
            ]
        ),
    )

    np.testing.assert_allclose(
        generalized[:3],
        legacy,
        atol=1e-15,
        rtol=0.0,
    )


def test_neutral_hydrostatics_are_zero_for_coincident_centers():
    wrench = hydrostatic_wrench(
        np.array(
            [
                1.0,
                -2.0,
                3.0,
                0.2,
                -0.15,
                0.4,
            ]
        ),
        MASS,
        RHO,
        NEUTRAL_VOLUME,
        np.zeros(3),
        np.zeros(3),
    )

    np.testing.assert_allclose(
        wrench,
        np.zeros(6),
        atol=1e-12,
        rtol=0.0,
    )


def test_cb_above_cg_generates_restoring_roll_moment():
    pose = np.array(
        [
            0.0,
            0.0,
            0.0,
            np.deg2rad(10.0),
            0.0,
            0.0,
        ]
    )

    wrench = hydrostatic_wrench(
        pose,
        MASS,
        RHO,
        NEUTRAL_VOLUME,
        np.zeros(3),
        np.array(
            [
                0.0,
                0.0,
                0.05,
            ]
        ),
    )

    assert wrench[3] < 0.0


def test_world_current_transforms_to_expected_body_direction():
    pose = np.array(
        [
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            np.pi / 2.0,
        ]
    )

    current_body = current_velocity_body(
        pose,
        np.array(
            [
                1.0,
                0.0,
                0.0,
            ]
        ),
    )

    np.testing.assert_allclose(
        current_body,
        np.array(
            [
                0.0,
                -1.0,
                0.0,
            ]
        ),
        atol=1e-14,
        rtol=0.0,
    )


def test_relative_velocity_reduces_to_notebook06_at_identity():
    vehicle_velocity = np.array(
        [
            1.1,
            -0.3,
            0.4,
        ]
    )

    current = np.array(
        [
            0.5,
            -0.25,
            0.1,
        ]
    )

    legacy = relative_water_velocity(
        vehicle_velocity,
        current,
    )

    generalized = relative_velocity_6dof(
        np.zeros(6),
        np.array(
            [
                1.1,
                -0.3,
                0.4,
                0.2,
                -0.1,
                0.05,
            ]
        ),
        current,
    )

    np.testing.assert_allclose(
        generalized[:3],
        legacy,
        atol=1e-15,
        rtol=0.0,
    )


def test_offset_thruster_produces_expected_pitch_moment():
    wrench = thruster_wrench_body(
        0.5,
        40.0,
        np.array(
            [
                1.0,
                0.0,
                0.0,
            ]
        ),
        np.array(
            [
                0.0,
                0.0,
                0.25,
            ]
        ),
    )

    np.testing.assert_allclose(
        wrench,
        np.array(
            [
                20.0,
                0.0,
                0.0,
                0.0,
                5.0,
                0.0,
            ]
        ),
        atol=1e-15,
        rtol=0.0,
    )


def test_complete_model_reduces_to_force_over_mass():
    _, nu_dot, _ = six_dof_state_derivative(
        np.zeros(6),
        np.zeros(6),
        mass=MASS,
        inertia=INERTIA,
        added_inertia=np.zeros(6),
        linear_damping=np.zeros(6),
        quadratic_damping=np.zeros(6),
        fluid_density=RHO,
        displaced_volume=NEUTRAL_VOLUME,
        center_of_gravity=np.zeros(3),
        center_of_buoyancy=np.zeros(3),
        current_velocity_world=np.zeros(3),
        actuator_wrench=np.array(
            [
                12.0,
                0.0,
                0.0,
                0.0,
                0.0,
                0.0,
            ]
        ),
    )

    np.testing.assert_allclose(
        nu_dot,
        np.array(
            [
                0.6,
                0.0,
                0.0,
                0.0,
                0.0,
                0.0,
            ]
        ),
        atol=1e-14,
        rtol=0.0,
    )


def test_complete_neutral_equilibrium_is_zero():
    eta_dot, nu_dot, _ = (
        six_dof_state_derivative(
            np.zeros(6),
            np.zeros(6),
            **canonical_args(),
        )
    )

    np.testing.assert_allclose(
        eta_dot,
        np.zeros(6),
        atol=1e-14,
        rtol=0.0,
    )

    np.testing.assert_allclose(
        nu_dot,
        np.zeros(6),
        atol=1e-14,
        rtol=0.0,
    )


def test_explicit_euler_uses_beginning_of_step_velocity():
    args = canonical_args()

    args["added_inertia"] = np.zeros(6)
    args["linear_damping"] = np.zeros(6)
    args["quadratic_damping"] = np.zeros(6)

    args["actuator_wrench"] = np.array(
        [
            12.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
        ]
    )

    pose_next, nu_next, _ = (
        integrate_six_dof_euler(
            np.zeros(6),
            np.zeros(6),
            0.1,
            **args,
        )
    )

    np.testing.assert_allclose(
        pose_next,
        np.zeros(6),
        atol=1e-15,
        rtol=0.0,
    )

    np.testing.assert_allclose(
        nu_next,
        np.array(
            [
                0.06,
                0.0,
                0.0,
                0.0,
                0.0,
                0.0,
            ]
        ),
        atol=1e-15,
        rtol=0.0,
    )


def test_forward_thrust_simulation_approaches_analytical_terminal_speed():
    args = canonical_args()

    args["actuator_wrench"] = (
        thruster_wrench_body(
            0.5,
            40.0,
            np.array(
                [
                    1.0,
                    0.0,
                    0.0,
                ]
            ),
            np.zeros(3),
        )
    )

    times, poses, velocities = (
        simulate_six_dof(
            np.zeros(6),
            np.zeros(6),
            20.0,
            0.01,
            args,
        )
    )

    expected_terminal = (
        -2.0 + np.sqrt(404.0)
    ) / 10.0

    assert len(times) == 2001

    assert abs(
        velocities[-1, 0]
        - expected_terminal
    ) < 0.03

    assert np.all(
        np.isfinite(poses)
    )

    assert np.all(
        np.isfinite(velocities)
    )


def test_simulation_is_deterministic():
    args = canonical_args()

    args["actuator_wrench"] = (
        thruster_wrench_body(
            0.25,
            40.0,
            np.array(
                [
                    1.0,
                    0.0,
                    0.0,
                ]
            ),
            np.zeros(3),
        )
    )

    result_a = simulate_six_dof(
        np.zeros(6),
        np.zeros(6),
        1.0,
        0.01,
        args,
    )

    result_b = simulate_six_dof(
        np.zeros(6),
        np.zeros(6),
        1.0,
        0.01,
        args,
    )

    for array_a, array_b in zip(
        result_a,
        result_b,
    ):
        np.testing.assert_array_equal(
            array_a,
            array_b,
        )
'''

test_path.write_text(
    textwrap.dedent(
        test_source
    ).lstrip()
)


# =====================================================================
# 5. Fresh-process public API check
# =====================================================================

api_check_script = r'''
import hydrolab3d.dynamics as dynamics
import hydrolab3d.world as world
import hydrolab3d.vehicles as vehicles

required_dynamics = {
    "acceleration_from_force",
    "added_mass_coriolis_matrix",
    "added_mass_matrix",
    "buoyancy_force",
    "coriolis_from_block_diagonal_mass",
    "euler_rate_matrix",
    "gravity_force",
    "hydrodynamic_damping_wrench",
    "hydrostatic_wrench",
    "integrate_position",
    "integrate_velocity",
    "kinematic_matrix_6dof",
    "pose_rate_6dof",
    "quadratic_drag_force",
    "rigid_body_coriolis_matrix",
    "rigid_body_mass_matrix",
    "skew",
    "thruster_force_body",
    "thruster_force_world",
    "thruster_wrench_body",
    "total_mass_matrix",
}

required_world = {
    "current_velocity_body",
    "generalized_current_velocity",
    "relative_velocity_6dof",
    "relative_water_velocity",
    "uniform_current_velocity",
}

required_vehicles = {
    "current_generalized_acceleration",
    "integrate_six_dof_euler",
    "simulate_six_dof",
    "six_dof_state_derivative",
}

assert required_dynamics.issubset(
    set(dynamics.__all__)
)

assert required_world.issubset(
    set(world.__all__)
)

assert required_vehicles.issubset(
    set(vehicles.__all__)
)

for name in required_dynamics:
    assert callable(
        getattr(
            dynamics,
            name,
        )
    )

for name in required_world:
    assert callable(
        getattr(
            world,
            name,
        )
    )

for name in required_vehicles:
    assert callable(
        getattr(
            vehicles,
            name,
        )
    )

print("PUBLIC_API_PASS")
print(
    "dynamics exports:",
    len(dynamics.__all__),
)
print(
    "world exports:",
    len(world.__all__),
)
print(
    "vehicles exports:",
    len(vehicles.__all__),
)
'''


env = __import__("os").environ.copy()

src_path = str(
    repo_root / "src"
)

existing_pythonpath = env.get(
    "PYTHONPATH",
    "",
)

if existing_pythonpath:
    env["PYTHONPATH"] = (
        src_path
        + __import__("os").pathsep
        + existing_pythonpath
    )
else:
    env["PYTHONPATH"] = src_path


api_result = subprocess.run(
    [
        sys.executable,
        "-c",
        api_check_script,
    ],
    cwd=repo_root,
    env=env,
    capture_output=True,
    text=True,
)


if api_result.returncode != 0:

    print(
        "PUBLIC API CHECK STDOUT:"
    )
    print(
        api_result.stdout
    )

    print(
        "PUBLIC API CHECK STDERR:"
    )
    print(
        api_result.stderr
    )

    raise RuntimeError(
        "Production public API validation failed."
    )


# =====================================================================
# 6. Run entire repository test suite
# =====================================================================

pytest_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
    ],
    cwd=repo_root,
    env=env,
    capture_output=True,
    text=True,
)


# Always show pytest output before deciding pass/fail.
print(
    "=" * 72
)

print(
    "STEP 17 - PUBLIC API AND FULL REPOSITORY REGRESSION"
)

print(
    "=" * 72
)


print(
    "\nUpdated production API files:"
)

print(
    " ",
    dynamics_init_path.relative_to(
        repo_root
    ),
)

print(
    " ",
    world_init_path.relative_to(
        repo_root
    ),
)

print(
    " ",
    vehicles_init_path.relative_to(
        repo_root
    ),
)


print(
    "\nNotebook 07 production test file:"
)

print(
    " ",
    test_path.relative_to(
        repo_root
    ),
)

print(
    "  size:",
    test_path.stat().st_size,
    "bytes",
)


print(
    "\nFresh-process public API check:"
)

print(
    api_result.stdout.strip()
)


print(
    "\nFull pytest output:"
)

print(
    pytest_result.stdout.strip()
)

if pytest_result.stderr.strip():

    print(
        "\nPytest STDERR:"
    )

    print(
        pytest_result.stderr.strip()
    )


print(
    "\nPytest return code:",
    pytest_result.returncode,
)


if pytest_result.returncode != 0:

    raise RuntimeError(
        "Full repository pytest regression failed. "
        "Do not proceed to the Notebook 07 conclusion."
    )


print(
    "\nPASS: Notebook 07 public production API is locked."
)

print(
    "PASS: dedicated Notebook 07 production tests are installed."
)

print(
    "PASS: the entire HydroLab-3D pytest suite passes."
)

print(
    "\nUse the ACTUAL pytest count printed above "
    "as the final Notebook 07 regression count."
)

STEP 17 - PUBLIC API AND FULL REPOSITORY REGRESSION

Updated production API files:
  src/hydrolab3d/dynamics/__init__.py
  src/hydrolab3d/world/__init__.py
  src/hydrolab3d/vehicles/__init__.py

Notebook 07 production test file:
  tests/test_six_dof_uuv.py
  size: 11743 bytes

Fresh-process public API check:
PUBLIC_API_PASS
dynamics exports: 21
world exports: 5
vehicles exports: 4

Full pytest output:
........................................................................ [ 63%]
.........................................                                [100%]
113 passed in 0.68s

Pytest return code: 0

PASS: Notebook 07 public production API is locked.
PASS: dedicated Notebook 07 production tests are installed.
PASS: the entire HydroLab-3D pytest suite passes.

Use the ACTUAL pytest count printed above as the final Notebook 07 regression count.


# Conclusion

Notebook 07 completed the transition from the earlier reduced-order HydroLab-3D building blocks to a complete, lightweight, current-aware **six-degree-of-freedom UUV dynamics model**.

The result is not a CFD simulator and is not intended to reproduce every hydrodynamic effect experienced by a real underwater vehicle.

Instead, it establishes a compact but physically structured marine-robotics dynamics layer that preserves the design philosophy of HydroLab-3D:

> **algorithm-first, transparent, deterministic, selectively faithful, and suitable for autonomy/control development.**

The completed model combines:

```text
6-DOF rigid-body kinematics
        +
rigid-body inertia
        +
added mass
        +
rigid-body Coriolis effects
        +
added-mass Coriolis effects
        +
linear + quadratic six-axis damping
        +
gravity
        +
buoyancy
        +
CG / CB restoring moments
        +
body-frame thruster wrench mapping
        +
world-frame ocean currents
        +
water-relative hydrodynamics
        +
fixed-step deterministic integration
```

into one reusable UUV simulation stack.

---

## Final State Representation

The vehicle pose is represented as:

$$
\boxed{
\eta
=
\begin{bmatrix}
x &
y &
z &
\phi &
\theta &
\psi
\end{bmatrix}^{T}
}
$$

where:

- $x,y,z$ are world-frame position coordinates;
- $\phi$ is roll;
- $\theta$ is pitch;
- $\psi$ is yaw.

The generalized body velocity is:

$$
\boxed{
\nu
=
\begin{bmatrix}
u &
v &
w &
p &
q &
r
\end{bmatrix}^{T}
}
$$

where:

- $u,v,w$ are body-frame translational velocities;
- $p,q,r$ are body-frame angular velocities.

The complete simulator state therefore contains twelve dynamic quantities:

$$
\boxed{
x_{state}
=
\begin{bmatrix}
\eta\\
\nu
\end{bmatrix}
\in\mathbb{R}^{12}
}
$$

---

# 1. Six-DOF Kinematics

The final production kinematic relation is:

$$
\boxed{
\dot{\eta}
=
J(\eta)\nu
}
$$

with:

$$
J(\eta)
=
\begin{bmatrix}
R_B^W & 0\\
0 & T(\phi,\theta)
\end{bmatrix}
$$

The translational block uses the same validated Z-Y-X body-to-world rotation convention introduced in Notebook 01:

$$
\boxed{
R_B^W
=
R_z(\psi)
R_y(\theta)
R_x(\phi)
}
$$

The Euler-angle-rate transformation is:

$$
T(\phi,\theta)
=
\begin{bmatrix}
1 &
\sin\phi\tan\theta &
\cos\phi\tan\theta
\\
0 &
\cos\phi &
-\sin\phi
\\
0 &
\frac{\sin\phi}{\cos\theta} &
\frac{\cos\phi}{\cos\theta}
\end{bmatrix}
$$

This maps:

$$
\begin{bmatrix}
p\\
q\\
r
\end{bmatrix}
$$

into:

$$
\begin{bmatrix}
\dot{\phi}\\
\dot{\theta}\\
\dot{\psi}
\end{bmatrix}
$$

A guard rejects states sufficiently close to the Z-Y-X singularity:

$$
\theta=\pm\frac{\pi}{2}
$$

rather than allowing undefined Euler-rate behavior to propagate silently.

---

# 2. Rigid-Body and Added-Mass Inertia

The initial HydroLab-3D UUV model uses a CG-centered diagonal rigid-body inertia:

$$
M_{RB}
=
\operatorname{diag}
\left(
m,\,
m,\,
m,\,
I_x,\,
I_y,\,
I_z
\right)
$$

Added mass is represented using positive effective inertia magnitudes:

$$
M_A
=
\operatorname{diag}
\left(
m_{A,u},
m_{A,v},
m_{A,w},
m_{A,p},
m_{A,q},
m_{A,r}
\right)
$$

so the total generalized inertia is:

$$
\boxed{
M=M_{RB}+M_A
}
$$

For the canonical Notebook 07 vehicle:

$$
m=20\text{ kg}
$$

$$
I
=
\begin{bmatrix}
1.2 &
1.6 &
2.0
\end{bmatrix}
$$

and:

$$
M_A
=
\operatorname{diag}
\left(
5,\,
8,\,
10,\,
0.2,\,
0.3,\,
0.4
\right)
$$

the validated total mass matrix is:

$$
\boxed{
M
=
\operatorname{diag}
\left(
25,\,
28,\,
30,\,
1.4,\,
1.9,\,
2.4
\right)
}
$$

The matrix remained positive definite and well-conditioned for the canonical model.

---

# 3. Coriolis and Centripetal Effects

Notebook 07 introduced generalized rigid-body and added-mass Coriolis terms.

For the selected block-diagonal inertia representation, the Coriolis matrix is assembled from generalized momentum.

If:

$$
M\nu
=
\begin{bmatrix}
\mathbf{p}\\
\mathbf{h}
\end{bmatrix}
$$

then:

$$
C(\nu)
=
\begin{bmatrix}
0 &
-S(\mathbf{p})
\\
-S(\mathbf{p}) &
-S(\mathbf{h})
\end{bmatrix}
$$

where:

$$
S(\mathbf{a})\mathbf{b}
=
\mathbf{a}\times\mathbf{b}
$$

The implementation was validated using the energy property:

$$
\boxed{
\nu^T C(\nu)\nu=0
}
$$

within numerical tolerance.

This confirms that the Coriolis terms redistribute generalized momentum but do not create or destroy mechanical energy.

The same formulation is used for added-mass Coriolis behavior with:

$$
\nu_r
$$

rather than absolute vehicle velocity.

---

# 4. Water-Relative Motion

A central modeling principle preserved from Notebook 06 is that hydrodynamic behavior depends on **vehicle motion relative to the surrounding water**, not merely relative to the world frame.

The ocean current is specified in world coordinates:

$$
\mathbf{v}_{c,W}
$$

and transformed into the body frame using:

$$
\boxed{
\mathbf{v}_{c,B}
=
(R_B^W)^T
\mathbf{v}_{c,W}
}
$$

The generalized current velocity is:

$$
\nu_c
=
\begin{bmatrix}
\mathbf{v}_{c,B}\\
0\\
0\\
0
\end{bmatrix}
$$

because the current model contains no ambient rotational fluid velocity.

The water-relative generalized velocity is then:

$$
\boxed{
\nu_r
=
\nu-\nu_c
}
$$

This means a vehicle moving exactly with the surrounding water experiences:

$$
\nu_r=0
$$

and therefore no translational drag simply because it is moving relative to the world.

That behavior was explicitly validated.

---

# 5. Current-Aware Added-Mass Correction

A world-fixed uniform current is not necessarily constant when observed from a rotating body frame.

Even if:

$$
\dot{\mathbf{v}}_{c,W}=0
$$

the body-frame current obeys:

$$
\boxed{
\dot{\mathbf{v}}_{c,B}
=
-\boldsymbol{\omega}
\times
\mathbf{v}_{c,B}
}
$$

where:

$$
\boldsymbol{\omega}
=
\begin{bmatrix}
p\\
q\\
r
\end{bmatrix}
$$

Therefore:

$$
\dot{\nu}_c
=
\begin{bmatrix}
-\boldsymbol{\omega}
\times
\mathbf{v}_{c,B}
\\
0\\
0\\
0
\end{bmatrix}
$$

The selected current-aware added-mass equation begins from:

$$
M_{RB}\dot{\nu}
+
M_A\dot{\nu}_r
+
C_{RB}(\nu)\nu
+
C_A(\nu_r)\nu_r
=
\tau_{ext}
$$

with:

$$
\dot{\nu}_r
=
\dot{\nu}
-
\dot{\nu}_c
$$

which gives:

$$
\boxed{
M\dot{\nu}
=
\tau_{ext}
-
C_{RB}(\nu)\nu
-
C_A(\nu_r)\nu_r
+
M_A\dot{\nu}_c
}
$$

This term captures the body-frame variation of a spatially uniform inertial current without introducing spatial current gradients or full fluid-field dynamics.

---

# 6. Six-Axis Hydrodynamic Damping

The generalized hydrodynamic damping model is:

$$
\boxed{
\tau_D
=
-d_L\odot\nu_r
-
d_Q\odot|\nu_r|\odot\nu_r
}
$$

where:

- $d_L$ contains linear damping coefficients;
- $d_Q$ contains quadratic damping coefficients;
- $\odot$ denotes component-wise multiplication.

The returned quantity is the **physical opposing wrench** itself.

Therefore:

$$
\nu_r^T\tau_D\le0
$$

for nonnegative damping coefficients.

This was validated numerically.

The model also reduces exactly to the Notebook 03 drag law in a one-axis case:

$$
F_D
=
-C_D|u|u
$$

when:

$$
d_{L,u}=0
$$

and:

$$
d_{Q,u}=C_D
$$

The Notebook 03 and Notebook 07 models are intentionally not claimed to be globally identical for arbitrary simultaneous multi-axis motion.

Notebook 03 uses a norm-coupled vector drag model, while Notebook 07 uses axis-wise six-DOF damping.

---

# 7. Gravity, Buoyancy, and Restoring Moments

Gravity and buoyancy continue to use the validated Notebook 04 production functions.

In world coordinates:

$$
\mathbf{F}_{G,W}
=
\begin{bmatrix}
0\\
0\\
-mg
\end{bmatrix}
$$

and:

$$
\mathbf{F}_{B,W}
=
\begin{bmatrix}
0\\
0\\
\rho gV
\end{bmatrix}
$$

These forces are transformed into body coordinates:

$$
\mathbf{F}_{G,B}
=
(R_B^W)^T
\mathbf{F}_{G,W}
$$

$$
\mathbf{F}_{B,B}
=
(R_B^W)^T
\mathbf{F}_{B,W}
$$

The total hydrostatic body wrench is:

$$
\boxed{
\tau_H
=
\begin{bmatrix}
\mathbf{F}_{G,B}
+
\mathbf{F}_{B,B}
\\
\mathbf{r}_G
\times
\mathbf{F}_{G,B}
+
\mathbf{r}_B
\times
\mathbf{F}_{B,B}
\end{bmatrix}
}
$$

This supports:

- positive buoyancy;
- neutral buoyancy;
- negative buoyancy;
- CG/CB offset restoring moments.

For a neutrally buoyant vehicle with the center of buoyancy above the center of gravity, a positive roll angle generated a negative restoring roll moment exactly as expected.

---

# 8. Generalized Thruster Wrench

Notebook 05's production body-force actuator model was preserved.

The validated body-frame thrust:

$$
\mathbf{F}_{T,B}
$$

is converted into a six-component generalized wrench using:

$$
\boxed{
\tau_T
=
\begin{bmatrix}
\mathbf{F}_{T,B}
\\
\mathbf{r}_T
\times
\mathbf{F}_{T,B}
\end{bmatrix}
}
$$

This allows actuator placement to produce physically meaningful moments.

For example, the canonical test used:

$$
\mathbf{F}_{T,B}
=
\begin{bmatrix}
20\\
0\\
0
\end{bmatrix}
\text{ N}
$$

at:

$$
\mathbf{r}_T
=
\begin{bmatrix}
0\\
0\\
0.25
\end{bmatrix}
\text{ m}
$$

giving:

$$
\boxed{
\mathbf{M}_T
=
\begin{bmatrix}
0\\
5\\
0
\end{bmatrix}
\text{ N m}
}
$$

Notebook 05 command saturation remains intact because Notebook 07 wraps the validated production thrust function rather than replacing it.

---

# 9. Complete Dynamic Equation

The final production acceleration model is:

$$
\boxed{
\dot{\nu}
=
M^{-1}
\left[
\tau_{act}
+
\tau_H
+
\tau_D
+
\tau_{env}
-
C_{RB}(\nu)\nu
-
C_A(\nu_r)\nu_r
+
M_A\dot{\nu}_c
\right]
}
$$

where:

- $\tau_{act}$ is the physical actuator wrench;
- $\tau_H$ is the physical gravity/buoyancy wrench;
- $\tau_D$ is the physical damping wrench;
- $\tau_{env}$ is an optional external disturbance wrench.

No extra sign reversal is applied to hydrostatics or damping because both functions already return forces and moments with their physical directions.

The complete state derivative is:

$$
\boxed{
\dot{x}_{state}
=
\begin{bmatrix}
J(\eta)\nu
\\
\dot{\nu}
\end{bmatrix}
}
$$

---

# 10. Numerical Integration

HydroLab-3D continues to use deterministic explicit Euler integration.

At step $k$:

$$
\eta_{k+1}
=
\eta_k
+
\dot{\eta}_k\Delta t
$$

and:

$$
\nu_{k+1}
=
\nu_k
+
\dot{\nu}_k\Delta t
$$

Both derivatives are evaluated from the same beginning-of-step state.

This preserves the numerical philosophy established in Notebooks 00 and 02.

The ordering was explicitly validated.

For:

$$
F_x=12\text{ N}
$$

$$
m=20\text{ kg}
$$

$$
\Delta t=0.1\text{ s}
$$

the first step from rest produced:

$$
u_1=0.06\text{ m/s}
$$

while:

$$
x_1=0
$$

because the beginning-of-step velocity was zero.

The second step then produced:

$$
x_2=0.006\text{ m}
$$

and:

$$
u_2=0.12\text{ m/s}
$$

exactly matching the inherited explicit-Euler update convention.

---

# 11. Canonical Multi-Step Experiment Results

Three production-level canonical experiments were used to validate the coupled model.

---

## Experiment A - Forward Thrust

The vehicle began:

- neutrally buoyant;
- at rest;
- at zero orientation;
- in still water.

A through-origin forward thruster produced:

$$
20\text{ N}
$$

of constant body-$x$ thrust.

The analytical steady-state surge equation was:

$$
20-2u-5u^2=0
$$

with positive solution:

$$
\boxed{
u_{\infty}
=
1.8099751242\text{ m/s}
}
$$

The production simulation produced:

$$
\boxed{
u_f
=
1.8099748021\text{ m/s}
}
$$

with terminal error:

$$
\boxed{
3.221\times10^{-7}\text{ m/s}
}
$$

The final surge acceleration was:

$$
2.590\times10^{-7}\text{ m/s}^2
$$

and the final damping force was approximately:

$$
-19.9999935\text{ N}
$$

which nearly exactly balances the applied:

$$
+20\text{ N}
$$

thrust.

The final position after 20 s was approximately:

$$
\begin{bmatrix}
33.2051\\
0\\
0
\end{bmatrix}
\text{ m}
$$

There was:

- zero sway velocity;
- zero heave velocity;
- zero unintended orientation change.

This confirmed the expected surge force balance and axis consistency.

---

## Experiment B - Passive Roll Recovery

The vehicle began at:

$$
\phi_0=15^\circ
$$

with:

$$
\mathbf{r}_B
=
\begin{bmatrix}
0\\
0\\
0.05
\end{bmatrix}
\text{ m}
$$

and:

$$
\mathbf{r}_G=0
$$

The initial restoring moment was:

$$
\boxed{
K_H=-2.5390\text{ N m}
}
$$

and the initial roll acceleration was:

$$
\boxed{
\dot{p}=-1.8136\text{ rad/s}^2
}
$$

so the dynamics immediately accelerated the vehicle toward upright equilibrium.

The response was lightly damped and oscillatory.

Observed bounds were approximately:

$$
\phi_{max}=15^\circ
$$

$$
\phi_{min}=-13.94^\circ
$$

The late roll envelope decreased to approximately:

$$
\boxed{
5.006^\circ
}
$$

from the initial:

$$
15^\circ
$$

disturbance.

The roll mechanical energy decreased from:

$$
0.3343\text{ J}
$$

to:

$$
0.02558\text{ J}
$$

corresponding to an overall reduction of approximately:

$$
\boxed{
92.35\%
}
$$

The final-quarter mean roll energy was only approximately:

$$
15.58\%
$$

of the first-quarter mean energy.

No pitch or yaw instability appeared.

This demonstrated bounded passive restoring behavior rather than merely satisfying an arbitrary final-angle condition.

---

## Experiment C - Passive Current Capture

The vehicle began stationary in the uniform world-frame current:

$$
\mathbf{v}_{c,W}
=
\begin{bmatrix}
0.50\\
-0.25\\
0.10
\end{bmatrix}
\text{ m/s}
$$

The initial relative-water speed was:

$$
\boxed{
0.56789\text{ m/s}
}
$$

Hydrodynamic forces accelerated the passive UUV toward the moving water.

After 30 s, the relative-water speed had fallen to:

$$
\boxed{
0.01786\text{ m/s}
}
$$

corresponding to a reduction of approximately:

$$
\boxed{
96.86\%
}
$$

The final world-frame vehicle velocity was approximately:

$$
\begin{bmatrix}
0.48797\\
-0.23680\\
0.09981
\end{bmatrix}
\text{ m/s}
$$

compared with the imposed current:

$$
\begin{bmatrix}
0.50\\
-0.25\\
0.10
\end{bmatrix}
\text{ m/s}
$$

The final velocity mismatch was:

$$
\boxed{
0.01786\text{ m/s}
}
$$

This demonstrates the intended Notebook 06 inheritance:

> the passive vehicle tends toward the water velocity because hydrodynamic forces depend on relative motion.

---

# 12. Backward Compatibility With Notebooks 00-06

Notebook 07 was explicitly reduced back to each earlier validated physical layer.

The following regressions passed:

### Notebook 00

6-DOF translational kinematics reduced exactly to:

```text
integrate_position()
```

### Notebook 01

The translational block of the 6-DOF kinematic matrix reproduced:

```text
rotation_body_to_world()
```

with the same orientation convention.

### Notebook 02

With higher-order marine effects disabled:

$$
\dot{\mathbf{v}}
=
\frac{\mathbf{F}}{m}
$$

and the explicit velocity update reproduced:

```text
acceleration_from_force()
integrate_velocity()
```

exactly.

### Notebook 03

Pure-axis Notebook 07 quadratic damping reproduced:

```text
quadratic_drag_force()
```

for both positive and negative velocities.

### Notebook 04

Notebook 07 hydrostatics reproduced the production gravity and buoyancy force balance for:

- positive buoyancy;
- neutral buoyancy;
- negative buoyancy.

### Notebook 05

Generalized through-origin thruster wrenches preserved:

```text
thruster_force_body()
```

including command saturation.

### Notebook 06

At identity orientation, the generalized current model reproduced:

```text
relative_water_velocity()
```

exactly.

A combined translational regression also produced the same complete legacy net force:

$$
\boxed{
\begin{bmatrix}
15.95\\
0\\
9.81
\end{bmatrix}
\text{ N}
}
$$

and acceleration:

$$
\boxed{
\begin{bmatrix}
0.7975\\
0\\
0.4905
\end{bmatrix}
\text{ m/s}^2
}
$$

from both the legacy production chain and the generalized Notebook 07 model.

---

# 13. Production Architecture

The validated functionality was promoted into the modular production package.

The final relevant source organization is:

```text
src/hydrolab3d/
│
├── dynamics/
│   ├── kinematics.py
│   ├── rigid_body.py
│   ├── drag.py
│   ├── buoyancy.py
│   ├── thrusters.py
│   └── hydrodynamics.py
│
├── world/
│   └── currents.py
│
└── vehicles/
    └── uuv.py
```

Notebook 07 added reusable production capabilities across these existing conceptual boundaries instead of creating one monolithic dynamics file.

The promoted public APIs include:

### `hydrolab3d.dynamics`

```text
euler_rate_matrix
kinematic_matrix_6dof
pose_rate_6dof
rigid_body_mass_matrix
added_mass_matrix
total_mass_matrix
rigid_body_coriolis_matrix
added_mass_coriolis_matrix
hydrodynamic_damping_wrench
hydrostatic_wrench
thruster_wrench_body
```

alongside the existing Notebook 00–05 production functions.

### `hydrolab3d.world`

```text
current_velocity_body
generalized_current_velocity
relative_velocity_6dof
```

alongside the existing Notebook 06 functions.

### `hydrolab3d.vehicles`

```text
current_generalized_acceleration
six_dof_state_derivative
integrate_six_dof_euler
simulate_six_dof
```

The tentative:

```text
core/state.py
```

was deliberately left empty.

A state-class abstraction was not introduced because the current implementation does not require one.

This avoids unnecessary architectural complexity before a concrete simulator-kernel requirement exists.

---

# 14. Fresh-Process Validation

After production promotion, cached HydroLab-3D modules were removed from the notebook process and the production implementation was imported again under explicit aliases.

The promoted functions resolved to:

```text
src/hydrolab3d/dynamics/kinematics.py
src/hydrolab3d/dynamics/hydrodynamics.py
src/hydrolab3d/dynamics/thrusters.py
src/hydrolab3d/world/currents.py
src/hydrolab3d/vehicles/uuv.py
```

A completely new Python interpreter was then launched.

It independently imported:

```text
/home/agniacolyte/HydroLab-3D/src/hydrolab3d/
```

and executed the promoted UUV implementation successfully.

The fresh process exited with:

```text
return code 0
```

This verifies that the successful Notebook 07 behavior does not depend on stale notebook-local functions or Jupyter kernel state.

---

# 15. Persistent Validation Artifacts

Production-generated validation artifacts were saved under:

```text
results/notebooks/07_six_dof_uuv/
```

The final artifact set is:

```text
forward_thrust.csv
roll_recovery.csv
current_capture.csv

forward_thrust_response.png
roll_recovery_response.png
current_capture_response.png

summary_metrics.csv
```

The trajectory sizes are:

```text
Forward thrust : 2001 samples
Roll recovery  : 4001 samples
Current capture: 3001 samples
```

The summary metrics preserve the principal quantitative Notebook 07 results:

```text
analytical terminal surge velocity
simulated terminal surge velocity
terminal surge error
final surge acceleration
early roll envelope
late roll envelope
initial roll mechanical energy
final roll mechanical energy
roll energy reduction
initial relative-water speed
final relative-water speed
relative-water-speed reduction
final current velocity error
```

These artifacts provide compact persistent evidence without making high-fidelity rendering or animation a requirement of the core simulator.

---

# 16. Automated Regression Suite

Notebook 07 added a dedicated production test module:

```text
tests/test_six_dof_uuv.py
```

The new tests cover high-information invariants including:

- identity kinematics;
- Euler singularity rejection;
- canonical generalized inertia;
- positive-definite mass matrix;
- Coriolis skew symmetry;
- zero Coriolis generalized power;
- dissipative hydrodynamic damping;
- Notebook 03 drag reduction;
- neutral hydrostatics;
- CG/CB restoring behavior;
- current-coordinate transformation;
- Notebook 06 relative-water reduction;
- offset-thruster moment generation;
- Notebook 02 force-over-mass reduction;
- complete neutral equilibrium;
- explicit-Euler ordering;
- canonical multi-step forward-thrust behavior;
- deterministic simulation.

The historical repository baseline before Notebook 07 was:

```text
95 passed
```

After Notebook 07, the complete repository regression produced:

```text
113 passed in 0.68s
```

Therefore Notebook 07 added:

$$
\boxed{
18
}
$$

automated tests while preserving every previously passing test.

The final repository status is:

$$
\boxed{
113/113\text{ tests passed}
}
$$

with:

```text
0 failures
pytest return code: 0
```

---

# 17. Model Scope and Deliberate Limitations

The completed model is intentionally selective.

It currently includes:

```text
rigid-body 6-DOF motion
added mass
Coriolis / centripetal effects
axis-wise hydrodynamic damping
gravity
buoyancy
CG/CB restoring moments
thruster force + mounting moment
uniform inertial-frame ocean current
current-relative dynamics
deterministic fixed-step simulation
```

It does **not** yet include:

```text
CFD
wave dynamics
free-surface effects
spatially varying currents
time-varying current fields
lift surfaces
cross-flow drag
full hydrodynamic derivative matrices
non-diagonal added mass
non-diagonal damping
propeller inflow dynamics
thruster-thruster interaction
thruster-hull interaction
cavitation
collision dynamics
seabed contact
tethers
sonar
DVL
IMU
pressure/depth sensors
sensor noise
state estimation
closed-loop control design
communication models
multi-UUV coordination
ROS 2 interfaces
reinforcement learning APIs
real-time rendering
```

These omissions are deliberate rather than accidental.

Notebook 07 establishes the vehicle-dynamics substrate on which those systems can later be built.

---

# 18. Numerical and Representation Limitations

The current orientation representation uses Z-Y-X Euler angles.

Therefore the simulator inherits the classical pitch singularity near:

$$
\theta=\pm90^\circ
$$

The production model detects the singularity but does not remove it.

A future version may introduce quaternion orientation while preserving the same external dynamics structure.

The simulator also currently uses fixed-step explicit Euler integration.

This is simple, transparent, and consistent with the earlier HydroLab-3D notebooks, but it is only first-order accurate.

Future work may compare:

```text
explicit Euler
semi-implicit Euler
RK2
RK4
adaptive integration
```

without changing the physical model itself.

For the present canonical scenarios:

$$
\Delta t=0.01\text{ s}
$$

produced stable and physically coherent trajectories.

---

# 19. What Notebook 07 Establishes

Before Notebook 07, HydroLab-3D had individually validated ingredients:

```text
position integration
orientation transforms
F = ma dynamics
drag
gravity
buoyancy
thrusters
ocean currents
```

After Notebook 07, these are no longer isolated demonstrations.

They now form a connected UUV dynamics system:

```text
              WORLD
                │
                │ current v_c,W
                ▼
      ┌─────────────────────┐
      │ body/current transform│
      └──────────┬──────────┘
                 │
                 ▼
              nu_r
                 │
       ┌─────────┴──────────┐
       │                    │
       ▼                    ▼
   damping             added-mass
                         Coriolis
       │                    │
       └─────────┬──────────┘
                 │
                 ▼
        generalized dynamics
                 ▲
       ┌─────────┼───────────┐
       │         │           │
       │         │           │
    thrusters hydrostatics rigid-body
                             Coriolis
                 │
                 ▼
              nu_dot
                 │
                 ▼
         explicit integration
                 │
          ┌──────┴──────┐
          ▼             ▼
         nu            eta
                         │
                         ▼
                   world motion
```

That transition is the central result of this notebook.

HydroLab-3D now has a reusable dynamic vehicle instead of a collection of disconnected physics primitives.

---

# 20. Readiness for Notebook 08

Notebook 07 intentionally stops before sensor simulation.

The next notebook can therefore treat the UUV's true state:

$$
\eta(t),\nu(t)
$$

as the latent physical state of the environment and construct sensor observations from it.

Notebook 08 can introduce sensor models such as:

```text
IMU
depth / pressure
DVL
heading
position fixes
possibly basic range sensing
```

while keeping a strict distinction between:

```text
ground-truth simulator state
```

and:

```text
measured / noisy sensor state
```

This is an important architectural transition.

Notebook 07 answers:

> **How does the underwater vehicle physically move?**

Notebook 08 can now ask:

> **What does the vehicle believe or measure about that motion?**

---

# Final Result

Notebook 07 successfully implemented, validated, promoted, and regression-tested a lightweight current-aware 6-DOF UUV dynamics model for HydroLab-3D.

The final system demonstrates:

$$
\boxed{
\text{kinematics}
+
\text{marine dynamics}
+
\text{actuation}
+
\text{hydrostatics}
+
\text{current interaction}
+
\text{deterministic simulation}
}
$$

while maintaining exact reduced-case compatibility with the validated physical lineage of Notebooks 00–06.

The canonical production results include:

$$
\boxed{
u_{\infty}
=
1.8099748\text{ m/s}
}
$$

for the 20 N forward-thrust experiment,

approximately:

$$
\boxed{
92.35\%
}
$$

roll mechanical-energy reduction during passive hydrostatic recovery,

and approximately:

$$
\boxed{
96.86\%
}
$$

reduction in water-relative speed during passive uniform-current capture.

Production code was independently verified in a fresh Python process, persistent artifacts were generated, and the complete HydroLab-3D repository finished with:

$$
\boxed{
113/113\text{ automated tests passing}
}
$$

The six-degree-of-freedom vehicle-dynamics foundation is therefore complete.

**Notebook 07 status: VALIDATED.**

**Next notebook: `08_sensor_models.ipynb`.**

---